<a href="https://colab.research.google.com/github/fbildirici/Bash-Cheat-Sheet/blob/main/SCALE_ICLR_FULL_MASTER_COLAB_ROBUST_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCALE — FULL MASTER FROM-SCRATCH COLAB
## Final ICLR Experiment, External Validation & Reviewer-Closure Artifact

This notebook is the **standalone canonical implementation** of the complete SCALE study.

It requires **no previous SCALE notebook, no previous Python state, and no manually
prepared local result file**. On a clean Google Colab runtime it:

1. checks GPU and network availability;
2. installs all Python dependencies;
3. creates a fresh experiment namespace;
4. generates the controlled workflow data and ontology from code;
5. downloads frozen/open external models and public external datasets when required;
6. trains every core learned baseline and SCALE variant;
7. runs the original G0–G7 program;
8. runs every external/reviewer-hardening analysis developed for the final paper;
9. writes detailed interpretation directly into the notebook;
10. generates one canonical results registry, claim ledger, reviewer-closure matrix,
    full results book, and ZIP artifact pack.

### Recommended runtime

**Google Colab A100** is strongly recommended.

Use:

**Runtime → Change runtime type → A100 GPU → Run all**

The full research run is intentionally expensive. Smaller reader/producer stages are
resumable through Drive, but the experiment namespace in this notebook is new and does
not depend on outputs from earlier notebook versions.

### Final scientific scope

The notebook tests:

- semantic interface compatibility;
- independently evolving constituent implementations;
- C1 natural-language paraphrase robustness;
- C2 schema/tool realization robustness;
- structural open-world ontology canonicalization;
- runtime semantic contract enforcement;
- calibration;
- decision-active semantic drift monitoring;
- arbitrary and typed readers;
- real LLM producers;
- independent learned and generative downstream executors;
- deployment routing and selective abstention;
- CRMArena-Pro offline external transfer;
- Schema.org, EDAM, PROV-O and ODRL structural evidence;
- real EDAM version evolution;
- official MCP/A2A release-history context;
- 120-base × 12-surface-family reviewer stress;
- message-level semantic counterfactuals;
- held-out Active-SCD threshold transfer;
- all original G0–G7 positive and negative results.

### Scientific integrity

Original prespecified G0/G2 failures are never post-hoc relabeled.
Closed-set superiority is never inferred from a numerical difference alone.
Generative-executor gates are frozen before drift evaluation.
Synthetic, external, and longitudinal evidence are labeled separately.



# 0 — Robust Colab Bootstrap

Bu hücre **notebook'u daha deney başlamadan durdurmaz**.

Yaptıkları:

- Colab/GPU'yu algılar;
- GPU yoksa core CPU testlerinin çalışabileceğini, büyük LLM aşamalarının skip edileceğini bildirir;
- disk ve internet erişimini *warning* olarak raporlar;
- Hugging Face / GitHub / W3C geçici erişim sorunlarında ana notebook'u öldürmez;
- gerekli paketları tek seferde kurar;
- paket importlarını doğrular;
- A100/L4/T4 farkına göre model aşamalarında kullanılacak güvenli runtime bayraklarını üretir.

**Önerilen runtime: A100.** Ancak başlangıç hücresi T4/L4 üzerinde de hata vermeden çalışır.


In [ ]:
# 0 — Robust one-click Colab bootstrap

import os, sys, gc, json, time, math, random, re, shutil, subprocess, itertools, hashlib, copy, platform
from pathlib import Path
from urllib.request import Request, urlopen

print("=" * 96)
print("SCALE — ROBUST COLAB BOOTSTRAP")
print("=" * 96)

IS_COLAB = Path("/content").exists()
print("Colab-like runtime:", IS_COLAB)

# ------------------------------------------------------------------
# GPU detection: informative, never an early hard failure.
# ------------------------------------------------------------------
GPU_AVAILABLE = False
GPU_NAME = "CPU"
GPU_MEMORY_GB = 0.0

try:
    q = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if q.returncode == 0 and q.stdout.strip():
        first = q.stdout.strip().splitlines()[0]
        parts = [x.strip() for x in first.split(",")]
        GPU_NAME = parts[0]
        GPU_MEMORY_GB = float(parts[1]) / 1024.0 if len(parts) > 1 else 0.0
        GPU_AVAILABLE = True
except Exception as exc:
    print("GPU detection warning:", type(exc).__name__)

print(f"GPU: {GPU_NAME} | VRAM ≈ {GPU_MEMORY_GB:.1f} GB")

# Runtime capabilities used later by strong-model sections.
FULL_STRONG_LLM_CAPABLE = bool(GPU_AVAILABLE and GPU_MEMORY_GB >= 20)
A100_CLASS_RUNTIME = bool(
    GPU_AVAILABLE
    and (
        "A100" in GPU_NAME.upper()
        or "H100" in GPU_NAME.upper()
        or GPU_MEMORY_GB >= 35
    )
)

if not GPU_AVAILABLE:
    print(
        "WARNING: No GPU detected. Core lightweight heads may run on CPU, "
        "but large generative-model stages will be marked unavailable."
    )
elif not FULL_STRONG_LLM_CAPABLE:
    print(
        "NOTE: GPU is available but VRAM is below the preferred strong-LLM level. "
        "Core SCALE experiments remain enabled; strong 7–8B stages may use safer "
        "memory settings or be skipped if loading fails."
    )

# ------------------------------------------------------------------
# Disk: warning, not fatal.
# ------------------------------------------------------------------
runtime_root = Path("/content") if IS_COLAB else Path.cwd()
try:
    du = shutil.disk_usage(runtime_root)
    FREE_DISK_GB = du.free / (1024**3)
except Exception:
    FREE_DISK_GB = float("nan")

print(f"Free disk: {FREE_DISK_GB:.1f} GB")
if FREE_DISK_GB == FREE_DISK_GB and FREE_DISK_GB < 12:
    print(
        "WARNING: less than 12 GB free. External 8B-model caching may fail; "
        "core controlled experiments remain usable."
    )

# ------------------------------------------------------------------
# Connectivity: best effort only.
# ------------------------------------------------------------------
NETWORK_ENDPOINTS = {
    "huggingface": "https://huggingface.co",
    "github_raw": "https://raw.githubusercontent.com",
    "w3c": "https://www.w3.org",
}
NETWORK_STATUS = {}

for name, url in NETWORK_ENDPOINTS.items():
    try:
        req = Request(
            url,
            headers={"User-Agent": "SCALE-ICLR-Colab/1.0"},
        )
        with urlopen(req, timeout=10) as resp:
            NETWORK_STATUS[name] = f"OK:{getattr(resp, 'status', 200)}"
    except Exception as exc:
        NETWORK_STATUS[name] = f"WARN:{type(exc).__name__}"

print("Network:", NETWORK_STATUS)
print(
    "Temporary network warnings do not stop the core experiment. "
    "Only the corresponding external stage may become inconclusive."
)

# ------------------------------------------------------------------
# Environment before pip.
# ------------------------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TQDM_DISABLE"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# Avoid unnecessary vision dependency paths for text-only experiments.
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

# ------------------------------------------------------------------
# Single compatible dependency installation.
# Keep numpy/torch supplied by Colab unless actually missing.
# ------------------------------------------------------------------
BOOTSTRAP_PACKAGES = [
    "transformers>=4.54,<5",
    "accelerate>=1.2,<2",
    "huggingface_hub>=0.27,<1",
    "safetensors>=0.4",
    "sentence-transformers>=3.0,<6",
    "scikit-learn>=1.4,<2",
    "rdflib>=7,<8",
    "jsonschema>=4.22",
    "requests>=2.31",
]

print("\nInstalling/validating Python dependencies...")
pip_cmd = [
    sys.executable, "-m", "pip", "install",
    "-q",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    *BOOTSTRAP_PACKAGES,
]

pip_result = subprocess.run(
    pip_cmd,
    capture_output=True,
    text=True,
)

if pip_result.returncode != 0:
    print("Primary pip install returned a warning.")
    print(pip_result.stderr[-2000:])
    print("Retrying once without quiet mode and without forced upgrade strategy...")
    retry = subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check",
            *BOOTSTRAP_PACKAGES,
        ],
        text=True,
    )
    if retry.returncode != 0:
        raise RuntimeError(
            "Python dependencies could not be installed after two attempts. "
            "This is a package-index/runtime problem rather than an experiment failure."
        )

# ------------------------------------------------------------------
# Imports after installation.
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import jsonschema

from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr, binomtest, wilcoxon

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)

print("\nPackage import validation: PASS")
print("Python:", platform.python_version())
print("torch:", torch.__version__)
print("CUDA available to torch:", torch.cuda.is_available())

# Update GPU information from torch after imports.
if torch.cuda.is_available():
    try:
        GPU_AVAILABLE = True
        GPU_NAME = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        GPU_MEMORY_GB = props.total_memory / (1024**3)
        FULL_STRONG_LLM_CAPABLE = GPU_MEMORY_GB >= 20
        A100_CLASS_RUNTIME = (
            "A100" in GPU_NAME.upper()
            or "H100" in GPU_NAME.upper()
            or GPU_MEMORY_GB >= 35
        )
    except Exception:
        pass

HEAD_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CAUSAL_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else (torch.float16 if torch.cuda.is_available() else torch.float32)
)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

print(
    f"Bootstrap complete | HEAD_DEVICE={HEAD_DEVICE} | "
    f"CAUSAL_DTYPE={CAUSAL_DTYPE} | "
    f"strong_llm_capable={FULL_STRONG_LLM_CAPABLE}"
)
print("=" * 96)


# SCALE — Final Hybrid Decisive A100 Experiment
## Closed-Set Discriminative Semantics + Open-World Ontology Prototypes + Directed Invariance + Offline CRMArena-Pro RQ5

This notebook is the **final paper-aligned controlled + external experiment**.

It is derived from the previous completed A100 runs and fixes the remaining empirical and implementation issues without weakening the evaluation gates.

---

# What the previous experiments established

The completed runs showed four strong mechanisms:

1. **Low-label implementation adaptation**
   - SCALE remained highly accurate when the evolved B implementation had 0–10% active semantic labels.
2. **Structural ontology generalization**
   - SCALE recovered opaque unseen ontology leaves and their canonical parent where ontology-free methods failed.
3. **Leakage-free semantic drift observability**
   - PrePolicy Active-SCD strongly predicted harmful semantic drift and downstream failure.
4. **Real-producer semantic recovery**
   - NLI-valid LLM handoffs were reliably converted into operational semantic contracts.

They also revealed three remaining issues:

1. **C2 schema/tool OOD**
   - a pure prototype decoder remained weaker than a discriminative dual-view classifier;
2. **G0/G1 measurement coupling**
   - semantic-vocabulary mistakes were being counted as parser/technical failures;
3. **CRMArena-Pro**
   - public data download/splitting worked, but SCALE-CRM stopped on a matrix-transpose implementation bug.

---

# Final architectural decision

The results support a **hybrid decoder** rather than forcing one decoder to solve two different problems.

## Closed-set known semantics

For known ontology classes and schema/tool OOD:

```text
dual-view BGE
    ↓
discriminative semantic bottleneck
    ↓
known active semantic class
```

This branch is intentionally similar to the strong `CB+DualView` control.

## Open-world ontology extension

For unseen ontology nodes:

```text
dual-view BGE
    ↓
relation-aware graph prototype head
    ↓
unseen ontology leaf
    ↓ is_a / ontology relation
canonical known parent
```

The graph-prototype branch is therefore used for the problem where it empirically adds value:
**open-world structural generalization**, not ordinary closed-set classification.

---

# Directed implementation invariance

The previous symmetric invariance loss could move both A and B representations.

The final loss instead uses A as a stop-gradient semantic teacher:

\[
L_{\mathrm{inv}}^{A\rightarrow B}
=
\mathrm{KL}\left(
p_A^{\mathrm{stop}}
\;\|\;
p_B
\right)
+
\lambda_z
\left(
1-\cos(z_A^{\mathrm{stop}},z_B)
\right).
\]

This preserves the well-supervised A decision geometry while adapting the low-label B implementation toward it.

The coefficient remains supervision-aware:

\[
\lambda_{\mathrm{inv}}
=
\lambda_0(1-\rho_B).
\]

---

# Learned logic simplification

Previous adversarial ablations did not show a measurable benefit from a differentiable `L_logic`.

Therefore **learned logic is no longer part of Full SCALE's primary objective**.

The ontology validator remains:

```text
semantic contract
    ↓
deterministic ontology validation
    ↓
repair safe structural violations
or
block unverifiable provenance violations
```

`CB+Logic` and `SCALE+Logic` are retained only as diagnostics.

This is a deliberate parsimony decision, not a hidden removal of a negative result.

---

# G0/G1 measurement correction

A technical parser gate should measure:

- model error;
- overflow;
- syntactic JSON parse success.

It should **not** classify a semantically invalid ontology label as a JSON parser failure.

The notebook therefore records separately:

```text
parse_success          = technical/syntactic validity
semantic_label_validity = allowed semantic vocabulary validity
```

G1 now tests SCALE's actual machine-readable/native interface competence.
A separate multi-reader stress table is still reported, but reader idiosyncrasy does not redefine whether the semantic contract itself is valid.

---

# Final controlled claims

The final experiment tests:

- **G0** technical transport/syntax validity;
- **G1** native SCALE interface competence under replacement;
- **G2** label-efficient directed invariance;
- **G3** structural ontology generalization;
- **G4** C1/C2 competitive utility against `CB+DualView`;
- **G5** deterministic semantic enforcement;
- **G6** leakage-free PrePolicy Active-SCD observability;
- **G7** real LLM producer semantic recovery.

---

# RQ5

After all controlled experiments finish, the notebook automatically:

1. pins the official `Salesforce/CRMArenaPro` Hugging Face revision;
2. downloads B2B/B2C tasks and schemas;
3. creates a conservative public-context-grounded external subset;
4. trains CRM-specific discriminative and SCALE-hybrid adapters;
5. runs `NL / JSON / Onto-RAG / CB-CRM / CB-Dual-CRM / SCALE-CRM`;
6. repeats progressive 0→1→2→3 constituent replacement;
7. reports task success, compositional retention, paired bootstrap confidence intervals, latency and token use.

This is explicitly reported as:

> **CRMArena-Pro offline dataset-grounded external validity**

and not as the official live Salesforce environment score.

In [ ]:
# 1/30 — Runtime configuration

# Standard-library imports are already available from bootstrap.
print(
    "Runtime configuration starting |",
    "GPU:", GPU_NAME,
    "| VRAM_GB:", round(GPU_MEMORY_GB, 1),
)

EXPERIMENT_VERSION = "scale-iclr-full-master-from-scratch-v3.1.0"
IMPLEMENTATION_PATCH = "standalone-cleanroom-v1"
ANALYSIS_PATCH = "full-reviewer-closure-v1"
MECHANISM_ROBUSTNESS_PATCH = "external-gate-master-v1"
SEED = 2026

# Main paired ablations: enough seeds for a final controlled result.
HEAD_SEEDS = [2026,2027,2028,2029,2030]
CURVE_SEEDS = HEAD_SEEDS  # final estimate: use all five paired seeds

N_TRAIN_BASE = 270
N_DEV_BASE = 45
N_CORE_TEST = 16
N_SHIFT_TEST = 12
N_ONTOLOGY_TEST = 12
N_DRIFT_BASE = 96

HEAD_EPOCHS = 48
CURVE_EPOCHS = 28

# Primary evolving-agent regime:
# only 10% of the new B implementation's role-active labels are visible.
MAIN_B_LABEL_VISIBILITY = 0.10
LABEL_VISIBILITY_GRID = [0.0,0.10,0.25,0.50,1.0]

BASE_INV_WEIGHT = 0.24
GRAPH_ANCHOR_WEIGHT = 0.06
GRAPH_GATE_INIT = -3.0

# Final hybrid SCALE.
PROTO_AUX_WEIGHT = 0.08
DIRECTED_LATENT_WEIGHT = 0.10


# C2 robustness: generic schema normalization + frozen semantic anchor.
DUAL_VIEW_RAW_WEIGHT = 0.55
DUAL_VIEW_NORMALIZED_WEIGHT = 0.45

# Fixed semantic-anchor contribution. It is NOT tuned on C1/C2.
SEMANTIC_ANCHOR_WEIGHT = 0.35


RUN_LLM_READERS = bool(torch.cuda.is_available())
RUN_C1_C2 = True
RUN_ONTOLOGY_SHIFT = True
RUN_REAL_PRODUCER = bool(torch.cuda.is_available())
RUN_LABEL_EFFICIENCY_CURVE = True
RUN_LOGIC_ADVERSARIAL = True

FULL_GPU_EVALUATION = bool(torch.cuda.is_available())
if FULL_GPU_EVALUATION:
    print("GPU evaluation mode: ENABLED — LLM readers/producers/executors can run.")
else:
    print(
        "GPU evaluation mode: DISABLED — CUDA is unavailable. "
        "Core SCALE experiments will continue; causal-LLM/CRM stages will be "
        "reported as SKIPPED/INCONCLUSIVE rather than raising an exception."
    )

RUN_CRM_PREP = False
USE_DRIVE = True

MAX_READER_INPUT = 900
MAX_READER_OUTPUT = 44
MAX_PRODUCER_INPUT = 520
MAX_PRODUCER_OUTPUT = 64

L0_COMPETENCE = 0.60
NLI_ENTAIL_THRESHOLD = 0.60
ONTOLOGY_NONINFERIOR_MARGIN = 0.03
KG_TOPK = 2

SEMANTIC_ENCODER_ID = "BAAI/bge-small-en-v1.5"
NLI_VALIDATOR_ID = "cross-encoder/nli-deberta-v3-xsmall"

READER_MODELS = [
    {
        "label":"Qwen3-1.7B",
        "primary":"Qwen/Qwen3-1.7B",
        "fallback":"Qwen/Qwen2.5-1.5B-Instruct",
        "trust_remote_code":False,
        "disable_thinking":True,
    },
    {
        "label":"Granite3.3-2B",
        "primary":"ibm-granite/granite-3.3-2b-instruct",
        "fallback":"Qwen/Qwen2.5-1.5B-Instruct",
        "trust_remote_code":False,
        "disable_thinking":False,
    },
    {
        "label":"Ministral-3B",
        "primary":"mistralai/Ministral-3b-instruct",
        "fallback":"Qwen/Qwen2.5-1.5B-Instruct",
        "trust_remote_code":False,
        "disable_thinking":False,
    },
]

PRODUCER_MODELS = [
    {
        "label":"Qwen3-0.6B",
        "primary":"Qwen/Qwen3-0.6B",
        "fallback":"Qwen/Qwen2.5-0.5B-Instruct",
        "trust_remote_code":False,
        "disable_thinking":True,
    },
    {
        "label":"Granite3.3-2B",
        "primary":"ibm-granite/granite-3.3-2b-instruct",
        "fallback":"Qwen/Qwen2.5-0.5B-Instruct",
        "trust_remote_code":False,
        "disable_thinking":False,
    },
]

MAIN_METHODS = [
    "NL","JSON","Onto-RAG","Symbolic",
    "CB-BGE","CB+DualView","CB+Logic","Proto-CB","SCALE"
]
STRUCTURED_METHODS = [
    "Symbolic","CB-BGE","CB+DualView","CB+Logic","Proto-CB","SCALE"
]
AGENTS = ["PLANNER","RETRIEVER","POLICY"]

CORE_COMPOSITIONS=[]
for bits in itertools.product("AB",repeat=3):
    mapping=dict(zip(AGENTS,bits))
    CORE_COMPOSITIONS.append({
        "composition_id":"".join(bits),
        "replacement_level":sum(v=="B" for v in bits),
        "mapping":mapping,
    })
CORE_COMPOSITIONS=sorted(
    CORE_COMPOSITIONS,
    key=lambda x:(x["replacement_level"],x["composition_id"])
)

def c_compositions(tag):
    return [
        {"composition_id":f"{tag}AA","replacement_level":1,
         "mapping":{"PLANNER":tag,"RETRIEVER":"A","POLICY":"A"}},
        {"composition_id":f"A{tag}A","replacement_level":1,
         "mapping":{"PLANNER":"A","RETRIEVER":tag,"POLICY":"A"}},
        {"composition_id":f"AA{tag}","replacement_level":1,
         "mapping":{"PLANNER":"A","RETRIEVER":"A","POLICY":tag}},
        {"composition_id":f"{tag}{tag}{tag}","replacement_level":3,
         "mapping":{"PLANNER":tag,"RETRIEVER":tag,"POLICY":tag}},
    ]

C1_COMPOSITIONS=c_compositions("C1")
C2_COMPOSITIONS=c_compositions("C2")

CONFIG={
    "version":EXPERIMENT_VERSION,
    "seed":SEED,
    "head_seeds":HEAD_SEEDS,
    "semantic_encoder":SEMANTIC_ENCODER_ID,
    "nli_validator":NLI_VALIDATOR_ID,
    "n_train":N_TRAIN_BASE,
    "n_dev":N_DEV_BASE,
    "n_core":N_CORE_TEST,
    "n_shift":N_SHIFT_TEST,
    "n_ontology":N_ONTOLOGY_TEST,
    "n_drift":N_DRIFT_BASE,
    "head_epochs":HEAD_EPOCHS,
    "main_b_visibility":MAIN_B_LABEL_VISIBILITY,
    "visibility_grid":LABEL_VISIBILITY_GRID,
    "base_inv_weight":BASE_INV_WEIGHT,
    "graph_anchor_weight":GRAPH_ANCHOR_WEIGHT,
    "readers":[x["primary"] for x in READER_MODELS],
    "methods":MAIN_METHODS,
}
CONFIG_HASH=hashlib.sha256(
    json.dumps(CONFIG,sort_keys=True).encode()
).hexdigest()[:12]

ROOT=Path(f"/content/{EXPERIMENT_VERSION}-{CONFIG_HASH}")
ROOT.mkdir(parents=True,exist_ok=True)

DRIVE_ROOT=None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive",force_remount=False)
        DRIVE_ROOT=Path(
            f"/content/drive/MyDrive/SCALE_Research/{EXPERIMENT_VERSION}-{CONFIG_HASH}"
        )
        DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
        print("Drive:",DRIVE_ROOT)
    except Exception as exc:
        print("Drive unavailable:",type(exc).__name__)

os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ.setdefault("HF_HOME","/content/hf_cache")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]="1"
os.environ["TRANSFORMERS_VERBOSITY"]="error"
os.environ["TQDM_DISABLE"]="1"
os.environ["OMP_NUM_THREADS"]="2"
os.environ["MKL_NUM_THREADS"]="2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True,max_split_size_mb:128"

random.seed(SEED)

core_calls=len(READER_MODELS)*len(MAIN_METHODS)*len(CORE_COMPOSITIONS)*N_CORE_TEST
shift_calls=(
    len(READER_MODELS)*len(MAIN_METHODS)*
    (len(C1_COMPOSITIONS)+len(C2_COMPOSITIONS))*N_SHIFT_TEST
    if RUN_C1_C2 else 0
)
ontology_calls=(
    len(READER_MODELS)*len(MAIN_METHODS)*2*N_ONTOLOGY_TEST
    if RUN_ONTOLOGY_SHIFT else 0
)

print("Experiment:",EXPERIMENT_VERSION)
print("Config hash:",CONFIG_HASH)
print("Primary B active-label visibility:",MAIN_B_LABEL_VISIBILITY)
print("Reader calls — core:",core_calls,
      "| C1/C2:",shift_calls,
      "| ontology:",ontology_calls)


In [ ]:
# =============================================================================
# SCALE FULL MASTER — TEST FAMILY MANIFEST
# =============================================================================

FULL_MASTER_TEST_FAMILIES = [
    # Core
    ("core_training", "MODELS"),
    ("representation_benchmark", "rep_df"),
    ("ontology_extension", "ontology_df"),
    ("label_efficiency", "curve_df"),
    ("native_receiver", "native_df"),
    ("runtime_constraints", "constraint_stress_df"),
    ("semantic_drift", "drift_df"),
    ("reader_g0", "reader_df"),
    ("real_producer_g7", "producer_df"),
    ("gate_ledger_original", "gates"),

    # External / robust
    ("crm_external", "crm_retention"),
    ("semantic_interventions", "intervention_df"),
    ("large_ontology", "large_ontology_df"),
    ("strict_router", "strict_router_df"),
    ("router_root_generalization", "router_root_generalization_df"),
    ("schemaorg_external", "schema_external_df"),
    ("naturalistic_version_drift", "naturalistic_df"),
    ("independent_executors", "independent_executor_df"),
    ("reader_failure_resolution", "reader_failure_summary"),
    ("selective_routing_safety", "safety_route_df"),
    ("corrected_role_mask", "corrected_mask_df"),

    # Reviewer stress
    ("small_n_paired", "smalln_df"),
    ("opaque_uncertainty", "onto_unc_df"),
    ("architecture_id_routing", "arch_replacement"),
    ("active_scd_incremental", "incremental_df"),
    ("dual_view_sensitivity", "dual_sweep_df"),

    # Comprehensive measurement
    ("expanded_heldout_96", "expanded_df"),
    ("dev_only_baseline_selection", "frozen_alpha_df"),
    ("active_scd_negative_controls", "scd_controls"),
    ("schema_structural_permutation", "schema_perm_summary"),
    ("risk_coverage", "risk_coverage_df"),
    ("claim_scorecard_comprehensive", "claim_scorecard"),

    # External gate validation
    ("strong_readers", "strong_reader_df"),
    ("strong_producers", "strong_producer_df"),
    ("generative_executors", "strong_executor_df"),
    ("g1_expanded_native", "g1_expanded_df"),
    ("g2_10seed", "g2_ext_df"),
    ("edam_static", "edam_static_df"),
    ("edam_longitudinal", "edam_longitudinal_df"),
    ("g5_combinatorial", "g5_combo_df"),
    ("real_protocol_history", "protocol_history_df"),
    ("external_gate_ledger", "external_gate_ledger"),

    # Master reviewer closure
    ("master_120x12_surface_stress", "master_surface_df"),
    ("message_counterfactuals", "message_cf_df"),
    ("active_scd_threshold_transfer", "threshold_summary"),
    ("w3c_external_ontologies", "w3c_external_summary"),
    ("master_claim_audit", "master_claim_audit"),
    ("canonical_master_registry", "master_registry"),
]

FULL_MASTER_RUN_ID = (
    EXPERIMENT_VERSION + "::" + CONFIG_HASH + "::FULL_ICLR_MASTER"
)

print("=" * 100)
print("SCALE FULL MASTER RUN")
print("Run ID:", FULL_MASTER_RUN_ID)
print("Declared test families:", len(FULL_MASTER_TEST_FAMILIES))
print("ROOT:", ROOT)
print("=" * 100)


In [ ]:
# 2/27 — Dependency state and shared utilities
# Packages are installed/imported by the robust bootstrap cell above.

print("Using bootstrap dependency state.")

_REQUIRED_IMPORTS = [
    ("numpy", np),
    ("pandas", pd),
    ("torch", torch),
    ("rdflib.Graph", Graph),
    ("SentenceTransformer", SentenceTransformer),
    ("AutoTokenizer", AutoTokenizer),
]

for name, obj in _REQUIRED_IMPORTS:
    assert obj is not None, f"Missing runtime import: {name}"

# Shared JSONL utilities used by long-running reader/producer stages.

def _decode_json_object_stream(text):
    decoder = json.JSONDecoder()
    rows = []
    bad = 0
    i = 0
    n = len(text)

    while i < n:
        progressed = True
        while progressed and i < n:
            progressed = False
            while i < n and text[i].isspace():
                i += 1
                progressed = True
            if text.startswith("\\n", i):
                i += 2
                progressed = True

        if i >= n:
            break

        try:
            obj, end = decoder.raw_decode(text, i)
        except Exception:
            bad += 1
            break

        if isinstance(obj, dict):
            rows.append(obj)
        else:
            bad += 1
        i = end

    return rows, bad


def safe_jsonl_records(path):
    path = Path(path)
    if not path.exists():
        return [], 0
    text = path.read_text(encoding="utf-8", errors="replace")
    return _decode_json_object_stream(text)


def safe_read_jsonl_df(path):
    rows, bad = safe_jsonl_records(path)
    if bad:
        print(
            f"Warning: {bad} unreadable JSON stream fragment(s) "
            f"in {Path(path).name}"
        )
    return pd.DataFrame(rows)


def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(row, ensure_ascii=False) + "\n"
    with path.open("a", encoding="utf-8") as f:
        f.write(payload)
        f.flush()
        try:
            os.fsync(f.fileno())
        except Exception:
            pass


def repair_jsonl_file(path, backup=True):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return 0

    text = path.read_text(encoding="utf-8", errors="replace")
    rows, bad = _decode_json_object_stream(text)
    if not rows:
        return 0

    canonical = "".join(
        json.dumps(r, ensure_ascii=False) + "\n"
        for r in rows
    )
    if text == canonical:
        return len(rows)

    if backup:
        bak = path.with_suffix(path.suffix + ".legacy.bak")
        if not bak.exists():
            bak.write_text(text, encoding="utf-8")

    path.write_text(canonical, encoding="utf-8")
    print(
        f"Recovered {len(rows)} JSONL record(s) in {path.name}"
        + (f"; unreadable fragments={bad}" if bad else "")
    )
    return len(rows)


def sync_result_file(local_path, remote_path):
    local_path = Path(local_path)
    remote_path = Path(remote_path)
    if local_path.exists():
        remote_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, remote_path)

# Serialization sanity check.
_probe = Path("/tmp/scale_jsonl_preflight.jsonl")
try:
    if _probe.exists():
        _probe.unlink()
    append_jsonl(_probe, {"i": 1, "text": "internal\\nnewline"})
    append_jsonl(_probe, {"i": 2, "text": "ok"})
    _rows, _bad = safe_jsonl_records(_probe)
    assert _bad == 0 and [x["i"] for x in _rows] == [1, 2]
    print("JSONL serialization/recovery preflight: PASS")
finally:
    if _probe.exists():
        _probe.unlink()

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)

print(
    "Runtime dependency state PASS |",
    "head device:", HEAD_DEVICE,
    "| causal dtype:", CAUSAL_DTYPE,
)



# Startup Smoke Test

Bu hücre ilk ağır model eğitiminden önce başlangıç altyapısını doğrular.
Hata burada çıkarsa problem doğrudan ortam/bağımlılık seviyesindedir; deney
sonuçlarıyla karışmaz.


In [ ]:
# Startup smoke test

print("=" * 80)
print("SCALE STARTUP SMOKE TEST")
print("=" * 80)

# Basic tensors.
_x = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=HEAD_DEVICE)
assert torch.isfinite(_x).all()

# RDF graph.
_g = Graph()
_ns = Namespace("https://example.org/smoke/")
_g.add((_ns.A, RDF.type, OWL.Class))
assert len(_g) == 1

# sklearn.
_le = LabelEncoder()
assert list(_le.fit_transform(["A", "B"])) == [0, 1]

# JSONL.
_tmp = Path("/tmp/scale_smoke.jsonl")
try:
    if _tmp.exists():
        _tmp.unlink()
    append_jsonl(_tmp, {"ok": True})
    _r, _b = safe_jsonl_records(_tmp)
    assert _b == 0 and _r[0]["ok"] is True
finally:
    if _tmp.exists():
        _tmp.unlink()

print("Core Python/torch/RDF/sklearn/JSONL smoke test: PASS")
print("GPU available:", torch.cuda.is_available())
print("Strong LLM capable runtime:", FULL_STRONG_LLM_CAPABLE)
print("=" * 80)


In [ ]:
# 3/27 — Base ontology, relations and explicit graph structure

SCL=Namespace("https://example.org/scale/")
PROV=Namespace("http://www.w3.org/ns/prov#")
ODRL=Namespace("http://www.w3.org/ns/odrl/2/")

INTENTS=["REQUEST","QUERY","RECOMMEND","AUTHORIZE","DENY","VERIFY"]
OBJECTS=["PURCHASE","ACCESS","CHANGE","INCIDENT","COMPLIANCE_ITEM","NO_OBJECT"]
ACTIONS=["PLAN","SEARCH","ASSESS","APPROVE","REJECT","VERIFY","UPDATE","SEND"]
ROLES=["PLANNER","RETRIEVER","POLICY","EXECUTOR","VERIFIER"]
AUTHORITIES=["NONE","PROPOSE","RECOMMEND","APPROVE","EXECUTE"]
PROVENANCE=["USER_REQUEST","CRM_DATA","POLICY_CONTEXT","SYSTEM_EVIDENCE"]
STATES=[
    "PROPOSED",
    "EVIDENCE_PASS","EVIDENCE_FAIL","EVIDENCE_UNCERTAIN",
    "POLICY_READY","AUTHORIZED","REJECTED","VERIFIED","EXECUTED",
]

AUTH_ORD={x:i for i,x in enumerate(AUTHORITIES)}

ACTION_REQUIRED_AUTH={
    "PLAN":"PROPOSE",
    "SEARCH":"NONE",
    "ASSESS":"NONE",
    "APPROVE":"APPROVE",
    "REJECT":"APPROVE",
    "VERIFY":"RECOMMEND",
    "UPDATE":"EXECUTE",
    "SEND":"EXECUTE",
}
ACTION_ALLOWED_ROLES={
    "PLAN":{"PLANNER"},
    "SEARCH":{"RETRIEVER"},
    "ASSESS":{"POLICY"},
    "APPROVE":{"EXECUTOR"},
    "REJECT":{"EXECUTOR"},
    "VERIFY":{"EXECUTOR","VERIFIER"},
    "UPDATE":{"EXECUTOR"},
    "SEND":{"EXECUTOR"},
}
ACTION_ALLOWED_STATES={
    "PLAN":{"PROPOSED"},
    "SEARCH":{"EVIDENCE_PASS","EVIDENCE_FAIL","EVIDENCE_UNCERTAIN"},
    "ASSESS":{"POLICY_READY"},
    "APPROVE":{"AUTHORIZED"},
    "REJECT":{"REJECTED"},
    "VERIFY":{"VERIFIED"},
    "UPDATE":{"EXECUTED"},
    "SEND":{"EXECUTED"},
}
ACTION_DEFAULT_STATE={a:sorted(v)[0] for a,v in ACTION_ALLOWED_STATES.items()}

TYPE_MAP={
    "intent":("Intent",INTENTS),
    "object":("ObjectType",OBJECTS),
    "action":("Action",ACTIONS),
    "role":("Role",ROLES),
    "authority":("Authority",AUTHORITIES),
    "provenance":("ProvenanceType",PROVENANCE),
    "state":("WorkflowState",STATES),
}

ontology=Graph()
ontology.bind("scale",SCL)
ontology.bind("prov",PROV)
ontology.bind("odrl",ODRL)

for cls in [
    "SemanticContract","Intent","ObjectType","Action","Role",
    "Authority","ProvenanceType","WorkflowState"
]:
    ontology.add((SCL[cls],RDF.type,OWL.Class))

for slot,(cls,vals) in TYPE_MAP.items():
    for v in vals:
        ontology.add((SCL[v],RDF.type,SCL[cls]))
        ontology.add((SCL[v],RDFS.label,Literal(v)))

REQ,PERM,NEXT=SCL.requiresAuthority,SCL.permittedRole,SCL.allowedState

for a,u in ACTION_REQUIRED_AUTH.items():
    ontology.add((SCL[a],REQ,SCL[u]))
for a,rs in ACTION_ALLOWED_ROLES.items():
    for r in rs:
        ontology.add((SCL[a],PERM,SCL[r]))
for a,ss in ACTION_ALLOWED_STATES.items():
    for s in ss:
        ontology.add((SCL[a],NEXT,SCL[s]))

ontology.add((SCL.SYSTEM_EVIDENCE,RDFS.subClassOf,PROV.Entity))
ontology.add((SCL.POLICY_CONTEXT,RDFS.subClassOf,PROV.Entity))
ontology.add((SCL.requiresAuthority,RDFS.seeAlso,ODRL.permission))

ONTOLOGY_PATH=ROOT/"scale_ontology.ttl"
ontology.serialize(destination=str(ONTOLOGY_PATH),format="turtle")

def contract_violations(c):
    needed=["intent","object","action","role","authority","provenance","state"]
    if c is None or any(c.get(k) is None for k in needed):
        return ["incomplete_contract"]

    a=c["action"]
    if a not in ACTION_REQUIRED_AUTH:
        return ["unknown_action"]

    out=[]
    if AUTH_ORD.get(c["authority"],-1)<AUTH_ORD[ACTION_REQUIRED_AUTH[a]]:
        out.append("insufficient_authority")
    if c["role"] not in ACTION_ALLOWED_ROLES[a]:
        out.append("role_action_mismatch")
    if c["state"] not in ACTION_ALLOWED_STATES[a]:
        out.append("state_action_mismatch")
    if c["intent"]=="AUTHORIZE" and c["provenance"] not in {
        "POLICY_CONTEXT","SYSTEM_EVIDENCE"
    }:
        out.append("invalid_authorization_provenance")
    return out

def ontology_repair(c):
    if c is None:
        return None
    c=dict(c)
    a=c.get("action")
    if a not in ACTION_REQUIRED_AUTH:
        return c

    req=ACTION_REQUIRED_AUTH[a]
    if AUTH_ORD.get(c.get("authority"),-1)<AUTH_ORD[req]:
        c["authority"]=req

    if c.get("role") not in ACTION_ALLOWED_ROLES[a]:
        c["role"]=sorted(ACTION_ALLOWED_ROLES[a])[0]

    if c.get("state") not in ACTION_ALLOWED_STATES[a]:
        c["state"]=ACTION_DEFAULT_STATE[a]

    # provenance is never fabricated.
    return c

print("Ontology triples:",len(ontology))

In [ ]:
# 4/27 — Controlled workflow generator
# object, evidence state and authority are independent latent variables

PASS_ACTION_BY_OBJECT={
    "PURCHASE":"APPROVE",
    "ACCESS":"APPROVE",
    "CHANGE":"UPDATE",
    "INCIDENT":"SEND",
    "COMPLIANCE_ITEM":"APPROVE",
}
EVIDENCE_TO_STATE={
    "PASS":"EVIDENCE_PASS",
    "FAIL":"EVIDENCE_FAIL",
    "UNCERTAIN":"EVIDENCE_UNCERTAIN",
}
OPERATIONAL_AUTHORITIES=["RECOMMEND","APPROVE","EXECUTE"]

EVIDENCE_TEXT={
    "PASS":[
        "all required evidence is present and checks pass",
        "supporting records satisfy the required controls",
        "the evidence package supports proceeding",
        "all mandatory checks succeeded",
    ],
    "FAIL":[
        "a blocking condition is present",
        "mandatory supporting evidence is missing",
        "a prohibiting control failed",
        "the evidence contains a blocking exception",
    ],
    "UNCERTAIN":[
        "the evidence remains incomplete",
        "the evidence is ambiguous",
        "additional verification is required",
        "the evidence does not support a final decision",
    ],
}

def desired_action(obj,status):
    if status=="FAIL":
        return "REJECT"
    if status=="UNCERTAIN":
        return "VERIFY"
    return PASS_ACTION_BY_OBJECT[obj]

def policy_from_semantics(obj,evidence_state,authority):
    if obj not in PASS_ACTION_BY_OBJECT:
        return None

    if evidence_state=="EVIDENCE_FAIL":
        desired="REJECT"
    elif evidence_state=="EVIDENCE_UNCERTAIN":
        desired="VERIFY"
    elif evidence_state=="EVIDENCE_PASS":
        desired=PASS_ACTION_BY_OBJECT[obj]
    else:
        return None

    req=ACTION_REQUIRED_AUTH[desired]
    final_action=(
        desired
        if AUTH_ORD.get(authority,-1)>=AUTH_ORD[req]
        else "VERIFY"
    )
    return {"object":obj,"action":final_action,"authority":authority}

def generate_tasks(n,seed,prefix):
    rng=random.Random(seed)
    objects=list(PASS_ACTION_BY_OBJECT)
    statuses=["PASS","FAIL","UNCERTAIN"]
    auths=OPERATIONAL_AUTHORITIES
    combos=list(itertools.product(objects,statuses,auths))
    rows=[]

    for i in range(n):
        obj,status,auth=combos[i%len(combos)]
        final=policy_from_semantics(obj,EVIDENCE_TO_STATE[status],auth)

        rows.append({
            "base_id":f"{prefix}-{i:04d}",
            "request":f"Process enterprise workflow item {10000+i}.",
            "object":obj,
            "evidence_status":status,
            "evidence":rng.choice(EVIDENCE_TEXT[status]),
            "authority":auth,
            "final_action":final["action"],
            "final_authority":auth,
        })

    rng.shuffle(rows)
    return rows

train_base=generate_tasks(N_TRAIN_BASE,SEED,"train")
dev_base=generate_tasks(N_DEV_BASE,SEED+1,"dev")
core_test_base=generate_tasks(N_CORE_TEST,SEED+2,"core")
shift_test_base=generate_tasks(N_SHIFT_TEST,SEED+3,"shift")
ontology_test_base=generate_tasks(N_ONTOLOGY_TEST,SEED+4,"ontology")
drift_base=generate_tasks(N_DRIFT_BASE,SEED+5,"drift")

def hop_contract(base,agent):
    if agent=="PLANNER":
        return {
            "intent":"REQUEST","object":base["object"],"action":"PLAN",
            "role":"PLANNER","authority":"PROPOSE",
            "provenance":"USER_REQUEST","state":"PROPOSED",
        }
    if agent=="RETRIEVER":
        return {
            "intent":"QUERY","object":"NO_OBJECT","action":"SEARCH",
            "role":"RETRIEVER","authority":"NONE",
            "provenance":"CRM_DATA",
            "state":EVIDENCE_TO_STATE[base["evidence_status"]],
        }
    return {
        "intent":"RECOMMEND","object":"NO_OBJECT","action":"ASSESS",
        "role":"POLICY","authority":base["authority"],
        "provenance":"POLICY_CONTEXT","state":"POLICY_READY",
    }

ACTIVE_SLOT={"PLANNER":"object","RETRIEVER":"state","POLICY":"authority"}

def global_semantics(base):
    return {
        "object":base["object"],
        "evidence_state":EVIDENCE_TO_STATE[base["evidence_status"]],
        "authority":base["authority"],
    }

def deterministic_policy(sem):
    if sem is None:
        return None
    return policy_from_semantics(
        sem.get("object"),
        sem.get("evidence_state"),
        sem.get("authority"),
    )

print("Train/dev/core/C/ontology/drift:",
      len(train_base),len(dev_base),len(core_test_base),
      len(shift_test_base),len(ontology_test_base),len(drift_base))

In [ ]:
# 5/27 — A/B training realizations and C1/C2 held-out implementations

OBJ_B={
    "PURCHASE":"procurement-case",
    "ACCESS":"privileged-access",
    "CHANGE":"configuration-change",
    "INCIDENT":"service-incident",
    "COMPLIANCE_ITEM":"control-item",
}
EVID_B={"PASS":"clear","FAIL":"blocked","UNCERTAIN":"needs-review"}
AUTH_B={
    "RECOMMEND":"advisory-right",
    "APPROVE":"decision-right",
    "EXECUTE":"execution-right",
}

OBJ_C1={
    "PURCHASE":"purchase request",
    "ACCESS":"privileged access request",
    "CHANGE":"deployment modification",
    "INCIDENT":"operational incident",
    "COMPLIANCE_ITEM":"assurance control item",
}
EVID_C1={
    "PASS":"all requirements are satisfied",
    "FAIL":"the evidence contains a blocking issue",
    "UNCERTAIN":"the evidence remains unresolved",
}
AUTH_C1={
    "RECOMMEND":"may recommend but not authorize",
    "APPROVE":"has approval permission",
    "EXECUTE":"has execution permission",
}

OBJ_C2={
    "PURCHASE":"procurement workflow",
    "ACCESS":"entitlement workflow",
    "CHANGE":"change-management workflow",
    "INCIDENT":"incident-response workflow",
    "COMPLIANCE_ITEM":"compliance-control workflow",
}
EVID_C2={
    "PASS":"evidence verdict pass",
    "FAIL":"evidence verdict fail",
    "UNCERTAIN":"evidence verdict unresolved",
}
AUTH_C2={
    "RECOMMEND":"authority scope recommendation",
    "APPROVE":"authority scope approval",
    "EXECUTE":"authority scope execution",
}

def realize_message(base,agent,impl,variant=0):
    v=variant%3

    if impl=="A":
        if agent=="PLANNER":
            opts=[
                f"Planner: workflow object is {base['object']}.",
                f"ROLE=PLANNER | object={base['object']}",
                f"Planning handoff identifies {base['object']} as the target object.",
            ]
        elif agent=="RETRIEVER":
            opts=[
                f"Retriever: evidence status is {base['evidence_status']}. {base['evidence']}.",
                f"ROLE=RETRIEVER | evidence={base['evidence_status']} | note={base['evidence']}",
                f"Evidence handoff reports {base['evidence_status']}; {base['evidence']}.",
            ]
        else:
            opts=[
                f"Policy: delegated authority is {base['authority']}.",
                f"ROLE=POLICY | authority={base['authority']}",
                f"Governance handoff grants {base['authority']} authority.",
            ]
        return opts[v]

    if impl=="B":
        if agent=="PLANNER":
            opts=[
                f"mission-router :: subject-class={OBJ_B[base['object']]}",
                f"planning-b reports category {OBJ_B[base['object']]}",
                f"workflow-router target={OBJ_B[base['object']]}",
            ]
        elif agent=="RETRIEVER":
            opts=[
                f"lookup-b :: signal-state={EVID_B[base['evidence_status']]} :: {base['evidence']}",
                f"retrieval service verdict {EVID_B[base['evidence_status']]}; {base['evidence']}",
                f"evidence-engine status={EVID_B[base['evidence_status']]}",
            ]
        else:
            opts=[
                f"governance-b :: privilege-band={AUTH_B[base['authority']]}",
                f"policy service assigns {AUTH_B[base['authority']]}",
                f"authority-band={AUTH_B[base['authority']]}",
            ]
        return opts[v]

    if impl=="C1":
        if agent=="PLANNER":
            return f"The next component should treat this as a {OBJ_C1[base['object']]}."
        if agent=="RETRIEVER":
            return f"Evidence review concludes that {EVID_C1[base['evidence_status']]}. {base['evidence']}."
        return f"The policy decision says the downstream component {AUTH_C1[base['authority']]}."

    if impl=="C2":
        if agent=="PLANNER":
            return f"routing.v4 | workflow-category: {OBJ_C2[base['object']]} | next=lookup"
        if agent=="RETRIEVER":
            return f"evidence.v4 | assessment: {EVID_C2[base['evidence_status']]} | next=governance"
        return f"policy.v4 | permission-field: {AUTH_C2[base['authority']]} | next=execution"

    raise ValueError(impl)

train_messages=[]
for base in train_base:
    for agent in AGENTS:
        gold=hop_contract(base,agent)
        for impl in ("A","B"):
            for variant in range(3):
                train_messages.append({
                    "base_id":base["base_id"],
                    "agent":agent,
                    "impl":impl,
                    "variant":variant,
                    "message":realize_message(base,agent,impl,variant),
                    "contract":gold,
                })

print("Training messages:",len(train_messages))

In [ ]:
# 6/27 — Frozen BGE semantic encoder and cache

print("Loading frozen semantic encoder:",SEMANTIC_ENCODER_ID)
semantic_encoder=SentenceTransformer(
    SEMANTIC_ENCODER_ID,
    device="cpu",
)
EMBED_CACHE={}

def semantic_encode(texts,batch_size=64):
    missing=[t for t in texts if t not in EMBED_CACHE]
    if missing:
        arr=semantic_encoder.encode(
            missing,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype("float32")
        for t,v in zip(missing,arr):
            EMBED_CACHE[t]=v
    return np.stack([EMBED_CACHE[t] for t in texts]).astype("float32")

def generic_surface_normalize(text):
    """
    Generic implementation/schema normalization.

    IMPORTANT:
    - no semantic aliases;
    - no C1/C2 concept mapping;
    - semantic values are preserved.
    """
    s=str(text)

    # tool.v4 / routing.v4 / evidence.v4 -> "tool version", etc.
    s=re.sub(
        r"\b([A-Za-z][A-Za-z0-9_-]*)\.v\d+\b",
        r"\1 version",
        s,
        flags=re.I
    )

    # expose compound schema keys to the language encoder.
    s=re.sub(r"[_\-/]+"," ",s)

    # structured delimiters become whitespace rather than opaque syntax.
    s=re.sub(r"[|:=;]+"," ",s)

    # brackets/quotes are not semantic.
    s=re.sub(r"[\[\]\{\}\"'`]+"," ",s)

    # collapse whitespace.
    s=re.sub(r"\s+"," ",s).strip()
    return s

def dual_view_encode(texts):
    raw=semantic_encode(texts)
    norm_texts=[generic_surface_normalize(t) for t in texts]
    norm=semantic_encode(norm_texts)

    fused=(
        DUAL_VIEW_RAW_WEIGHT*raw
        +DUAL_VIEW_NORMALIZED_WEIGHT*norm
    )
    fused=fused/np.maximum(
        np.linalg.norm(fused,axis=1,keepdims=True),
        1e-8
    )
    return fused.astype("float32")

train_texts=[x["message"] for x in train_messages]

X_np=semantic_encode(train_texts)
X_DUAL_np=dual_view_encode(train_texts)

X_CPU=torch.tensor(X_np,dtype=torch.float32)
X_DUAL_CPU=torch.tensor(X_DUAL_np,dtype=torch.float32)

print("Frozen raw embeddings:",X_CPU.shape)
print("Frozen dual-view embeddings:",X_DUAL_CPU.shape)

# Sanity check: normalization cannot erase the original semantic values.
for t in train_texts[:50]:
    assert len(generic_surface_normalize(t))>0

In [ ]:
# 7/30 — Relation-aware ontology graph and true unseen-leaf extension

SLOTS=["intent","object","action","role","authority","provenance","state"]

encoders={}
Y={}
for slot in SLOTS:
    le=LabelEncoder()
    y=le.fit_transform([x["contract"][slot] for x in train_messages])
    encoders[slot]=le
    Y[slot]=torch.tensor(y,dtype=torch.long)

def concept_description(slot,label):
    if slot=="action":
        return (
            f"Action {label}. Required authority {ACTION_REQUIRED_AUTH[label]}. "
            f"Permitted roles {' '.join(sorted(ACTION_ALLOWED_ROLES[label]))}. "
            f"Valid states {' '.join(sorted(ACTION_ALLOWED_STATES[label]))}."
        )
    if slot=="authority":
        return f"Authority {label}. Permission rank {AUTH_ORD[label]}."
    if slot=="role":
        acts=[a for a,rs in ACTION_ALLOWED_ROLES.items() if label in rs]
        return f"Agent role {label}. Permitted actions {' '.join(acts)}."
    if slot=="state":
        acts=[a for a,ss in ACTION_ALLOWED_STATES.items() if label in ss]
        return f"Workflow state {label}. Compatible actions {' '.join(acts)}."
    if slot=="object":
        return f"Enterprise workflow object type {label}."
    if slot=="provenance":
        return f"Evidence provenance type {label}."
    if slot=="intent":
        return f"Communication intent {label}."
    return f"{slot} {label}"

BASE_NODE_KEYS=[
    (slot,str(label))
    for slot in SLOTS
    for label in encoders[slot].classes_
]

RELATION_TYPES=[
    "requires_authority",
    "permitted_role",
    "allowed_state",
    "role_action",
    "provenance_role",
    "object_action",
    "is_a",
]

def _normalize_adj(a):
    deg=a.sum(1)
    inv=np.zeros_like(deg)
    nz=deg>0
    inv[nz]=1.0/np.sqrt(deg[nz])
    return (inv[:,None]*a)*inv[None,:]

def build_graph_context(extra_leaves=None):
    """
    extra_leaves:
      label: unseen leaf label
      parent: known canonical object parent
      description: text feature with NO parent-name leakage
    """
    node_keys=list(BASE_NODE_KEYS)
    descriptions=[concept_description(s,l) for s,l in node_keys]

    if extra_leaves:
        for leaf in extra_leaves:
            node_keys.append(("object_leaf",leaf["label"]))
            descriptions.append(leaf["description"])

    idx={k:i for i,k in enumerate(node_keys)}
    n=len(node_keys)

    rel_adj={
        r:np.zeros((n,n),dtype="float32")
        for r in RELATION_TYPES
    }

    def link(rel,k1,k2,bidir=True):
        if k1 in idx and k2 in idx:
            i,j=idx[k1],idx[k2]
            rel_adj[rel][i,j]=1.0
            if bidir:
                rel_adj[rel][j,i]=1.0

    for action,auth in ACTION_REQUIRED_AUTH.items():
        link("requires_authority",("action",action),("authority",auth))

    for action,roles in ACTION_ALLOWED_ROLES.items():
        for role in roles:
            link("permitted_role",("action",action),("role",role))

    for action,states in ACTION_ALLOWED_STATES.items():
        for state in states:
            link("allowed_state",("action",action),("state",state))

    for role,action in {
        "PLANNER":"PLAN",
        "RETRIEVER":"SEARCH",
        "POLICY":"ASSESS",
    }.items():
        link("role_action",("role",role),("action",action))

    for prov,role in {
        "USER_REQUEST":"PLANNER",
        "CRM_DATA":"RETRIEVER",
        "POLICY_CONTEXT":"POLICY",
    }.items():
        link("provenance_role",("provenance",prov),("role",role))

    for obj,action in PASS_ACTION_BY_OBJECT.items():
        link("object_action",("object",obj),("action",action))

    leaf_parent={}
    leaf_indices=[]

    if extra_leaves:
        for leaf in extra_leaves:
            link(
                "is_a",
                ("object_leaf",leaf["label"]),
                ("object",leaf["parent"]),
            )
            leaf_parent[leaf["label"]]=leaf["parent"]
            leaf_indices.append(idx[("object_leaf",leaf["label"])])

    rel_adj={
        r:torch.tensor(_normalize_adj(a),dtype=torch.float32)
        for r,a in rel_adj.items()
    }

    features=torch.tensor(
        semantic_encode(descriptions),
        dtype=torch.float32
    )

    class_indices={
        slot:[
            idx[(slot,str(label))]
            for label in encoders[slot].classes_
        ]
        for slot in SLOTS
    }

    return {
        "node_keys":node_keys,
        "idx":idx,
        "features":features,
        "rel_adj":rel_adj,
        "class_indices":class_indices,
        "leaf_parent":leaf_parent,
        "leaf_indices":leaf_indices,
    }

# Known ontology children are present in the TRAINING graph so the typed
# is_a relation itself is learned before unseen-leaf evaluation.
# These nodes are never task labels and never appear in training messages.
KNOWN_GRAPH_LEAVES=[
    {"label":"ROUTINE_ORDER","parent":"PURCHASE",
     "description":"Ontology concept ROUTINE_ORDER."},
    {"label":"STANDARD_ENTITLEMENT","parent":"ACCESS",
     "description":"Ontology concept STANDARD_ENTITLEMENT."},
    {"label":"SCHEDULED_MODIFICATION","parent":"CHANGE",
     "description":"Ontology concept SCHEDULED_MODIFICATION."},
    {"label":"SERVICE_ALERT","parent":"INCIDENT",
     "description":"Ontology concept SERVICE_ALERT."},
    {"label":"INTERNAL_ASSURANCE","parent":"COMPLIANCE_ITEM",
     "description":"Ontology concept INTERNAL_ASSURANCE."},
]

BASE_GRAPH=build_graph_context(KNOWN_GRAPH_LEAVES)

# Transparent leaf names: meaningful lexical form, but parent is never stated.
TRANSPARENT_LEAVES=[
    {"label":"CAPEX_ORDER","parent":"PURCHASE",
     "description":"Ontology concept CAPEX_ORDER."},
    {"label":"PRIVILEGED_SESSION","parent":"ACCESS",
     "description":"Ontology concept PRIVILEGED_SESSION."},
    {"label":"EMERGENCY_PATCH","parent":"CHANGE",
     "description":"Ontology concept EMERGENCY_PATCH."},
    {"label":"SECURITY_EVENT","parent":"INCIDENT",
     "description":"Ontology concept SECURITY_EVENT."},
    {"label":"REGULATORY_CHECK","parent":"COMPLIANCE_ITEM",
     "description":"Ontology concept REGULATORY_CHECK."},
]

# Opaque names: relationship is available only in the ontology graph.
OPAQUE_LEAVES=[
    {"label":"ZX17","parent":"PURCHASE",
     "description":"Ontology concept ZX17."},
    {"label":"QK42","parent":"ACCESS",
     "description":"Ontology concept QK42."},
    {"label":"MN58","parent":"CHANGE",
     "description":"Ontology concept MN58."},
    {"label":"RT91","parent":"INCIDENT",
     "description":"Ontology concept RT91."},
    {"label":"PV33","parent":"COMPLIANCE_ITEM",
     "description":"Ontology concept PV33."},
]

TRANSPARENT_GRAPH=build_graph_context(KNOWN_GRAPH_LEAVES+TRANSPARENT_LEAVES)
OPAQUE_GRAPH=build_graph_context(KNOWN_GRAPH_LEAVES+OPAQUE_LEAVES)

TRANSPARENT_BY_PARENT={x["parent"]:x for x in TRANSPARENT_LEAVES}
OPAQUE_BY_PARENT={x["parent"]:x for x in OPAQUE_LEAVES}

def ontology_shift_planner_message(base,tier):
    leaf=(
        TRANSPARENT_BY_PARENT[base["object"]]
        if tier=="transparent"
        else OPAQUE_BY_PARENT[base["object"]]
    )
    # Intentionally no parent name or description in the handoff.
    return f"Planner ontology-v3 handoff: target concept is {leaf['label']}."

print("Base graph nodes:",len(BASE_GRAPH["node_keys"]))
print("Transparent expanded graph:",len(TRANSPARENT_GRAPH["node_keys"]))
print("Opaque expanded graph:",len(OPAQUE_GRAPH["node_keys"]))

In [ ]:
# 8/27 — Shared supervision masks and semantic-pair structure

def build_label_masks(b_visibility):
    masks={
        slot:torch.ones(len(train_messages),dtype=torch.float32)
        for slot in SLOTS
    }
    for i,item in enumerate(train_messages):
        if item["impl"]!="B":
            continue

        active=ACTIVE_SLOT[item["agent"]]
        key=f'{item["base_id"]}|{item["agent"]}|{item["variant"]}|{SEED}|{b_visibility}'
        h=int(hashlib.sha256(key.encode()).hexdigest()[:8],16)/0xFFFFFFFF

        if h>b_visibility:
            masks[active][i]=0.0

    return masks

MAIN_LABEL_MASK=build_label_masks(MAIN_B_LABEL_VISIBILITY)

group_names=[f'{x["base_id"]}|{x["agent"]}' for x in train_messages]
group_map={k:i for i,k in enumerate(sorted(set(group_names)))}
GROUP=torch.tensor([group_map[k] for k in group_names],dtype=torch.long)

POS_INDEX=np.full(len(train_messages),-1,dtype=np.int64)
NEG_INDEX=np.full(len(train_messages),-1,dtype=np.int64)

for i,item in enumerate(train_messages):
    positives=[
        j for j,o in enumerate(train_messages)
        if j!=i and o["base_id"]==item["base_id"] and o["agent"]==item["agent"]
    ]
    if positives:
        POS_INDEX[i]=positives[0]

    ci=item["contract"]
    candidates=[]
    for j,o in enumerate(train_messages):
        if j==i or o["agent"]!=item["agent"]:
            continue
        cj=o["contract"]
        h=sum(ci[s]!=cj[s] for s in SLOTS)
        if h>0:
            candidates.append((h,j))
    if candidates:
        candidates.sort(key=lambda x:(x[0],x[1]))
        NEG_INDEX[i]=candidates[0][1]

POS_T=torch.tensor(POS_INDEX,dtype=torch.long)
NEG_T=torch.tensor(NEG_INDEX,dtype=torch.long)

masked=sum(
    int(MAIN_LABEL_MASK[ACTIVE_SLOT[x["agent"]]][i].item()==0)
    for i,x in enumerate(train_messages)
    if x["impl"]=="B"
)
total_b=sum(x["impl"]=="B" for x in train_messages)
print("Main B active label visibility:",
      1-masked/max(1,total_b))

# Exact A→B teacher/student pairs: same base, role and surface variant.
# A is the well-supervised teacher and is stop-gradient in the final invariance loss.
AB_A_INDEX=[]
AB_B_INDEX=[]
AB_ROLE=[]

lookup={}
for idx,item in enumerate(train_messages):
    lookup[
        (item["base_id"],item["agent"],item["variant"],item["impl"])
    ]=idx

for item in train_messages:
    if item["impl"]!="A":
        continue
    key_b=(item["base_id"],item["agent"],item["variant"],"B")
    key_a=(item["base_id"],item["agent"],item["variant"],"A")
    if key_b in lookup:
        AB_A_INDEX.append(lookup[key_a])
        AB_B_INDEX.append(lookup[key_b])
        AB_ROLE.append(item["agent"])

AB_A_T=torch.tensor(AB_A_INDEX,dtype=torch.long)
AB_B_T=torch.tensor(AB_B_INDEX,dtype=torch.long)

print("Directed A→B invariance pairs:",len(AB_A_INDEX))


In [ ]:
# 9/31 — CB, prototype baseline, and final hybrid SCALE decoder

class CBHead(nn.Module):
    def __init__(self,in_dim,hidden=128,zdim=64,dual_view=False):
        super().__init__()
        self.dual_view=bool(dual_view)
        self.backbone=nn.Sequential(
            nn.Linear(in_dim,hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden,zdim),
        )
        self.heads=nn.ModuleDict({
            s:nn.Linear(zdim,len(encoders[s].classes_))
            for s in SLOTS
        })
        self.conf=nn.Linear(zdim,1)

    def forward(self,x):
        z=self.backbone(x)
        logits={s:self.heads[s](z) for s in SLOTS}
        conf=torch.sigmoid(self.conf(z)).squeeze(-1)
        return z,logits,conf


class ResidualRGCN(nn.Module):
    """Small typed relation-aware residual graph correction."""
    def __init__(self,dim,relations):
        super().__init__()
        self.relations=list(relations)
        self.r1=nn.ModuleDict({
            r:nn.Linear(dim,dim,bias=False) for r in self.relations
        })
        self.r2=nn.ModuleDict({
            r:nn.Linear(dim,dim,bias=False) for r in self.relations
        })
        self.self1=nn.Linear(dim,dim,bias=False)
        self.self2=nn.Linear(dim,dim,bias=False)
        self.norm1=nn.LayerNorm(dim)
        self.norm2=nn.LayerNorm(dim)

    def _aggregate(self,h,rel_adj,self_layer,relation_layers):
        out=self_layer(h)
        used=0
        for r in self.relations:
            a=rel_adj[r].to(h.device)
            if int(torch.count_nonzero(a).item())==0:
                continue
            out=out+relation_layers[r](a@h)
            used+=1
        return out/max(1,used+1)

    def forward(self,h,rel_adj):
        h1=self.norm1(F.gelu(
            self._aggregate(h,rel_adj,self.self1,self.r1)
        ))
        h2=self.norm2(
            self._aggregate(h1,rel_adj,self.self2,self.r2)
        )
        return h2


class ProtoContractHead(nn.Module):
    """Strong prototype-only baseline."""
    def __init__(
        self,
        in_dim,
        graph_dim=128,
        graph_mode=False,
        dual_view=False,
        semantic_anchor=False,
    ):
        super().__init__()
        self.graph_mode=bool(graph_mode)
        self.dual_view=bool(dual_view)
        self.semantic_anchor=bool(semantic_anchor)

        self.slot_proj=nn.ModuleDict({
            s:nn.Sequential(
                nn.Linear(in_dim,graph_dim),
                nn.GELU(),
                nn.LayerNorm(graph_dim),
            )
            for s in SLOTS
        })
        self.rgcn=ResidualRGCN(graph_dim,RELATION_TYPES)
        self.graph_gate=nn.ParameterDict({
            s:nn.Parameter(torch.tensor(float(GRAPH_GATE_INIT)))
            for s in SLOTS
        })
        self.log_tau=nn.Parameter(torch.tensor(math.log(.08)))

        self.conf=nn.Sequential(
            nn.Linear(in_dim,64),
            nn.GELU(),
            nn.Linear(64,1),
        )

    def all_node_repr(self,graph_ctx,slot):
        feats=graph_ctx["features"].to(next(self.parameters()).device)
        text=self.slot_proj[slot](feats)
        if not self.graph_mode:
            return text,text
        delta=self.rgcn(text,graph_ctx["rel_adj"])
        return text+torch.sigmoid(self.graph_gate[slot])*delta,text

    def prototype_matrix(self,graph_ctx,slot):
        nodes,_=self.all_node_repr(graph_ctx,slot)
        idx=torch.tensor(
            graph_ctx["class_indices"][slot],
            dtype=torch.long,
            device=nodes.device
        )
        return nodes[idx]

    def query(self,x,slot):
        return F.normalize(self.slot_proj[slot](x),p=2,dim=-1)

    def graph_anchor_loss(self,graph_ctx):
        if not self.graph_mode:
            return next(self.parameters()).new_tensor(0.)
        vals=[]
        for slot in SLOTS:
            g,t=self.all_node_repr(graph_ctx,slot)
            idx=torch.tensor(
                graph_ctx["class_indices"][slot],
                dtype=torch.long,
                device=g.device
            )
            vals.append(
                (
                    1-F.cosine_similarity(
                        F.normalize(g[idx],p=2,dim=-1),
                        F.normalize(t[idx].detach(),p=2,dim=-1),
                        dim=-1
                    )
                ).mean()
            )
        return torch.stack(vals).mean()

    def forward(self,x,graph_ctx):
        logits={}
        lat=[]
        tau=torch.exp(self.log_tau).clamp(.03,.50)

        for slot in SLOTS:
            q=self.query(x,slot)
            p=F.normalize(
                self.prototype_matrix(graph_ctx,slot),
                p=2,dim=-1
            )
            logits[slot]=(q@p.T)/tau
            lat.append(q)

        z=torch.cat(lat,dim=-1)
        conf=torch.sigmoid(self.conf(x)).squeeze(-1)
        return z,logits,conf


class HybridScaleHead(nn.Module):
    """
    Final SCALE:
      - discriminative closed-set decoder for known concepts / C2;
      - separate graph-prototype decoder for open-world ontology extension.
    """
    def __init__(
        self,
        in_dim,
        hidden=128,
        zdim=64,
        graph_dim=128,
        graph_mode=True,
        dual_view=True,
    ):
        super().__init__()
        self.dual_view=bool(dual_view)
        self.graph_mode=bool(graph_mode)

        # Closed-set branch: deliberately comparable to CB+DualView.
        self.backbone=nn.Sequential(
            nn.Linear(in_dim,hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden,zdim),
        )
        self.closed_heads=nn.ModuleDict({
            s:nn.Linear(zdim,len(encoders[s].classes_))
            for s in SLOTS
        })
        self.conf=nn.Linear(zdim,1)

        # Open-world ontology branch.
        self.slot_proj=nn.ModuleDict({
            s:nn.Sequential(
                nn.Linear(in_dim,graph_dim),
                nn.GELU(),
                nn.LayerNorm(graph_dim),
            )
            for s in SLOTS
        })
        self.rgcn=ResidualRGCN(graph_dim,RELATION_TYPES)
        self.graph_gate=nn.ParameterDict({
            s:nn.Parameter(torch.tensor(float(GRAPH_GATE_INIT)))
            for s in SLOTS
        })
        self.log_tau=nn.Parameter(torch.tensor(math.log(.08)))

    def forward(self,x,graph_ctx=None):
        z=self.backbone(x)
        logits={s:self.closed_heads[s](z) for s in SLOTS}
        conf=torch.sigmoid(self.conf(z)).squeeze(-1)
        return z,logits,conf

    def query(self,x,slot):
        return F.normalize(self.slot_proj[slot](x),p=2,dim=-1)

    def all_node_repr(self,graph_ctx,slot):
        feats=graph_ctx["features"].to(next(self.parameters()).device)
        text=self.slot_proj[slot](feats)

        if not self.graph_mode:
            return text,text

        delta=self.rgcn(text,graph_ctx["rel_adj"])
        graph=text+torch.sigmoid(self.graph_gate[slot])*delta
        return graph,text

    def open_logits(self,x,graph_ctx):
        logits={}
        tau=torch.exp(self.log_tau).clamp(.03,.50)

        for slot in SLOTS:
            q=self.query(x,slot)
            nodes,_=self.all_node_repr(graph_ctx,slot)
            idx=torch.tensor(
                graph_ctx["class_indices"][slot],
                dtype=torch.long,
                device=x.device
            )
            proto=F.normalize(nodes[idx],p=2,dim=-1)
            logits[slot]=(q@proto.T)/tau

        return logits

    def graph_anchor_loss(self,graph_ctx):
        if not self.graph_mode:
            return next(self.parameters()).new_tensor(0.)

        vals=[]
        for slot in SLOTS:
            graph,text=self.all_node_repr(graph_ctx,slot)
            idx=torch.tensor(
                graph_ctx["class_indices"][slot],
                dtype=torch.long,
                device=graph.device
            )
            vals.append(
                (
                    1-F.cosine_similarity(
                        F.normalize(graph[idx],p=2,dim=-1),
                        F.normalize(text[idx].detach(),p=2,dim=-1),
                        dim=-1
                    )
                ).mean()
            )
        return torch.stack(vals).mean()

In [ ]:
# 10/31 — Directed A→B invariance, hybrid ontology auxiliary loss, active calibration

ROW_AGENT=[x["agent"] for x in train_messages]

def masked_ce(logits,y,mask):
    loss=F.cross_entropy(logits,y,reduction="none")
    mask=mask.to(logits.device)
    return (loss*mask).sum()/mask.sum().clamp_min(1.0)

def b_active_visibility(label_masks):
    vals=[]
    for i,item in enumerate(train_messages):
        if item["impl"]=="B":
            vals.append(
                float(label_masks[ACTIVE_SLOT[item["agent"]]][i])
            )
    return float(np.mean(vals)) if vals else 1.0

def directed_ab_invariance(logits,device):
    """
    A is stop-gradient teacher.
    Only the role-active semantic distribution is aligned.
    """
    a_idx=AB_A_T.to(device)
    b_idx=AB_B_T.to(device)
    losses=[]

    for role in AGENTS:
        slot=ACTIVE_SLOT[role]
        mask=torch.tensor(
            [r==role for r in AB_ROLE],
            dtype=torch.bool,
            device=device
        )
        if not mask.any():
            continue

        aa=a_idx[mask]
        bb=b_idx[mask]

        teacher=F.softmax(logits[slot][aa].detach(),dim=-1)
        student_log=F.log_softmax(logits[slot][bb],dim=-1)
        losses.append(
            F.kl_div(
                student_log,
                teacher,
                reduction="batchmean"
            )
        )

    return torch.stack(losses).mean()

def directed_latent_invariance(z,device):
    a=AB_A_T.to(device)
    b=AB_B_T.to(device)
    za=F.normalize(z[a].detach(),p=2,dim=-1)
    zb=F.normalize(z[b],p=2,dim=-1)
    return (1-F.cosine_similarity(za,zb,dim=-1)).mean()

def invalid_joint_mass(logits):
    # Diagnostic only in the final method. Full SCALE does not optimize this loss.
    pa=F.softmax(logits["action"],dim=-1)
    pu=F.softmax(logits["authority"],dim=-1)
    pr=F.softmax(logits["role"],dim=-1)
    ps=F.softmax(logits["state"],dim=-1)
    pi=F.softmax(logits["intent"],dim=-1)
    pp=F.softmax(logits["provenance"],dim=-1)

    ac=list(encoders["action"].classes_)
    uc=list(encoders["authority"].classes_)
    rc=list(encoders["role"].classes_)
    sc=list(encoders["state"].classes_)
    ic=list(encoders["intent"].classes_)
    pc=list(encoders["provenance"].classes_)

    terms=[]

    for ai,a in enumerate(ac):
        req=AUTH_ORD[ACTION_REQUIRED_AUTH[a]]
        bad_u=[j for j,u in enumerate(uc) if AUTH_ORD[u]<req]
        bad_r=[j for j,r in enumerate(rc) if r not in ACTION_ALLOWED_ROLES[a]]
        bad_s=[j for j,s in enumerate(sc) if s not in ACTION_ALLOWED_STATES[a]]

        if bad_u:
            terms.append((pa[:,ai]*pu[:,bad_u].sum(-1)).mean())
        if bad_r:
            terms.append((pa[:,ai]*pr[:,bad_r].sum(-1)).mean())
        if bad_s:
            terms.append((pa[:,ai]*ps[:,bad_s].sum(-1)).mean())

    if "AUTHORIZE" in ic:
        ii=ic.index("AUTHORIZE")
        bad_p=[
            j for j,p in enumerate(pc)
            if p not in {"POLICY_CONTEXT","SYSTEM_EVIDENCE"}
        ]
        if bad_p:
            terms.append((pi[:,ii]*pp[:,bad_p].sum(-1)).mean())

    return torch.stack(terms).mean() if terms else pa.new_tensor(0.)

def active_correctness_tensor(logits,y,device):
    out=torch.zeros(
        len(train_messages),
        device=device,
        dtype=torch.float32
    )

    for agent in AGENTS:
        ids=torch.tensor(
            [i for i,a in enumerate(ROW_AGENT) if a==agent],
            dtype=torch.long,
            device=device
        )
        slot=ACTIVE_SLOT[agent]
        out[ids]=(
            logits[slot][ids].argmax(-1).eq(y[slot][ids])
        ).float()

    return out

def prototype_auxiliary_loss(model,x,y,label_masks):
    if not isinstance(model,HybridScaleHead):
        return x.new_tensor(0.)

    open_logits=model.open_logits(x,BASE_GRAPH)

    # Keep the graph branch grounded in known semantic labels without letting it
    # replace the closed-set discriminative decoder.
    vals=[]
    for slot in SLOTS:
        vals.append(
            masked_ce(
                open_logits[slot],
                y[slot],
                label_masks[slot]
            )
        )
    return torch.stack(vals).mean()

def train_model(
    method,
    seed,
    label_masks,
    epochs=HEAD_EPOCHS,
    use_inv=False,
    use_logic=False,
    use_cal=True,
    graph_mode=None,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    dual_methods={
        "CB+DualView",
        "SCALE","SCALE-NoInv","SCALE-NoOnt",
        "SCALE-NoCal","SCALE+Logic",
    }
    is_dual=method in dual_methods

    if method in {"CB-BGE","CB+Logic","CB+DualView"}:
        model=CBHead(
            X_CPU.shape[1],
            dual_view=is_dual
        )
    elif method=="Proto-CB":
        model=ProtoContractHead(
            X_CPU.shape[1],
            graph_dim=128,
            graph_mode=False,
            dual_view=False,
        )
    else:
        model=HybridScaleHead(
            X_CPU.shape[1],
            graph_mode=bool(graph_mode),
            dual_view=True,
        )

    model=model.to(HEAD_DEVICE)
    x=(X_DUAL_CPU if is_dual else X_CPU).to(HEAD_DEVICE)
    y={s:Y[s].to(HEAD_DEVICE) for s in SLOTS}

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4
    )

    visibility=b_active_visibility(label_masks)
    lambda_inv=BASE_INV_WEIGHT*max(0.,1.-visibility)

    hist=[]

    for ep in range(epochs):
        model.train()

        if isinstance(model,(ProtoContractHead,HybridScaleHead)):
            z,logits,conf=model(x,BASE_GRAPH)
        else:
            z,logits,conf=model(x)

        l_task=masked_ce(
            logits["action"],
            y["action"],
            label_masks["action"]
        )

        l_con=sum(
            masked_ce(logits[s],y[s],label_masks[s])
            for s in [
                "intent","object","role",
                "authority","provenance","state"
            ]
        )/6.

        linv=(
            directed_ab_invariance(logits,HEAD_DEVICE)
            if use_inv and lambda_inv>0
            else z.new_tensor(0.)
        )
        llat=(
            directed_latent_invariance(z,HEAD_DEVICE)
            if use_inv and lambda_inv>0
            else z.new_tensor(0.)
        )

        llogic=(
            invalid_joint_mass(logits)
            if use_logic
            else z.new_tensor(0.)
        )

        lproto=prototype_auxiliary_loss(
            model,x,y,label_masks
        )

        lanchor=(
            model.graph_anchor_loss(BASE_GRAPH)
            if isinstance(model,HybridScaleHead) and model.graph_mode
            else z.new_tensor(0.)
        )

        with torch.no_grad():
            active_ok=active_correctness_tensor(
                logits,y,HEAD_DEVICE
            )

        lcal=(
            ((conf-active_ok)**2).mean()
            if use_cal
            else z.new_tensor(0.)
        )

        loss=l_task+.70*l_con

        if use_inv and lambda_inv>0:
            loss += lambda_inv*linv
            loss += (
                DIRECTED_LATENT_WEIGHT
                *lambda_inv
                *llat
            )

        if isinstance(model,HybridScaleHead):
            loss += PROTO_AUX_WEIGHT*lproto
            if model.graph_mode:
                loss += GRAPH_ANCHOR_WEIGHT*lanchor

        if use_logic:
            loss += .15*llogic

        if use_cal:
            loss += .05*lcal

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),1.
        )
        opt.step()

        if ep in {0,9,29,epochs-1}:
            gates={}
            if isinstance(model,HybridScaleHead):
                gates={
                    f"graph_gate_{s}":
                    float(torch.sigmoid(
                        model.graph_gate[s]
                    ).detach())
                    for s in ["object","state","authority"]
                }

            hist.append({
                "method":method,
                "seed":seed,
                "epoch":ep+1,
                "b_visibility":visibility,
                "lambda_inv":lambda_inv,
                "loss":float(loss.detach()),
                "task":float(l_task.detach()),
                "concept":float(l_con.detach()),
                "directed_inv":float(linv.detach()),
                "directed_latent":float(llat.detach()),
                "proto_aux":float(lproto.detach()),
                "logic_diagnostic":float(llogic.detach()),
                "graph_anchor":float(lanchor.detach()),
                "active_brier":float(lcal.detach()),
                **gates,
            })

    model=model.cpu().eval()
    del x,y
    # Safe even when this cell is executed independently.
    if "cleanup_gpu" in globals():
        cleanup_gpu()
    else:
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return model,pd.DataFrame(hist)

In [ ]:
# Preflight — utilities required by the long training stage

assert "cleanup_gpu" in globals(), (
    "cleanup_gpu is missing. Run the Dependencies cell before training."
)
assert callable(cleanup_gpu), "cleanup_gpu must be callable."
assert "train_model" in globals() and callable(train_model)
assert "torch" in globals()

# A real no-op invocation catches ordering/runtime issues before multi-seed training.
cleanup_gpu()

print(
    "Training preflight PASS — cleanup_gpu is defined, callable, "
    "and executed successfully before model training."
)

# Model-shape smoke tests.
# These execute one tiny forward pass before the expensive five-seed training loop.
print("Model-shape smoke tests...")

_smoke_raw=X_CPU[:4]
_smoke_dual=X_DUAL_CPU[:4]

with torch.inference_mode():
    _cb=CBHead(X_CPU.shape[1],dual_view=False)
    _z,_logits,_conf=_cb(_smoke_raw)
    assert _z.shape[0]==4 and _conf.shape[0]==4
    for _slot in SLOTS:
        assert _logits[_slot].shape==(4,len(encoders[_slot].classes_))

    _cbd=CBHead(X_CPU.shape[1],dual_view=True)
    _z,_logits,_conf=_cbd(_smoke_dual)
    for _slot in SLOTS:
        assert _logits[_slot].shape==(4,len(encoders[_slot].classes_))

    _proto=ProtoContractHead(
        X_CPU.shape[1],graph_dim=128,graph_mode=False,dual_view=False
    )
    _z,_logits,_conf=_proto(_smoke_raw,BASE_GRAPH)
    for _slot in SLOTS:
        assert _logits[_slot].shape==(4,len(encoders[_slot].classes_))

    _scale=HybridScaleHead(
        X_CPU.shape[1],graph_mode=True,dual_view=True
    )
    _z,_closed,_conf=_scale(_smoke_dual,BASE_GRAPH)
    _open=_scale.open_logits(_smoke_dual,BASE_GRAPH)
    for _slot in SLOTS:
        expected=(4,len(encoders[_slot].classes_))
        assert _closed[_slot].shape==expected
        assert _open[_slot].shape==expected

del _cb,_cbd,_proto,_scale,_z,_logits,_closed,_open,_conf
cleanup_gpu()

print("Training preflight PASS — utilities + decoder dimensions are valid.")


In [ ]:
# 11/31 — Train paired multi-seed final models and diagnostics

MODEL_SPECS={
    "CB-BGE":dict(
        use_inv=False,use_logic=False,use_cal=True,graph_mode=None
    ),
    "CB+DualView":dict(
        use_inv=False,use_logic=False,use_cal=True,graph_mode=None
    ),
    "CB+Logic":dict(
        use_inv=False,use_logic=True,use_cal=True,graph_mode=None
    ),
    "Proto-CB":dict(
        use_inv=False,use_logic=False,use_cal=True,graph_mode=False
    ),
    "SCALE-NoInv":dict(
        use_inv=False,use_logic=False,use_cal=True,graph_mode=True
    ),
    "SCALE-NoOnt":dict(
        use_inv=True,use_logic=False,use_cal=True,graph_mode=False
    ),
    "SCALE-NoCal":dict(
        use_inv=True,use_logic=False,use_cal=False,graph_mode=True
    ),
    # Diagnostic only: learned logic is not part of Full SCALE.
    "SCALE+Logic":dict(
        use_inv=True,use_logic=True,use_cal=True,graph_mode=True
    ),
    "SCALE":dict(
        use_inv=True,use_logic=False,use_cal=True,graph_mode=True
    ),
}

MODELS={m:[] for m in MODEL_SPECS}
histories=[]

for seed in HEAD_SEEDS:
    print("HEAD SEED",seed)

    for method,spec in MODEL_SPECS.items():
        model,h=train_model(
            method=method,
            seed=seed,
            label_masks=MAIN_LABEL_MASK,
            epochs=HEAD_EPOCHS,
            **spec
        )
        MODELS[method].append(model)
        histories.append(h)

training_history=pd.concat(
    histories,
    ignore_index=True
)
display(training_history.round(4))

training_history.to_csv(
    ROOT/"head_training_history.csv",
    index=False
)

print(
    "Trained models:",
    {k:len(v) for k,v in MODELS.items()}
)

In [ ]:
# 12/27 — Ensemble prediction, Symbolic, CtD-lite, calibration

def single_predict(model,texts,graph_ctx=BASE_GRAPH):
    use_dual=bool(getattr(model,"dual_view",False))
    arr=dual_view_encode(texts) if use_dual else semantic_encode(texts)
    emb=torch.tensor(arr,dtype=torch.float32)

    with torch.inference_mode():
        if isinstance(model,(ProtoContractHead,HybridScaleHead)):
            z,logits,conf=model(emb,graph_ctx)
        else:
            z,logits,conf=model(emb)

    out=[]
    for i in range(len(texts)):
        probs={}
        contract={}
        for s in SLOTS:
            p=F.softmax(logits[s][i],dim=-1).cpu().numpy()
            probs[s]=p
            contract[s]=str(
                encoders[s].inverse_transform([int(np.argmax(p))])[0]
            )

        out.append({
            "contract":contract,
            "probs":probs,
            "confidence":float(conf[i]),
            "embedding":z[i].cpu().numpy(),
        })
    return out

def ensemble_predict(method,texts,graph_ctx=BASE_GRAPH):
    members=MODELS[method]
    per_member=[
        single_predict(m,texts,graph_ctx)
        for m in members
    ]

    out=[]
    for i in range(len(texts)):
        probs={}
        contract={}
        for s in SLOTS:
            p=np.mean(
                [member[i]["probs"][s] for member in per_member],
                axis=0
            )
            probs[s]=p
            contract[s]=str(
                encoders[s].inverse_transform([int(np.argmax(p))])[0]
            )

        out.append({
            "contract":contract,
            "probs":probs,
            "confidence":float(np.mean(
                [member[i]["confidence"] for member in per_member]
            )),
            "member_contracts":[m[i]["contract"] for m in per_member],
        })
    return out

OBJ_B_REV={v:k for k,v in OBJ_B.items()}
EVID_B_REV={v:k for k,v in EVID_B.items()}
AUTH_B_REV={v:k for k,v in AUTH_B.items()}

def symbolic_parse(message,agent):
    up=message.upper()
    low=message.lower()

    if agent=="PLANNER":
        obj=next(
            (o for o in PASS_ACTION_BY_OBJECT if re.search(rf"\b{re.escape(o)}\b",up)),
            None
        )
        if obj is None:
            for alias,canonical in OBJ_B_REV.items():
                if alias in low:
                    obj=canonical
                    break
        return {
            "intent":"REQUEST","object":obj,"action":"PLAN","role":"PLANNER",
            "authority":"PROPOSE","provenance":"USER_REQUEST","state":"PROPOSED",
        }

    if agent=="RETRIEVER":
        status=next(
            (s for s in ["PASS","FAIL","UNCERTAIN"] if re.search(rf"\b{s}\b",up)),
            None
        )
        if status is None:
            for alias,canonical in EVID_B_REV.items():
                if alias in low:
                    status=canonical
                    break
        return {
            "intent":"QUERY","object":"NO_OBJECT","action":"SEARCH","role":"RETRIEVER",
            "authority":"NONE","provenance":"CRM_DATA",
            "state":EVIDENCE_TO_STATE.get(status),
        }

    auth=next(
        (a for a in AUTHORITIES if re.search(rf"\b{a}\b",up)),
        None
    )
    if auth is None:
        for alias,canonical in AUTH_B_REV.items():
            if alias in low:
                auth=canonical
                break

    return {
        "intent":"RECOMMEND","object":"NO_OBJECT","action":"ASSESS","role":"POLICY",
        "authority":auth,"provenance":"POLICY_CONTEXT","state":"POLICY_READY",
    }

kmeans=MiniBatchKMeans(
    n_clusters=42,
    random_state=SEED,
    batch_size=128,
    n_init="auto",
)
codes=kmeans.fit_predict(X_np)
CODEBOOK={}

for code in range(42):
    ids=np.where(codes==code)[0]
    if len(ids):
        CODEBOOK[code]={}
        for s in SLOTS:
            vals=[train_messages[i]["contract"][s] for i in ids]
            CODEBOOK[code][s]=max(set(vals),key=vals.count)

def ctd_predict(text):
    code=int(kmeans.predict(semantic_encode([text]))[0])
    return CODEBOOK.get(code)

def slot_accuracy(gold,pred):
    if pred is None:
        return 0.0
    return float(np.mean([pred.get(s)==gold[s] for s in SLOTS]))

def exact_contract(gold,pred):
    return int(pred is not None and all(pred.get(s)==gold[s] for s in SLOTS))

def active_correct(gold,pred,agent):
    slot=ACTIVE_SLOT[agent]
    return int(pred is not None and pred.get(slot)==gold[slot])

def ece(conf,correct,bins=10):
    conf=np.asarray(conf,float)
    correct=np.asarray(correct,float)
    edges=np.linspace(0,1,bins+1)
    score=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(conf>=lo)&(conf<(hi if hi<1 else hi+1e-9))
        if m.any():
            score += m.mean()*abs(conf[m].mean()-correct[m].mean())
    return float(score)

In [ ]:
# 13/27 — Representation benchmark: A/B, C1, C2 + seed-level ablations

REP_METHODS=[
    "Symbolic","CtD-lite",
    "CB-BGE","CB+DualView","CB+Logic","Proto-CB",
    "SCALE-NoInv","SCALE-NoOnt","SCALE-NoCal","SCALE+Logic",
    "SCALE-Raw","SCALE-Full"
]

rep_rows=[]
seed_rows=[]

def prediction_for(method,msg,agent,graph_ctx=BASE_GRAPH,repair=True):
    if method=="Symbolic":
        return symbolic_parse(msg,agent),None,None

    if method=="CtD-lite":
        return ctd_predict(msg),None,None

    model_name={
        "SCALE-Raw":"SCALE",
        "SCALE-Full":"SCALE",
    }.get(method,method)

    p=ensemble_predict(model_name,[msg],graph_ctx)[0]
    c=p["contract"]

    if repair and method in {
        "CB+Logic","SCALE-NoInv","SCALE-NoOnt",
        "SCALE-NoCal","SCALE+Logic","SCALE-Full"
    }:
        c=ontology_repair(c)

    return c,p["confidence"],p

def evaluate_rep(base,agent,impl,split_name,variant=0,graph_ctx=BASE_GRAPH):
    msg=(
        ontology_shift_planner_message(base)
        if impl=="ONT" and agent=="PLANNER"
        else realize_message(base,agent,impl,variant)
        if impl!="ONT"
        else realize_message(base,agent,"C1",variant)
    )
    gold=hop_contract(base,agent)

    for method in REP_METHODS:
        pred,conf,pobj=prediction_for(
            method,msg,agent,graph_ctx,
            repair=(method!="SCALE-Raw")
        )

        rep_rows.append({
            "split":split_name,
            "base_id":base["base_id"],
            "agent":agent,
            "implementation":impl,
            "variant":variant,
            "method":method,
            "active_correct":active_correct(gold,pred,agent),
            "slot_accuracy":slot_accuracy(gold,pred),
            "exact_contract":exact_contract(gold,pred),
            "semantic_violation":int(bool(contract_violations(pred))),
            "confidence":conf,
        })

        # Per-seed rows for learned methods to support paired ablations.
        base_method={
            "SCALE-Raw":"SCALE",
            "SCALE-Full":"SCALE",
        }.get(method,method)

        if base_method in MODELS:
            for seed,member in zip(HEAD_SEEDS,MODELS[base_method]):
                pm=single_predict(member,[msg],graph_ctx)[0]
                cm=pm["contract"]

                if method in {
                    "CB+Logic","SCALE-NoInv","SCALE-NoOnt",
                    "SCALE-NoCal","SCALE+Logic","SCALE-Full"
                }:
                    cm=ontology_repair(cm)

                seed_rows.append({
                    "split":split_name,
                    "base_id":base["base_id"],
                    "agent":agent,
                    "method":method,
                    "seed":seed,
                    "active_correct":active_correct(gold,cm,agent),
                    "exact_contract":exact_contract(gold,cm),
                })

for base in core_test_base:
    for agent in AGENTS:
        for impl in ("A","B"):
            for variant in range(3):
                evaluate_rep(base,agent,impl,"known_ab",variant)

if RUN_C1_C2:
    for base in shift_test_base:
        for agent in AGENTS:
            evaluate_rep(base,agent,"C1","c1_natural",0)
            evaluate_rep(base,agent,"C2","c2_schema",0)

rep_df=pd.DataFrame(rep_rows)
seed_rep_df=pd.DataFrame(seed_rows)

rep_summary=(
    rep_df.groupby(["split","method"],as_index=False)
    .agg(
        active_accuracy=("active_correct","mean"),
        slot_accuracy=("slot_accuracy","mean"),
        exact_contract=("exact_contract","mean"),
        semantic_violation_rate=("semantic_violation","mean"),
        n=("base_id","size"),
    )
)

cal_rows=[]
for (split_name,method),g in rep_df.dropna(subset=["confidence"]).groupby(["split","method"]):
    cal_rows.append({
        "split":split_name,
        "method":method,
        "active_ece":ece(g["confidence"],g["active_correct"]),
        "n":len(g),
    })
calibration_df=pd.DataFrame(cal_rows)

display(rep_summary.round(4))
display(calibration_df.round(4))

rep_df.to_csv(ROOT/"representation_results.csv",index=False)
seed_rep_df.to_csv(ROOT/"representation_seed_results.csv",index=False)
rep_summary.to_csv(ROOT/"representation_summary.csv",index=False)
calibration_df.to_csv(ROOT/"calibration_summary.csv",index=False)

In [ ]:
# 14/31 — True ontology extension with the final open-world graph decoder

def dynamic_object_member(model,text,graph_ctx):
    use_dual=bool(getattr(model,"dual_view",False))
    arr=(
        dual_view_encode([text])
        if use_dual
        else semantic_encode([text])
    )
    x=torch.tensor(arr,dtype=torch.float32)

    candidate_keys=[
        k for k in graph_ctx["node_keys"]
        if k[0] in {"object","object_leaf"}
    ]

    with torch.inference_mode():
        if isinstance(model,HybridScaleHead):
            q=model.query(x,"object")[0]
            all_nodes,_=model.all_node_repr(
                graph_ctx,"object"
            )
        elif isinstance(model,ProtoContractHead):
            q=model.query(x,"object")[0]
            all_nodes,_=model.all_node_repr(
                graph_ctx,"object"
            )
        else:
            raise TypeError(type(model).__name__)

    ids=torch.tensor(
        [graph_ctx["idx"][k] for k in candidate_keys],
        dtype=torch.long
    )
    proto=F.normalize(
        all_nodes[ids],
        p=2,dim=-1
    )
    scores=(q@proto.T).cpu().numpy()
    j=int(np.argmax(scores))

    return candidate_keys[j][1],scores

def ontology_object_prediction(method,text,graph_ctx):
    if method=="Symbolic":
        c=symbolic_parse(text,"PLANNER")
        return c.get("object"),c.get("object")

    if method in {
        "CB-BGE","CB+DualView","CB+Logic"
    }:
        p=ensemble_predict(method,[text],BASE_GRAPH)[0]
        obj=p["contract"].get("object")
        return obj,obj

    model_name={
        "SCALE-Full":"SCALE",
        "SCALE-NoOnt":"SCALE-NoOnt",
        "Proto-CB":"Proto-CB",
        "Proto+Ancestor":"Proto-CB",
    }[method]

    votes=[]
    for member in MODELS[model_name]:
        label,_=dynamic_object_member(
            member,text,graph_ctx
        )
        votes.append(label)

    predicted=max(set(votes),key=votes.count)

    # SCALE-Full and Proto+Ancestor are permitted to traverse the
    # explicitly supplied unseen leaf→parent ontology relation.
    if (
        method in {"SCALE-Full","Proto+Ancestor"}
        and predicted in graph_ctx["leaf_parent"]
    ):
        canonical=graph_ctx["leaf_parent"][predicted]
    elif predicted in OBJECTS:
        canonical=predicted
    else:
        canonical=predicted

    return predicted,canonical

ontology_rows=[]

if RUN_ONTOLOGY_SHIFT:
    for tier,graph_ctx,leaf_map in [
        ("transparent",TRANSPARENT_GRAPH,TRANSPARENT_BY_PARENT),
        ("opaque",OPAQUE_GRAPH,OPAQUE_BY_PARENT),
    ]:
        for base in ontology_test_base:
            gold_parent=base["object"]
            gold_leaf=leaf_map[gold_parent]["label"]
            msg=ontology_shift_planner_message(
                base,tier
            )

            for method in [
                "Symbolic","CB-BGE","CB+DualView","CB+Logic",
                "Proto-CB","Proto+Ancestor",
                "SCALE-NoOnt","SCALE-Full"
            ]:
                pred_node,canonical=ontology_object_prediction(
                    method,msg,graph_ctx
                )

                ontology_rows.append({
                    "tier":tier,
                    "base_id":base["base_id"],
                    "gold_leaf":gold_leaf,
                    "gold_parent":gold_parent,
                    "method":method,
                    "predicted_node":pred_node,
                    "canonical_object":canonical,
                    "leaf_detection":int(pred_node==gold_leaf),
                    "canonical_parent_accuracy":int(
                        canonical==gold_parent
                    ),
                })

ontology_df=pd.DataFrame(ontology_rows)

ontology_summary=(
    ontology_df.groupby(
        ["tier","method"],
        as_index=False
    )
    .agg(
        leaf_detection=("leaf_detection","mean"),
        canonical_parent_accuracy=(
            "canonical_parent_accuracy","mean"
        ),
        n=("base_id","size"),
    )
) if len(ontology_df) else pd.DataFrame()

if len(ontology_summary):
    display(ontology_summary.round(4))
    ontology_df.to_csv(
        ROOT/"ontology_shift_results.csv",
        index=False
    )
    ontology_summary.to_csv(
        ROOT/"ontology_shift_summary.csv",
        index=False
    )

In [ ]:
# 15/30 — Label-efficiency curve: direct invariance-mechanism test

curve_rows=[]

def active_eval_set():
    rows=[]
    for base in shift_test_base:
        for agent in AGENTS:
            rows.append((base,agent,"B","heldout_b"))
            rows.append((base,agent,"C1","c1_natural"))
            rows.append((base,agent,"C2","c2_schema"))
    return rows

if RUN_LABEL_EFFICIENCY_CURVE:
    eval_items=active_eval_set()

    for visibility in LABEL_VISIBILITY_GRID:
        masks=build_label_masks(visibility)

        for seed in CURVE_SEEDS:
            trained={}

            trained["CB-BGE"],_=train_model(
                "CB-BGE",seed,masks,
                epochs=CURVE_EPOCHS,
                use_inv=False,use_logic=False,use_cal=True,graph_mode=None
            )

            trained["CB+DualView"],_=train_model(
                "CB+DualView",seed,masks,
                epochs=CURVE_EPOCHS,
                use_inv=False,use_logic=False,use_cal=True,graph_mode=None
            )

            trained["SCALE-NoInv"],_=train_model(
                "SCALE-NoInv",seed,masks,
                epochs=CURVE_EPOCHS,
                use_inv=False,use_logic=False,use_cal=True,graph_mode=True
            )

            # Compatibility-only control:
            # directed invariance is retained while ontology graph propagation is removed.
            # This is the closest internal control to backward-compatible
            # representation learning without explicit ontology structure.
            trained["SCALE-NoOnt"],_=train_model(
                "SCALE-NoOnt",seed,masks,
                epochs=CURVE_EPOCHS,
                use_inv=True,use_logic=False,use_cal=True,graph_mode=False
            )

            trained["SCALE"],_=train_model(
                "SCALE",seed,masks,
                epochs=CURVE_EPOCHS,
                use_inv=True,use_logic=False,use_cal=True,graph_mode=True
            )

            for method,model in trained.items():
                for base,agent,impl,split_name in eval_items:
                    msg=realize_message(base,agent,impl,0)
                    pred=single_predict(model,[msg],BASE_GRAPH)[0]["contract"]

                    if method in {"SCALE","SCALE-NoInv","SCALE-NoOnt"}:
                        pred=ontology_repair(pred)

                    gold=hop_contract(base,agent)

                    curve_rows.append({
                        "visibility":visibility,
                        "seed":seed,
                        "method":method,
                        "split":split_name,
                        "base_id":base["base_id"],
                        "agent":agent,
                        "implementation":impl,
                        "active_correct":active_correct(gold,pred,agent),
                    })

            del trained
            gc.collect()

curve_df=pd.DataFrame(curve_rows)

if len(curve_df):
    curve_summary=(
        curve_df.groupby(
            ["visibility","method","split"],
            as_index=False
        )
        .agg(
            active_accuracy=("active_correct","mean"),
            active_sd=("active_correct","std"),
            n=("active_correct","size"),
        )
    )
    display(curve_summary.round(4))
    curve_df.to_csv(ROOT/"label_efficiency_results.csv",index=False)
    curve_summary.to_csv(ROOT/"label_efficiency_summary.csv",index=False)
else:
    curve_summary=pd.DataFrame()

# Final label-efficiency diagnostics.
# These do NOT alter G2 or tune a threshold; they expose seed stability,
# paired deltas and the role of ontology vs compatibility alignment.
curve_seed_summary=pd.DataFrame()
curve_paired_deltas=pd.DataFrame()

if len(curve_df):
    curve_seed_summary=(
        curve_df.groupby(
            ["visibility","seed","method","split"],
            as_index=False
        )
        .agg(active_accuracy=("active_correct","mean"))
    )

    paired_rows=[]
    baselines=[
        "CB-BGE",
        "CB+DualView",
        "SCALE-NoInv",
        "SCALE-NoOnt",
    ]

    for visibility in LABEL_VISIBILITY_GRID:
        for split_name in ["heldout_b","c1_natural","c2_schema"]:
            sub=curve_df[
                (curve_df.visibility==visibility)
                &(curve_df.split==split_name)
            ].copy()

            for baseline in baselines:
                a=sub[sub.method=="SCALE"][
                    ["seed","base_id","agent","active_correct"]
                ].rename(columns={"active_correct":"scale"})
                b=sub[sub.method==baseline][
                    ["seed","base_id","agent","active_correct"]
                ].rename(columns={"active_correct":"baseline"})

                pair=a.merge(
                    b,on=["seed","base_id","agent"],how="inner"
                )
                if not len(pair):
                    continue

                d=(pair["scale"]-pair["baseline"]).to_numpy(dtype=float)
                rng=np.random.default_rng(
                    int(SEED+round(1000*visibility)+len(paired_rows))
                )
                boots=[]
                for _ in range(2000):
                    ids=rng.integers(0,len(d),len(d))
                    boots.append(float(d[ids].mean()))

                lo,hi=np.percentile(boots,[2.5,97.5])
                paired_rows.append({
                    "visibility":visibility,
                    "split":split_name,
                    "baseline":baseline,
                    "n_pairs":len(d),
                    "scale_minus_baseline":float(d.mean()),
                    "ci95_low":float(lo),
                    "ci95_high":float(hi),
                })

    curve_paired_deltas=pd.DataFrame(paired_rows)

    display(
        curve_seed_summary.groupby(
            ["visibility","method","split"],
            as_index=False
        ).agg(
            mean_accuracy=("active_accuracy","mean"),
            seed_sd=("active_accuracy","std"),
            n_seeds=("seed","nunique"),
        ).round(4)
    )
    display(curve_paired_deltas.round(4))

    curve_seed_summary.to_csv(
        ROOT/"label_efficiency_seed_summary.csv",index=False
    )
    curve_paired_deltas.to_csv(
        ROOT/"label_efficiency_paired_deltas.csv",index=False
    )

    # Flag non-monotonicity for interpretation rather than hiding it.
    scale_held=(
        curve_seed_summary[
            (curve_seed_summary.method=="SCALE")
            &(curve_seed_summary.split=="heldout_b")
        ]
        .groupby("visibility")["active_accuracy"]
        .mean()
        .sort_index()
    )
    if len(scale_held)>1:
        diffs=np.diff(scale_held.values)
        if (diffs < -1e-9).any():
            print(
                "NOTE: SCALE held-out-B curve is non-monotonic across label "
                "visibility. Treat G2 as a stability question; do not infer "
                "monotonic sample-efficiency without additional evidence."
            )


In [ ]:
# 16/30 — Active interface, Onto-RAG, and native structured receiver

def active_payload(contract,agent,confidence=None):
    slot=ACTIVE_SLOT[agent]
    key="evidence_state" if agent=="RETRIEVER" else slot
    value=None if contract is None else contract.get(slot)

    return json.dumps({
        "role":agent,
        "active":{key:value},
        "confidence":None if confidence is None else round(float(confidence),4),
    },ensure_ascii=False,separators=(",",":"))

# Strong role-filtered ontology retrieval baseline.
ONTO_RAG_DOCS=[]

def add_doc(role,kind,canonical,text):
    ONTO_RAG_DOCS.append({
        "role":role,"kind":kind,"canonical":canonical,"text":text
    })

for obj in PASS_ACTION_BY_OBJECT:
    add_doc(
        "PLANNER","object",obj,
        f"{obj.lower()} {OBJ_B[obj]} enterprise workflow object"
    )

for status,state in EVIDENCE_TO_STATE.items():
    add_doc(
        "RETRIEVER","evidence_state",state,
        f"{status.lower()} {EVID_B[status]} evidence state {state.lower()}"
    )

for auth in OPERATIONAL_AUTHORITIES:
    add_doc(
        "POLICY","authority",auth,
        f"{auth.lower()} {AUTH_B[auth]} delegated authority"
    )

ONTO_RAG_EMB=semantic_encode([x["text"] for x in ONTO_RAG_DOCS])

def onto_rag_retrieve(raw,agent,topk=KG_TOPK):
    q=semantic_encode([raw])[0]
    ids=[i for i,x in enumerate(ONTO_RAG_DOCS) if x["role"]==agent]
    scores=[float(ONTO_RAG_EMB[i]@q) for i in ids]
    ranked=np.argsort(-np.asarray(scores))[:topk]

    return [
        {
            "kind":ONTO_RAG_DOCS[ids[j]]["kind"],
            "concept":ONTO_RAG_DOCS[ids[j]]["canonical"],
            "score":round(scores[j],4),
        }
        for j in ranked
    ]

def learned_contract(msg,agent,method,graph_ctx=BASE_GRAPH):
    if method=="Symbolic":
        return symbolic_parse(msg,agent),None

    model_method=method
    p=ensemble_predict(model_method,[msg],graph_ctx)[0]
    c=p["contract"]

    if method in {"CB+Logic","SCALE"}:
        c=ontology_repair(c)

    return c,p["confidence"]

def interface_message(base,agent,impl,method,graph_ctx=BASE_GRAPH):
    raw=(
        ontology_shift_planner_message(base)
        if impl=="ONT" and agent=="PLANNER"
        else realize_message(base,agent,impl if impl!="ONT" else "C1",0)
    )

    if method=="NL":
        return raw

    if method=="JSON":
        return json.dumps(
            {"agent":agent,"implementation":impl,"message":raw},
            ensure_ascii=False,separators=(",",":")
        )

    if method=="Onto-RAG":
        return json.dumps(
            {"message":raw,"ontology_hits":onto_rag_retrieve(raw,agent)},
            ensure_ascii=False,separators=(",",":")
        )

    c,conf=learned_contract(raw,agent,method,graph_ctx)
    return active_payload(c,agent,conf)

def build_interface(base,comp,method,graph_ctx=BASE_GRAPH):
    packets=[]
    for agent in AGENTS:
        impl=comp["mapping"][agent]
        packets.append(
            f"[{agent}]\n{interface_message(base,agent,impl,method,graph_ctx)}"
        )
    return "\n\n".join(packets)

def native_semantics(base,comp,method,graph_ctx=BASE_GRAPH):
    if method not in STRUCTURED_METHODS:
        return None

    out={}
    for agent in AGENTS:
        impl=comp["mapping"][agent]
        raw=realize_message(base,agent,impl,0)
        c,_=learned_contract(raw,agent,method,graph_ctx)

        if agent=="PLANNER":
            out["object"]=None if c is None else c.get("object")
        elif agent=="RETRIEVER":
            out["evidence_state"]=None if c is None else c.get("state")
        else:
            out["authority"]=None if c is None else c.get("authority")

    return out

In [ ]:
# 17/27 — Native receiver benchmark

native_rows=[]

splits=[("core",core_test_base,CORE_COMPOSITIONS)]
if RUN_C1_C2:
    splits += [
        ("c1_natural",shift_test_base,C1_COMPOSITIONS),
        ("c2_schema",shift_test_base,C2_COMPOSITIONS),
    ]

for split_name,tasks,comps in splits:
    for base in tasks:
        gold_sem=global_semantics(base)

        for comp in comps:
            for method in STRUCTURED_METHODS:
                sem=native_semantics(base,comp,method,BASE_GRAPH)
                final=deterministic_policy(sem)

                native_rows.append({
                    "split":split_name,
                    "base_id":base["base_id"],
                    "composition_id":comp["composition_id"],
                    "replacement_level":comp["replacement_level"],
                    "method":method,
                    "semantic_success":int(sem==gold_sem),
                    "task_success":int(
                        final is not None
                        and final["object"]==base["object"]
                        and final["action"]==base["final_action"]
                        and final["authority"]==base["final_authority"]
                    ),
                })

native_df=pd.DataFrame(native_rows)
native_summary=(
    native_df.groupby(["split","method","replacement_level"],as_index=False)
    .agg(
        semantic_success=("semantic_success","mean"),
        task_success=("task_success","mean"),
        n=("task_success","size"),
    )
)
display(native_summary.round(4))
native_df.to_csv(ROOT/"native_receiver_results.csv",index=False)
native_summary.to_csv(ROOT/"native_receiver_summary.csv",index=False)

In [ ]:
# 18/27 — Guaranteed constraint-stress benchmark

def final_contract(base):
    sem=global_semantics(base)
    final=deterministic_policy(sem)
    a=final["action"]

    intent=(
        "AUTHORIZE" if a=="APPROVE"
        else "DENY" if a=="REJECT"
        else "VERIFY" if a=="VERIFY"
        else "REQUEST"
    )

    return {
        "intent":intent,
        "object":base["object"],
        "action":a,
        "role":"EXECUTOR",
        "authority":base["authority"],
        "provenance":"SYSTEM_EVIDENCE",
        "state":ACTION_DEFAULT_STATE[a],
    }

def strictly_below_required(action):
    req_idx=AUTH_ORD[ACTION_REQUIRED_AUTH[action]]
    return AUTHORITIES[max(0,req_idx-1)]

def corrupt_contract(c,kind):
    x=dict(c)

    if kind=="authority_below_required":
        x["authority"]=strictly_below_required(x["action"])

    elif kind=="role_mismatch":
        valid=ACTION_ALLOWED_ROLES[x["action"]]
        x["role"]=next(r for r in ROLES if r not in valid)

    elif kind=="state_mismatch":
        x["state"]=next(
            s for s in STATES
            if s not in ACTION_ALLOWED_STATES[x["action"]]
        )

    elif kind=="provenance_corruption":
        x["intent"]="AUTHORIZE"
        x["provenance"]="USER_REQUEST"

    return x

stress_rows=[]
for base in dev_base:
    gold=final_contract(base)

    for kind in [
        "authority_below_required",
        "role_mismatch",
        "state_mismatch",
        "provenance_corruption",
    ]:
        bad=corrupt_contract(gold,kind)
        before=contract_violations(bad)
        repaired=ontology_repair(bad)
        after=contract_violations(repaired)

        stress_rows.append({
            "base_id":base["base_id"],
            "corruption":kind,
            "detected":int(bool(before)),
            "fully_repaired":int(not bool(after)),
            "residual_violation":int(bool(after)),
            "should_block":int(bool(after)),
        })

stress_df=pd.DataFrame(stress_rows)
stress_summary=(
    stress_df.groupby("corruption",as_index=False)
    .agg(
        n=("base_id","size"),
        detection_rate=("detected","mean"),
        repair_rate=("fully_repaired","mean"),
        residual_violation_rate=("residual_violation","mean"),
        block_rate=("should_block","mean"),
    )
)
display(stress_summary.round(4))
stress_df.to_csv(ROOT/"constraint_stress_results.csv",index=False)
stress_summary.to_csv(ROOT/"constraint_stress_summary.csv",index=False)

In [ ]:
# 19/30 — Pre-repair adversarial semantic-consistency benchmark

logic_rows=[]

def adversarial_messages(base):
    return [
        (
            "planner_action_role_conflict",
            "PLANNER",
            f"ROLE=PLANNER | object={base['object']} | local action=SEARCH | provenance=USER_REQUEST"
        ),
        (
            "retriever_action_role_conflict",
            "RETRIEVER",
            f"ROLE=RETRIEVER | evidence={base['evidence_status']} | local action=PLAN | provenance=CRM_DATA"
        ),
        (
            "policy_provenance_conflict",
            "POLICY",
            f"ROLE=POLICY | intent=AUTHORIZE | provenance=USER_REQUEST | authority={base['authority']} | local action=ASSESS"
        ),
        (
            "executor_authority_conflict",
            "POLICY",
            "ROLE=EXECUTOR | action=UPDATE | authority=APPROVE | state=EXECUTED | provenance=SYSTEM_EVIDENCE"
        ),
    ]

if RUN_LOGIC_ADVERSARIAL:
    for base in dev_base:
        for corruption,agent,msg in adversarial_messages(base):

            for method in [
                "CB-BGE","CB+Logic",
                "SCALE","SCALE+Logic"
            ]:
                p=ensemble_predict(method,[msg],BASE_GRAPH)[0]
                raw=p["contract"]
                violations=contract_violations(raw)

                logic_rows.append({
                    "base_id":base["base_id"],
                    "corruption":corruption,
                    "method":method,
                    "invalid_contract":int(bool(violations)),
                    "n_violations":len(violations),
                    "violations":"|".join(violations),
                })

logic_df=pd.DataFrame(logic_rows)

if len(logic_df):
    logic_adversarial_summary=(
        logic_df.groupby("method",as_index=False)
        .agg(
            invalid_contract_rate=("invalid_contract","mean"),
            mean_violations=("n_violations","mean"),
            n=("base_id","size"),
        )
    )
    display(logic_adversarial_summary.round(4))
    logic_df.to_csv(ROOT/"logic_adversarial_results.csv",index=False)
    logic_adversarial_summary.to_csv(
        ROOT/"logic_adversarial_summary.csv",index=False
    )
else:
    logic_adversarial_summary=pd.DataFrame()

In [ ]:
# 19/30 — Leakage-free PrePolicy Active-SCD

def jsd(p,q,eps=1e-8):
    p=np.asarray(p,float)+eps
    q=np.asarray(q,float)+eps
    p/=p.sum();q/=q.sum()
    m=.5*(p+q)
    return float(
        .5*np.sum(p*np.log(p/m))
        +.5*np.sum(q*np.log(q/m))
    )

def object_distance(a,b):
    return 0.0 if a==b else 1.0

STATE_DIST={
    ("EVIDENCE_PASS","EVIDENCE_UNCERTAIN"):.5,
    ("EVIDENCE_UNCERTAIN","EVIDENCE_PASS"):.5,
    ("EVIDENCE_FAIL","EVIDENCE_UNCERTAIN"):.5,
    ("EVIDENCE_UNCERTAIN","EVIDENCE_FAIL"):.5,
    ("EVIDENCE_PASS","EVIDENCE_FAIL"):1.0,
    ("EVIDENCE_FAIL","EVIDENCE_PASS"):1.0,
}

def state_distance(a,b):
    if a==b:
        return 0.0
    return STATE_DIST.get((a,b),1.0)

def authority_distance(a,b):
    if a not in AUTH_ORD or b not in AUTH_ORD:
        return 1.0
    return abs(AUTH_ORD[a]-AUTH_ORD[b])/(len(AUTHORITIES)-1)

def prepolicy_active_scd(reference,current):
    """
    PRIMARY RQ4 metric.
    No final-action / downstream-policy term is allowed here.
    """
    slots={
        "PLANNER":"object",
        "RETRIEVER":"state",
        "POLICY":"authority",
    }

    posterior=[]
    semantic=[]

    for agent,slot in slots.items():
        posterior.append(jsd(
            reference[agent]["probs"][slot],
            current[agent]["probs"][slot]
        ))

        a=reference[agent]["contract"].get(slot)
        b=current[agent]["contract"].get(slot)

        if slot=="object":
            semantic.append(object_distance(a,b))
        elif slot=="state":
            semantic.append(state_distance(a,b))
        else:
            semantic.append(authority_distance(a,b))

    return (
        .65*float(np.mean(posterior))
        +.35*float(np.mean(semantic))
    )

def policy_augmented_active_scd(reference,current):
    """
    SECONDARY DIAGNOSTIC ONLY.
    Never used for the primary failure-prediction claim.
    """
    pre=prepolicy_active_scd(reference,current)

    ref_sem={
        "object":reference["PLANNER"]["contract"].get("object"),
        "evidence_state":reference["RETRIEVER"]["contract"].get("state"),
        "authority":reference["POLICY"]["contract"].get("authority"),
    }
    cur_sem={
        "object":current["PLANNER"]["contract"].get("object"),
        "evidence_state":current["RETRIEVER"]["contract"].get("state"),
        "authority":current["POLICY"]["contract"].get("authority"),
    }

    policy_impact=float(
        deterministic_policy(ref_sem)!=deterministic_policy(cur_sem)
    )

    return .80*pre+.20*policy_impact

def full_scd(reference,current):
    vals=[]
    for agent in AGENTS:
        vals.append(np.mean([
            jsd(
                reference[agent]["probs"][s],
                current[agent]["probs"][s]
            )
            for s in SLOTS
        ]))
    return float(np.mean(vals))

In [ ]:
# 20/30 — Calibrated D0–D3 drift benchmark

OBJECT_DRIFT_MAP={
    "PURCHASE":"CHANGE",
    "ACCESS":"INCIDENT",
    "CHANGE":"INCIDENT",
    "INCIDENT":"CHANGE",
    "COMPLIANCE_ITEM":"CHANGE",
}

def flip_status(status):
    return {
        "PASS":"FAIL",
        "FAIL":"PASS",
        "UNCERTAIN":"FAIL",
    }[status]

def authority_below_desired(base):
    desired=desired_action(base["object"],base["evidence_status"])
    req_idx=AUTH_ORD[ACTION_REQUIRED_AUTH[desired]]
    return AUTHORITIES[max(0,req_idx-1)]

def drifted_base(base,ops):
    x=copy.deepcopy(base)

    for op in ops:
        if op=="object_substitution":
            x["object"]=OBJECT_DRIFT_MAP[x["object"]]

        elif op=="evidence_flip":
            x["evidence_status"]=flip_status(x["evidence_status"])
            x["evidence"]=f"the changed evidence now indicates {x['evidence_status']}"

        elif op=="authority_below_required":
            x["authority"]=authority_below_desired(x)

    return x

def drift_messages(base,severity):
    if severity==0:
        x=copy.deepcopy(base)
        impl="A"
        ops=[]

    elif severity==1:
        x=copy.deepcopy(base)
        impl="C1"
        ops=["benign_surface_evolution"]

    elif severity==2:
        choices=[
            ["object_substitution"],
            ["evidence_flip"],
            ["authority_below_required"],
        ]
        j=int(hashlib.sha256(base["base_id"].encode()).hexdigest()[:4],16)%3
        ops=choices[j]
        x=drifted_base(base,ops)
        impl="A"

    elif severity==3:
        choices=[
            ["object_substitution","evidence_flip"],
            ["evidence_flip","authority_below_required"],
            ["object_substitution","authority_below_required"],
        ]
        j=int(hashlib.sha256((base["base_id"]+"d3").encode()).hexdigest()[:4],16)%3
        ops=choices[j]
        x=drifted_base(base,ops)
        impl="A"

    else:
        raise ValueError(severity)

    return {
        agent:realize_message(x,agent,impl,0)
        for agent in AGENTS
    },ops

drift_rows=[]

for base in drift_base:
    gold_sem=global_semantics(base)
    gold_final=deterministic_policy(gold_sem)

    reference={
        agent:ensemble_predict(
            "SCALE",
            [realize_message(base,agent,"A",0)],
            BASE_GRAPH
        )[0]
        for agent in AGENTS
    }

    for severity in [0,1,2,3]:
        msgs,ops=drift_messages(base,severity)

        current={
            agent:ensemble_predict("SCALE",[msgs[agent]],BASE_GRAPH)[0]
            for agent in AGENTS
        }

        repaired={
            agent:{
                **current[agent],
                "contract":ontology_repair(current[agent]["contract"])
            }
            for agent in AGENTS
        }

        sem={
            "object":repaired["PLANNER"]["contract"].get("object"),
            "evidence_state":repaired["RETRIEVER"]["contract"].get("state"),
            "authority":repaired["POLICY"]["contract"].get("authority"),
        }
        pred_final=deterministic_policy(sem)

        drift_rows.append({
            "base_id":base["base_id"],
            "severity":severity,
            "harmful_drift":int(severity>=2),
            "operators":"|".join(ops),
            "prepolicy_active_scd":prepolicy_active_scd(reference,repaired),
            "policy_augmented_active_scd":policy_augmented_active_scd(reference,repaired),
            "full_scd":full_scd(reference,repaired),
            "confidence":float(np.mean([
                current[a]["confidence"] for a in AGENTS
            ])),
            "semantic_success":int(sem==gold_sem),
            "failure":int(pred_final!=gold_final),
        })

drift_df=pd.DataFrame(drift_rows)

drift_summary=(
    drift_df.groupby("severity",as_index=False)
    .agg(
        n=("base_id","size"),
        harmful_rate=("harmful_drift","mean"),
        failure_rate=("failure","mean"),
        semantic_success=("semantic_success","mean"),
        mean_prepolicy_active_scd=("prepolicy_active_scd","mean"),
        mean_policy_augmented_active_scd=("policy_augmented_active_scd","mean"),
        mean_full_scd=("full_scd","mean"),
        mean_confidence=("confidence","mean"),
    )
)
display(drift_summary.round(4))
drift_df.to_csv(ROOT/"semantic_drift_results.csv",index=False)
drift_summary.to_csv(ROOT/"semantic_drift_summary.csv",index=False)

In [ ]:
# 21/30 — Harmful-drift detection and leakage-free system-health observability

def grouped_auc(df,target,features):
    xv=df[list(features)].astype(float).values
    y=df[target].astype(int).values
    groups=df["base_id"].values

    if len(np.unique(y))<2 or len(np.unique(groups))<3:
        return np.nan

    n_splits=min(5,len(np.unique(groups)))
    gkf=GroupKFold(n_splits=n_splits)
    pred=np.full(len(df),np.nan)

    for tr,te in gkf.split(xv,y,groups):
        if len(np.unique(y[tr]))<2:
            continue
        clf=LogisticRegression(max_iter=1000)
        clf.fit(xv[tr],y[tr])
        pred[te]=clf.predict_proba(xv[te])[:,1]

    valid=~np.isnan(pred)
    if valid.sum()<5 or len(np.unique(y[valid]))<2:
        return np.nan

    return roc_auc_score(y[valid],pred[valid])

obs_df=drift_df.assign(
    neg_confidence=1-drift_df["confidence"]
)

severity_corr=spearmanr(
    drift_df["severity"],
    drift_df["prepolicy_active_scd"]
).statistic

observability=pd.DataFrame([{
    "n":len(drift_df),
    "failures":int(drift_df.failure.sum()),
    "harmful_cases":int(drift_df.harmful_drift.sum()),

    "spearman_severity_prepolicy_active_scd":severity_corr,

    "auc_prepolicy_active_scd_harmful_drift":grouped_auc(
        obs_df,"harmful_drift",["prepolicy_active_scd"]
    ),
    "auc_full_scd_harmful_drift":grouped_auc(
        obs_df,"harmful_drift",["full_scd"]
    ),

    "auc_neg_confidence_failure":grouped_auc(
        obs_df,"failure",["neg_confidence"]
    ),
    "auc_prepolicy_active_scd_failure":grouped_auc(
        obs_df,"failure",["prepolicy_active_scd"]
    ),
    "auc_confidence_plus_prepolicy_scd_failure":grouped_auc(
        obs_df,"failure",["neg_confidence","prepolicy_active_scd"]
    ),

    # Secondary only; shown explicitly so it cannot be confused with the primary.
    "auc_policy_augmented_scd_failure_secondary":grouped_auc(
        obs_df,"failure",["policy_augmented_active_scd"]
    ),
}])

display(observability.round(4))
observability.to_csv(ROOT/"observability_summary.csv",index=False)

In [ ]:
# 22/30 — Stable sequential causal-model runner


def gpu_mem():
    if not torch.cuda.is_available():
        return "CPU runtime / CUDA unavailable"
    try:
        free_b, total_b = torch.cuda.mem_get_info()
        return f"{free_b/1024**3:.1f}/{total_b/1024**3:.1f} GB free"
    except Exception:
        return "CUDA available"


def load_model_safe(spec):
    """
    Sequential causal-model loader.

    Scientific behavior:
    - GPU run: try primary then declared fallback.
    - CPU run: do not attempt thousands of causal generations; raise a short,
      explicit runtime-unavailable error that caller stages catch and record
      as SKIPPED/INCONCLUSIVE.
    - Model-repository/load errors are reported with both attempted IDs.
    """
    cleanup_gpu()

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA_UNAVAILABLE: causal LLM evaluation requires a GPU runtime. "
            "Core SCALE tests remain valid and continue."
        )

    errors = []

    for model_id, primary_loaded in [
        (spec["primary"], True),
        (spec["fallback"], False),
    ]:
        if not model_id:
            continue

        try:
            trust = bool(spec.get("trust_remote_code", False))
            print("Loading:", model_id, "|", gpu_mem())

            tok = AutoTokenizer.from_pretrained(
                model_id,
                use_fast=True,
                trust_remote_code=trust,
            )

            kwargs = dict(
                torch_dtype=CAUSAL_DTYPE,
                low_cpu_mem_usage=True,
                device_map={"": 0},
                trust_remote_code=trust,
            )

            try:
                mdl = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    attn_implementation="sdpa",
                    **kwargs,
                )
            except Exception as first_exc:
                cleanup_gpu()
                mdl = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    **kwargs,
                )

            mdl.eval()
            if tok.pad_token_id is None:
                tok.pad_token_id = tok.eos_token_id

            print("Ready:", model_id, "|", gpu_mem())
            return model_id, primary_loaded, tok, mdl

        except Exception as exc:
            errors.append(
                f"{model_id}:{type(exc).__name__}:{str(exc)[:180]}"
            )
            print(
                "Model load warning:",
                model_id,
                "|",
                type(exc).__name__,
                str(exc)[:180],
            )
            cleanup_gpu()

    raise RuntimeError(
        "MODEL_LOAD_UNAVAILABLE | " + " | ".join(errors)
    )


def chat_text(tok,messages,spec=None):
    kwargs=dict(
        tokenize=False,
        add_generation_prompt=True,
    )

    if spec and spec.get("disable_thinking"):
        kwargs["enable_thinking"]=False

    try:
        return tok.apply_chat_template(messages,**kwargs)
    except TypeError:
        kwargs.pop("enable_thinking",None)
        return tok.apply_chat_template(messages,**kwargs)
    except Exception:
        return "\n\n".join(
            f'{m["role"].upper()}: {m["content"]}'
            for m in messages
        )+"\nASSISTANT:"

def generate_text(tok,mdl,messages,spec,max_input,max_new):
    prompt=chat_text(tok,messages,spec)
    enc=tok(prompt,return_tensors="pt",truncation=False)
    n_in=int(enc["input_ids"].shape[-1])

    if n_in>max_input:
        return {
            "text":"","input_tokens":n_in,"output_tokens":0,
            "latency_s":0.0,"overflow":True,
        }

    try:
        model_device = next(mdl.parameters()).device
    except Exception:
        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    enc = {k: v.to(model_device) for k, v in enc.items()}

    t0=time.time()
    with torch.inference_mode():
        out=mdl.generate(
            **enc,
            max_new_tokens=max_new,
            do_sample=False,
            use_cache=True,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    dt=time.time()-t0

    new_ids=out[0,n_in:]
    return {
        "text":tok.decode(new_ids,skip_special_tokens=True).strip(),
        "input_tokens":n_in,
        "output_tokens":int(new_ids.numel()),
        "latency_s":dt,
        "overflow":False,
    }

def extract_json(text):
    s=re.sub(r"```(?:json)?","",text.strip(),flags=re.I).replace("```","")

    try:
        obj=json.loads(s)
        if isinstance(obj,dict):
            return obj
    except Exception:
        pass

    starts=[i for i,ch in enumerate(s) if ch=="{"]
    for start in starts:
        depth=0
        for end in range(start,len(s)):
            if s[end]=="{":
                depth+=1
            elif s[end]=="}":
                depth-=1
                if depth==0:
                    try:
                        obj=json.loads(s[start:end+1])
                        if isinstance(obj,dict):
                            return obj
                    except Exception:
                        break
    return None

In [ ]:
# 23/30 — Common semantic receiver, preflight, and ontology-shift reader case

READER_SYSTEM="""You are the semantic receiver in a multi-agent workflow.

Recover exactly three global semantic fields:
1. object — from PLANNER
2. evidence_state — from RETRIEVER
3. authority — from POLICY

CRITICAL INTERFACE RULE:
If an agent block explicitly contains a structured semantic payload such as
{"role":"PLANNER","active":{"object":"CHANGE"}}
then COPY the value in `active` exactly. Do not reinterpret, rename, paraphrase,
or replace an explicitly supplied active semantic value.

Only infer semantics when the interface contains raw natural language rather
than an explicit active semantic field.

Return exactly one JSON object:
{"object":"<OBJECT>","evidence_state":"<STATE>","authority":"<AUTHORITY>"}

Allowed object:
PURCHASE, ACCESS, CHANGE, INCIDENT, COMPLIANCE_ITEM

Allowed evidence_state:
EVIDENCE_PASS, EVIDENCE_FAIL, EVIDENCE_UNCERTAIN

Allowed authority:
NONE, PROPOSE, RECOMMEND, APPROVE, EXECUTE

Examples of OUTPUT FORMAT only:
{"object":"CHANGE","evidence_state":"EVIDENCE_PASS","authority":"EXECUTE"}
{"object":"ACCESS","evidence_state":"EVIDENCE_UNCERTAIN","authority":"APPROVE"}

No explanation. No markdown. No additional keys."""

def parse_semantics(text):
    obj=extract_json(text)

    if isinstance(obj,dict):
        return {
            "object":str(obj.get("object","")).strip().upper(),
            "evidence_state":str(obj.get("evidence_state","")).strip().upper(),
            "authority":str(obj.get("authority","")).strip().upper(),
        },True

    up=text.upper()
    o=next((x for x in PASS_ACTION_BY_OBJECT if re.search(rf"\b{re.escape(x)}\b",up)),"")
    s=next((x for x in ["EVIDENCE_PASS","EVIDENCE_FAIL","EVIDENCE_UNCERTAIN"] if x in up),"")
    a=next((x for x in AUTHORITIES if re.search(rf"\b{x}\b",up)),"")

    return {"object":o,"evidence_state":s,"authority":a},False


def semantics_parse_valid(sem):
    return (
        sem.get("object") in PASS_ACTION_BY_OBJECT
        and sem.get("evidence_state") in {
            "EVIDENCE_PASS","EVIDENCE_FAIL","EVIDENCE_UNCERTAIN"
        }
        and sem.get("authority") in AUTHORITIES
    )

def reader_repair_messages(original_output):
    return [
        {"role":"system","content":READER_SYSTEM},
        {"role":"user","content":
            "The previous answer was malformed or outside the allowed labels. "
            "Repair FORMAT ONLY. Preserve the semantic values already present "
            "when they are valid. Return exactly one allowed JSON object.\\n\\n"
            f"PREVIOUS OUTPUT:\\n{original_output}"
        }
    ]

def reader_messages(base,comp,method,graph_ctx=BASE_GRAPH):
    return [
        {"role":"system","content":READER_SYSTEM},
        {"role":"user","content":
            f"INTERFACE={method}\nCOMPOSITION={comp['composition_id']}\n\n"
            f"{build_interface(base,comp,method,graph_ctx)}"
        }
    ]


def ontology_structured_planner_payload(base,method,tier):
    graph_ctx=TRANSPARENT_GRAPH if tier=="transparent" else OPAQUE_GRAPH
    msg=ontology_shift_planner_message(base,tier)

    if method in {"Symbolic","CB-BGE","CB+DualView","CB+Logic"}:
        pred_node,canonical=ontology_object_prediction(method,msg,graph_ctx)
        return json.dumps({
            "role":"PLANNER",
            "active":{"object":canonical},
            "confidence":None,
        },ensure_ascii=False,separators=(",",":"))

    if method in {"Proto-CB","SCALE"}:
        key="Proto-CB" if method=="Proto-CB" else "SCALE-Full"
        pred_node,canonical=ontology_object_prediction(key,msg,graph_ctx)
        return json.dumps({
            "role":"PLANNER",
            "active":{"object":canonical},
            "ontology_node":pred_node,
            "confidence":None,
        },ensure_ascii=False,separators=(",",":"))

    raise ValueError(method)

def ontology_reader_messages(base,method,tier):
    packets=[]

    # Planner uses the unseen ontology leaf.
    raw=ontology_shift_planner_message(base,tier)

    if method=="NL":
        planner_payload=raw
    elif method=="JSON":
        planner_payload=json.dumps(
            {"agent":"PLANNER","implementation":f"ONT_{tier}","message":raw},
            ensure_ascii=False,separators=(",",":")
        )
    elif method=="Onto-RAG":
        planner_payload=json.dumps(
            {"message":raw,"ontology_hits":onto_rag_retrieve(raw,"PLANNER")},
            ensure_ascii=False,separators=(",",":")
        )
    else:
        planner_payload=ontology_structured_planner_payload(base,method,tier)

    packets.append(f"[PLANNER]\n{planner_payload}")

    # Retriever and Policy remain known implementation A.
    for agent in ["RETRIEVER","POLICY"]:
        packets.append(
            f"[{agent}]\n{interface_message(base,agent,'A',method,BASE_GRAPH)}"
        )

    return [
        {"role":"system","content":READER_SYSTEM},
        {"role":"user","content":
            f"INTERFACE={method}\n"
            f"COMPOSITION=ONTOLOGY_{tier.upper()}\n\n"
            +"\n\n".join(packets)
        }
    ]

# Preflight
assert len(MAIN_METHODS)==len(set(MAIN_METHODS)), "Duplicate MAIN_METHODS entry."
assert set(STRUCTURED_METHODS).issubset(set(MAIN_METHODS)), (
    "STRUCTURED_METHODS must be a subset of MAIN_METHODS."
)
assert {"NL","JSON","Onto-RAG"}.issubset(set(MAIN_METHODS))
assert {"Symbolic","CB-BGE","CB+DualView","CB+Logic","Proto-CB","SCALE"}.issubset(
    set(MAIN_METHODS)
)

sample=core_test_base[0]

# Core interface dispatch.
for method in MAIN_METHODS:
    txt=build_interface(sample,CORE_COMPOSITIONS[0],method,BASE_GRAPH)
    assert isinstance(txt,str) and len(txt)>0, f"Core interface failed: {method}"

# Ontology-reader dispatch.
# This catches missing method registrations (e.g. CB+DualView) BEFORE any
# expensive reader model is loaded.
for tier in ["transparent","opaque"]:
    for method in MAIN_METHODS:
        msgs=ontology_reader_messages(sample,method,tier)
        assert isinstance(msgs,list) and len(msgs)==2, (
            f"Ontology reader dispatch failed: tier={tier}, method={method}"
        )
        assert all(
            isinstance(m,dict) and "role" in m and "content" in m
            for m in msgs
        ), f"Malformed ontology messages: tier={tier}, method={method}"

probe='{"object":"CHANGE","evidence_state":"EVIDENCE_PASS","authority":"EXECUTE"}'
sem,ok=parse_semantics(probe)
assert ok and sem["object"]=="CHANGE"

print(
    "PRE-FLIGHT PASS | "
    f"{len(MAIN_METHODS)} methods | core + transparent ontology + opaque ontology"
)


# Runtime Availability Check Before Generative Tests

The controlled SCALE experiments do **not** require a causal LLM GPU.

The following sections do:

- arbitrary LLM readers,
- real upstream LLM producers,
- CRM offline generative executors,
- strong 7–8B readers/producers/executors.

If CUDA is unavailable, those sections are recorded as **SKIPPED / INCONCLUSIVE**
instead of throwing an exception. For the complete paper-strength run, use an
A100 runtime and rerun the notebook from the beginning.


In [ ]:
GEN_RUNTIME_STATUS = pd.DataFrame([
    {
        "component": "Core SCALE / learned heads / ontology / Active-SCD",
        "requires_cuda": False,
        "enabled": True,
        "runtime_status": "READY",
    },
    {
        "component": "Small causal LLM readers",
        "requires_cuda": True,
        "enabled": bool(RUN_LLM_READERS),
        "runtime_status": "READY" if RUN_LLM_READERS else "SKIPPED_NO_CUDA",
    },
    {
        "component": "Real causal LLM producers",
        "requires_cuda": True,
        "enabled": bool(RUN_REAL_PRODUCER),
        "runtime_status": "READY" if RUN_REAL_PRODUCER else "SKIPPED_NO_CUDA",
    },
    {
        "component": "Strong 7–8B external LLM suite",
        "requires_cuda": True,
        "enabled": bool(FULL_STRONG_LLM_CAPABLE),
        "runtime_status": (
            "READY"
            if FULL_STRONG_LLM_CAPABLE
            else "SKIPPED_INSUFFICIENT_GPU"
        ),
    },
])
display(GEN_RUNTIME_STATUS)


In [ ]:
# 24/30 — Multi-reader experiment: core + C1/C2 + ontology extension

READER_RESULTS=ROOT/"reader_results.jsonl"

if DRIVE_ROOT is not None and not READER_RESULTS.exists():
    remote=DRIVE_ROOT/READER_RESULTS.name
    if remote.exists():
        shutil.copy2(remote,READER_RESULTS)

# Salvage the completed reader generations from the legacy serialization bug.
repair_jsonl_file(READER_RESULTS)

def done_keys():
    rows,_=safe_jsonl_records(READER_RESULTS)
    return {x["run_key"] for x in rows if x.get("run_key")}

def sync_reader():
    if DRIVE_ROOT is not None and READER_RESULTS.exists():
        try:
            shutil.copy2(READER_RESULTS,DRIVE_ROOT/READER_RESULTS.name)
        except Exception as exc:
            print("Sync warning:",type(exc).__name__)

def record_reader_case(
    spec,active_model,primary,tok,mdl,
    base,split_name,method,composition_id,replacement_level,messages
):
    key=(
        f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|{spec['label']}|"
        f"{split_name}|{method}|{composition_id}|{base['base_id']}"
    )

    error=None
    try:
        gen=generate_text(
            tok,mdl,messages,spec,
            MAX_READER_INPUT,MAX_READER_OUTPUT
        )
    except torch.cuda.OutOfMemoryError:
        cleanup_gpu()
        gen={"text":"","input_tokens":0,"output_tokens":0,"latency_s":0.0,"overflow":False}
        error="OOM"
    except Exception as exc:
        gen={"text":"","input_tokens":0,"output_tokens":0,"latency_s":0.0,"overflow":False}
        error=f"{type(exc).__name__}:{str(exc)[:120]}"

    pred_sem,parse_ok=parse_semantics(gen["text"])
    repair_attempted=False
    repair_success=False

    # One uniform retry for syntactically malformed output only.
    # A well-formed but semantically invalid prediction is NOT retried.
    if (
        error is None
        and not gen["overflow"]
        and not parse_ok
    ):
        repair_attempted=True
        try:
            repaired_gen=generate_text(
                tok,mdl,
                reader_repair_messages(gen["text"]),
                spec,
                MAX_READER_INPUT,
                MAX_READER_OUTPUT
            )
            repaired_sem,repaired_ok=parse_semantics(repaired_gen["text"])

            if repaired_ok and semantics_parse_valid(repaired_sem):
                pred_sem=repaired_sem
                parse_ok=True
                repair_success=True
                gen["text"]=repaired_gen["text"]
                gen["input_tokens"]+=repaired_gen["input_tokens"]
                gen["output_tokens"]+=repaired_gen["output_tokens"]
                gen["latency_s"]+=repaired_gen["latency_s"]
        except Exception:
            pass

    gold_sem=global_semantics(base)
    sem_ok=pred_sem==gold_sem

    final=deterministic_policy(pred_sem)
    task_ok=(
        final is not None
        and final["object"]==base["object"]
        and final["action"]==base["final_action"]
        and final["authority"]==base["final_authority"]
    )

    return key,{
        "run_key":key,
        "experiment_version":EXPERIMENT_VERSION,
        "config_hash":CONFIG_HASH,
        "reader_model":spec["label"],
        "active_model_id":active_model,
        "primary_model_loaded":bool(primary),
        "split":split_name,
        "method":method,
        "composition_id":composition_id,
        "replacement_level":replacement_level,
        "base_id":base["base_id"],
        "parse_success":int(parse_ok),
        "semantic_label_validity":int(
            parse_ok and semantics_parse_valid(pred_sem)
        ),
        "repair_attempted":int(repair_attempted),
        "repair_success":int(repair_success),
        "object_correct":int(pred_sem.get("object")==gold_sem["object"]),
        "evidence_correct":int(pred_sem.get("evidence_state")==gold_sem["evidence_state"]),
        "authority_correct":int(pred_sem.get("authority")==gold_sem["authority"]),
        "semantic_success":int(sem_ok),
        "task_success":int(task_ok),
        "raw_output":gen["text"],
        "input_tokens":gen["input_tokens"],
        "output_tokens":gen["output_tokens"],
        "latency_s":round(float(gen["latency_s"]),4),
        "overflow":int(gen["overflow"]),
        "error":error,
    }

if RUN_LLM_READERS:
    done=done_keys()

    for spec in READER_MODELS:
        try:
            active_model, primary, tok, mdl = load_model_safe(spec)
        except Exception as exc:
            print(
                "\nREADER SKIPPED:",
                spec["label"],
                "|",
                type(exc).__name__,
                str(exc)[:240],
            )
            continue

        print("\nREADER:", spec["label"], "|", active_model)

        # Core A/B.
        for method in MAIN_METHODS:
            for base in core_test_base:
                for comp in CORE_COMPOSITIONS:
                    key=(
                        f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|{spec['label']}|"
                        f"core|{method}|{comp['composition_id']}|{base['base_id']}"
                    )
                    if key in done:
                        continue

                    key,rec=record_reader_case(
                        spec,active_model,primary,tok,mdl,
                        base,"core",method,
                        comp["composition_id"],comp["replacement_level"],
                        reader_messages(base,comp,method,BASE_GRAPH)
                    )
                    append_jsonl(READER_RESULTS,rec)
                    done.add(key)

            print(method,"core complete")

        # C1/C2.
        if RUN_C1_C2:
            for split_name,comps in [
                ("c1_natural",C1_COMPOSITIONS),
                ("c2_schema",C2_COMPOSITIONS),
            ]:
                for method in MAIN_METHODS:
                    for base in shift_test_base:
                        for comp in comps:
                            key=(
                                f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|{spec['label']}|"
                                f"{split_name}|{method}|{comp['composition_id']}|{base['base_id']}"
                            )
                            if key in done:
                                continue

                            key,rec=record_reader_case(
                                spec,active_model,primary,tok,mdl,
                                base,split_name,method,
                                comp["composition_id"],comp["replacement_level"],
                                reader_messages(base,comp,method,BASE_GRAPH)
                            )
                            append_jsonl(READER_RESULTS,rec)
                            done.add(key)

        # Genuine ontology extension: transparent and opaque unseen leaves.
        if RUN_ONTOLOGY_SHIFT:
            for tier in ["transparent","opaque"]:
                split_name=f"ontology_{tier}"

                for method in MAIN_METHODS:
                    for base in ontology_test_base:
                        key=(
                            f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|{spec['label']}|"
                            f"{split_name}|{method}|ONT-AA|{base['base_id']}"
                        )
                        if key in done:
                            continue

                        key,rec=record_reader_case(
                            spec,active_model,primary,tok,mdl,
                            base,split_name,method,
                            "ONT-AA",1,
                            ontology_reader_messages(base,method,tier)
                        )
                        append_jsonl(READER_RESULTS,rec)
                        done.add(key)

        sync_reader()
        del mdl,tok
        cleanup_gpu()
else:
    print(
        "LLM readers SKIPPED: CUDA/GPU evaluation is unavailable. "
        "This is a runtime availability result, not a semantic model failure."
    )

In [ ]:
# 25/30 — Reader summaries, retention and paired statistics

reader_raw=safe_read_jsonl_df(READER_RESULTS)

# Backfill fields defensively if a same-version interrupted checkpoint was
# created before a diagnostic field was added.
if len(reader_raw):
    if "primary_model_loaded" not in reader_raw.columns:
        reader_raw["primary_model_loaded"]=False
    if "parse_success" not in reader_raw.columns:
        reader_raw["parse_success"]=0
    if "semantic_label_validity" not in reader_raw.columns:
        reader_raw["semantic_label_validity"]=np.nan
    for _col in [
        "repair_attempted","repair_success",
        "object_correct","evidence_correct","authority_correct",
        "semantic_success","task_success","overflow"
    ]:
        if _col not in reader_raw.columns:
            reader_raw[_col]=0
    if "error" not in reader_raw.columns:
        reader_raw["error"]=None
    if "input_tokens" not in reader_raw.columns:
        reader_raw["input_tokens"]=0
    if "latency_s" not in reader_raw.columns:
        reader_raw["latency_s"]=0.0

reader_df=(
    reader_raw[reader_raw.primary_model_loaded==True].copy()
    if len(reader_raw)
    else pd.DataFrame()
)

if len(reader_df):
    technical=(
        reader_df.groupby(["reader_model","split","method"],as_index=False)
        .agg(
            n=("task_success","size"),
            semantic_success=("semantic_success","mean"),
            task_success=("task_success","mean"),
            parse_success=("parse_success","mean"),
            semantic_label_validity=(
                "semantic_label_validity","mean"
            ),
            repair_attempt_rate=("repair_attempted","mean"),
            repair_success_rate=("repair_success","mean"),
            object_accuracy=("object_correct","mean"),
            evidence_accuracy=("evidence_correct","mean"),
            authority_accuracy=("authority_correct","mean"),
            overflow_rate=("overflow","mean"),
            error_rate=("error",lambda x:x.notna().mean()),
            mean_input_tokens=("input_tokens","mean"),
            mean_latency_s=("latency_s","mean"),
        )
    )

    core_comp=(
        reader_df[reader_df.split=="core"]
        .groupby(
            ["reader_model","method","composition_id","replacement_level"],
            as_index=False
        )
        .agg(task_success=("task_success","mean"))
    )

    core_summary=(
        core_comp.groupby(
            ["reader_model","method","replacement_level"],
            as_index=False
        )
        .agg(
            task_success=("task_success","mean"),
            task_success_sd=("task_success","std"),
            n_compositions=("composition_id","size"),
        )
    )

    iid=(
        core_summary[core_summary.replacement_level==0][
            ["reader_model","method","task_success"]
        ]
        .rename(columns={"task_success":"iid_success"})
    )
    core_summary=core_summary.merge(
        iid,on=["reader_model","method"],how="left"
    )
    core_summary["compositional_retention"]=core_summary.apply(
        lambda r:r.task_success/r.iid_success
        if pd.notna(r.iid_success) and r.iid_success>0
        else np.nan,
        axis=1
    )

    heldout_summary=(
        reader_df[reader_df.split.isin([
            "c1_natural","c2_schema",
            "ontology_transparent","ontology_opaque"
        ])]
        .groupby(["reader_model","split","method"],as_index=False)
        .agg(
            semantic_success=("semantic_success","mean"),
            task_success=("task_success","mean"),
            n=("task_success","size"),
        )
    )

    def paired_bootstrap(split_name,model_label,rival,B=2500):
        a=reader_df[
            (reader_df.reader_model==model_label)&
            (reader_df.split==split_name)&
            (reader_df.method=="SCALE")
        ]
        b=reader_df[
            (reader_df.reader_model==model_label)&
            (reader_df.split==split_name)&
            (reader_df.method==rival)
        ]

        keys=["reader_model","split","base_id"]
        if split_name=="core":
            keys+=["composition_id","replacement_level"]

        m=a.merge(b,on=keys,suffixes=("_scale","_rival"))
        if not len(m):
            return None

        rng=np.random.default_rng(SEED)
        vals=[]
        for _ in range(B):
            ids=rng.integers(0,len(m),len(m))
            s=m.iloc[ids]
            vals.append(float(
                (s.task_success_scale-s.task_success_rival).mean()
            ))

        return {
            "split":split_name,
            "reader_model":model_label,
            "rival":rival,
            "n":len(m),
            "delta_success":float(
                (m.task_success_scale-m.task_success_rival).mean()
            ),
            "ci_low":float(np.quantile(vals,.025)),
            "ci_high":float(np.quantile(vals,.975)),
        }

    boots=[]
    for ml in sorted(reader_df.reader_model.unique()):
        for split_name in [
            "core","c1_natural","c2_schema",
            "ontology_transparent","ontology_opaque"
        ]:
            for rival in [m for m in MAIN_METHODS if m!="SCALE"]:
                x=paired_bootstrap(split_name,ml,rival)
                if x:
                    boots.append(x)

    bootstrap_df=pd.DataFrame(boots)

    display(technical.round(4))
    display(core_summary.round(4))
    display(heldout_summary.round(4))
    display(bootstrap_df.round(4))

    technical.to_csv(ROOT/"reader_technical_diagnostics.csv",index=False)
    core_summary.to_csv(ROOT/"reader_core_robustness.csv",index=False)
    heldout_summary.to_csv(ROOT/"reader_heldout_summary.csv",index=False)
    bootstrap_df.to_csv(ROOT/"reader_paired_bootstrap.csv",index=False)
else:
    technical=core_summary=heldout_summary=bootstrap_df=pd.DataFrame()
    print("No reader results.")

In [ ]:
# 28/30 — Real producer C3 with NLI fidelity validation

NLI_TOKENIZER=None
NLI_MODEL=None
NLI_ENTAIL_ID=None

def load_nli():
    global NLI_TOKENIZER,NLI_MODEL,NLI_ENTAIL_ID
    if NLI_MODEL is not None:
        return

    print("Loading NLI validator:",NLI_VALIDATOR_ID)
    NLI_TOKENIZER=AutoTokenizer.from_pretrained(NLI_VALIDATOR_ID,use_fast=True)
    NLI_MODEL=AutoModelForSequenceClassification.from_pretrained(
        NLI_VALIDATOR_ID
    ).to("cpu").eval()

    id2label={
        int(k):str(v).lower()
        for k,v in NLI_MODEL.config.id2label.items()
    }
    matches=[i for i,l in id2label.items() if "entail" in l]
    if not matches:
        raise RuntimeError(f"Entailment label not found: {id2label}")
    NLI_ENTAIL_ID=matches[0]
    print("NLI labels:",id2label)

def active_hypothesis(base,agent):
    if agent=="PLANNER":
        return f"The workflow object is {base['object']}."
    if agent=="RETRIEVER":
        return f"The evidence state is {EVIDENCE_TO_STATE[base['evidence_status']]}."
    return f"The authority level is {base['authority']}."

def entailment_prob(premise,hypothesis):
    load_nli()
    inp=NLI_TOKENIZER(
        premise,hypothesis,
        return_tensors="pt",
        truncation=True,max_length=256
    )
    with torch.inference_mode():
        logits=NLI_MODEL(**inp).logits[0]
    return float(F.softmax(logits,dim=-1)[NLI_ENTAIL_ID])

PRODUCER_SYSTEM="""You are one upstream agent.
Communicate only the supplied operational fact in one short natural-language sentence.
Preserve its meaning exactly.
Do not output JSON and do not add other workflow facts."""

def producer_prompt(base,agent):
    return [
        {"role":"system","content":PRODUCER_SYSTEM},
        {"role":"user","content":
            f"Operational fact: {active_hypothesis(base,agent)}"
        }
    ]


PRODUCER_RESULTS=ROOT/"real_producer_results.jsonl"

PRODUCER_REMOTE=None
if DRIVE_ROOT is not None:
    PRODUCER_REMOTE=DRIVE_ROOT/PRODUCER_RESULTS.name
    if not PRODUCER_RESULTS.exists() and PRODUCER_REMOTE.exists():
        shutil.copy2(PRODUCER_REMOTE,PRODUCER_RESULTS)

repair_jsonl_file(PRODUCER_RESULTS)

# Cheap representation preflight before loading causal producer models.
_prod_probe=core_test_base[0]
_prod_probe_msg=realize_message(_prod_probe,"PLANNER","C1",0)
for _method in ["Symbolic","CB-BGE","CB+DualView","CB+Logic","Proto-CB","SCALE"]:
    _pred,_,_=prediction_for(
        _method,_prod_probe_msg,"PLANNER",BASE_GRAPH,repair=True
    )
    assert isinstance(_pred,dict), f"Producer representation dispatch failed: {_method}"
print("Producer representation preflight PASS")

if RUN_REAL_PRODUCER:
    try:
        load_nli()
    except Exception as exc:
        print(
            "REAL PRODUCER SKIPPED: NLI validator unavailable |",
            type(exc).__name__,
            str(exc)[:240],
        )
        RUN_REAL_PRODUCER = False

if RUN_REAL_PRODUCER:
    producer_tasks=core_test_base[:min(12,len(core_test_base))]
    done=set()

    for _row in safe_jsonl_records(PRODUCER_RESULTS)[0]:
        if _row.get("run_key"):
            done.add(_row["run_key"])

    for spec in PRODUCER_MODELS:
        try:
            active_model, primary, tok, mdl = load_model_safe(spec)
        except Exception as exc:
            print(
                "PRODUCER MODEL SKIPPED:",
                spec["label"],
                "|",
                type(exc).__name__,
                str(exc)[:240],
            )
            continue

        for base in producer_tasks:
            for agent in AGENTS:
                key=(
                    f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|"
                    f"{spec['label']}|{base['base_id']}|{agent}"
                )
                if key in done:
                    continue

                error=None
                try:
                    gen=generate_text(
                        tok,mdl,producer_prompt(base,agent),spec,
                        MAX_PRODUCER_INPUT,MAX_PRODUCER_OUTPUT
                    )
                except Exception as exc:
                    gen={"text":"","input_tokens":0,"output_tokens":0,"latency_s":0.0,"overflow":False}
                    error=f"{type(exc).__name__}:{str(exc)[:100]}"

                ent=(
                    entailment_prob(gen["text"],active_hypothesis(base,agent))
                    if gen["text"] else 0.0
                )
                accepted=(
                    error is None
                    and not gen["overflow"]
                    and ent>=NLI_ENTAIL_THRESHOLD
                )

                rec={
                    "run_key":key,
                    "producer_model":spec["label"],
                    "active_model_id":active_model,
                    "primary_model_loaded":bool(primary),
                    "base_id":base["base_id"],
                    "agent":agent,
                    "message":gen["text"],
                    "entailment_prob":ent,
                    "accepted":int(accepted),
                    "error":error,
                }
                append_jsonl(PRODUCER_RESULTS,rec)
                if PRODUCER_REMOTE is not None:
                    sync_result_file(PRODUCER_RESULTS,PRODUCER_REMOTE)
                done.add(key)

        del mdl,tok
        cleanup_gpu()

    producer_df=safe_read_jsonl_df(PRODUCER_RESULTS)
    if len(producer_df) and "primary_model_loaded" in producer_df.columns:
        producer_df=producer_df[
            producer_df.primary_model_loaded==True
        ].copy()
    else:
        producer_df=pd.DataFrame(columns=[
            "producer_model","active_model_id","primary_model_loaded",
            "base_id","agent","message","entailment_prob","accepted","error"
        ])

    acceptance=(
        producer_df.groupby(["producer_model","agent"],as_index=False)
        .agg(
            n=("base_id","size"),
            acceptance_rate=("accepted","mean"),
            mean_entailment=("entailment_prob","mean"),
        )
    )

    c3_rows=[]
    for _,r in producer_df[producer_df.accepted==1].iterrows():
        base=next(b for b in core_test_base if b["base_id"]==r["base_id"])
        agent=r["agent"]
        gold=hop_contract(base,agent)
        msg=r["message"]

        for method in ["Symbolic","CB-BGE","CB+DualView","CB+Logic","Proto-CB","SCALE"]:
            pred,_,_=prediction_for(method,msg,agent,BASE_GRAPH,repair=True)

            c3_rows.append({
                "producer_model":r["producer_model"],
                "base_id":r["base_id"],
                "agent":agent,
                "method":method,
                "active_correct":active_correct(gold,pred,agent),
                "slot_accuracy":slot_accuracy(gold,pred),
                "exact_contract":exact_contract(gold,pred),
            })

    c3_df=pd.DataFrame(c3_rows)
    c3_summary=(
        c3_df.groupby(["producer_model","method","agent"],as_index=False)
        .agg(
            n=("base_id","size"),
            active_accuracy=("active_correct","mean"),
            slot_accuracy=("slot_accuracy","mean"),
            exact_contract=("exact_contract","mean"),
        )
        if len(c3_df) else pd.DataFrame()
    )

    display(acceptance.round(4))
    if len(c3_summary):
        display(c3_summary.round(4))

    acceptance.to_csv(ROOT/"real_producer_acceptance.csv",index=False)
    c3_df.to_csv(ROOT/"real_producer_representation_results.csv",index=False)
    c3_summary.to_csv(ROOT/"real_producer_representation_summary.csv",index=False)
else:
    producer_df = pd.DataFrame(columns=[
        "producer_model","active_model_id","primary_model_loaded",
        "base_id","agent","message","entailment_prob","accepted","error"
    ])
    acceptance = pd.DataFrame()
    c3_df = pd.DataFrame()
    c3_summary = pd.DataFrame()
    print(
        "Real-producer evaluation SKIPPED/INCONCLUSIVE: "
        "GPU causal generation is unavailable in this runtime."
    )

In [ ]:
# 29/31 — Final paper-aligned claim gates and diagnostics

gates=[]
diagnostics=[]

# G0 — transport/syntactic validity only.
g0=(
    len(technical)>0
    and technical.error_rate.max()<0.01
    and technical.overflow_rate.max()<0.01
    and technical.parse_success.min()>=0.95
)
gates.append(("G0 Technical transport/syntax validity",g0))

# Semantic label validity is reported separately; it is a method error, not a parser error.
diagnostics.append((
    "Reader semantic-label minimum validity",
    float(technical.semantic_label_validity.min())
    if len(technical) else np.nan
))

# G1 — SCALE's native machine-readable interface competence.
# This tests the interface itself, not whether a particular LLM chooses to copy it.
g1=False
if len(native_summary):
    scale_core=native_summary[
        (native_summary.split=="core")
        &(native_summary.method=="SCALE")
    ]
    if len(scale_core):
        g1=(
            scale_core.task_success.min()>=0.95
            and scale_core.semantic_success.min()>=0.95
        )
gates.append(("G1 Native SCALE interface competence",g1))

# Cross-reader SCALE stress remains an explicit diagnostic.
reader_scale_core=core_summary[
    core_summary.method=="SCALE"
] if len(core_summary) else pd.DataFrame()

if len(reader_scale_core):
    diagnostics.append((
        "Cross-reader SCALE core minimum task success",
        float(reader_scale_core.task_success.min())
    ))

# G2 — label-efficient directed implementation invariance.
g2=False
if len(curve_summary):
    low=curve_summary[
        (curve_summary.visibility<=0.10)
        &(curve_summary.split.isin([
            "heldout_b","c2_schema"
        ]))
    ]

    avg=(
        low.groupby("method")[
            "active_accuracy"
        ].mean()
        if len(low)
        else pd.Series(dtype=float)
    )

    full=curve_summary[
        (curve_summary.visibility==1.0)
        &(curve_summary.split=="heldout_b")
    ]

    full_avg=(
        full.groupby("method")[
            "active_accuracy"
        ].mean()
        if len(full)
        else pd.Series(dtype=float)
    )

    needed=[
        "SCALE","CB-BGE",
        "CB+DualView","SCALE-NoInv"
    ]

    if all(m in avg.index for m in needed):
        low_adv=(
            avg["SCALE"]
            >max(
                avg["CB-BGE"],
                avg["CB+DualView"],
                avg["SCALE-NoInv"]
            )+.05
        )

        noninferior=True
        if all(
            m in full_avg.index
            for m in ["SCALE","CB+DualView"]
        ):
            noninferior=(
                full_avg["SCALE"]
                >=full_avg["CB+DualView"]-.05
            )

        g2=low_adv and noninferior

gates.append(("G2 Label-efficient directed invariance",g2))

# G3 — open-world structural ontology generalization.
g3=False
graph_added_value=np.nan

if len(ontology_summary):
    opaque=ontology_summary[
        ontology_summary.tier=="opaque"
    ].set_index("method")

    needed=[
        "SCALE-Full",
        "Proto-CB",
        "Proto+Ancestor",
        "SCALE-NoOnt",
        "CB+DualView",
    ]

    if all(m in opaque.index for m in needed):
        scale=float(
            opaque.loc[
                "SCALE-Full",
                "canonical_parent_accuracy"
            ]
        )
        proto=float(
            opaque.loc[
                "Proto-CB",
                "canonical_parent_accuracy"
            ]
        )
        ancestor=float(
            opaque.loc[
                "Proto+Ancestor",
                "canonical_parent_accuracy"
            ]
        )
        noont=float(
            opaque.loc[
                "SCALE-NoOnt",
                "canonical_parent_accuracy"
            ]
        )

        g3=(
            scale>=.75
            and scale>proto+.15
            and scale>noont+.15
        )
        graph_added_value=scale-ancestor

gates.append(("G3 Structural ontology generalization",g3))

# G4 — competitive C1/C2 utility against the strongest learned closed-set control.
g4=False

if len(rep_summary):
    held=rep_summary[
        rep_summary.split.isin([
            "c1_natural","c2_schema"
        ])
    ]

    learned=(
        held.groupby("method")[
            "active_accuracy"
        ].mean()
        if len(held)
        else pd.Series(dtype=float)
    )

    needed=[
        "SCALE-Full",
        "CB-BGE",
        "CB+DualView",
        "Proto-CB",
    ]

    if all(m in learned.index for m in needed):
        strongest=max(
            learned["CB-BGE"],
            learned["CB+DualView"],
            learned["Proto-CB"]
        )
        g4=(
            learned["SCALE-Full"]
            >=strongest-ONTOLOGY_NONINFERIOR_MARGIN
        )

gates.append(("G4 Competitive C1/C2 utility",g4))

# G5 — deterministic runtime ontology enforcement.
g5=False

if len(stress_summary):
    s=stress_summary.set_index("corruption")

    needed=[
        "authority_below_required",
        "role_mismatch",
        "state_mismatch",
        "provenance_corruption",
    ]

    if all(x in s.index for x in needed):
        structural_ok=all(
            float(s.loc[x,"detection_rate"])>=.99
            and float(s.loc[x,"repair_rate"])>=.99
            for x in [
                "authority_below_required",
                "role_mismatch",
                "state_mismatch",
            ]
        )
        provenance_ok=(
            float(
                s.loc[
                    "provenance_corruption",
                    "detection_rate"
                ]
            )>=.99
            and float(
                s.loc[
                    "provenance_corruption",
                    "block_rate"
                ]
            )>=.99
        )
        g5=structural_ok and provenance_ok

gates.append(("G5 Runtime semantic enforcement",g5))

# Learned logic stays a negative/diagnostic result.
if len(logic_adversarial_summary):
    diagnostics.append((
        "Learned-logic diagnostic table available",
        True
    ))

# G6 — leakage-free PrePolicy Active-SCD.
g6=False

if len(observability):
    r=observability.iloc[0]
    g6=(
        pd.notna(
            r["auc_prepolicy_active_scd_harmful_drift"]
        )
        and r[
            "auc_prepolicy_active_scd_harmful_drift"
        ]>=.75
        and pd.notna(
            r["auc_prepolicy_active_scd_failure"]
        )
        and r[
            "auc_prepolicy_active_scd_failure"
        ]>=.75
        and pd.notna(
            r["auc_neg_confidence_failure"]
        )
        and r[
            "auc_prepolicy_active_scd_failure"
        ]>=r[
            "auc_neg_confidence_failure"
        ]+.10
        and r[
            "spearman_severity_prepolicy_active_scd"
        ]>=.40
    )

gates.append(("G6 Leakage-free Active-SCD observability",g6))

# G7 — real producer recovery.
g7=False

if len(c3_summary):
    scale_rows=c3_summary[
        c3_summary.method=="SCALE"
    ]
    symbolic_rows=c3_summary[
        c3_summary.method=="Symbolic"
    ]

    if len(scale_rows):
        scale_active=float(np.average(
            scale_rows["active_accuracy"],
            weights=scale_rows["n"]
        ))
        symbolic_active=(
            float(np.average(
                symbolic_rows["active_accuracy"],
                weights=symbolic_rows["n"]
            ))
            if len(symbolic_rows)
            else 1.
        )

        g7=(
            scale_active>=.85
            and scale_active>symbolic_active+.05
        )

gates.append(("G7 Real-producer semantic recovery",g7))

gate_df=pd.DataFrame(
    gates,
    columns=["gate","pass"]
)
diagnostic_df=pd.DataFrame(
    diagnostics,
    columns=["diagnostic","value"]
)

display(gate_df)
display(diagnostic_df)

overall=bool(gate_df["pass"].all())

status=(
    "CONTROLLED SCALE CLAIM SUPPORTED"
    if overall
    else "CONTROLLED CLAIM NOT YET FULLY SUPPORTED"
)

report=f"""
# SCALE Final Hybrid — Controlled Claim Gate

Experiment: `{EXPERIMENT_VERSION}`
Config hash: `{CONFIG_HASH}`

Overall: **{status}**

## Required gates
{chr(10).join(
    f"- {g}: {'PASS' if p else 'FAIL / INCONCLUSIVE'}"
    for g,p in gates
)}

## Methodological changes justified by previous results

- Closed-set semantics now use a **discriminative dual-view decoder**.
- Open-world ontology extension uses a separate **relation-aware graph prototype decoder**.
- Implementation invariance is **directed A→B** with A stop-gradient.
- `L_logic` is not part of Full SCALE; deterministic ontology enforcement remains.
- G0 measures syntactic/transport validity only.
- semantic-vocabulary validity is reported separately.
- G1 evaluates the machine-readable SCALE interface directly.
- PrePolicy Active-SCD remains the primary drift metric.

## Architecture diagnostic
Graph added value over Proto+Ancestor on opaque leaves:
`{graph_added_value if pd.notna(graph_added_value) else "NA"}`

## Interpretation rule
A PASS does not mean SCALE must outperform every fully supervised classifier
on every IID example. It means the complete controlled evidence supports the
specific paper claims: robust closed-set utility, low-label implementation
adaptation, structural ontology extension, runtime semantic enforcement,
and semantic-drift observability.
"""

(ROOT/"CLAIM_GATE.md").write_text(
    report.strip()+"\n",
    encoding="utf-8"
)

(ROOT/"run_manifest.json").write_text(
    json.dumps(
        CONFIG|{
            "config_hash":CONFIG_HASH,
            "final_architecture":
            "hybrid_closed_discriminative_open_graph",
            "learned_logic_in_full_scale":False,
            "invariance":
            "directed_A_to_B_stop_gradient",
        },
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

from IPython.display import Markdown,display
display(Markdown(report))

if DRIVE_ROOT is not None:
    for p in ROOT.iterdir():
        if p.is_file() and p.stat().st_size<80_000_000:
            try:
                shutil.copy2(
                    p,
                    DRIVE_ROOT/p.name
                )
            except Exception as exc:
                print(
                    "Final sync warning:",
                    p.name,
                    type(exc).__name__
                )

cleanup_gpu()
print("\nCONTROLLED STAGE DONE")
print("Outputs:",ROOT)
print("GPU:",gpu_mem())

In [ ]:
# 30/30 — Paper-ready diagnostic figures

import matplotlib.pyplot as plt

FIG_DIR=ROOT/"figures"
FIG_DIR.mkdir(exist_ok=True)

if len(core_summary):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    avg=(
        core_summary.groupby(["method","replacement_level"],as_index=False)
        .agg(task_success=("task_success","mean"))
    )
    for method in MAIN_METHODS:
        g=avg[avg.method==method].sort_values("replacement_level")
        if len(g):
            ax.plot(
                g["replacement_level"],
                g["task_success"],
                marker="o",
                label=method
            )
    ax.set_xlabel("Number of independently replaced agents")
    ax.set_ylabel("Task success")
    ax.set_ylim(-0.03,1.03)
    ax.set_xticks([0,1,2,3])
    ax.legend(ncol=2,fontsize=8)
    ax.set_title("Compositional robustness")
    fig.tight_layout()
    fig.savefig(FIG_DIR/"robustness_curve.png",dpi=220)
    plt.show()

if len(curve_summary):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    avg=(
        curve_summary.groupby(["visibility","method"],as_index=False)
        .agg(active_accuracy=("active_accuracy","mean"))
    )
    for method in ["CB-BGE","CB+DualView","SCALE-NoInv","SCALE-NoOnt","SCALE"]:
        g=avg[avg.method==method].sort_values("visibility")
        if len(g):
            ax.plot(
                100*g["visibility"],
                g["active_accuracy"],
                marker="o",
                label=method
            )
    ax.set_xlabel("Observed B active-slot labels (%)")
    ax.set_ylabel("Held-out active semantic accuracy")
    ax.set_ylim(-0.03,1.03)
    ax.legend()
    ax.set_title("Label-efficient implementation adaptation")
    fig.tight_layout()
    fig.savefig(FIG_DIR/"label_efficiency_curve.png",dpi=220)
    plt.show()

if len(drift_summary):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    ax.plot(
        drift_summary["severity"],
        drift_summary["mean_prepolicy_active_scd"],
        marker="o",
        label="PrePolicy Active-SCD"
    )
    ax.plot(
        drift_summary["severity"],
        drift_summary["failure_rate"],
        marker="o",
        label="Failure rate"
    )
    ax.set_xlabel("Drift severity (D0–D3)")
    ax.set_ylabel("Mean value")
    ax.set_xticks([0,1,2,3])
    ax.set_ylim(-0.03,1.03)
    ax.legend()
    ax.set_title("Leakage-free semantic drift and failure")
    fig.tight_layout()
    fig.savefig(FIG_DIR/"drift_curve.png",dpi=220)
    plt.show()

if len(ontology_summary):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    opaque=ontology_summary[
        ontology_summary.tier=="opaque"
    ].sort_values("canonical_parent_accuracy",ascending=False)

    ax.bar(
        opaque["method"],
        opaque["canonical_parent_accuracy"]
    )
    ax.set_ylabel("Canonical-parent accuracy")
    ax.set_ylim(-0.03,1.03)
    ax.tick_params(axis="x",rotation=35)
    ax.set_title("Opaque unseen ontology leaves")
    fig.tight_layout()
    fig.savefig(FIG_DIR/"opaque_ontology_shift.png",dpi=220)
    plt.show()

print("Figures:",FIG_DIR)

# RQ5 — CRMArena-Pro Offline External Validity
## Public-source, Salesforce-free, end-to-end external transfer test

This stage runs **only after all controlled SCALE experiments have finished**.

It requires **no private Salesforce environment and no Salesforce credentials**.

The notebook downloads the official public CRMArena-Pro release from:

```text
Salesforce/CRMArenaPro
```

and freezes the exact Hugging Face dataset revision in the run manifest.

Official public files used:

```text
tasks_b2b.json
tasks_b2c.json
b2b_schema.json
b2c_schema.json
```

The public records expose:

```text
task
query
metadata
answer
reward_metric
persona
```

The benchmark card states that `metadata` is intended to be part of the system-prompt context.

---

## Important scientific boundary

Many full CRMArena-Pro tasks normally require live Salesforce records/tools.

Without a Salesforce org, the notebook does **not** pretend that those hidden records are available.

Instead it automatically constructs a conservative:

```text
CRMArena-Pro Offline Context-Grounded Subset
```

A non-privacy example is admitted to the answer-level offline benchmark only when its public
`query + persona + metadata` already contains the ground-truth answer evidence after the same
context truncation used by the model.

Privacy-rejection tasks are separately admissible because the benchmark target is refusal rather
than retrieval of a hidden Salesforce value.

Therefore:

```text
full CRMArena-Pro official environment score  !=  offline RQ5 score
```

The paper should report this stage as:

> **CRMArena-Pro offline dataset-grounded external-validity evaluation**

and keep the official live Salesforce environment evaluation as a limitation / future extension.

---

# External RQ5 contains two complementary tests

## RQ5-A — Semantic-interface transfer

A CRM-specific SCALE adapter is trained only on the public training partition.

Semantic contract:

```text
Planner   → CRM task family
Retriever → answer/evidence mode
Policy    → confidentiality posture
```

The same learned principles are retained:

- frozen BGE semantics;
- dual-view schema robustness;
- low-label A↔B implementation invariance;
- ontology/prototype alignment;
- residual graph correction.

Held-out CRM records are evaluated under:

```text
A
B
C2 unseen schema realization
```

with:

```text
CB-CRM
SCALE-CRM
```

This asks whether the SCALE representation mechanism transfers to a new enterprise domain.

---

## RQ5-B — End-to-end offline task test

For each frozen context-grounded CRM task:

```text
User
  ↓
Planner
  ↓
Retriever
  ↓
Policy
  ↓
Executor
  ↓
Ground-truth scorer
```

The same task, context, model, replacement level and ground truth are used for:

```text
NL
JSON
Onto-RAG
CB-CRM
SCALE-CRM
```

Agent implementations are replaced progressively:

```text
0 → 1 → 2 → 3
```

The replaced role is balanced deterministically across records.

Primary outputs:

- task success;
- compositional retention;
- semantic recovery;
- exact/fuzzy/privacy score;
- tokens;
- latency;
- paired SCALE-vs-baseline differences.

No ground-truth answer is placed in a prompt.

# Robustness Study A — Semantic Concept Interventions

A concept bottleneck is useful only if intervening on the semantic representation
predictably changes downstream behavior.

This experiment therefore treats the three role-active semantic fields as explicit
intervention variables:

- Planner → `object`
- Retriever → `evidence_state`
- Policy → `authority`

For each held-out C1/C2 example and each learned semantic interface, we:

1. decode the active semantic contract;
2. evaluate the downstream deterministic policy;
3. replace 0, 1, 2, or 3 active concepts with their gold values;
4. measure task recovery as more semantic concepts are corrected.

This is reported as an **intervention recovery curve**, analogous to concept-intervention
evaluation in concept-bottleneck models.

We additionally perturb an inactive field (`intent`) while keeping all active fields fixed.
A valid role-active interface should remain invariant to such an inactive intervention.

This is a mechanism test, not a claim of causal discovery.

In [ ]:
# 31A — Semantic concept-intervention faithfulness

from itertools import combinations

INTERVENTION_METHODS=["CB+DualView","SCALE"]
INTERVENTION_SPLITS={"c1_natural":"C1","c2_schema":"C2"}
ACTIVE_SEMANTIC_KEYS={
    "PLANNER":"object",
    "RETRIEVER":"evidence_state",
    "POLICY":"authority",
}

def decoded_active_semantics(base,impl,method):
    contracts={}
    semantics={}

    for agent in AGENTS:
        msg=realize_message(base,agent,impl,0)
        pred=ensemble_predict(method,[msg],BASE_GRAPH)[0]
        contract=pred["contract"]

        if method=="SCALE":
            contract=ontology_repair(contract)

        contracts[agent]=copy.deepcopy(contract)

        if agent=="PLANNER":
            semantics["object"]=contract.get("object")
        elif agent=="RETRIEVER":
            semantics["evidence_state"]=contract.get("state")
        else:
            semantics["authority"]=contract.get("authority")

    return contracts,semantics


def perturb_inactive_intent(contracts):
    """
    Change only an inactive semantic slot. The role-active interface should
    therefore produce exactly the same downstream semantics.
    """
    out=copy.deepcopy(contracts)
    candidates=["INFORM","QUERY","VERIFY","RECOMMEND","REQUEST"]

    for agent in AGENTS:
        current=out[agent].get("intent")
        replacement=next(
            (x for x in candidates if x!=current and x in INTENTS),
            current
        )
        out[agent]["intent"]=replacement

    return out


def active_semantics_from_contracts(contracts):
    return {
        "object":contracts["PLANNER"].get("object"),
        "evidence_state":contracts["RETRIEVER"].get("state"),
        "authority":contracts["POLICY"].get("authority"),
    }


intervention_rows=[]

for split_name,impl in INTERVENTION_SPLITS.items():
    for base in shift_test_base:
        gold=global_semantics(base)
        gold_policy=deterministic_policy(gold)

        for method in INTERVENTION_METHODS:
            contracts,pred_sem=decoded_active_semantics(
                base,impl,method
            )
            base_policy=deterministic_policy(pred_sem)

            keys=["object","evidence_state","authority"]

            # Evaluate every subset of active semantic corrections.
            for r in range(4):
                for subset in combinations(keys,r):
                    edited=copy.deepcopy(pred_sem)
                    for key in subset:
                        edited[key]=gold[key]

                    post=deterministic_policy(edited)

                    intervention_rows.append({
                        "split":split_name,
                        "base_id":base["base_id"],
                        "method":method,
                        "intervention_type":"gold_active_correction",
                        "n_active_intervened":r,
                        "intervened_slots":"|".join(subset),
                        "baseline_success":int(base_policy==gold_policy),
                        "post_success":int(post==gold_policy),
                        "policy_changed":int(post!=base_policy),
                    })

            # Inactive-slot intervention.
            inactive_contracts=perturb_inactive_intent(contracts)
            inactive_sem=active_semantics_from_contracts(
                inactive_contracts
            )
            inactive_policy=deterministic_policy(inactive_sem)

            intervention_rows.append({
                "split":split_name,
                "base_id":base["base_id"],
                "method":method,
                "intervention_type":"inactive_intent_perturbation",
                "n_active_intervened":0,
                "intervened_slots":"inactive_intent",
                "baseline_success":int(base_policy==gold_policy),
                "post_success":int(inactive_policy==gold_policy),
                "policy_changed":int(inactive_policy!=base_policy),
            })

intervention_df=pd.DataFrame(intervention_rows)

active_curve=(
    intervention_df[
        intervention_df.intervention_type=="gold_active_correction"
    ]
    .groupby(
        ["split","method","n_active_intervened"],
        as_index=False
    )
    .agg(
        task_success=("post_success","mean"),
        policy_change_rate=("policy_changed","mean"),
        n=("base_id","size"),
    )
)

inactive_summary=(
    intervention_df[
        intervention_df.intervention_type=="inactive_intent_perturbation"
    ]
    .groupby(["split","method"],as_index=False)
    .agg(
        inactive_invariance=(
            "policy_changed",
            lambda x:1-float(np.mean(x))
        ),
        task_success=("post_success","mean"),
        n=("base_id","size"),
    )
)

# Rescue rate among examples that initially fail.
rescue_rows=[]
for (split_name,method),g in intervention_df[
    intervention_df.intervention_type=="gold_active_correction"
].groupby(["split","method"]):
    base=g[g.n_active_intervened==0][
        ["base_id","post_success"]
    ].rename(columns={"post_success":"base_success"})
    full=g[g.n_active_intervened==3][
        ["base_id","post_success"]
    ].rename(columns={"post_success":"full_success"})
    pair=base.merge(full,on="base_id")
    failures=pair[pair.base_success==0]
    rescue=(
        float(failures.full_success.mean())
        if len(failures)
        else np.nan
    )
    rescue_rows.append({
        "split":split_name,
        "method":method,
        "baseline_task_success":float(pair.base_success.mean()),
        "full_intervention_task_success":float(pair.full_success.mean()),
        "failure_rescue_rate":rescue,
        "n_initial_failures":int(len(failures)),
    })

intervention_rescue=pd.DataFrame(rescue_rows)

display(active_curve.round(4))
display(inactive_summary.round(4))
display(intervention_rescue.round(4))

intervention_df.to_csv(
    ROOT/"semantic_intervention_results.csv",
    index=False
)
active_curve.to_csv(
    ROOT/"semantic_intervention_curve.csv",
    index=False
)
inactive_summary.to_csv(
    ROOT/"semantic_intervention_inactive_summary.csv",
    index=False
)
intervention_rescue.to_csv(
    ROOT/"semantic_intervention_rescue.csv",
    index=False
)

print("Semantic intervention study complete.")

# Robustness Study B — Large Hierarchical Ontology Extension

The original opaque ontology test uses a small number of unseen leaf concepts.
This robustness study expands the ontology to **100 previously unseen opaque nodes**
across five canonical object families and three hierarchy depths.

The test deliberately separates two problems:

1. **identifier grounding** — can the system identify an unseen ontology node?
2. **structural interpretation** — can the identified node be mapped through `is_a`
   relations to the correct canonical root?

To avoid overstating the learned contribution, two strong deterministic controls are added:

- `ExactID+Ancestor`
- `NormalizedID+Ancestor`

If an opaque identifier is explicitly transmitted, symbolic ontology lookup is a legitimate
ceiling baseline and must be reported.

We therefore also include a `semantic_alias` condition where the exact opaque identifier is
not present. In that condition only canonical-root accuracy is meaningful; exact leaf
detection is intentionally undefined.

The expanded benchmark is exploratory robustness evidence and does not replace the original
prespecified ontology gate.

In [ ]:


# ==============================================================================# 31B — Large hierarchical ontology-extension stress test

import hashlib as _hashlib

LARGE_ROOTS=[x for x in OBJECTS if x!="NO_OBJECT"]
LARGE_BRANCHES_PER_ROOT=4

SEMANTIC_ALIAS_BY_ROOT={
    "PURCHASE":"capital expenditure procurement request",
    "ACCESS":"privileged entitlement session",
    "CHANGE":"scheduled production modification",
    "INCIDENT":"security service alert",
    "COMPLIANCE_ITEM":"regulatory assurance review",
}

def opaque_label(root,path):
    token=_hashlib.sha256(
        f"{root}|{path}|{SEED}".encode()
    ).hexdigest()[:8].upper()
    return f"X{token}"

# Construct 20 nodes/root:
# 4 depth-1 + 8 depth-2 + 8 depth-3 = 100 total.
LARGE_OPAQUE_NODES=[]
for root in LARGE_ROOTS:
    for b in range(LARGE_BRANCHES_PER_ROOT):
        d1=opaque_label(root,f"{b}")
        LARGE_OPAQUE_NODES.append({
            "label":d1,
            "direct_parent":root,
            "canonical_root":root,
            "depth":1,
            "description":f"Ontology concept {d1}.",
        })

        for c in range(2):
            d2=opaque_label(root,f"{b}.{c}")
            LARGE_OPAQUE_NODES.append({
                "label":d2,
                "direct_parent":d1,
                "canonical_root":root,
                "depth":2,
                "description":f"Ontology concept {d2}.",
            })

            d3=opaque_label(root,f"{b}.{c}.0")
            LARGE_OPAQUE_NODES.append({
                "label":d3,
                "direct_parent":d2,
                "canonical_root":root,
                "depth":3,
                "description":f"Ontology concept {d3}.",
            })

assert len(LARGE_OPAQUE_NODES)==100, f"Expected 100 nodes, got {len(LARGE_OPAQUE_NODES)} from roots={LARGE_ROOTS}"


def build_hierarchical_object_graph(extra_nodes):
    """
    Independent robustness graph. It leaves the primary ontology experiment
    unchanged and supports multi-hop unseen object hierarchies.
    """
    all_extra=[
        {
            "label":x["label"],
            "direct_parent":x.get("direct_parent",x.get("parent")),
            "canonical_root":x.get("canonical_root",x.get("parent")),
            "description":x["description"],
            "depth":x.get("depth",1),
        }
        for x in (
            [
                {
                    "label":k["label"],
                    "direct_parent":k["parent"],
                    "canonical_root":k["parent"],
                    "description":k["description"],
                    "depth":1,
                }
                for k in KNOWN_GRAPH_LEAVES
            ]
            +extra_nodes
        )
    ]

    node_keys=list(BASE_NODE_KEYS)
    descriptions=[
        concept_description(s,l)
        for s,l in node_keys
    ]

    for node in all_extra:
        node_keys.append(("object_leaf",node["label"]))
        descriptions.append(node["description"])

    idx={k:i for i,k in enumerate(node_keys)}
    n=len(node_keys)

    rel_adj={
        r:np.zeros((n,n),dtype="float32")
        for r in RELATION_TYPES
    }

    def link(rel,k1,k2,bidir=True):
        if k1 not in idx or k2 not in idx:
            return
        i,j=idx[k1],idx[k2]
        rel_adj[rel][i,j]=1.
        if bidir:
            rel_adj[rel][j,i]=1.

    for action,auth in ACTION_REQUIRED_AUTH.items():
        link(
            "requires_authority",
            ("action",action),
            ("authority",auth)
        )

    for action,roles in ACTION_ALLOWED_ROLES.items():
        for role in roles:
            link(
                "permitted_role",
                ("action",action),
                ("role",role)
            )

    for action,states in ACTION_ALLOWED_STATES.items():
        for state in states:
            link(
                "allowed_state",
                ("action",action),
                ("state",state)
            )

    for role,action in {
        "PLANNER":"PLAN",
        "RETRIEVER":"SEARCH",
        "POLICY":"ASSESS",
    }.items():
        link(
            "role_action",
            ("role",role),
            ("action",action)
        )

    for prov,role in {
        "USER_REQUEST":"PLANNER",
        "CRM_DATA":"RETRIEVER",
        "POLICY_CONTEXT":"POLICY",
    }.items():
        link(
            "provenance_role",
            ("provenance",prov),
            ("role",role)
        )

    for obj,action in PASS_ACTION_BY_OBJECT.items():
        link(
            "object_action",
            ("object",obj),
            ("action",action)
        )

    root_by_leaf={}
    direct_parent={}
    depth_by_leaf={}

    extra_labels={x["label"] for x in all_extra}

    for node in all_extra:
        label=node["label"]
        parent=node["direct_parent"]

        child_key=("object_leaf",label)
        parent_key=(
            ("object",parent)
            if parent in OBJECTS
            else ("object_leaf",parent)
        )
        link("is_a",child_key,parent_key)

        root_by_leaf[label]=node["canonical_root"]
        direct_parent[label]=parent
        depth_by_leaf[label]=node["depth"]

    rel_adj={
        r:torch.tensor(
            _normalize_adj(a),
            dtype=torch.float32
        )
        for r,a in rel_adj.items()
    }

    features=torch.tensor(
        semantic_encode(descriptions),
        dtype=torch.float32
    )

    class_indices={
        slot:[
            idx[(slot,str(label))]
            for label in encoders[slot].classes_
        ]
        for slot in SLOTS
    }

    return {
        "node_keys":node_keys,
        "idx":idx,
        "features":features,
        "rel_adj":rel_adj,
        "class_indices":class_indices,
        "leaf_parent":root_by_leaf,
        "direct_parent":direct_parent,
        "depth_by_leaf":depth_by_leaf,
        "leaf_indices":[
            idx[("object_leaf",x["label"])]
            for x in all_extra
        ],
    }


LARGE_OPAQUE_GRAPH=build_hierarchical_object_graph(
    LARGE_OPAQUE_NODES
)

LARGE_NODE_BY_LABEL={
    x["label"]:x
    for x in LARGE_OPAQUE_NODES
}

def punctuate_identifier(label):
    # deterministic surface change with no semantic alias table
    mid=max(2,len(label)//2)
    return label[:mid]+"-"+label[mid:]

def large_ontology_message(node,surface):
    label=node["label"]
    root=node["canonical_root"]

    if surface=="exact_id":
        return f"Planner ontology-v4 handoff: target concept is {label}."
    if surface=="wrapped_id":
        return f"planner.tool.v4|entity=[{label.lower()}]|mode=route"
    if surface=="punctuated_id":
        return (
            "schema/tool handoff target="
            +punctuate_identifier(label.lower())
        )
    if surface=="semantic_alias":
        return (
            "Planner handoff describes a "
            +SEMANTIC_ALIAS_BY_ROOT[root]
            +"."
        )

    raise ValueError(surface)


def exact_id_lookup(text,graph_ctx):
    candidates=list(graph_ctx["leaf_parent"])
    for label in candidates:
        if label in text:
            return label
    return None


def normalized_id_lookup(text,graph_ctx):
    normalized=re.sub(
        r"[^A-Z0-9]","",str(text).upper()
    )
    hits=[]
    for label in graph_ctx["leaf_parent"]:
        key=re.sub(r"[^A-Z0-9]","",label.upper())
        if key and key in normalized:
            hits.append(label)
    # Longest match avoids accidental short-prefix selection.
    return max(hits,key=len) if hits else None


def canonicalize_lookup(label,graph_ctx):
    if label is None:
        return None
    if label in OBJECTS:
        return label
    return graph_ctx["leaf_parent"].get(label)


def learned_large_prediction(method,text,graph_ctx):
    if method=="CB+DualView":
        p=ensemble_predict(
            "CB+DualView",[text],BASE_GRAPH
        )[0]
        obj=p["contract"].get("object")
        return obj,obj

    model_name={
        "Proto-CB":"Proto-CB",
        "Proto+Ancestor":"Proto-CB",
        "SCALE-NoOnt":"SCALE-NoOnt",
        "SCALE-Full":"SCALE",
    }[method]

    votes=[]
    for member in MODELS[model_name]:
        label,_=dynamic_object_member(
            member,text,graph_ctx
        )
        votes.append(label)

    pred=max(set(votes),key=votes.count)

    if method in {"Proto+Ancestor","SCALE-Full"}:
        canonical=canonicalize_lookup(
            pred,graph_ctx
        )
    else:
        canonical=pred if pred in OBJECTS else pred

    return pred,canonical


LARGE_ONTOLOGY_METHODS=[
    "ExactID+Ancestor",
    "NormalizedID+Ancestor",
    "CB+DualView",
    "Proto-CB",
    "Proto+Ancestor",
    "SCALE-NoOnt",
    "SCALE-Full",
]

LARGE_SURFACES=[
    "exact_id",
    "wrapped_id",
    "punctuated_id",
    "semantic_alias",
]

large_onto_rows=[]

for node in LARGE_OPAQUE_NODES:
    gold_leaf=node["label"]
    gold_root=node["canonical_root"]
    depth=node["depth"]

    for surface in LARGE_SURFACES:
        text=large_ontology_message(node,surface)

        for method in LARGE_ONTOLOGY_METHODS:
            if method=="ExactID+Ancestor":
                pred=exact_id_lookup(
                    text,LARGE_OPAQUE_GRAPH
                )
                canonical=canonicalize_lookup(
                    pred,LARGE_OPAQUE_GRAPH
                )

            elif method=="NormalizedID+Ancestor":
                pred=normalized_id_lookup(
                    text,LARGE_OPAQUE_GRAPH
                )
                canonical=canonicalize_lookup(
                    pred,LARGE_OPAQUE_GRAPH
                )

            else:
                pred,canonical=learned_large_prediction(
                    method,text,LARGE_OPAQUE_GRAPH
                )

            leaf_defined=(surface!="semantic_alias")

            large_onto_rows.append({
                "gold_leaf":gold_leaf,
                "gold_root":gold_root,
                "depth":depth,
                "surface":surface,
                "method":method,
                "predicted_node":pred,
                "canonical_root":canonical,
                "leaf_detection":(
                    int(pred==gold_leaf)
                    if leaf_defined
                    else np.nan
                ),
                "canonical_root_accuracy":int(
                    canonical==gold_root
                ),
            })

large_ontology_df=pd.DataFrame(
    large_onto_rows
)

large_ontology_summary=(
    large_ontology_df.groupby(
        ["surface","method"],
        as_index=False
    )
    .agg(
        canonical_root_accuracy=(
            "canonical_root_accuracy","mean"
        ),
        leaf_detection=("leaf_detection","mean"),
        n=("gold_leaf","size"),
    )
)

large_ontology_depth=(
    large_ontology_df[
        large_ontology_df.surface!="semantic_alias"
    ]
    .groupby(
        ["depth","method"],
        as_index=False
    )
    .agg(
        canonical_root_accuracy=(
            "canonical_root_accuracy","mean"
        ),
        leaf_detection=("leaf_detection","mean"),
        n=("gold_leaf","size"),
    )
)

display(large_ontology_summary.round(4))
display(large_ontology_depth.round(4))

large_ontology_df.to_csv(
    ROOT/"large_ontology_results.csv",
    index=False
)
large_ontology_summary.to_csv(
    ROOT/"large_ontology_summary.csv",
    index=False
)
large_ontology_depth.to_csv(
    ROOT/"large_ontology_depth_summary.csv",
    index=False
)

print(
    "Large ontology stress complete:",
    len(LARGE_OPAQUE_NODES),
    "unseen nodes x",
    len(LARGE_SURFACES),
    "surface conditions."
)

# 31C — Leave-one-drift-family-out semantic observability

from sklearn.metrics import balanced_accuracy_score

DRIFT_FAMILIES=[
    "object_substitution",
    "evidence_flip",
    "authority_below_required",
]

def _operator_set(x):
    return {
        y for y in str(x).split("|")
        if y and y!="benign_surface_evolution"
    }


# Add a generic raw semantic-embedding drift baseline.
# This asks whether ordinary message embedding distance is sufficient.
_raw_embedding_drift={}

_ref_texts=[]
_cur_texts=[]
_keys=[]

base_lookup={
    b["base_id"]:b
    for b in drift_base
}

for _,row in drift_df.iterrows():
    base=base_lookup[row["base_id"]]
    severity=int(row["severity"])

    current_msgs,_=drift_messages(
        base,severity
    )

    for agent in AGENTS:
        _ref_texts.append(
            realize_message(base,agent,"A",0)
        )
        _cur_texts.append(
            current_msgs[agent]
        )
        _keys.append(
            (base["base_id"],severity,agent)
        )

_ref_emb=semantic_encode(_ref_texts)
_cur_emb=semantic_encode(_cur_texts)

_cos=(
    1-np.sum(_ref_emb*_cur_emb,axis=1)
)

_tmp={}
for key,val in zip(_keys,_cos):
    b,s,a=key
    _tmp.setdefault((b,s),[]).append(
        float(val)
    )

for key,vals in _tmp.items():
    _raw_embedding_drift[key]=float(
        np.mean(vals)
    )

lofo_df=drift_df.copy()
lofo_df["raw_bge_drift"]=[
    _raw_embedding_drift[
        (r.base_id,int(r.severity))
    ]
    for r in lofo_df.itertuples()
]
lofo_df["neg_confidence"]=1-lofo_df["confidence"]
lofo_df["operator_set"]=[
    _operator_set(x)
    for x in lofo_df["operators"]
]


def best_balanced_threshold(y,score):
    y=np.asarray(y,int)
    score=np.asarray(score,float)

    if len(np.unique(y))<2:
        return np.nan,np.nan

    candidates=np.unique(score)
    if len(candidates)>200:
        candidates=np.quantile(
            score,np.linspace(0,1,200)
        )

    best_t=None
    best=-1

    for t in candidates:
        pred=(score>=t).astype(int)
        b=balanced_accuracy_score(y,pred)
        if b>best:
            best=b
            best_t=float(t)

    return best_t,float(best)


LOFO_METRICS={
    "PrePolicy Active-SCD":"prepolicy_active_scd",
    "Full SCD":"full_scd",
    "Negative confidence":"neg_confidence",
    "Raw BGE drift":"raw_bge_drift",
}

lofo_rows=[]

all_bases=sorted(
    lofo_df.base_id.unique()
)

for split_seed in range(20):
    rng=np.random.default_rng(
        SEED+9000+split_seed
    )
    shuffled=np.array(
        all_bases,
        dtype=object
    )
    rng.shuffle(shuffled)
    cut=len(shuffled)//2
    train_ids=set(shuffled[:cut])
    test_ids=set(shuffled[cut:])

    for heldout in DRIFT_FAMILIES:
        # Clean family-level protocol:
        # train = D0/D1 + D2 examples from other single-op families
        # test  = D0/D1 + D2 examples from the held-out single-op family
        train=lofo_df[
            lofo_df.base_id.isin(train_ids)
            &(
                (lofo_df.severity<=1)
                |(
                    (lofo_df.severity==2)
                    &lofo_df.operator_set.apply(
                        lambda s:
                        heldout not in s
                        and len(s)==1
                    )
                )
            )
        ].copy()

        test=lofo_df[
            lofo_df.base_id.isin(test_ids)
            &(
                (lofo_df.severity<=1)
                |(
                    (lofo_df.severity==2)
                    &lofo_df.operator_set.apply(
                        lambda s:
                        heldout in s
                        and len(s)==1
                    )
                )
            )
        ].copy()

        train_y=(
            train.severity==2
        ).astype(int).to_numpy()
        test_y=(
            test.severity==2
        ).astype(int).to_numpy()

        if (
            len(np.unique(train_y))<2
            or len(np.unique(test_y))<2
        ):
            continue

        for metric_name,col in LOFO_METRICS.items():
            t,train_bal=best_balanced_threshold(
                train_y,
                train[col].to_numpy()
            )

            score=test[col].to_numpy()
            auc=roc_auc_score(
                test_y,score
            )
            pred=(score>=t).astype(int)
            bal=balanced_accuracy_score(
                test_y,pred
            )

            # Failure prediction on the same unseen-family test population.
            failure_y=test.failure.astype(
                int
            ).to_numpy()
            failure_auc=(
                roc_auc_score(
                    failure_y,score
                )
                if len(np.unique(failure_y))>=2
                else np.nan
            )

            lofo_rows.append({
                "split_seed":split_seed,
                "heldout_family":heldout,
                "metric":metric_name,
                "threshold":t,
                "train_balanced_accuracy":train_bal,
                "test_harmful_auc":auc,
                "test_balanced_accuracy":bal,
                "test_failure_auc":failure_auc,
                "n_train":len(train),
                "n_test":len(test),
                "n_test_harmful":int(test_y.sum()),
            })

lofo_results=pd.DataFrame(
    lofo_rows
)

lofo_summary=(
    lofo_results.groupby(
        ["heldout_family","metric"],
        as_index=False
    )
    .agg(
        harmful_auc_mean=("test_harmful_auc","mean"),
        harmful_auc_sd=("test_harmful_auc","std"),
        harmful_auc_min=("test_harmful_auc","min"),
        balanced_accuracy_mean=("test_balanced_accuracy","mean"),
        failure_auc_mean=("test_failure_auc","mean"),
        failure_auc_min=("test_failure_auc","min"),
        splits=("split_seed","nunique"),
    )
)

lofo_overall=(
    lofo_results.groupby(
        "metric",
        as_index=False
    )
    .agg(
        harmful_auc_mean=("test_harmful_auc","mean"),
        harmful_auc_min=("test_harmful_auc","min"),
        balanced_accuracy_mean=("test_balanced_accuracy","mean"),
        failure_auc_mean=("test_failure_auc","mean"),
        failure_auc_min=("test_failure_auc","min"),
    )
)

display(lofo_summary.round(4))
display(lofo_overall.round(4))

lofo_results.to_csv(
    ROOT/"leave_one_drift_family_out_results.csv",
    index=False
)
lofo_summary.to_csv(
    ROOT/"leave_one_drift_family_out_summary.csv",
    index=False
)
lofo_overall.to_csv(
    ROOT/"leave_one_drift_family_out_overall.csv",
    index=False
)

print("Leave-one-drift-family-out observability complete.")

# 31D — Mechanism robustness report and paper-ready figures

MECH_FIG_DIR=ROOT/"figures"
MECH_FIG_DIR.mkdir(exist_ok=True)

# Intervention curve
if len(active_curve):
    fig,ax=plt.subplots(figsize=(7.2,4.6))
    for (split_name,method),g in active_curve.groupby(
        ["split","method"]
    ):
        g=g.sort_values(
            "n_active_intervened"
        )
        ax.plot(
            g["n_active_intervened"],
            g["task_success"],
            marker="o",
            label=f"{method} / {split_name}"
        )
    ax.set_xlabel(
        "Number of gold active concepts intervened"
    )
    ax.set_ylabel("Task success")
    ax.set_xticks([0,1,2,3])
    ax.set_ylim(-0.03,1.03)
    ax.legend(fontsize=8)
    ax.set_title(
        "Semantic concept-intervention recovery"
    )
    fig.tight_layout()
    fig.savefig(
        MECH_FIG_DIR/"semantic_intervention_curve.png",
        dpi=220
    )
    plt.show()

# Large ontology
if len(large_ontology_summary):
    fig,ax=plt.subplots(figsize=(8.4,4.8))
    pivot=large_ontology_summary.pivot(
        index="method",
        columns="surface",
        values="canonical_root_accuracy"
    )
    pivot.plot(kind="bar",ax=ax)
    ax.set_ylabel("Canonical-root accuracy")
    ax.set_ylim(-0.03,1.03)
    ax.set_title(
        "100-node hierarchical ontology stress test"
    )
    ax.tick_params(axis="x",rotation=35)
    fig.tight_layout()
    fig.savefig(
        MECH_FIG_DIR/"large_ontology_stress.png",
        dpi=220
    )
    plt.show()

# LOFO SCD
if len(lofo_summary):
    fig,ax=plt.subplots(figsize=(8.0,4.8))
    primary=lofo_summary[
        lofo_summary.metric.isin([
            "PrePolicy Active-SCD",
            "Full SCD",
            "Negative confidence",
            "Raw BGE drift",
        ])
    ]
    pivot=primary.pivot(
        index="heldout_family",
        columns="metric",
        values="harmful_auc_mean"
    )
    pivot.plot(kind="bar",ax=ax)
    ax.set_ylabel(
        "Held-out-family harmful-drift AUROC"
    )
    ax.set_ylim(0,1.03)
    ax.set_title(
        "Leave-one-drift-family-out observability"
    )
    ax.tick_params(axis="x",rotation=20)
    fig.tight_layout()
    fig.savefig(
        MECH_FIG_DIR/"lofo_drift_auc.png",
        dpi=220
    )
    plt.show()

# Exploratory support diagnostics. These are NOT primary claim gates.
support_lines=[]

if len(intervention_rescue):
    scale_c2=intervention_rescue[
        (intervention_rescue.method=="SCALE")
        &(intervention_rescue.split=="c2_schema")
    ]
    if len(scale_c2):
        r=scale_c2.iloc[0]
        support_lines.append(
            "Semantic intervention: SCALE C2 baseline "
            f"{r.baseline_task_success:.3f}, "
            f"full active-concept intervention "
            f"{r.full_intervention_task_success:.3f}, "
            f"failure rescue "
            f"{r.failure_rescue_rate if pd.notna(r.failure_rescue_rate) else float('nan'):.3f}."
        )

if len(large_ontology_summary):
    q=large_ontology_summary[
        (large_ontology_summary.method=="SCALE-Full")
        &(large_ontology_summary.surface=="semantic_alias")
    ]
    if len(q):
        support_lines.append(
            "Large ontology semantic-alias canonical-root accuracy: "
            f"{q.iloc[0].canonical_root_accuracy:.3f}."
        )

if len(lofo_overall):
    q=lofo_overall[
        lofo_overall.metric=="PrePolicy Active-SCD"
    ]
    if len(q):
        support_lines.append(
            "LOFO Active-SCD harmful-drift AUROC mean/min: "
            f"{q.iloc[0].harmful_auc_mean:.3f}/"
            f"{q.iloc[0].harmful_auc_min:.3f}."
        )

mechanism_report = """
# SCALE Exploratory Mechanism Robustness

These analyses are supplementary robustness studies and do not alter G0–G7.

""" + "\n".join(
    f"- {x}" for x in support_lines
)

(ROOT/"MECHANISM_ROBUSTNESS_REPORT.md").write_text(
    mechanism_report.strip()+"\n",
    encoding="utf-8"
)

from IPython.display import Markdown,display
display(Markdown(mechanism_report))

print(
    "Mechanism robustness outputs complete:",
    ROOT
)

# 31/39 — CRMArena-Pro offline configuration and source pinning

RUN_CRM_OFFLINE = bool(torch.cuda.is_available())
if not RUN_CRM_OFFLINE:
    print(
        "CRM offline generative execution SKIPPED/INCONCLUSIVE: "
        "CUDA is unavailable. CRM semantic-transfer analyses that do not "
        "require causal generation remain available."
    )

# Keep the external stage meaningful but still practical on one A100.
CRM_SEMANTIC_TRAIN_LIMIT = 800
CRM_SEMANTIC_DEV_LIMIT = 160
CRM_OFFLINE_TEST_LIMIT = 48

CRM_HEAD_SEEDS = [2026, 2027, 2028]
CRM_HEAD_EPOCHS = 30
CRM_B_LABEL_VISIBILITY = 0.10

# Two independent executor families are enough for the external transfer stage.
CRM_EXECUTOR_SPECS = READER_MODELS[:2]

CRM_OFFLINE_METHODS = [
    "NL",
    "JSON",
    "Onto-RAG",
    "CB-CRM",
    "CB-Dual-CRM",
    "SCALE-CRM",
]

CRM_MAX_INPUT = 2800
CRM_MAX_OUTPUT = 96
CRM_CONTEXT_CHAR_BUDGET = 6500
CRM_SCHEMA_TOPK = 2

CRM_FUZZY_SUCCESS_THRESHOLD = 0.67
CRM_NONINFERIOR_MARGIN = 0.03

CRM_ROOT = ROOT / "crmarena_pro_offline"
CRM_ROOT.mkdir(exist_ok=True)

CRM_SOURCE_REPO = "Salesforce/CRMArenaPro"
CRM_SOURCE_FILES = [
    "tasks_b2b.json",
    "tasks_b2c.json",
    "b2b_schema.json",
    "b2c_schema.json",
]

if RUN_CRM_OFFLINE:
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    info = api.dataset_info(CRM_SOURCE_REPO)
    CRM_SOURCE_REVISION = info.sha

    print("Official CRMArena-Pro revision:", CRM_SOURCE_REVISION)

    crm_source_paths = {}
    for filename in CRM_SOURCE_FILES:
        path = hf_hub_download(
            repo_id=CRM_SOURCE_REPO,
            filename=filename,
            repo_type="dataset",
            revision=CRM_SOURCE_REVISION,
        )
        crm_source_paths[filename] = Path(path)
        print("Downloaded:", filename)

    source_manifest = {
        "repo": CRM_SOURCE_REPO,
        "revision": CRM_SOURCE_REVISION,
        "files": {
            name: str(path)
            for name, path in crm_source_paths.items()
        },
        "evaluation_type": "offline_dataset_grounded_external_validity",
        "salesforce_credentials_required": False,
    }

    (CRM_ROOT/"crm_source_manifest.json").write_text(
        json.dumps(source_manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
else:
    CRM_SOURCE_REVISION = None
    crm_source_paths = {}

# 32/39 — Parse official records and construct the conservative offline-answerable pool

from difflib import SequenceMatcher

def load_json_file(path):
    obj=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(obj,dict):
        obj=obj.get("data",obj.get("tasks",obj.get("records",obj)))
    if not isinstance(obj,list):
        raise TypeError(f"Expected a JSON list, got {type(obj).__name__}")
    return obj

def compact_json(obj):
    return json.dumps(obj,ensure_ascii=False,separators=(",",":"),default=str)

def norm_text(x):
    s=str(x).lower()
    s=re.sub(r"[^a-z0-9]+"," ",s)
    return re.sub(r"\s+"," ",s).strip()

def answer_list(x):
    if isinstance(x,list):
        return [str(v) for v in x]
    if x is None:
        return []
    return [str(x)]

def crm_public_context(record):
    # This is exactly the public context available to the offline executor.
    metadata=record.get("metadata",{})
    persona=record.get("persona","")
    query=record.get("query","")

    text=(
        f"PERSONA:\n{persona}\n\n"
        f"QUERY:\n{query}\n\n"
        f"PUBLIC METADATA:\n{compact_json(metadata)}"
    )
    return text[:CRM_CONTEXT_CHAR_BUDGET]

def answer_supported_by_public_context(record):
    metric=str(record.get("reward_metric","")).lower()

    # Privacy tasks test refusal, not retrieval of a hidden record value.
    if metric=="privacy_rejection":
        return True

    gold=[norm_text(x) for x in answer_list(record.get("answer")) if norm_text(x)]
    if not gold:
        return False

    ctx=norm_text(crm_public_context(record))

    # Conservative admission: every gold item must already occur in the
    # exact public context given to the model.
    return all(g in ctx for g in gold)

def record_hash(record):
    raw=(
        f"{record['org']}|{record['source_index']}|"
        f"{record.get('task')}|{record.get('query')}"
    )
    return hashlib.sha256(raw.encode()).hexdigest()

raw_records=[]

if RUN_CRM_OFFLINE:
    for org,filename in [
        ("b2b","tasks_b2b.json"),
        ("b2c","tasks_b2c.json"),
    ]:
        data=load_json_file(crm_source_paths[filename])

        for i,item in enumerate(data):
            r=dict(item)
            r["org"]=org
            r["source_index"]=i
            r["record_id"]=f"{org}:{i}"
            r["source_hash"]=record_hash(r)
            r["offline_context_supported"]=answer_supported_by_public_context(r)
            raw_records.append(r)

crm_all_df=pd.DataFrame([
    {
        "record_id":r["record_id"],
        "org":r["org"],
        "source_index":r["source_index"],
        "task":r.get("task"),
        "reward_metric":r.get("reward_metric"),
        "offline_context_supported":r["offline_context_supported"],
        "source_hash":r["source_hash"],
    }
    for r in raw_records
])

print("Official public records:",len(crm_all_df))
print("Offline context-grounded records:",
      int(crm_all_df.offline_context_supported.sum()))

display(
    crm_all_df.groupby(
        ["org","reward_metric","offline_context_supported"],
        dropna=False
    ).size().reset_index(name="n")
)

crm_all_df.to_csv(CRM_ROOT/"crm_public_record_audit.csv",index=False)

# 33/39 — Reproducible stratified train/dev/test split and schema retrieval index

def stratified_cap(records,limit,group_keys=("org","task")):
    if len(records)<=limit:
        return list(records)

    groups={}
    for r in records:
        key=tuple(str(r.get(k,"")) for k in group_keys)
        groups.setdefault(key,[]).append(r)

    for key in groups:
        groups[key]=sorted(groups[key],key=lambda x:x["source_hash"])

    selected=[]
    keys=sorted(groups)
    pos={k:0 for k in keys}

    while len(selected)<limit:
        progressed=False
        for k in keys:
            if pos[k]<len(groups[k]) and len(selected)<limit:
                selected.append(groups[k][pos[k]])
                pos[k]+=1
                progressed=True
        if not progressed:
            break

    return selected

# Split within org/task so task families are represented on both sides.
by_group={}
for r in raw_records:
    key=(r["org"],str(r.get("task","")))
    by_group.setdefault(key,[]).append(r)

crm_train_pool=[]
crm_dev_pool=[]
crm_test_pool=[]

for key,rows in by_group.items():
    rows=sorted(rows,key=lambda x:x["source_hash"])
    n=len(rows)

    if n<5:
        crm_train_pool.extend(rows)
        continue

    n_test=max(1,int(round(.20*n)))
    n_dev=max(1,int(round(.15*n)))

    crm_test_pool.extend(rows[-n_test:])
    crm_dev_pool.extend(rows[-(n_test+n_dev):-n_test])
    crm_train_pool.extend(rows[:-(n_test+n_dev)])

crm_train_records=stratified_cap(
    crm_train_pool,CRM_SEMANTIC_TRAIN_LIMIT
)
crm_dev_records=stratified_cap(
    crm_dev_pool,CRM_SEMANTIC_DEV_LIMIT
)

# Answer-level RQ5 uses ONLY held-out public-context-supported records.
crm_test_supported=[
    r for r in crm_test_pool
    if r["offline_context_supported"]
]
crm_test_records=stratified_cap(
    crm_test_supported,CRM_OFFLINE_TEST_LIMIT,
    group_keys=("org","task","reward_metric")
)

assert not (
    {r["record_id"] for r in crm_train_records}
    & {r["record_id"] for r in crm_test_records}
)

split_manifest=[]

for split,rows in [
    ("train",crm_train_records),
    ("dev",crm_dev_records),
    ("offline_test",crm_test_records),
]:
    for r in rows:
        split_manifest.append({
            "record_id":r["record_id"],
            "split":split,
            "org":r["org"],
            "task":r.get("task"),
            "reward_metric":r.get("reward_metric"),
            "context_supported":r["offline_context_supported"],
            "source_hash":r["source_hash"],
        })

crm_split_df=pd.DataFrame(split_manifest)
crm_split_df.to_csv(CRM_ROOT/"crm_split_manifest.csv",index=False)

print(
    "CRM split sizes:",
    len(crm_train_records),
    len(crm_dev_records),
    len(crm_test_records)
)
display(
    crm_split_df.groupby(
        ["split","org","reward_metric"],
        dropna=False
    ).size().reset_index(name="n")
)

# ---------- Schema index ----------
b2b_schema=load_json_file(crm_source_paths["b2b_schema.json"])
b2c_schema=load_json_file(crm_source_paths["b2c_schema.json"])

CRM_SCHEMA_BY_ORG={"b2b":b2b_schema,"b2c":b2c_schema}
CRM_SCHEMA_DOCS={}

for org,schema in CRM_SCHEMA_BY_ORG.items():
    docs=[]
    for item in schema:
        obj=item.get("object","UNKNOWN")
        fields=item.get("fields",{})
        desc=f"CRM object {obj}. Fields: "+", ".join(
            f"{k}: {v}" for k,v in fields.items()
        )
        docs.append({
            "object":obj,
            "text":desc[:3000],
        })

    emb=semantic_encode([d["text"] for d in docs])
    CRM_SCHEMA_DOCS[org]={"docs":docs,"emb":emb}

def crm_schema_retrieve(record,topk=CRM_SCHEMA_TOPK):
    org=record["org"]
    q=semantic_encode([str(record.get("query",""))])[0]
    m=CRM_SCHEMA_DOCS[org]
    sims=m["emb"]@q
    idx=np.argsort(-sims)[:topk]
    return [m["docs"][int(i)] for i in idx]

# 34/39 — CRM semantic contract and independently evolving role realizations

CRM_ROLES=["PLANNER","RETRIEVER","POLICY"]
CRM_SLOTS=["task","reward","privacy"]

def crm_privacy_label(record):
    return (
        "RESTRICTED"
        if str(record.get("reward_metric","")).lower()=="privacy_rejection"
        else "STANDARD"
    )

def crm_contract(record):
    return {
        "task":str(record.get("task","unknown")),
        "reward":str(record.get("reward_metric","unknown")),
        "privacy":crm_privacy_label(record),
    }

def humanize(x):
    return str(x).replace("_"," ").replace("-"," ").strip()

def crm_role_message(record,role,impl):
    c=crm_contract(record)
    q=str(record.get("query",""))
    q_short=q[:220]

    if role=="PLANNER":
        if impl=="A":
            return (
                f"Planning assessment: this CRM request belongs to the "
                f"{humanize(c['task'])} work category. Request: {q_short}"
            )
        if impl=="B":
            return (
                f"workflow.v2 | work-kind={c['task'].replace('_','-')} | "
                f"request-fragment={q_short}"
            )
        if impl=="C2":
            return (
                f"dispatch.v9::category={c['task'].replace('_','.')}::"
                f"payload={q_short}"
            )

    if role=="RETRIEVER":
        if impl=="A":
            return (
                f"Evidence handling should use the "
                f"{humanize(c['reward'])} evaluation mode. "
                "Evidence provenance is the public benchmark context."
            )
        if impl=="B":
            return (
                f"evidence.v2 | scoring-mode={c['reward']} | "
                "source=public-benchmark-context"
            )
        if impl=="C2":
            return (
                f"resolver.v9::metric={c['reward'].replace('_','-')}::"
                "source=benchmark.context"
            )

    if role=="POLICY":
        label=c["privacy"]
        if impl=="A":
            return (
                "Disclosure posture is restricted; do not reveal protected data."
                if label=="RESTRICTED"
                else "Disclosure posture is standard for this benchmark request."
            )
        if impl=="B":
            return f"policy.v2 | disclosure-class={label.lower()}"
        if impl=="C2":
            return f"guardrail.v9::class={label.lower()}"

    raise ValueError((role,impl))

CRM_ACTIVE_SLOT={
    "PLANNER":"task",
    "RETRIEVER":"reward",
    "POLICY":"privacy",
}

# Progressive, role-balanced A→B replacement.
def crm_composition(record,replacement_level):
    assert replacement_level in {0,1,2,3}

    if replacement_level==0:
        return {r:"A" for r in CRM_ROLES}
    if replacement_level==3:
        return {r:"B" for r in CRM_ROLES}

    offset=int(record["source_hash"][:8],16)%3
    order=CRM_ROLES[offset:]+CRM_ROLES[:offset]

    mapping={r:"A" for r in CRM_ROLES}
    for r in order[:replacement_level]:
        mapping[r]="B"
    return mapping

# External semantic OOD diagnostic.
def crm_c2_messages(record):
    return {
        role:crm_role_message(record,role,"C2")
        for role in CRM_ROLES
    }

sample=crm_train_records[0]
print(crm_contract(sample))
for role in CRM_ROLES:
    print(role,"A:",crm_role_message(sample,role,"A")[:180])
    print(role,"B:",crm_role_message(sample,role,"B")[:180])
    print(role,"C2:",crm_role_message(sample,role,"C2")[:180])

# 35/40 — Train CRM-specific CB, dual-view CB and final hybrid SCALE adapters

crm_label_encoders={}
for slot in CRM_SLOTS:
    le=LabelEncoder()
    le.fit([
        crm_contract(r)[slot]
        for r in crm_train_records
    ])
    crm_label_encoders[slot]=le

known_tasks=set(
    crm_label_encoders["task"].classes_
)

crm_test_records=[
    r for r in crm_test_records
    if crm_contract(r)["task"] in known_tasks
]

CRM_RELATIONS=[
    "task_reward",
    "task_privacy",
]

def crm_graph_context():
    node_keys=[
        (slot,str(label))
        for slot in CRM_SLOTS
        for label in crm_label_encoders[slot].classes_
    ]
    idx={k:i for i,k in enumerate(node_keys)}

    descriptions=[]
    for slot,label in node_keys:
        if slot=="task":
            descriptions.append(
                f"CRM professional task category {humanize(label)}."
            )
        elif slot=="reward":
            descriptions.append(
                f"CRM answer evaluation and evidence mode {humanize(label)}."
            )
        else:
            descriptions.append(
                f"CRM confidentiality posture {humanize(label)}."
            )

    n=len(node_keys)
    rel_adj={
        r:np.zeros((n,n),dtype="float32")
        for r in CRM_RELATIONS
    }

    def link(rel,a,b):
        if a not in idx or b not in idx:
            return
        i,j=idx[a],idx[b]
        rel_adj[rel][i,j]=1.
        rel_adj[rel][j,i]=1.

    for rec in crm_train_records:
        c=crm_contract(rec)
        link(
            "task_reward",
            ("task",c["task"]),
            ("reward",c["reward"])
        )
        link(
            "task_privacy",
            ("task",c["task"]),
            ("privacy",c["privacy"])
        )

    def norm_adj(a):
        deg=a.sum(1)
        inv=np.zeros_like(deg)
        nz=deg>0
        inv[nz]=1/np.sqrt(deg[nz])
        return (inv[:,None]*a)*inv[None,:]

    rel_adj={
        r:torch.tensor(
            norm_adj(a),
            dtype=torch.float32
        )
        for r,a in rel_adj.items()
    }

    features=torch.tensor(
        semantic_encode(descriptions),
        dtype=torch.float32
    )

    class_indices={
        slot:[
            idx[(slot,str(label))]
            for label in crm_label_encoders[slot].classes_
        ]
        for slot in CRM_SLOTS
    }

    return {
        "node_keys":node_keys,
        "features":features,
        "rel_adj":rel_adj,
        "class_indices":class_indices,
    }

CRM_GRAPH=crm_graph_context()

# ---------- Training rows ----------
crm_train_rows=[]

for rec in crm_train_records:
    for role in CRM_ROLES:
        for impl in ["A","B"]:
            crm_train_rows.append({
                "record_id":rec["record_id"],
                "role":role,
                "impl":impl,
                "message":crm_role_message(
                    rec,role,impl
                ),
                "contract":crm_contract(rec),
            })

crm_raw_X=torch.tensor(
    semantic_encode([
        x["message"]
        for x in crm_train_rows
    ]),
    dtype=torch.float32
)

crm_dual_X=torch.tensor(
    dual_view_encode([
        x["message"]
        for x in crm_train_rows
    ]),
    dtype=torch.float32
)

crm_y={}
for slot in CRM_SLOTS:
    crm_y[slot]=torch.tensor(
        crm_label_encoders[slot].transform([
            x["contract"][slot]
            for x in crm_train_rows
        ]),
        dtype=torch.long
    )

# Directed A→B CRM pairs.
crm_lookup={}
for idx,row in enumerate(crm_train_rows):
    crm_lookup[
        (row["record_id"],row["role"],row["impl"])
    ]=idx

CRM_A_IDX=[]
CRM_B_IDX=[]
CRM_PAIR_ROLE=[]

for row in crm_train_rows:
    if row["impl"]!="A":
        continue
    a=crm_lookup[
        (row["record_id"],row["role"],"A")
    ]
    b=crm_lookup[
        (row["record_id"],row["role"],"B")
    ]
    CRM_A_IDX.append(a)
    CRM_B_IDX.append(b)
    CRM_PAIR_ROLE.append(row["role"])

CRM_A_T=torch.tensor(
    CRM_A_IDX,dtype=torch.long
)
CRM_B_T=torch.tensor(
    CRM_B_IDX,dtype=torch.long
)

class CRMHead(nn.Module):
    def __init__(
        self,
        in_dim=384,
        hidden=128,
        zdim=64,
        dual=False
    ):
        super().__init__()
        self.dual_view=bool(dual)
        self.backbone=nn.Sequential(
            nn.Linear(in_dim,hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden,zdim),
        )
        self.heads=nn.ModuleDict({
            s:nn.Linear(
                zdim,
                len(crm_label_encoders[s].classes_)
            )
            for s in CRM_SLOTS
        })

    def forward(self,x):
        z=self.backbone(x)
        return z,{
            s:self.heads[s](z)
            for s in CRM_SLOTS
        }


class CRMScaleHead(nn.Module):
    """CRM transfer of the final hybrid SCALE architecture."""
    def __init__(
        self,
        in_dim=384,
        hidden=128,
        zdim=64,
        graph_dim=128
    ):
        super().__init__()
        self.dual_view=True

        # Closed-set discriminative branch.
        self.backbone=nn.Sequential(
            nn.Linear(in_dim,hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden,zdim),
        )
        self.closed_heads=nn.ModuleDict({
            s:nn.Linear(
                zdim,
                len(crm_label_encoders[s].classes_)
            )
            for s in CRM_SLOTS
        })

        # Relational prototype auxiliary branch.
        self.slot_proj=nn.ModuleDict({
            s:nn.Sequential(
                nn.Linear(in_dim,graph_dim),
                nn.GELU(),
                nn.LayerNorm(graph_dim),
            )
            for s in CRM_SLOTS
        })
        self.rgcn=ResidualRGCN(
            graph_dim,
            CRM_RELATIONS
        )
        self.graph_gate=nn.ParameterDict({
            s:nn.Parameter(
                torch.tensor(float(GRAPH_GATE_INIT))
            )
            for s in CRM_SLOTS
        })
        self.log_tau=nn.Parameter(
            torch.tensor(math.log(.08))
        )

    def forward(self,x):
        z=self.backbone(x)
        return z,{
            s:self.closed_heads[s](z)
            for s in CRM_SLOTS
        }

    def all_nodes(self,slot):
        feats=CRM_GRAPH[
            "features"
        ].to(next(self.parameters()).device)

        text=self.slot_proj[slot](feats)
        delta=self.rgcn(
            text,
            CRM_GRAPH["rel_adj"]
        )
        graph=(
            text
            +torch.sigmoid(
                self.graph_gate[slot]
            )*delta
        )
        return graph,text

    def open_logits(self,x):
        logits={}
        tau=torch.exp(
            self.log_tau
        ).clamp(.03,.50)

        for slot in CRM_SLOTS:
            q=F.normalize(
                self.slot_proj[slot](x),
                p=2,dim=-1
            )
            nodes,_=self.all_nodes(slot)
            ids=torch.tensor(
                CRM_GRAPH[
                    "class_indices"
                ][slot],
                dtype=torch.long,
                device=x.device
            )
            proto=F.normalize(
                nodes[ids],
                p=2,dim=-1
            )

            # Correct matrix orientation:
            # [batch,dim] @ [dim,n_classes]
            logits[slot]=(q@proto.T)/tau

        return logits

    def anchor_loss(self):
        vals=[]

        for slot in CRM_SLOTS:
            graph,text=self.all_nodes(slot)
            ids=torch.tensor(
                CRM_GRAPH[
                    "class_indices"
                ][slot],
                dtype=torch.long,
                device=graph.device
            )
            vals.append(
                (
                    1-F.cosine_similarity(
                        F.normalize(
                            graph[ids],
                            p=2,dim=-1
                        ),
                        F.normalize(
                            text[ids].detach(),
                            p=2,dim=-1
                        ),
                        dim=-1
                    )
                ).mean()
            )

        return torch.stack(vals).mean()

def crm_masks(seed):
    rng=np.random.default_rng(seed)
    masks={
        s:torch.zeros(
            len(crm_train_rows)
        )
        for s in CRM_SLOTS
    }

    for i,row in enumerate(crm_train_rows):
        active=CRM_ACTIVE_SLOT[
            row["role"]
        ]

        if row["impl"]=="A":
            masks[active][i]=1.
        elif rng.random()<CRM_B_LABEL_VISIBILITY:
            masks[active][i]=1.

    return masks

def crm_directed_invariance(logits,device):
    a=CRM_A_T.to(device)
    b=CRM_B_T.to(device)
    vals=[]

    for role in CRM_ROLES:
        slot=CRM_ACTIVE_SLOT[role]
        mask=torch.tensor(
            [r==role for r in CRM_PAIR_ROLE],
            dtype=torch.bool,
            device=device
        )

        aa=a[mask]
        bb=b[mask]

        teacher=F.softmax(
            logits[slot][aa].detach(),
            dim=-1
        )
        student=F.log_softmax(
            logits[slot][bb],
            dim=-1
        )
        vals.append(
            F.kl_div(
                student,
                teacher,
                reduction="batchmean"
            )
        )

    return torch.stack(vals).mean()

def crm_directed_latent(z,device):
    a=CRM_A_T.to(device)
    b=CRM_B_T.to(device)
    za=F.normalize(
        z[a].detach(),
        p=2,dim=-1
    )
    zb=F.normalize(
        z[b],
        p=2,dim=-1
    )
    return (
        1-F.cosine_similarity(
            za,zb,dim=-1
        )
    ).mean()

def crm_proto_aux(model,x,y,masks):
    if not isinstance(
        model,CRMScaleHead
    ):
        return x.new_tensor(0.)

    logits=model.open_logits(x)
    vals=[]

    for slot in CRM_SLOTS:
        vals.append(
            masked_ce(
                logits[slot],
                y[slot],
                masks[slot]
            )
        )

    return torch.stack(vals).mean()

def train_crm_adapter(kind,seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    if kind=="CB-CRM":
        model=CRMHead(dual=False)
        x=crm_raw_X.to(HEAD_DEVICE)
    elif kind=="CB-Dual-CRM":
        model=CRMHead(dual=True)
        x=crm_dual_X.to(HEAD_DEVICE)
    elif kind=="SCALE-CRM":
        model=CRMScaleHead()
        x=crm_dual_X.to(HEAD_DEVICE)
    else:
        raise ValueError(kind)

    model=model.to(HEAD_DEVICE)
    y={
        s:crm_y[s].to(HEAD_DEVICE)
        for s in CRM_SLOTS
    }
    masks=crm_masks(seed)

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-4
    )

    hist=[]

    for ep in range(CRM_HEAD_EPOCHS):
        model.train()
        z,logits=model(x)

        supervised=[]

        for role in CRM_ROLES:
            slot=CRM_ACTIVE_SLOT[role]
            supervised.append(
                masked_ce(
                    logits[slot],
                    y[slot],
                    masks[slot]
                )
            )

        lsup=torch.stack(
            supervised
        ).mean()

        linv=(
            crm_directed_invariance(
                logits,HEAD_DEVICE
            )
            if kind=="SCALE-CRM"
            else z.new_tensor(0.)
        )

        llat=(
            crm_directed_latent(
                z,HEAD_DEVICE
            )
            if kind=="SCALE-CRM"
            else z.new_tensor(0.)
        )

        lproto=crm_proto_aux(
            model,x,y,masks
        )

        lanchor=(
            model.anchor_loss()
            if kind=="SCALE-CRM"
            else z.new_tensor(0.)
        )

        loss=lsup

        if kind=="SCALE-CRM":
            inv_weight=(
                BASE_INV_WEIGHT
                *(1-CRM_B_LABEL_VISIBILITY)
            )
            loss += inv_weight*linv
            loss += (
                DIRECTED_LATENT_WEIGHT
                *inv_weight
                *llat
            )
            loss += PROTO_AUX_WEIGHT*lproto
            loss += GRAPH_ANCHOR_WEIGHT*lanchor

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),1.
        )
        opt.step()

        if ep in {
            0,9,19,CRM_HEAD_EPOCHS-1
        }:
            hist.append({
                "kind":kind,
                "seed":seed,
                "epoch":ep+1,
                "loss":float(loss.detach()),
                "supervised":float(lsup.detach()),
                "directed_inv":float(linv.detach()),
                "directed_latent":float(llat.detach()),
                "proto_aux":float(lproto.detach()),
                "anchor":float(lanchor.detach()),
            })

    model=model.cpu().eval()
    # Safe even when this cell is executed independently.
    if "cleanup_gpu" in globals():
        cleanup_gpu()
    else:
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return model,hist

CRM_MODELS={
    "CB-CRM":[],
    "CB-Dual-CRM":[],
    "SCALE-CRM":[],
}
crm_history=[]

assert set(CRM_MODELS)=={"CB-CRM","CB-Dual-CRM","SCALE-CRM"}

# CRM shape smoke test catches matrix-orientation problems before the
# three-seed adapter training starts.
_crm_n=min(4,len(crm_train_rows))
assert _crm_n>0, "No CRM training rows were constructed."

with torch.inference_mode():
    _raw=crm_raw_X[:_crm_n]
    _dual=crm_dual_X[:_crm_n]

    _m=CRMHead(dual=False)
    _z,_l=_m(_raw)
    for _slot in CRM_SLOTS:
        assert _l[_slot].shape==(
            _crm_n,len(crm_label_encoders[_slot].classes_)
        )

    _m2=CRMHead(dual=True)
    _z,_l=_m2(_dual)
    for _slot in CRM_SLOTS:
        assert _l[_slot].shape==(
            _crm_n,len(crm_label_encoders[_slot].classes_)
        )

    _ms=CRMScaleHead()
    _z,_closed=_ms(_dual)
    _open=_ms.open_logits(_dual)
    for _slot in CRM_SLOTS:
        expected=(_crm_n,len(crm_label_encoders[_slot].classes_))
        assert _closed[_slot].shape==expected
        assert _open[_slot].shape==expected

del _m,_m2,_ms,_z,_l,_closed,_open,_raw,_dual
cleanup_gpu()
print("CRM adapter shape preflight PASS")

for seed in CRM_HEAD_SEEDS:
    for kind in CRM_MODELS:
        model,h=train_crm_adapter(
            kind,seed
        )
        CRM_MODELS[kind].append(model)
        crm_history.extend(h)
        print(
            "CRM adapter trained:",
            kind,seed
        )

pd.DataFrame(
    crm_history
).to_csv(
    CRM_ROOT/
    "crm_adapter_training_history.csv",
    index=False
)

print(
    "CRM training complete:",
    {k:len(v) for k,v in CRM_MODELS.items()}
)

# 36/39 — RQ5-A: held-out CRM semantic-interface transfer

def crm_single_predict(model,message,role):
    arr=(
        dual_view_encode([message])
        if getattr(model,"dual_view",False)
        else semantic_encode([message])
    )
    x=torch.tensor(arr,dtype=torch.float32)

    with torch.inference_mode():
        _,logits=model(x)

    slot=CRM_ACTIVE_SLOT[role]
    p=F.softmax(logits[slot],dim=-1)[0].cpu().numpy()
    idx=int(np.argmax(p))

    return (
        str(crm_label_encoders[slot].inverse_transform([idx])[0]),
        p
    )

def crm_ensemble_predict(kind,message,role):
    votes=[]
    probs=[]

    for model in CRM_MODELS[kind]:
        label,p=crm_single_predict(model,message,role)
        votes.append(label)
        probs.append(p)

    mean_p=np.mean(probs,axis=0)
    idx=int(np.argmax(mean_p))
    label=str(
        crm_label_encoders[
            CRM_ACTIVE_SLOT[role]
        ].inverse_transform([idx])[0]
    )
    return label,mean_p


# Validate every learned CRM adapter before full semantic evaluation.
assert len(crm_test_pool)>0, "CRM semantic test pool is empty."
_crm_probe=next(
    r for r in crm_test_pool
    if crm_contract(r)["task"] in known_tasks
)
for _kind in ["CB-CRM","CB-Dual-CRM","SCALE-CRM"]:
    assert _kind in CRM_MODELS and len(CRM_MODELS[_kind])==len(CRM_HEAD_SEEDS), (
        f"Incomplete CRM model ensemble: {_kind}"
    )
    for _role in CRM_ROLES:
        _label,_p=crm_ensemble_predict(
            _kind,
            crm_role_message(_crm_probe,_role,"C2"),
            _role
        )
        assert _label in set(
            crm_label_encoders[CRM_ACTIVE_SLOT[_role]].classes_
        )
        assert np.isfinite(_p).all()
print("CRM semantic-prediction preflight PASS")

crm_sem_rows=[]

for rec in crm_test_pool:
    if crm_contract(rec)["task"] not in known_tasks:
        continue

    gold=crm_contract(rec)

    for impl in ["A","B","C2"]:
        for role in CRM_ROLES:
            msg=crm_role_message(rec,role,impl)
            slot=CRM_ACTIVE_SLOT[role]

            for kind in ["CB-CRM","CB-Dual-CRM","SCALE-CRM"]:
                pred,_=crm_ensemble_predict(kind,msg,role)

                crm_sem_rows.append({
                    "record_id":rec["record_id"],
                    "org":rec["org"],
                    "task":rec.get("task"),
                    "implementation":impl,
                    "role":role,
                    "slot":slot,
                    "method":kind,
                    "correct":int(pred==gold[slot]),
                    "predicted":pred,
                    "gold":gold[slot],
                })

crm_sem_df=pd.DataFrame(crm_sem_rows)

crm_sem_summary=(
    crm_sem_df.groupby(
        ["method","implementation","role"],
        as_index=False
    )
    .agg(
        semantic_accuracy=("correct","mean"),
        n=("correct","size"),
    )
)

display(crm_sem_summary.round(4))

crm_sem_df.to_csv(
    CRM_ROOT/"crm_semantic_transfer_results.csv",
    index=False
)
crm_sem_summary.to_csv(
    CRM_ROOT/"crm_semantic_transfer_summary.csv",
    index=False
)

# 37/39 — RQ5-B interface construction and offline ground-truth scorer

CRM_EXECUTOR_SYSTEM="""You are the final executor in an offline CRM benchmark.

Use ONLY:
- the user's public benchmark query,
- the public benchmark persona/metadata,
- the supplied inter-agent handoffs,
- and, when present, retrieved CRM schema descriptions.

Do not assume access to hidden Salesforce records.

If the POLICY handoff states that disclosure is restricted, refuse disclosure.

Return exactly:
{"answer":["item 1","item 2"]}

For a refusal return:
{"answer":["REFUSE"]}

No explanation and no markdown."""

def crm_predicted_contract(rec,comp,kind):
    out={}
    for role in CRM_ROLES:
        msg=crm_role_message(rec,role,comp[role])
        slot=CRM_ACTIVE_SLOT[role]
        label,_=crm_ensemble_predict(kind,msg,role)
        out[slot]=label
    return out

def crm_interface(rec,replacement_level,method):
    comp=crm_composition(rec,replacement_level)
    packets=[]

    if method in {"CB-CRM","CB-Dual-CRM","SCALE-CRM"}:
        pred=crm_predicted_contract(rec,comp,method)

        packets=[
            "[PLANNER]\n"+compact_json({
                "role":"PLANNER",
                "active":{"task":pred["task"]}
            }),
            "[RETRIEVER]\n"+compact_json({
                "role":"RETRIEVER",
                "active":{"reward":pred["reward"]}
            }),
            "[POLICY]\n"+compact_json({
                "role":"POLICY",
                "active":{"privacy":pred["privacy"]}
            }),
        ]

    else:
        for role in CRM_ROLES:
            raw=crm_role_message(rec,role,comp[role])

            if method=="NL":
                payload=raw
            elif method=="JSON":
                payload=compact_json({
                    "role":role,
                    "implementation":comp[role],
                    "message":raw,
                })
            elif method=="Onto-RAG":
                payload=raw
            else:
                raise ValueError(method)

            packets.append(f"[{role}]\n{payload}")

    extra=""
    if method=="Onto-RAG":
        hits=crm_schema_retrieve(rec)
        extra="\n\nRETRIEVED CRM SCHEMA:\n"+compact_json(hits)

    return (
        "\n\n".join(packets)+extra
    )

def crm_executor_messages(rec,replacement_level,method):
    return [
        {"role":"system","content":CRM_EXECUTOR_SYSTEM},
        {"role":"user","content":
            f"ORG={rec['org'].upper()}\n"
            f"INTERFACE={method}\n"
            f"REPLACEMENT_LEVEL={replacement_level}\n\n"
            f"PUBLIC CONTEXT:\n{crm_public_context(rec)}\n\n"
            f"INTER-AGENT HANDOFFS:\n"
            f"{crm_interface(rec,replacement_level,method)}"
        }
    ]

def parse_answer_output(text):
    obj=extract_json(text)
    if isinstance(obj,dict) and "answer" in obj:
        a=obj["answer"]
        if isinstance(a,list):
            return [str(x) for x in a],True
        return [str(a)],True
    return [],False

def token_f1(a,b):
    aa=norm_text(a).split()
    bb=norm_text(b).split()

    if not aa or not bb:
        return 0.0

    from collections import Counter
    ca,cb=Counter(aa),Counter(bb)
    common=sum((ca&cb).values())

    if common==0:
        return 0.0

    p=common/len(aa)
    r=common/len(bb)
    return 2*p*r/(p+r)

REFUSAL_MARKERS=[
    "refuse",
    "cannot provide",
    "can't provide",
    "not authorized",
    "unable to disclose",
    "cannot disclose",
    "restricted",
]

def crm_score(rec,pred_answers,raw_text):
    metric=str(rec.get("reward_metric","")).lower()
    gold=answer_list(rec.get("answer"))

    if metric=="privacy_rejection":
        blob=norm_text(
            " ".join(pred_answers)+" "+str(raw_text)
        )
        ok=(
            any(norm_text(x) in blob for x in REFUSAL_MARKERS)
            or any(norm_text(x)=="refuse" for x in pred_answers)
        )
        return float(ok),int(ok)

    pg=[norm_text(x) for x in pred_answers if norm_text(x)]
    gg=[norm_text(x) for x in gold if norm_text(x)]

    if metric=="exact_match":
        ok=(set(pg)==set(gg))
        return float(ok),int(ok)

    # Public benchmark's fuzzy metric is approximated here because the official
    # Salesforce evaluator is not available offline. The approximation and
    # threshold are fixed before the run.
    if metric=="fuzzy_match":
        if not gg or not pg:
            return 0.0,0

        vals=[]
        for g in gg:
            vals.append(max(token_f1(g,p) for p in pg))
        score=float(np.mean(vals))
        return score,int(score>=CRM_FUZZY_SUCCESS_THRESHOLD)

    # Unknown public reward metric: conservative exact-set fallback.
    ok=(set(pg)==set(gg))
    return float(ok),int(ok)

# Full end-to-end dispatch preflight before loading expensive CRM executors.
assert len(crm_test_records)>0, (
    "No context-grounded CRM test records remain after known-task filtering."
)
assert len(CRM_OFFLINE_METHODS)==len(set(CRM_OFFLINE_METHODS))
assert set(CRM_OFFLINE_METHODS)=={
    "NL","JSON","Onto-RAG","CB-CRM","CB-Dual-CRM","SCALE-CRM"
}

probe_rec=crm_test_records[0]

for _method in CRM_OFFLINE_METHODS:
    for _k in [0,1,2,3]:
        _msgs=crm_executor_messages(probe_rec,_k,_method)
        assert isinstance(_msgs,list) and len(_msgs)==2
        assert all(
            isinstance(m,dict) and "role" in m and "content" in m
            for m in _msgs
        )
        _prompt=_msgs[1]["content"]
        assert "GROUND_TRUTH" not in _prompt.upper()

# Scorer smoke tests on the real probe record.
_score,_ok=crm_score(
    probe_rec,
    answer_list(probe_rec.get("answer")),
    " ".join(answer_list(probe_rec.get("answer")))
)
assert np.isfinite(_score)
assert _ok in {0,1}

print(
    "Offline CRM executor preflight PASS | "
    f"{len(CRM_OFFLINE_METHODS)} methods x 4 replacement levels"
)

# 38/39 — RQ5-B: run the frozen offline external benchmark

CRM_RESULTS_FILE=CRM_ROOT/"crm_offline_results.jsonl"

CRM_REMOTE_RESULTS=None
if DRIVE_ROOT is not None:
    CRM_REMOTE_DIR=DRIVE_ROOT/"crmarena_pro_offline"
    CRM_REMOTE_DIR.mkdir(parents=True,exist_ok=True)
    CRM_REMOTE_RESULTS=CRM_REMOTE_DIR/CRM_RESULTS_FILE.name
    if not CRM_RESULTS_FILE.exists() and CRM_REMOTE_RESULTS.exists():
        shutil.copy2(CRM_REMOTE_RESULTS,CRM_RESULTS_FILE)

# If the prior 2304-generation run is available, recover it in place.
repair_jsonl_file(CRM_RESULTS_FILE)

def crm_done_keys():
    rows,_=safe_jsonl_records(CRM_RESULTS_FILE)
    return {
        r["run_key"]
        for r in rows
        if r.get("run_key")
    }

done=crm_done_keys()
total_planned=(
    len(CRM_EXECUTOR_SPECS)
    *len(crm_test_records)
    *len(CRM_OFFLINE_METHODS)
    *4
)

print(
    "Offline CRM generations planned:",
    total_planned,
    "| already complete:",
    len(done)
)

completed=0

if RUN_CRM_OFFLINE:
    for spec in CRM_EXECUTOR_SPECS:
        try:
            active_model, primary, tok, mdl = load_model_safe(spec)
        except Exception as exc:
            print(
                "CRM EXECUTOR SKIPPED:",
                spec["label"],
                "|",
                type(exc).__name__,
                str(exc)[:240],
            )
            continue

        print(
            "\nCRM EXECUTOR:",
            spec["label"],
            "|",active_model
        )

        for rec in crm_test_records:
            for replacement_level in [0,1,2,3]:
                for method in CRM_OFFLINE_METHODS:
                    key=(
                        f"{EXPERIMENT_VERSION}|{CONFIG_HASH}|"
                        f"CRM-OFFLINE|{CRM_SOURCE_REVISION}|"
                        f"{spec['label']}|{rec['record_id']}|"
                        f"{replacement_level}|{method}"
                    )

                    if key in done:
                        continue

                    error=None
                    try:
                        gen=generate_text(
                            tok,mdl,
                            crm_executor_messages(
                                rec,replacement_level,method
                            ),
                            spec,
                            CRM_MAX_INPUT,
                            CRM_MAX_OUTPUT
                        )
                    except torch.cuda.OutOfMemoryError:
                        cleanup_gpu()
                        gen={
                            "text":"",
                            "input_tokens":0,
                            "output_tokens":0,
                            "latency_s":0,
                            "overflow":False,
                        }
                        error="OOM"
                    except Exception as exc:
                        gen={
                            "text":"",
                            "input_tokens":0,
                            "output_tokens":0,
                            "latency_s":0,
                            "overflow":False,
                        }
                        error=(
                            f"{type(exc).__name__}:"
                            f"{str(exc)[:160]}"
                        )

                    pred,parse_ok=parse_answer_output(gen["text"])
                    score,task_ok=crm_score(
                        rec,pred,gen["text"]
                    )

                    row={
                        "run_key":key,
                        "experiment_version":EXPERIMENT_VERSION,
                        "config_hash":CONFIG_HASH,
                        "crm_source_revision":CRM_SOURCE_REVISION,
                        "executor_model":spec["label"],
                        "active_model_id":active_model,
                        "primary_model_loaded":bool(primary),
                        "record_id":rec["record_id"],
                        "org":rec["org"],
                        "task":rec.get("task"),
                        "reward_metric":rec.get("reward_metric"),
                        "method":method,
                        "replacement_level":replacement_level,
                        "composition":crm_composition(
                            rec,replacement_level
                        ),
                        "parse_success":int(parse_ok),
                        "reward_score":score,
                        "task_success":task_ok,
                        "predicted_answer":pred,
                        "gold_answer":answer_list(rec.get("answer")),
                        "input_tokens":gen["input_tokens"],
                        "output_tokens":gen["output_tokens"],
                        "latency_s":float(gen["latency_s"]),
                        "overflow":int(gen["overflow"]),
                        "error":error,
                    }

                    append_jsonl(CRM_RESULTS_FILE,row)

                    if (
                        CRM_REMOTE_RESULTS is not None
                        and (completed+1)%50==0
                    ):
                        sync_result_file(
                            CRM_RESULTS_FILE,
                            CRM_REMOTE_RESULTS
                        )

                    done.add(key)
                    completed+=1

                    if completed%50==0:
                        print(
                            f"CRM progress: "
                            f"{len(done)}/{total_planned}"
                        )

        del mdl,tok
        cleanup_gpu()

    print("Offline CRM benchmark complete.")
else:
    print("RUN_CRM_OFFLINE=False")

# 39/39 — RQ5 offline summaries, compositional retention, paired statistics and report

repair_jsonl_file(CRM_RESULTS_FILE)
crm_results=safe_read_jsonl_df(CRM_RESULTS_FILE)

if len(crm_results):
    # Scientific summaries use successfully loaded primary models only.
    sci=crm_results[
        crm_results["primary_model_loaded"]==True
    ].copy()

    crm_summary=(
        sci.groupby(
            ["executor_model","method","replacement_level"],
            as_index=False
        )
        .agg(
            n=("task_success","size"),
            task_success=("task_success","mean"),
            reward_score=("reward_score","mean"),
            parse_success=("parse_success","mean"),
            overflow_rate=("overflow","mean"),
            error_rate=("error",lambda x:x.notna().mean()),
            mean_input_tokens=("input_tokens","mean"),
            mean_output_tokens=("output_tokens","mean"),
            mean_latency_s=("latency_s","mean"),
        )
    )

    display(crm_summary.round(4))

    # Compositional retention: shifted k=3 / IID k=0.
    retention=[]
    for model in crm_summary.executor_model.unique():
        for method in crm_summary.method.unique():
            g=crm_summary[
                (crm_summary.executor_model==model)
                &(crm_summary.method==method)
            ].set_index("replacement_level")

            if 0 in g.index and 3 in g.index:
                iid=float(g.loc[0,"task_success"])
                shifted=float(g.loc[3,"task_success"])
                retention.append({
                    "executor_model":model,
                    "method":method,
                    "iid_success":iid,
                    "k3_success":shifted,
                    "compositional_retention":(
                        shifted/iid if iid>0 else np.nan
                    ),
                })

    crm_retention=pd.DataFrame(retention)
    display(crm_retention.round(4))

    # Paired bootstrap on per-record SCALE-vs-baseline differences.
    rng=np.random.default_rng(2026)
    paired=[]

    for model in sci.executor_model.unique():
        for k in [0,1,2,3]:
            sub=sci[
                (sci.executor_model==model)
                &(sci.replacement_level==k)
            ]

            scale=sub[sub.method=="SCALE-CRM"][
                ["record_id","task_success"]
            ].rename(columns={"task_success":"scale"})

            for baseline in [
                "NL","JSON","Onto-RAG",
                "CB-CRM","CB-Dual-CRM"
            ]:
                b=sub[sub.method==baseline][
                    ["record_id","task_success"]
                ].rename(columns={"task_success":"base"})

                m=scale.merge(b,on="record_id")
                if not len(m):
                    continue

                d=(m["scale"]-m["base"]).to_numpy(float)
                boots=[]

                for _ in range(3000):
                    ids=rng.integers(0,len(d),len(d))
                    boots.append(float(d[ids].mean()))

                paired.append({
                    "executor_model":model,
                    "replacement_level":k,
                    "baseline":baseline,
                    "n":len(d),
                    "mean_difference":float(d.mean()),
                    "ci_low":float(np.quantile(boots,.025)),
                    "ci_high":float(np.quantile(boots,.975)),
                })

    crm_paired=pd.DataFrame(paired)

    # Conservative RQ5 offline gate.
    primary_models=sorted(sci.executor_model.unique())

    technical=(
        crm_summary.parse_success.min()>=.95
        and crm_summary.error_rate.max()<.01
        and crm_summary.overflow_rate.max()<.01
    )

    scale_ret=crm_retention[
        crm_retention.method=="SCALE-CRM"
    ]

    nl_ret=crm_retention[
        crm_retention.method=="NL"
    ]
    json_ret=crm_retention[
        crm_retention.method=="JSON"
    ]

    retention_ok=False
    if (
        len(primary_models)>=2
        and len(scale_ret)
        and len(nl_ret)
        and len(json_ret)
    ):
        sr=scale_ret.groupby("executor_model")[
            "compositional_retention"
        ].mean()
        nr=nl_ret.groupby("executor_model")[
            "compositional_retention"
        ].mean()
        jr=json_ret.groupby("executor_model")[
            "compositional_retention"
        ].mean()

        common=sorted(
            set(sr.index)&set(nr.index)&set(jr.index)
        )
        retention_ok=(
            len(common)>=2
            and all(
                sr[m]>max(nr[m],jr[m])
                for m in common
            )
        )

    # At k=3 SCALE must remain non-inferior to the strongest CRM learned baseline.
    shifted=sci[
        sci.replacement_level==3
    ]
    shifted_avg=shifted.groupby(
        "method"
    )["task_success"].mean()

    learned_refs=[
        m for m in [
            "CB-CRM","CB-Dual-CRM"
        ]
        if m in shifted_avg.index
    ]

    cb_noninferior=(
        "SCALE-CRM" in shifted_avg.index
        and len(learned_refs)>0
        and shifted_avg["SCALE-CRM"]
            >=max(
                shifted_avg[m]
                for m in learned_refs
            )-CRM_NONINFERIOR_MARGIN
    )

    RQ5_OFFLINE_PASS=(
        technical
        and retention_ok
        and cb_noninferior
    )

    crm_summary.to_csv(
        CRM_ROOT/"crm_offline_summary.csv",
        index=False
    )
    crm_retention.to_csv(
        CRM_ROOT/"crm_compositional_retention.csv",
        index=False
    )
    crm_paired.to_csv(
        CRM_ROOT/"crm_paired_bootstrap.csv",
        index=False
    )

    # Paper-ready external-validity robustness figure.
    import matplotlib.pyplot as plt

    fig,ax=plt.subplots(figsize=(7.4,4.6))
    avg=(
        crm_summary.groupby(
            ["method","replacement_level"],
            as_index=False
        )
        .agg(task_success=("task_success","mean"))
    )

    for method in CRM_OFFLINE_METHODS:
        g=avg[
            avg.method==method
        ].sort_values("replacement_level")
        if len(g):
            ax.plot(
                g.replacement_level,
                g.task_success,
                marker="o",
                label=method
            )

    ax.set_xlabel("Number of replaced CRM agents")
    ax.set_ylabel("Offline task success")
    ax.set_xticks([0,1,2,3])
    ax.set_ylim(-.03,1.03)
    ax.legend(fontsize=8,ncol=2)
    ax.set_title(
        "CRMArena-Pro offline context-grounded external validity"
    )
    fig.tight_layout()

    fig_path=ROOT/"figures"/"crm_offline_robustness.png"
    fig.savefig(fig_path,dpi=220)
    plt.show()

    report=f"""
# RQ5 — CRMArena-Pro Offline External Validity

Official source:
- repository: `{CRM_SOURCE_REPO}`
- revision: `{CRM_SOURCE_REVISION}`

Evaluation type:
**offline dataset-grounded external-validity subset**

No Salesforce credentials or live Salesforce org were used.

Public records downloaded: **{len(raw_records)}**

Public-context-supported held-out tasks tested: **{len(crm_test_records)}**

Primary executor families: **{", ".join(primary_models)}**

## Scientific boundary

This is **not** the official live CRMArena-Pro Salesforce environment score.

Non-privacy tasks were admitted only when their ground-truth answer was already supported
by the exact public `query + persona + metadata` context provided to the model.
Privacy-rejection tasks were evaluated as refusal tasks.

The official live environment remains a separate future extension.

## Offline RQ5 checks

- technical validity: **{"PASS" if technical else "FAIL"}**
- SCALE retention > NL/JSON in both primary executors: **{"PASS" if retention_ok else "FAIL / INCONCLUSIVE"}**
- SCALE k=3 non-inferior to strongest CRM learned baseline: **{"PASS" if cb_noninferior else "FAIL / INCONCLUSIVE"}**

Overall offline RQ5:
**{"SUPPORTED" if RQ5_OFFLINE_PASS else "NOT YET FULLY SUPPORTED"}**

## Paper wording if supported

> On a frozen, context-grounded subset of the public CRMArena-Pro release, SCALE preserved
> task success more effectively under progressive constituent replacement than raw natural-language
> and fixed-JSON communication, while remaining competitive with a CRM-specific learned bottleneck.

Do not describe this result as an official CRMArena-Pro environment score.
"""

    (CRM_ROOT/"RQ5_OFFLINE_REPORT.md").write_text(
        report.strip()+"\n",
        encoding="utf-8"
    )

    from IPython.display import Markdown,display
    display(Markdown(report))

    if DRIVE_ROOT is not None:
        try:
            dest=DRIVE_ROOT/"crmarena_pro_offline"
            dest.mkdir(exist_ok=True)

            for p in CRM_ROOT.iterdir():
                if p.is_file() and p.stat().st_size<80_000_000:
                    shutil.copy2(p,dest/p.name)

            if fig_path.exists():
                shutil.copy2(
                    fig_path,
                    dest/fig_path.name
                )
        except Exception as exc:
            print(
                "CRM offline sync warning:",
                type(exc).__name__,
                str(exc)[:120]
            )
else:
    print("No CRM offline results were produced.")

# Final methodological changes made in this notebook

The final notebook incorporates the evidence from all prior experimental iterations rather than preserving unsupported complexity.

## 1. C2 schema/tool robustness

**Problem found:** a pure ontology-prototype decoder was excellent for natural paraphrases and ontology extension but weaker than a discriminative classifier on schema/tool OOD.

**Final solution:** SCALE is hybrid.

```text
known semantic class
→ discriminative dual-view decoder

unseen ontology class
→ graph-prototype open-world decoder
→ ontology canonicalization
```

This keeps the strength of `CB+DualView` where closed-set discrimination is appropriate and keeps SCALE's graph mechanism where structural ontology extrapolation is required.

## 2. Invariance

**Problem found:** symmetric invariance could unnecessarily move the well-supervised A representation.

**Final solution:** directed A→B stop-gradient invariance.

A acts as the semantic teacher; B adapts toward A.

## 3. Learned logic

**Problem found:** `L_logic` showed no measurable independent benefit.

**Final solution:** remove it from Full SCALE's primary objective and retain it only as a diagnostic.

The deterministic ontology validator/repair/block layer remains because its stress-test contribution was strongly supported.

## 4. Reader gates

**Problem found:** a semantically invalid label was being counted as a parser/technical failure.

**Final solution:** separate:

```text
technical parse validity
semantic label validity
```

G0 is now genuinely technical.

G1 evaluates the machine-readable semantic interface itself; cross-reader behavior remains separately reported.

## 5. Confidence

Active semantic correctness remains the calibration target because downstream agents consume role-active semantics rather than seven-slot exact contracts.

## 6. Drift observability

PrePolicy Active-SCD remains the primary RQ4 metric.

It does not use downstream policy outcome and therefore avoids target leakage.

## 7. CRMArena-Pro

The official public source is revision-pinned and downloaded automatically.

The previous CRM matrix bug is fixed.

RQ5 now includes:

```text
CB-CRM
CB-Dual-CRM
SCALE-CRM
```

so SCALE is compared against a CRM-specific dual-view discriminative control, not only a weaker raw-CB baseline.

The final RQ5 remains explicitly **offline dataset-grounded external validity**, not an official live Salesforce environment score.

# Recovery + final-analysis patch

This notebook contains two categories of changes.

## Technical corrections

- Correct JSONL serialization uses a real newline rather than the literal characters `\n`.
- Existing malformed reader / producer / CRM result streams are decoded with `JSONDecoder.raw_decode`, so escaped newlines inside generated text are preserved safely.
- Legacy files are backed up before canonical rewriting.
- Reader, producer, and CRM runs are resumable from recovered result keys.
- Producer and CRM result files are checkpointed to Drive.
- A two-record serialization preflight runs before expensive stages.

These corrections do **not** change any model prediction, gate threshold, task label, or metric.

## Statistical / baseline strengthening

- The label-efficiency experiment now uses all five paired seeds rather than three.
- `SCALE-NoOnt` is added to the label-efficiency curve as a compatibility-only control: it retains directed implementation invariance but removes ontology graph propagation.
- Seed-level summaries and paired bootstrap confidence intervals are exported.
- Non-monotonic label-efficiency behavior is explicitly flagged rather than hidden.

The primary G2 gate is intentionally unchanged. These additions improve inference about the mechanism without moving the success criterion after observing results.

# Final robustness additions

This version adds three reviewer-oriented mechanism studies without altering the
original G0–G7 success gates.

## A. Semantic intervention faithfulness
Measures whether correcting role-active semantic concepts predictably repairs downstream
task behavior and whether inactive-field perturbations leave the active interface unchanged.

## B. Large hierarchical ontology extension
Expands the ontology test to 100 unseen opaque nodes across three hierarchy depths and
adds exact/normalized symbolic ancestor controls to prevent overclaiming a learned advantage
when an explicit ontology identifier alone is sufficient.

## C. Leave-one-drift-family-out observability
Selects operating thresholds using other drift families, then evaluates a completely held-out
single-operation family. Active-SCD is compared with full SCD, confidence, and raw BGE message drift.

These studies are exploratory robustness evidence. They are reported separately from the
prespecified primary gates and must be interpreted even if they produce negative results.

In [ ]:

# =============================================================================
# 40/40 — Reviewer-hardening continuation
# Run this ONE CELL only after the notebook above has completed.
#
# Adds four paper-facing tests without retraining the main SCALE heads:
#   T1. Deployable hybrid routing (explicit-ID -> symbolic; known -> CB; unseen -> ontology)
#   T2. PrePolicy Active-SCD alpha sensitivity
#   T3. Independent learned downstream executors (raw/dual-view BGE -> task decision)
#   T4. Hard semantic-alias stress test
#
# Outputs are written under ROOT/"reviewer_hardening".
# =============================================================================

from pathlib import Path
import copy, hashlib, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)

import matplotlib.pyplot as plt

REVIEW_DIR = ROOT / "reviewer_hardening"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("SCALE REVIEWER-HARDENING TESTS")
print("Output:", REVIEW_DIR)
print("=" * 88)

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def _stable_group_fold(groups, n_splits=5):
    groups = np.asarray(groups, dtype=object)
    unique = np.unique(groups)
    return GroupKFold(n_splits=min(n_splits, len(unique)))

def _safe_auc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

def _best_balanced_threshold(y, score):
    y = np.asarray(y, int)
    score = np.asarray(score, float)
    if len(np.unique(y)) < 2:
        return 0.5, np.nan
    vals = np.unique(score)
    if len(vals) > 250:
        vals = np.quantile(score, np.linspace(0, 1, 250))
    best_t, best_b = 0.5, -1.0
    for t in vals:
        pred = (score >= t).astype(int)
        b = balanced_accuracy_score(y, pred)
        if b > best_b:
            best_t, best_b = float(t), float(b)
    return best_t, float(best_b)

def _mean_scale_dynamic_scores(text, graph_ctx):
    """
    Returns mean prototype scores over SCALE ensemble members for all object/object_leaf
    candidates in graph_ctx. Uses existing trained SCALE members only.
    """
    candidate_keys = [
        k for k in graph_ctx["node_keys"]
        if k[0] in {"object", "object_leaf"}
    ]

    per_member = []
    for member in MODELS["SCALE"]:
        use_dual = bool(getattr(member, "dual_view", False))
        arr = dual_view_encode([text]) if use_dual else semantic_encode([text])
        x = torch.tensor(arr, dtype=torch.float32)

        with torch.inference_mode():
            q = member.query(x, "object")[0]
            all_nodes, _ = member.all_node_repr(graph_ctx, "object")

        ids = torch.tensor(
            [graph_ctx["idx"][k] for k in candidate_keys],
            dtype=torch.long
        )
        proto = F.normalize(all_nodes[ids], p=2, dim=-1)
        per_member.append((q @ proto.T).detach().cpu().numpy())

    return candidate_keys, np.mean(np.stack(per_member, axis=0), axis=0)

def _router_features(text):
    """
    Open-set routing features:
      - closed-set CB+DualView object confidence
      - closed-set top1-top2 margin
      - ontology score advantage of unseen leaves over known roots
      - best unseen ontology score
    """
    cb = ensemble_predict("CB+DualView", [text], BASE_GRAPH)[0]
    p = np.asarray(cb["probs"]["object"], float)
    order = np.sort(p)
    cb_max = float(order[-1])
    cb_margin = float(order[-1] - order[-2]) if len(order) > 1 else cb_max

    keys, scores = _mean_scale_dynamic_scores(text, LARGE_OPAQUE_GRAPH)
    known_scores = [
        float(s) for k, s in zip(keys, scores)
        if k[0] == "object"
    ]
    unseen_scores = [
        float(s) for k, s in zip(keys, scores)
        if k[0] == "object_leaf"
        and k[1] in LARGE_NODE_BY_LABEL
    ]

    best_known = max(known_scores) if known_scores else -1.0
    best_unseen = max(unseen_scores) if unseen_scores else -1.0

    return np.array([
        cb_max,
        cb_margin,
        best_unseen - best_known,
        best_unseen,
    ], dtype=float)

# =============================================================================
# T1 — DEPLOYABLE HYBRID ROUTING
# =============================================================================
#
# Important design:
#   1) If an explicit ontology identifier is recoverable, deterministic normalization wins.
#   2) Otherwise a learned open-set router decides known vs unseen semantics.
#   3) Known -> CB+DualView
#      Unseen -> SCALE graph-prototype + ontology canonicalization
#
# This removes oracle branch selection from the paper architecture.
# =============================================================================

router_rows = []

# Known closed-set planner messages. Group by base_id so variants of the same task
# never appear in both train and test folds.
for base in shift_test_base:
    for impl in ["B", "C1", "C2"]:
        text = realize_message(base, "PLANNER", impl, 0)
        router_rows.append({
            "kind": "known",
            "is_unseen": 0,
            "group": f"known::{base['base_id']}",
            "gold_root": base["object"],
            "surface": impl,
            "text": text,
            "explicit_id_hit": None,
        })

# Unseen semantic aliases. Group by canonical root, not node, so identical/root-specific
# semantic aliases cannot leak across folds.
for node in LARGE_OPAQUE_NODES:
    text = large_ontology_message(node, "semantic_alias")
    router_rows.append({
        "kind": "unseen",
        "is_unseen": 1,
        "group": f"unseen_root::{node['canonical_root']}",
        "gold_root": node["canonical_root"],
        "surface": "semantic_alias",
        "text": text,
        "explicit_id_hit": None,
    })

router_df = pd.DataFrame(router_rows)

feat = np.stack([_router_features(t) for t in router_df["text"]])
router_df["cb_maxprob"] = feat[:, 0]
router_df["cb_margin"] = feat[:, 1]
router_df["onto_advantage"] = feat[:, 2]
router_df["best_unseen_score"] = feat[:, 3]

X = feat
y = router_df["is_unseen"].to_numpy(int)
groups = router_df["group"].to_numpy(object)

router_df["p_unseen_oof"] = np.nan

gkf = _stable_group_fold(groups, n_splits=5)
for tr, te in gkf.split(X, y, groups):
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        )),
    ])
    pipe.fit(X[tr], y[tr])
    router_df.loc[te, "p_unseen_oof"] = pipe.predict_proba(X[te])[:, 1]

valid = router_df["p_unseen_oof"].notna()
router_auc = _safe_auc(
    router_df.loc[valid, "is_unseen"],
    router_df.loc[valid, "p_unseen_oof"]
)

router_threshold, router_balacc = _best_balanced_threshold(
    router_df.loc[valid, "is_unseen"],
    router_df.loc[valid, "p_unseen_oof"]
)

router_pred = (
    router_df.loc[valid, "p_unseen_oof"].to_numpy(float)
    >= router_threshold
).astype(int)

router_f1 = f1_score(
    router_df.loc[valid, "is_unseen"].to_numpy(int),
    router_pred
)
known_false_open = float(
    router_pred[
        router_df.loc[valid, "is_unseen"].to_numpy(int) == 0
    ].mean()
)
unseen_recall = float(
    router_pred[
        router_df.loc[valid, "is_unseen"].to_numpy(int) == 1
    ].mean()
)

router_metrics = pd.DataFrame([{
    "router_auc": router_auc,
    "selected_threshold": router_threshold,
    "balanced_accuracy": router_balacc,
    "f1_unseen": router_f1,
    "known_false_open_rate": known_false_open,
    "unseen_recall": unseen_recall,
    "n": int(valid.sum()),
    "n_groups": int(router_df.loc[valid, "group"].nunique()),
}])

# Threshold sensitivity (OOF only).
threshold_rows = []
for t in np.linspace(0.05, 0.95, 19):
    yy = router_df.loc[valid, "is_unseen"].to_numpy(int)
    pp = router_df.loc[valid, "p_unseen_oof"].to_numpy(float)
    pred = (pp >= t).astype(int)
    threshold_rows.append({
        "threshold": float(t),
        "balanced_accuracy": float(balanced_accuracy_score(yy, pred)),
        "f1_unseen": float(f1_score(yy, pred)),
        "known_false_open_rate": float(pred[yy == 0].mean()),
        "unseen_recall": float(pred[yy == 1].mean()),
    })
router_threshold_df = pd.DataFrame(threshold_rows)

# Fit final router on all non-ID calibration rows only for a deployable callable.
FINAL_ROUTER = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    )),
])
FINAL_ROUTER.fit(X, y)

def scale_hybrid_route(text, graph_ctx=LARGE_OPAQUE_GRAPH, threshold=None):
    """
    Deployable three-way route:
      explicit ontology ID -> deterministic normalization/canonicalization
      no ID + known semantics -> CB+DualView
      no ID + unseen semantics -> SCALE open-world ontology branch
    """
    if threshold is None:
        threshold = router_threshold

    # First use the mechanism proven strongest for recoverable IDs.
    id_hit = normalized_id_lookup(text, graph_ctx)
    if id_hit is not None:
        return {
            "route": "symbolic_id",
            "predicted_node": id_hit,
            "canonical_root": canonicalize_lookup(id_hit, graph_ctx),
            "p_unseen": 1.0,
        }

    p_unseen = float(
        FINAL_ROUTER.predict_proba(
            _router_features(text).reshape(1, -1)
        )[0, 1]
    )

    if p_unseen >= threshold:
        node, root = learned_large_prediction(
            "SCALE-Full", text, graph_ctx
        )
        return {
            "route": "open_world_ontology",
            "predicted_node": node,
            "canonical_root": root,
            "p_unseen": p_unseen,
        }

    cb = ensemble_predict("CB+DualView", [text], BASE_GRAPH)[0]
    root = cb["contract"].get("object")
    return {
        "route": "known_discriminative",
        "predicted_node": root,
        "canonical_root": root,
        "p_unseen": p_unseen,
    }

# Mixed routing benchmark: known C1/C2 + all four unseen surface conditions.
hybrid_rows = []

for base in shift_test_base:
    for impl in ["C1", "C2"]:
        text = realize_message(base, "PLANNER", impl, 0)
        out = scale_hybrid_route(text)
        hybrid_rows.append({
            "kind": "known",
            "surface": impl,
            "gold_root": base["object"],
            **out,
            "correct": int(out["canonical_root"] == base["object"]),
        })

for node in LARGE_OPAQUE_NODES:
    for surface in LARGE_SURFACES:
        text = large_ontology_message(node, surface)
        out = scale_hybrid_route(text)
        hybrid_rows.append({
            "kind": "unseen",
            "surface": surface,
            "gold_root": node["canonical_root"],
            **out,
            "correct": int(out["canonical_root"] == node["canonical_root"]),
        })

hybrid_route_df = pd.DataFrame(hybrid_rows)
hybrid_route_summary = (
    hybrid_route_df
    .groupby(["kind", "surface", "route"], as_index=False)
    .agg(
        accuracy=("correct", "mean"),
        n=("correct", "size"),
        mean_p_unseen=("p_unseen", "mean"),
    )
)

print("\n[T1] OOF OPEN-SET ROUTER")
display(router_metrics.round(4))
display(router_threshold_df.round(4))
print("\n[T1] DEPLOYABLE THREE-WAY HYBRID ROUTING")
display(hybrid_route_summary.round(4))

router_df.to_csv(REVIEW_DIR / "router_oof_predictions.csv", index=False)
router_metrics.to_csv(REVIEW_DIR / "router_metrics.csv", index=False)
router_threshold_df.to_csv(REVIEW_DIR / "router_threshold_sensitivity.csv", index=False)
hybrid_route_df.to_csv(REVIEW_DIR / "hybrid_route_results.csv", index=False)
hybrid_route_summary.to_csv(REVIEW_DIR / "hybrid_route_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(
    router_threshold_df["threshold"],
    router_threshold_df["balanced_accuracy"],
    marker="o",
    label="Balanced accuracy"
)
ax.plot(
    router_threshold_df["threshold"],
    router_threshold_df["unseen_recall"],
    marker="o",
    label="Unseen recall"
)
ax.plot(
    router_threshold_df["threshold"],
    1 - router_threshold_df["known_false_open_rate"],
    marker="o",
    label="Known specificity"
)
ax.set_xlabel("Router threshold")
ax.set_ylabel("Score")
ax.set_ylim(-0.03, 1.03)
ax.legend()
ax.set_title("Open-set router threshold sensitivity")
fig.tight_layout()
fig.savefig(REVIEW_DIR / "router_threshold_sensitivity.png", dpi=220)
plt.show()

# =============================================================================
# T2 — PREPOLICY ACTIVE-SCD ALPHA SENSITIVITY
# =============================================================================
#
# Recomputes the two leakage-free components separately:
#   posterior JS drift
#   role-active semantic distance
# then sweeps alpha in alpha*JS + (1-alpha)*semantic_distance.
# =============================================================================

def _prepolicy_components(reference, current):
    slots = {
        "PLANNER": "object",
        "RETRIEVER": "state",
        "POLICY": "authority",
    }
    posterior, semantic = [], []

    for agent, slot in slots.items():
        posterior.append(jsd(
            reference[agent]["probs"][slot],
            current[agent]["probs"][slot]
        ))

        a = reference[agent]["contract"].get(slot)
        b = current[agent]["contract"].get(slot)

        if slot == "object":
            semantic.append(object_distance(a, b))
        elif slot == "state":
            semantic.append(state_distance(a, b))
        else:
            semantic.append(authority_distance(a, b))

    return float(np.mean(posterior)), float(np.mean(semantic))

component_rows = []

for base in drift_base:
    reference = {
        agent: ensemble_predict(
            "SCALE",
            [realize_message(base, agent, "A", 0)],
            BASE_GRAPH
        )[0]
        for agent in AGENTS
    }

    for severity in [0, 1, 2, 3]:
        msgs, ops = drift_messages(base, severity)

        current = {
            agent: ensemble_predict(
                "SCALE", [msgs[agent]], BASE_GRAPH
            )[0]
            for agent in AGENTS
        }
        repaired = {
            agent: {
                **current[agent],
                "contract": ontology_repair(current[agent]["contract"])
            }
            for agent in AGENTS
        }

        js_part, semantic_part = _prepolicy_components(reference, repaired)

        row0 = drift_df[
            (drift_df["base_id"] == base["base_id"])
            & (drift_df["severity"] == severity)
        ].iloc[0]

        component_rows.append({
            "base_id": base["base_id"],
            "severity": severity,
            "harmful_drift": int(row0["harmful_drift"]),
            "failure": int(row0["failure"]),
            "posterior_js": js_part,
            "active_semantic_distance": semantic_part,
        })

scd_components_df = pd.DataFrame(component_rows)

alpha_grid = sorted(set(
    [0.0, 0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.65,
     0.70, 0.75, 0.80, 0.90, 1.0]
))

alpha_rows = []

for alpha in alpha_grid:
    tmp = scd_components_df.copy()
    tmp["score"] = (
        alpha * tmp["posterior_js"]
        + (1.0 - alpha) * tmp["active_semantic_distance"]
    )

    # grouped_auc exists earlier in the notebook and keeps base_id together.
    harmful_auc = grouped_auc(tmp, "harmful_drift", ["score"])
    failure_auc = grouped_auc(tmp, "failure", ["score"])

    alpha_rows.append({
        "alpha_js": alpha,
        "alpha_semantic": 1.0 - alpha,
        "harmful_auc": harmful_auc,
        "failure_auc": failure_auc,
    })

scd_alpha_df = pd.DataFrame(alpha_rows)

base_alpha_row = scd_alpha_df[
    np.isclose(scd_alpha_df["alpha_js"], 0.65)
].iloc[0]

print("\n[T2] PREPOLICY ACTIVE-SCD WEIGHT SENSITIVITY")
display(scd_alpha_df.round(4))

scd_components_df.to_csv(
    REVIEW_DIR / "scd_components.csv", index=False
)
scd_alpha_df.to_csv(
    REVIEW_DIR / "scd_alpha_sensitivity.csv", index=False
)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(
    scd_alpha_df["alpha_js"],
    scd_alpha_df["failure_auc"],
    marker="o",
    label="Failure AUROC"
)
ax.plot(
    scd_alpha_df["alpha_js"],
    scd_alpha_df["harmful_auc"],
    marker="o",
    label="Harmful-drift AUROC"
)
ax.axvline(0.65, linestyle="--", linewidth=1, label="Paper alpha = 0.65")
ax.set_xlabel("Weight on posterior JS component (alpha)")
ax.set_ylabel("AUROC")
ax.set_ylim(0.0, 1.03)
ax.legend()
ax.set_title("PrePolicy Active-SCD weight sensitivity")
fig.tight_layout()
fig.savefig(REVIEW_DIR / "scd_alpha_sensitivity.png", dpi=220)
plt.show()

# =============================================================================
# T3 — INDEPENDENT LEARNED DOWNSTREAM EXECUTOR
# =============================================================================
#
# Reviewer concern addressed:
# Active-SCD should not look strong merely because failure is computed by the same
# deterministic semantic rule. Here the downstream executor does NOT consume SCALE
# contracts. It directly consumes concatenated frozen message embeddings and is trained
# independently to predict final action and authority.
#
# Two executor families:
#   - Raw-BGE Logistic Executor
#   - DualView-BGE Logistic Executor
# =============================================================================

def _executor_vector(messages_by_agent, dual=False):
    texts = [messages_by_agent[a] for a in AGENTS]
    arr = dual_view_encode(texts) if dual else semantic_encode(texts)
    return arr.reshape(-1).astype(np.float32)

def _clean_messages(base, impl):
    return {
        a: realize_message(base, a, impl, 0)
        for a in AGENTS
    }

def _fit_executor(dual=False):
    X_train, y_action, y_auth = [], [], []

    # Train on clean independent realizations A and B only.
    for base in train_base:
        for impl in ["A", "B"]:
            X_train.append(
                _executor_vector(_clean_messages(base, impl), dual=dual)
            )
            y_action.append(base["final_action"])
            y_auth.append(base["final_authority"])

    X_train = np.stack(X_train)

    action_clf = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED
        )),
    ])
    auth_clf = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED + 1
        )),
    ])

    action_clf.fit(X_train, np.asarray(y_action, object))
    auth_clf.fit(X_train, np.asarray(y_auth, object))
    return action_clf, auth_clf

independent_executor_rows = []

for executor_name, dual in [
    ("Raw-BGE-LogReg", False),
    ("DualView-BGE-LogReg", True),
]:
    print(f"\nTraining independent executor: {executor_name}")
    action_clf, auth_clf = _fit_executor(dual=dual)

    for base in drift_base:
        for severity in [0, 1, 2, 3]:
            msgs, ops = drift_messages(base, severity)
            x = _executor_vector(msgs, dual=dual).reshape(1, -1)

            pred_action = str(action_clf.predict(x)[0])
            pred_auth = str(auth_clf.predict(x)[0])

            success = int(
                pred_action == base["final_action"]
                and pred_auth == base["final_authority"]
            )

            original = drift_df[
                (drift_df["base_id"] == base["base_id"])
                & (drift_df["severity"] == severity)
            ].iloc[0]

            independent_executor_rows.append({
                "executor": executor_name,
                "base_id": base["base_id"],
                "severity": severity,
                "operators": "|".join(ops),
                "pred_action": pred_action,
                "pred_authority": pred_auth,
                "gold_action": base["final_action"],
                "gold_authority": base["final_authority"],
                "task_success": success,
                "failure": 1 - success,
                "prepolicy_active_scd": float(
                    original["prepolicy_active_scd"]
                ),
                "full_scd": float(original["full_scd"]),
                "neg_confidence": float(1.0 - original["confidence"]),
            })

independent_executor_df = pd.DataFrame(independent_executor_rows)

independent_summary_rows = []

for executor_name, sub in independent_executor_df.groupby("executor"):
    by_sev = (
        sub.groupby("severity", as_index=False)
        .agg(
            n=("base_id", "size"),
            task_success=("task_success", "mean"),
            failure_rate=("failure", "mean"),
        )
    )
    print(f"\n[T3] {executor_name} severity curve")
    display(by_sev.round(4))

    independent_summary_rows.append({
        "executor": executor_name,
        "d0_accuracy": float(
            sub[sub["severity"] == 0]["task_success"].mean()
        ),
        "failure_auc_active_scd": grouped_auc(
            sub, "failure", ["prepolicy_active_scd"]
        ),
        "failure_auc_full_scd": grouped_auc(
            sub, "failure", ["full_scd"]
        ),
        "failure_auc_neg_confidence": grouped_auc(
            sub, "failure", ["neg_confidence"]
        ),
        "n": len(sub),
    })

independent_executor_summary = pd.DataFrame(independent_summary_rows)

print("\n[T3] INDEPENDENT EXECUTOR FAILURE PREDICTION")
display(independent_executor_summary.round(4))

independent_executor_df.to_csv(
    REVIEW_DIR / "independent_executor_results.csv", index=False
)
independent_executor_summary.to_csv(
    REVIEW_DIR / "independent_executor_summary.csv", index=False
)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for executor_name, sub in independent_executor_df.groupby("executor"):
    g = (
        sub.groupby("severity", as_index=False)
        .agg(failure_rate=("failure", "mean"))
    )
    ax.plot(
        g["severity"], g["failure_rate"],
        marker="o", label=executor_name
    )
ax.set_xlabel("Drift severity")
ax.set_ylabel("Independent-executor failure rate")
ax.set_xticks([0, 1, 2, 3])
ax.set_ylim(-0.03, 1.03)
ax.legend()
ax.set_title("Independent learned executor under semantic drift")
fig.tight_layout()
fig.savefig(REVIEW_DIR / "independent_executor_failure_curve.png", dpi=220)
plt.show()

# =============================================================================
# T4 — HARD SEMANTIC-ALIAS STRESS
# =============================================================================
#
# This is deliberately harder than the original root aliases and contains no
# explicit ontology identifier. It is still a controlled stress test, NOT a
# human-authored external ontology benchmark; the report labels it accordingly.
# =============================================================================

HARD_ALIAS_BY_ROOT = {
    "PURCHASE": [
        "a request to obtain budgeted equipment from an external supplier",
        "an acquisition need that commits approved funds to obtain operational goods",
        "a sourcing request for a required asset from a vendor",
    ],
    "ACCESS": [
        "temporary elevation of a user's permissions to a restricted resource",
        "a request to grant privileged entitlement to a protected system",
        "controlled permission for a user to reach a restricted capability",
    ],
    "CHANGE": [
        "a planned alteration to a production configuration requiring controlled rollout",
        "a governed modification to an operational deployment",
        "an approved alteration to the configuration of a running service",
    ],
    "INCIDENT": [
        "an unexpected operational event requiring triage and response",
        "a service disruption that requires investigation and containment",
        "an abnormal operational event requiring coordinated remediation",
    ],
    "COMPLIANCE_ITEM": [
        "an evidence item assessed against a regulatory control obligation",
        "an assurance checkpoint used to verify conformity with a required control",
        "a governed review item tied to an external or internal obligation",
    ],
}

hard_alias_rows = []

for root, aliases in HARD_ALIAS_BY_ROOT.items():
    for alias_id, alias in enumerate(aliases):
        text = f"Planner handoff: the target is {alias}."

        for method in [
            "CB+DualView",
            "Proto-CB",
            "Proto+Ancestor",
            "SCALE-NoOnt",
            "SCALE-Full",
        ]:
            pred, canonical = learned_large_prediction(
                method, text, LARGE_OPAQUE_GRAPH
            )
            hard_alias_rows.append({
                "gold_root": root,
                "alias_id": alias_id,
                "text": text,
                "method": method,
                "predicted_node": pred,
                "canonical_root": canonical,
                "canonical_root_accuracy": int(canonical == root),
            })

hard_alias_df = pd.DataFrame(hard_alias_rows)
hard_alias_summary = (
    hard_alias_df
    .groupby("method", as_index=False)
    .agg(
        canonical_root_accuracy=("canonical_root_accuracy", "mean"),
        n=("canonical_root_accuracy", "size"),
    )
    .sort_values("canonical_root_accuracy", ascending=False)
)

print("\n[T4] HARD CONTROLLED SEMANTIC-ALIAS STRESS")
display(hard_alias_summary.round(4))

hard_alias_df.to_csv(
    REVIEW_DIR / "hard_alias_results.csv", index=False
)
hard_alias_summary.to_csv(
    REVIEW_DIR / "hard_alias_summary.csv", index=False
)

# =============================================================================
# FINAL PAPER-FACING SUMMARY
# =============================================================================

# SCD sensitivity robustness:
alpha_fail_min = float(scd_alpha_df["failure_auc"].min())
alpha_fail_max = float(scd_alpha_df["failure_auc"].max())
alpha_harm_min = float(scd_alpha_df["harmful_auc"].min())
alpha_harm_max = float(scd_alpha_df["harmful_auc"].max())

# Hybrid route headline:
mixed_acc = float(hybrid_route_df["correct"].mean())
known_acc = float(
    hybrid_route_df[hybrid_route_df["kind"] == "known"]["correct"].mean()
)
unseen_acc = float(
    hybrid_route_df[hybrid_route_df["kind"] == "unseen"]["correct"].mean()
)

# Independent executor headline:
indep_min_auc = float(
    independent_executor_summary["failure_auc_active_scd"].min()
)
indep_max_auc = float(
    independent_executor_summary["failure_auc_active_scd"].max()
)
indep_min_d0 = float(independent_executor_summary["d0_accuracy"].min())

hard_scale = hard_alias_summary[
    hard_alias_summary["method"] == "SCALE-Full"
]["canonical_root_accuracy"]
hard_scale = float(hard_scale.iloc[0]) if len(hard_scale) else np.nan

report = f"""
# SCALE Reviewer-Hardening Report

## T1 — Deployable hybrid routing

The final routing stack is now operational rather than oracle-defined:

1. recoverable ontology identifier -> deterministic normalized-ID canonicalization;
2. otherwise, learned OOF open-set routing distinguishes known from unseen semantics;
3. known semantics -> CB+DualView;
4. unseen semantics -> SCALE ontology branch.

OOF router AUROC: **{router_auc:.4f}**

OOF balanced accuracy at the selected threshold ({router_threshold:.4f}):
**{router_balacc:.4f}**

Unseen recall: **{unseen_recall:.4f}**

Known false-open rate: **{known_false_open:.4f}**

Mixed deployable routing canonical-root accuracy:
**{mixed_acc:.4f}**

Known-condition accuracy: **{known_acc:.4f}**

Unseen-condition accuracy: **{unseen_acc:.4f}**

Interpretation:
This test directly addresses the reviewer concern that the hybrid architecture otherwise appears
to rely on oracle knowledge of whether an input is known or open-world.

## T2 — Active-SCD alpha sensitivity

Paper setting alpha_JS=0.65:

- harmful-drift AUROC: **{float(base_alpha_row["harmful_auc"]):.4f}**
- failure AUROC: **{float(base_alpha_row["failure_auc"]):.4f}**

Across the full tested alpha grid:

- failure AUROC range: **[{alpha_fail_min:.4f}, {alpha_fail_max:.4f}]**
- harmful-drift AUROC range: **[{alpha_harm_min:.4f}, {alpha_harm_max:.4f}]**

Interpretation:
If performance remains high over a broad alpha interval, the Active-SCD result is not dependent
on the exact 0.65/0.35 weighting choice. If it varies sharply, the paper should report this
sensitivity rather than treating 0.65 as universal.

## T3 — Independent learned downstream executors

Minimum clean D0 task accuracy across independent executors:
**{indep_min_d0:.4f}**

Active-SCD failure AUROC across independent learned executors:
**[{indep_min_auc:.4f}, {indep_max_auc:.4f}]**

These executors never consume SCALE semantic contracts; they predict task decisions directly
from frozen message embeddings. This test therefore weakens the objection that the very high
Active-SCD result is only a restatement of the deterministic semantic policy used in the original
controlled benchmark.

Important qualification:
The target labels still come from the controlled task generator, so this is an independent learned
executor robustness test, not a substitute for a live longitudinal deployment.

## T4 — Hard controlled semantic aliases

SCALE-Full canonical-root accuracy:
**{hard_scale:.4f}**

This stress set avoids explicit identifiers and exact canonical phrases. It is intentionally harder
than the original semantic-alias templates.

Important qualification:
These aliases are controlled, manually specified stress cases inside the notebook. Do not call this
a human-authored external ontology benchmark.

## Recommended manuscript use

If T1 is strong:
- add the three-way routing rule to Method 4.2;
- report router AUROC / balanced accuracy and the mixed canonicalization result;
- remove any appearance of oracle known-vs-unseen branch selection.

If T2 is stable:
- retain alpha=0.65 but add the sensitivity range in the appendix.

If T3 remains strong:
- add an "Independent downstream executor" robustness paragraph to RQ4;
- this materially strengthens the Active-SCD claim.

If T4 separates SCALE from SCALE-NoOnt / CB+DualView:
- use it as a robustness extension of the ontology mechanism result.
If it does not separate them:
- keep the small opaque-leaf intervention as the primary ontology mechanism evidence
  and report T4 only as a boundary result.

Generated files are in:
`{REVIEW_DIR}`
"""

(REVIEW_DIR / "REVIEWER_HARDENING_REPORT.md").write_text(
    report.strip() + "\n",
    encoding="utf-8"
)

from IPython.display import display, Markdown
display(Markdown(report))

print("\nSaved reviewer-hardening artifacts:")
for p in sorted(REVIEW_DIR.iterdir()):
    print(" -", p.name)


In [ ]:

# =============================================================================
# 41/46 — Strict routing integrity + corrected deployable hybrid router
# =============================================================================

from pathlib import Path
import json, re, math, copy, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)

FINAL_TEST_DIR = ROOT / "final_additional_tests"
FINAL_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 96)
print("FINAL SCALE ADDITIONAL TESTS")
print("Output:", FINAL_TEST_DIR)
print("=" * 96)

# ---------- preflight ----------
_REQUIRED = [
    "MODELS", "LARGE_OPAQUE_GRAPH", "LARGE_OPAQUE_NODES",
    "LARGE_NODE_BY_LABEL", "large_ontology_message",
    "learned_large_prediction", "ensemble_predict",
    "shift_test_base", "realize_message", "BASE_GRAPH",
    "semantic_encode", "dual_view_encode", "drift_df",
    "drift_base", "drift_messages", "grouped_auc",
    "_router_features", "_prepolicy_components",
]
_missing = [x for x in _REQUIRED if x not in globals()]
assert not _missing, (
    "Run every previous notebook cell first. Missing: " + ", ".join(_missing)
)

def _safe_auc_local(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

def _select_balanced_threshold(y, score):
    y = np.asarray(y, int)
    score = np.asarray(score, float)
    vals = np.unique(score)
    if len(vals) > 300:
        vals = np.quantile(score, np.linspace(0, 1, 300))
    best = (0.5, -1.0)
    for t in vals:
        pred = (score >= t).astype(int)
        b = balanced_accuracy_score(y, pred)
        if b > best[1]:
            best = (float(t), float(b))
    return best

# -------------------------------------------------------------------------
# Why the old normalized matcher could fail
# -------------------------------------------------------------------------
# Old behavior removes all punctuation/whitespace from the WHOLE sentence and
# then checks whether an ontology ID is a substring of the result. That can
# accidentally create an ID across token boundaries.
#
# New rule:
#   * extract identifier-like lexical tokens/spans;
#   * normalize EACH span independently;
#   * require an exact normalized match to one complete ontology identifier.
# This still recovers exact, punctuated, and wrapped IDs, but cannot fabricate
# an ID by concatenating unrelated words in a semantic alias.
# -------------------------------------------------------------------------

def strict_explicit_id_lookup(text, graph_ctx):
    labels = list(graph_ctx["leaf_parent"].keys())
    key_to_label = {
        re.sub(r"[^A-Z0-9]", "", str(label).upper()): label
        for label in labels
    }

    # Candidate lexical spans. Allows XAB12-CD34, xab12_cd34, etc.,
    # but does not concatenate separate ordinary words.
    spans = re.findall(
        r"[A-Za-z0-9]+(?:[-_.:][A-Za-z0-9]+)*",
        str(text)
    )

    hits = []
    for span in spans:
        key = re.sub(r"[^A-Z0-9]", "", span.upper())
        if key in key_to_label:
            hits.append((span, key_to_label[key]))

    # Exact identifier match; longest raw span only resolves duplicate surfaces.
    if not hits:
        return None
    hits = sorted(hits, key=lambda x: len(x[0]), reverse=True)
    return hits[0][1]

# ---------- exhaustive identifier audit ----------
STRICT_SURFACES = (
    list(LARGE_SURFACES)
    if "LARGE_SURFACES" in globals()
    else ["exact_id", "punctuated_id", "semantic_alias", "wrapped_id"]
)

id_audit_rows = []

for node in LARGE_OPAQUE_NODES:
    for surface in STRICT_SURFACES:
        text = large_ontology_message(node, surface)

        old_hit = normalized_id_lookup(text, LARGE_OPAQUE_GRAPH)
        strict_hit = strict_explicit_id_lookup(text, LARGE_OPAQUE_GRAPH)

        should_have_id = surface in {
            "exact_id", "punctuated_id", "wrapped_id"
        }

        id_audit_rows.append({
            "node": node["label"],
            "gold_root": node["canonical_root"],
            "surface": surface,
            "text": text,
            "should_have_explicit_id": int(should_have_id),
            "old_hit": old_hit,
            "strict_hit": strict_hit,
            "old_false_positive": int(
                (not should_have_id) and old_hit is not None
            ),
            "strict_false_positive": int(
                (not should_have_id) and strict_hit is not None
            ),
            "strict_false_negative": int(
                should_have_id and strict_hit != node["label"]
            ),
            "strict_correct_id": int(
                strict_hit == node["label"]
                if should_have_id
                else strict_hit is None
            ),
        })

id_audit_df = pd.DataFrame(id_audit_rows)

id_audit_summary = (
    id_audit_df.groupby("surface", as_index=False)
    .agg(
        n=("node", "size"),
        old_false_positive_rate=("old_false_positive", "mean"),
        strict_false_positive_rate=("strict_false_positive", "mean"),
        strict_false_negative_rate=("strict_false_negative", "mean"),
        strict_integrity_accuracy=("strict_correct_id", "mean"),
    )
)

print("\n[41A] STRICT IDENTIFIER AUDIT")
display(id_audit_summary.round(4))

# Hard correctness condition: semantic aliases must NEVER be symbolic IDs.
alias_fp = id_audit_df[
    id_audit_df["surface"] == "semantic_alias"
]["strict_false_positive"].sum()
assert alias_fp == 0, (
    f"Strict matcher still produced {alias_fp} semantic-alias false ID hits."
)

id_audit_df.to_csv(
    FINAL_TEST_DIR / "strict_identifier_audit.csv", index=False
)
id_audit_summary.to_csv(
    FINAL_TEST_DIR / "strict_identifier_audit_summary.csv", index=False
)

# -------------------------------------------------------------------------
# Rebuild router calibration set with hard aliases added as unseen examples.
# Grouping prevents root-specific alias leakage across folds.
# -------------------------------------------------------------------------

STRICT_ROUTER_ROWS = []

for base in shift_test_base:
    for impl in ["B", "C1", "C2"]:
        txt = realize_message(base, "PLANNER", impl, 0)
        STRICT_ROUTER_ROWS.append({
            "kind": "known",
            "is_unseen": 0,
            "group": f"known::{base['base_id']}",
            "gold_root": base["object"],
            "surface": impl,
            "text": txt,
        })

# Original semantic aliases.
for node in LARGE_OPAQUE_NODES:
    STRICT_ROUTER_ROWS.append({
        "kind": "unseen",
        "is_unseen": 1,
        "group": f"unseen_root::{node['canonical_root']}",
        "gold_root": node["canonical_root"],
        "surface": "semantic_alias",
        "text": large_ontology_message(node, "semantic_alias"),
    })

# Hard aliases from the previous reviewer-hardening cell, if available.
if "HARD_ALIAS_BY_ROOT" in globals():
    for root, aliases in HARD_ALIAS_BY_ROOT.items():
        for j, alias in enumerate(aliases):
            STRICT_ROUTER_ROWS.append({
                "kind": "unseen",
                "is_unseen": 1,
                "group": f"hard_root::{root}",
                "gold_root": root,
                "surface": "hard_semantic_alias",
                "text": f"Planner handoff: the target is {alias}.",
            })

strict_router_df = pd.DataFrame(STRICT_ROUTER_ROWS)
Xr = np.stack([
    _router_features(t) for t in strict_router_df["text"]
])
yr = strict_router_df["is_unseen"].to_numpy(int)
gr = strict_router_df["group"].to_numpy(object)

strict_router_df["p_unseen_oof"] = np.nan

gkf = GroupKFold(n_splits=min(5, strict_router_df["group"].nunique()))
for tr, te in gkf.split(Xr, yr, gr):
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2500,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])
    pipe.fit(Xr[tr], yr[tr])
    strict_router_df.loc[te, "p_unseen_oof"] = (
        pipe.predict_proba(Xr[te])[:, 1]
    )

_valid = strict_router_df["p_unseen_oof"].notna()
strict_router_auc = _safe_auc_local(
    strict_router_df.loc[_valid, "is_unseen"],
    strict_router_df.loc[_valid, "p_unseen_oof"]
)

strict_router_threshold, strict_router_balacc = _select_balanced_threshold(
    strict_router_df.loc[_valid, "is_unseen"],
    strict_router_df.loc[_valid, "p_unseen_oof"]
)

_y = strict_router_df.loc[_valid, "is_unseen"].to_numpy(int)
_p = strict_router_df.loc[_valid, "p_unseen_oof"].to_numpy(float)
_pred = (_p >= strict_router_threshold).astype(int)

strict_router_metrics = pd.DataFrame([{
    "router_auc": strict_router_auc,
    "selected_threshold": strict_router_threshold,
    "balanced_accuracy": strict_router_balacc,
    "f1_unseen": float(f1_score(_y, _pred)),
    "unseen_recall": float(_pred[_y == 1].mean()),
    "known_false_open_rate": float(_pred[_y == 0].mean()),
    "n": int(len(_y)),
    "groups": int(strict_router_df.loc[_valid, "group"].nunique()),
}])

print("\n[41B] STRICT OOF ROUTER")
display(strict_router_metrics.round(4))

STRICT_FINAL_ROUTER = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        random_state=SEED,
    )),
])
STRICT_FINAL_ROUTER.fit(Xr, yr)

def scale_hybrid_route_strict(
    text,
    graph_ctx=LARGE_OPAQUE_GRAPH,
    threshold=None
):
    if threshold is None:
        threshold = strict_router_threshold

    # 1. Deterministic exact identifier path.
    id_hit = strict_explicit_id_lookup(text, graph_ctx)
    if id_hit is not None:
        return {
            "route": "symbolic_id",
            "predicted_node": id_hit,
            "canonical_root": canonicalize_lookup(id_hit, graph_ctx),
            "p_unseen": 1.0,
        }

    # 2. Learned known/open-world decision.
    p_unseen = float(
        STRICT_FINAL_ROUTER.predict_proba(
            _router_features(text).reshape(1, -1)
        )[0, 1]
    )

    # 3. Open-world ontology branch.
    if p_unseen >= threshold:
        node, root = learned_large_prediction(
            "SCALE-Full", text, graph_ctx
        )
        return {
            "route": "open_world_ontology",
            "predicted_node": node,
            "canonical_root": root,
            "p_unseen": p_unseen,
        }

    # 4. Known closed-set branch.
    cb = ensemble_predict(
        "CB+DualView", [text], BASE_GRAPH
    )[0]
    root = cb["contract"].get("object")

    return {
        "route": "known_discriminative",
        "predicted_node": root,
        "canonical_root": root,
        "p_unseen": p_unseen,
    }

# ---------- corrected mixed deployable benchmark ----------
strict_hybrid_rows = []

# Known C1/C2.
for base in shift_test_base:
    for impl in ["C1", "C2"]:
        text = realize_message(base, "PLANNER", impl, 0)
        out = scale_hybrid_route_strict(text)
        strict_hybrid_rows.append({
            "kind": "known",
            "surface": impl,
            "gold_root": base["object"],
            **out,
            "correct": int(out["canonical_root"] == base["object"]),
        })

# Unseen four-surface large ontology.
for node in LARGE_OPAQUE_NODES:
    for surface in STRICT_SURFACES:
        text = large_ontology_message(node, surface)
        out = scale_hybrid_route_strict(text)

        strict_hybrid_rows.append({
            "kind": "unseen",
            "surface": surface,
            "gold_root": node["canonical_root"],
            **out,
            "correct": int(
                out["canonical_root"] == node["canonical_root"]
            ),
        })

# Hard aliases.
if "HARD_ALIAS_BY_ROOT" in globals():
    for root, aliases in HARD_ALIAS_BY_ROOT.items():
        for alias in aliases:
            text = f"Planner handoff: the target is {alias}."
            out = scale_hybrid_route_strict(text)
            strict_hybrid_rows.append({
                "kind": "unseen",
                "surface": "hard_semantic_alias",
                "gold_root": root,
                **out,
                "correct": int(out["canonical_root"] == root),
            })

strict_hybrid_df = pd.DataFrame(strict_hybrid_rows)

strict_hybrid_summary = (
    strict_hybrid_df.groupby(
        ["kind", "surface", "route"], as_index=False
    )
    .agg(
        canonical_root_accuracy=("correct", "mean"),
        n=("correct", "size"),
        mean_p_unseen=("p_unseen", "mean"),
    )
)

print("\n[41C] CORRECTED THREE-WAY ROUTING")
display(strict_hybrid_summary.round(4))

# Critical assertion: no ID route for semantic aliases.
_bad_alias_routes = strict_hybrid_df[
    strict_hybrid_df["surface"].isin(
        ["semantic_alias", "hard_semantic_alias"]
    )
    & (strict_hybrid_df["route"] == "symbolic_id")
]
assert len(_bad_alias_routes) == 0, (
    "A semantic alias still entered symbolic_id route."
)

strict_router_df.to_csv(
    FINAL_TEST_DIR / "strict_router_oof.csv", index=False
)
strict_router_metrics.to_csv(
    FINAL_TEST_DIR / "strict_router_metrics.csv", index=False
)
strict_hybrid_df.to_csv(
    FINAL_TEST_DIR / "strict_hybrid_results.csv", index=False
)
strict_hybrid_summary.to_csv(
    FINAL_TEST_DIR / "strict_hybrid_summary.csv", index=False
)

print("STRICT ROUTING INTEGRITY: PASS")



# =============================================================================
# 42/46 — Naturalistic version-evolution drift benchmark
# =============================================================================
#
# Purpose:
# The original drift study uses explicit corruption operators. This test instead
# simulates ordinary software/agent evolution:
#
# BENIGN:
#   - prompt rewrite (C1)
#   - schema/tool wrapper upgrade (C2)
#
# SEMANTIC VERSION CHANGES:
#   - object taxonomy reinterpretation
#   - evidence-status interpretation change
#   - authority-boundary change
#   - compound semantic version change
#
# The harmful label is NOT "which transform was used".
# It is computed from whether the underlying semantic version change alters the
# original deterministic task decision. Therefore a semantic version change can
# be benign for a particular base case.
# =============================================================================

def _rotate_evidence(status):
    return {
        "PASS": "FAIL",
        "FAIL": "UNCERTAIN",
        "UNCERTAIN": "PASS",
    }[status]

def _lower_authority(auth):
    order = ["RECOMMEND", "APPROVE", "EXECUTE"]
    i = order.index(auth)
    return order[max(0, i - 1)]

def _upper_authority(auth):
    order = ["RECOMMEND", "APPROVE", "EXECUTE"]
    i = order.index(auth)
    return order[min(len(order) - 1, i + 1)]

def _swap_object(obj):
    if "OBJECT_DRIFT_MAP" in globals():
        return OBJECT_DRIFT_MAP[obj]
    fallback = {
        "PURCHASE": "CHANGE",
        "ACCESS": "INCIDENT",
        "CHANGE": "INCIDENT",
        "INCIDENT": "CHANGE",
        "COMPLIANCE_ITEM": "CHANGE",
    }
    return fallback[obj]

def _version_case(base, family):
    """
    Returns:
      messages: raw Planner/Retriever/Policy handoffs
      gt_semantics_after_version: semantic meaning intended by this version
      metadata
    """
    b = copy.deepcopy(base)
    impl = {"PLANNER": "A", "RETRIEVER": "A", "POLICY": "A"}

    if family == "baseline_v1":
        pass

    elif family == "prompt_rewrite_v2":
        impl = {a: "C1" for a in AGENTS}

    elif family == "tool_schema_v2":
        impl = {a: "C2" for a in AGENTS}

    elif family == "object_taxonomy_v2":
        b["object"] = _swap_object(base["object"])

    elif family == "evidence_semantics_v2":
        b["evidence_status"] = _rotate_evidence(base["evidence_status"])
        b["evidence"] = EVIDENCE_TEXT[b["evidence_status"]][0]

    elif family == "authority_boundary_v2":
        # Lower when possible; otherwise raise. This guarantees a version change
        # but does not guarantee a task failure.
        new_auth = _lower_authority(base["authority"])
        if new_auth == base["authority"]:
            new_auth = _upper_authority(base["authority"])
        b["authority"] = new_auth

    elif family == "compound_semantic_v3":
        b["object"] = _swap_object(base["object"])
        b["evidence_status"] = _rotate_evidence(base["evidence_status"])
        b["evidence"] = EVIDENCE_TEXT[b["evidence_status"]][1]
        new_auth = _lower_authority(base["authority"])
        if new_auth == base["authority"]:
            new_auth = _upper_authority(base["authority"])
        b["authority"] = new_auth
        # Also introduce a realistic implementation/schema release.
        impl = {"PLANNER": "C1", "RETRIEVER": "C2", "POLICY": "C2"}

    else:
        raise ValueError(family)

    messages = {
        agent: realize_message(b, agent, impl[agent], 0)
        for agent in AGENTS
    }

    sem_after = {
        "object": b["object"],
        "evidence_state": EVIDENCE_TO_STATE[b["evidence_status"]],
        "authority": b["authority"],
    }

    return messages, sem_after, {
        "family": family,
        "impl": impl,
    }

NATURALISTIC_FAMILIES = [
    "baseline_v1",
    "prompt_rewrite_v2",
    "tool_schema_v2",
    "object_taxonomy_v2",
    "evidence_semantics_v2",
    "authority_boundary_v2",
    "compound_semantic_v3",
]

def _scale_bundle_from_messages(msgs):
    bundle = {}
    for agent in AGENTS:
        p = ensemble_predict("SCALE", [msgs[agent]], BASE_GRAPH)[0]
        p = copy.deepcopy(p)
        p["contract"] = ontology_repair(p["contract"])
        bundle[agent] = p
    return bundle

def _bundle_semantics(bundle):
    return {
        "object": bundle["PLANNER"]["contract"].get("object"),
        "evidence_state": bundle["RETRIEVER"]["contract"].get("state"),
        "authority": bundle["POLICY"]["contract"].get("authority"),
    }

naturalistic_rows = []

for base in drift_base:
    ref_msgs, ref_gt_sem, _ = _version_case(base, "baseline_v1")
    ref_bundle = _scale_bundle_from_messages(ref_msgs)

    original_gold_policy = deterministic_policy(global_semantics(base))

    for family in NATURALISTIC_FAMILIES:
        msgs, gt_sem_after, meta = _version_case(base, family)
        cur_bundle = _scale_bundle_from_messages(msgs)

        js_part, sem_part = _prepolicy_components(
            ref_bundle, cur_bundle
        )
        active_scd_065 = 0.65 * js_part + 0.35 * sem_part

        pred_sem = _bundle_semantics(cur_bundle)
        pred_policy = deterministic_policy(pred_sem)

        gt_version_policy = deterministic_policy(gt_sem_after)

        # Ground-truth "harmful version change":
        # did the intended semantic release change the original decision?
        gt_harmful = int(gt_version_policy != original_gold_policy)

        # SCALE task failure remains relative to the ORIGINAL task, which is the
        # monitoring use case: did independent evolution break behavior?
        scale_failure = int(pred_policy != original_gold_policy)

        naturalistic_rows.append({
            "base_id": base["base_id"],
            "family": family,
            "gt_harmful_version_change": gt_harmful,
            "scale_failure": scale_failure,
            "posterior_js": js_part,
            "active_semantic_distance": sem_part,
            "prepolicy_active_scd": active_scd_065,
            "original_action": base["final_action"],
            "original_authority": base["final_authority"],
            "version_action": (
                None if gt_version_policy is None
                else gt_version_policy["action"]
            ),
            "version_authority": (
                None if gt_version_policy is None
                else gt_version_policy["authority"]
            ),
            "pred_action": (
                None if pred_policy is None
                else pred_policy["action"]
            ),
            "pred_authority": (
                None if pred_policy is None
                else pred_policy["authority"]
            ),
        })

naturalistic_df = pd.DataFrame(naturalistic_rows)

naturalistic_family_summary = (
    naturalistic_df.groupby("family", as_index=False)
    .agg(
        n=("base_id", "size"),
        harmful_rate=("gt_harmful_version_change", "mean"),
        scale_failure_rate=("scale_failure", "mean"),
        mean_active_scd=("prepolicy_active_scd", "mean"),
        mean_js=("posterior_js", "mean"),
        mean_active_distance=("active_semantic_distance", "mean"),
    )
)

nat_harm_auc = grouped_auc(
    naturalistic_df,
    "gt_harmful_version_change",
    ["prepolicy_active_scd"]
)
nat_failure_auc = grouped_auc(
    naturalistic_df,
    "scale_failure",
    ["prepolicy_active_scd"]
)

# LOFO by version family. Threshold is selected only on OTHER families.
nat_lofo_rows = []

for held_family in [
    f for f in NATURALISTIC_FAMILIES
    if f != "baseline_v1"
]:
    train = naturalistic_df[
        ~naturalistic_df["family"].isin(
            ["baseline_v1", held_family]
        )
    ].copy()
    test = naturalistic_df[
        naturalistic_df["family"].isin(
            ["baseline_v1", held_family]
        )
    ].copy()

    if (
        train["gt_harmful_version_change"].nunique() < 2
        or test["gt_harmful_version_change"].nunique() < 2
    ):
        harm_auc = np.nan
        bal = np.nan
    else:
        t, _ = _select_balanced_threshold(
            train["gt_harmful_version_change"],
            train["prepolicy_active_scd"]
        )
        pred = (
            test["prepolicy_active_scd"].to_numpy(float) >= t
        ).astype(int)

        harm_auc = _safe_auc_local(
            test["gt_harmful_version_change"],
            test["prepolicy_active_scd"]
        )
        bal = float(
            balanced_accuracy_score(
                test["gt_harmful_version_change"],
                pred
            )
        )

    fail_auc = _safe_auc_local(
        test["scale_failure"],
        test["prepolicy_active_scd"]
    )

    nat_lofo_rows.append({
        "held_family": held_family,
        "harmful_auc": harm_auc,
        "balanced_accuracy": bal,
        "failure_auc": fail_auc,
        "n": len(test),
    })

naturalistic_lofo_df = pd.DataFrame(nat_lofo_rows)

print("\n[42A] NATURALISTIC VERSION-EVOLUTION SUMMARY")
display(naturalistic_family_summary.round(4))

print("\n[42B] NATURALISTIC OVERALL")
display(pd.DataFrame([{
    "harmful_version_change_auc": nat_harm_auc,
    "scale_failure_auc": nat_failure_auc,
    "n": len(naturalistic_df),
}]).round(4))

print("\n[42C] NATURALISTIC LEAVE-ONE-VERSION-FAMILY-OUT")
display(naturalistic_lofo_df.round(4))

naturalistic_df.to_csv(
    FINAL_TEST_DIR / "naturalistic_version_drift_results.csv",
    index=False
)
naturalistic_family_summary.to_csv(
    FINAL_TEST_DIR / "naturalistic_version_drift_summary.csv",
    index=False
)
naturalistic_lofo_df.to_csv(
    FINAL_TEST_DIR / "naturalistic_version_drift_lofo.csv",
    index=False
)



# =============================================================================
# 43/46 — Independent LLM downstream-executor robustness
# =============================================================================
#
# This is the expensive final observability test.
#
# The LLM executor receives ONLY raw Planner/Retriever/Policy messages.
# It never receives:
#   - SCALE contracts,
#   - Active-SCD,
#   - ontology predictions,
#   - the gold semantic tuple.
#
# We report failure-AUROC only after additionally conditioning on bases the same
# LLM solved correctly at severity 0. This prevents a generally weak LLM from
# making Active-SCD appear predictive simply because it already fails clean data.
# =============================================================================

RUN_FINAL_LLM_EXECUTOR = True
LLM_EXECUTOR_N_BASES = min(24, len(drift_base))
LLM_EXECUTOR_SPECS = READER_MODELS[:2]  # Qwen + Granite
LLM_EXECUTOR_FILE = FINAL_TEST_DIR / "llm_executor_results.jsonl"

LLM_EXECUTOR_SYSTEM = """You are the final decision executor in a multi-agent workflow.

You receive raw handoffs from PLANNER, RETRIEVER, and POLICY.
Infer the operational object, evidence state, and authority, then determine
the final workflow action.

Valid final actions:
APPROVE, UPDATE, SEND, REJECT, VERIFY

Valid final authorities:
RECOMMEND, APPROVE, EXECUTE

Return exactly one JSON object:
{"action":"ACTION","authority":"AUTHORITY"}

No explanation. No markdown.
"""

def llm_executor_messages(raw_msgs):
    body = "\n\n".join(
        f"[{agent}]\n{raw_msgs[agent]}"
        for agent in AGENTS
    )
    return [
        {"role": "system", "content": LLM_EXECUTOR_SYSTEM},
        {"role": "user", "content": body},
    ]

def parse_executor_decision(text):
    obj = extract_json(text)
    if not isinstance(obj, dict):
        return None, None, False

    action = str(obj.get("action", "")).strip().upper()
    auth = str(obj.get("authority", "")).strip().upper()

    valid_actions = {
        "APPROVE", "UPDATE", "SEND", "REJECT", "VERIFY"
    }
    valid_auth = {
        "RECOMMEND", "APPROVE", "EXECUTE"
    }

    ok = action in valid_actions and auth in valid_auth
    return action, auth, bool(ok)

def _llm_done():
    rows, _ = safe_jsonl_records(LLM_EXECUTOR_FILE)
    return {
        (r["reader"], r["base_id"], int(r["severity"]))
        for r in rows
        if "reader" in r
    }

if RUN_FINAL_LLM_EXECUTOR:
    done = _llm_done()
    selected_bases = drift_base[:LLM_EXECUTOR_N_BASES]

    for spec in LLM_EXECUTOR_SPECS:
        model_id = None
        tok = None
        mdl = None
        try:
            model_id, primary_loaded, tok, mdl = load_model_safe(spec)

            for base in selected_bases:
                for severity in [0, 1, 2, 3]:
                    key = (spec["label"], base["base_id"], severity)
                    if key in done:
                        continue

                    raw_msgs, ops = drift_messages(base, severity)

                    gen = generate_text(
                        tok, mdl,
                        llm_executor_messages(raw_msgs),
                        spec,
                        max_input=700,
                        max_new=36,
                    )

                    action, authority, parse_ok = parse_executor_decision(
                        gen["text"]
                    )

                    success = int(
                        parse_ok
                        and action == base["final_action"]
                        and authority == base["final_authority"]
                    )

                    # Reuse the exact leakage-free SCD computed in the main run.
                    drow = drift_df[
                        (drift_df["base_id"] == base["base_id"])
                        & (drift_df["severity"] == severity)
                    ].iloc[0]

                    row = {
                        "reader": spec["label"],
                        "loaded_model": model_id,
                        "primary_model_loaded": bool(primary_loaded),
                        "base_id": base["base_id"],
                        "severity": severity,
                        "operators": "|".join(ops),
                        "parse_ok": int(parse_ok),
                        "pred_action": action,
                        "pred_authority": authority,
                        "gold_action": base["final_action"],
                        "gold_authority": base["final_authority"],
                        "task_success": success,
                        "failure": 1 - success,
                        "prepolicy_active_scd": float(
                            drow["prepolicy_active_scd"]
                        ),
                        "full_scd": float(drow["full_scd"]),
                        "neg_confidence": float(
                            1.0 - drow["confidence"]
                        ),
                        "input_tokens": int(gen["input_tokens"]),
                        "output_tokens": int(gen["output_tokens"]),
                        "latency_s": float(gen["latency_s"]),
                        "overflow": int(gen["overflow"]),
                        "raw_output": gen["text"][:500],
                    }

                    append_jsonl(LLM_EXECUTOR_FILE, row)
                    done.add(key)

        finally:
            try:
                del mdl
            except Exception:
                pass
            try:
                del tok
            except Exception:
                pass
            cleanup_gpu()

llm_executor_df = safe_read_jsonl_df(LLM_EXECUTOR_FILE)

if len(llm_executor_df):
    llm_executor_df["severity"] = llm_executor_df["severity"].astype(int)

    llm_exec_summary_rows = []

    for reader, g in llm_executor_df.groupby("reader"):
        # Identify cases the executor solved cleanly.
        clean = g[g["severity"] == 0][
            ["base_id", "task_success"]
        ].rename(columns={"task_success": "clean_success"})

        gg = g.merge(clean, on="base_id", how="left")
        conditioned = gg[gg["clean_success"] == 1].copy()

        llm_exec_summary_rows.append({
            "reader": reader,
            "n_all": len(g),
            "n_baseline_correct_bases": int(
                conditioned["base_id"].nunique()
            ),
            "parse_success": float(g["parse_ok"].mean()),
            "clean_accuracy": float(
                g[g["severity"] == 0]["task_success"].mean()
            ),
            "failure_auc_active_scd_all": grouped_auc(
                g, "failure", ["prepolicy_active_scd"]
            ),
            "failure_auc_active_scd_baseline_correct": (
                grouped_auc(
                    conditioned, "failure",
                    ["prepolicy_active_scd"]
                )
                if len(conditioned) else np.nan
            ),
            "failure_auc_full_scd_baseline_correct": (
                grouped_auc(
                    conditioned, "failure",
                    ["full_scd"]
                )
                if len(conditioned) else np.nan
            ),
            "failure_auc_confidence_baseline_correct": (
                grouped_auc(
                    conditioned, "failure",
                    ["neg_confidence"]
                )
                if len(conditioned) else np.nan
            ),
        })

    llm_executor_summary = pd.DataFrame(llm_exec_summary_rows)

    llm_executor_severity = (
        llm_executor_df.groupby(
            ["reader", "severity"], as_index=False
        )
        .agg(
            task_success=("task_success", "mean"),
            failure_rate=("failure", "mean"),
            parse_success=("parse_ok", "mean"),
            n=("base_id", "size"),
        )
    )

    print("\n[43A] INDEPENDENT LLM EXECUTOR SUMMARY")
    display(llm_executor_summary.round(4))

    print("\n[43B] LLM EXECUTOR SEVERITY CURVES")
    display(llm_executor_severity.round(4))

    llm_executor_summary.to_csv(
        FINAL_TEST_DIR / "llm_executor_summary.csv", index=False
    )
    llm_executor_severity.to_csv(
        FINAL_TEST_DIR / "llm_executor_severity.csv", index=False
    )
else:
    print("No LLM executor rows were produced.")



# =============================================================================
# 44/46 — Human-authored external ontology diagnostic (W3C PROV-O)
# =============================================================================
#
# IMPORTANT INTERPRETATION:
# This is NOT presented as a new SCALE benchmark and does not retrain SCALE on
# PROV-O. It is a human-authored external ontology sanity test for the structural
# premise used by SCALE:
#
#   semantic grounding -> explicit ontology ancestry -> canonical parent/root
#
# It uses:
#   * W3C PROV-O class labels/comments as human-authored ontology content;
#   * the same frozen BGE semantic encoder used by SCALE;
#   * graph ancestry from PROV-O.
#
# It compares:
#   A) direct text-to-root classification
#   B) text-to-leaf grounding followed by graph canonicalization
#
# If internet is unavailable, this section skips cleanly.
# =============================================================================

RUN_EXTERNAL_PROVO = True
PROVO_URL = "https://www.w3.org/ns/prov-o.ttl"

provo_summary = pd.DataFrame()
provo_rows_df = pd.DataFrame()

if RUN_EXTERNAL_PROVO:
    try:
        from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef

        pg = Graph()
        pg.parse(PROVO_URL, format="turtle")

        PROV = Namespace("http://www.w3.org/ns/prov#")
        ROOT_URIS = {
            "Entity": PROV.Entity,
            "Activity": PROV.Activity,
            "Agent": PROV.Agent,
        }

        def _label(uri):
            vals = list(pg.objects(uri, RDFS.label))
            if vals:
                return str(vals[0])
            return str(uri).split("#")[-1]

        def _comment(uri):
            vals = list(pg.objects(uri, RDFS.comment))
            if vals:
                return str(vals[0]).strip()
            return ""

        # Build parent map over rdfs:subClassOf.
        parent_map = {}
        classes = set(pg.subjects(RDF.type, OWL.Class))
        classes |= set(pg.subjects(RDFS.subClassOf, None))
        classes |= set(pg.objects(None, RDFS.subClassOf))

        for c in classes:
            if not isinstance(c, URIRef):
                continue
            parent_map[c] = [
                p for p in pg.objects(c, RDFS.subClassOf)
                if isinstance(p, URIRef)
            ]

        def _ancestors(uri, max_hops=12):
            seen = set()
            frontier = [uri]
            for _ in range(max_hops):
                new = []
                for x in frontier:
                    for p in parent_map.get(x, []):
                        if p not in seen:
                            seen.add(p)
                            new.append(p)
                frontier = new
                if not frontier:
                    break
            return seen

        external_items = []

        for c in sorted(classes, key=str):
            if not isinstance(c, URIRef):
                continue
            if c in ROOT_URIS.values():
                continue

            anc = _ancestors(c)
            roots = [
                name for name, root_uri in ROOT_URIS.items()
                if root_uri in anc
            ]

            # Keep unambiguous descendants with human-authored comments.
            comment = _comment(c)
            label = _label(c)

            if len(roots) == 1 and len(comment) >= 35:
                external_items.append({
                    "uri": c,
                    "label": label,
                    "comment": comment,
                    "root": roots[0],
                })

        # Avoid extremely large/duplicated diagnostics.
        # Deterministic ordering keeps the test reproducible.
        external_items = external_items[:80]

        if len(external_items) < 6:
            raise RuntimeError(
                f"Too few usable PROV-O subclasses: {len(external_items)}"
            )

        leaf_labels = [
            x["label"] for x in external_items
        ]
        leaf_proto = semantic_encode([
            f"PROV-O concept {x['label']}."
            for x in external_items
        ])
        leaf_proto = leaf_proto / np.maximum(
            np.linalg.norm(leaf_proto, axis=1, keepdims=True), 1e-12
        )

        root_names = list(ROOT_URIS)
        root_proto = semantic_encode([
            f"PROV-O {r} concept."
            for r in root_names
        ])
        root_proto = root_proto / np.maximum(
            np.linalg.norm(root_proto, axis=1, keepdims=True), 1e-12
        )

        query_emb = semantic_encode([
            x["comment"] for x in external_items
        ])
        query_emb = query_emb / np.maximum(
            np.linalg.norm(query_emb, axis=1, keepdims=True), 1e-12
        )

        leaf_sim = query_emb @ leaf_proto.T
        root_sim = query_emb @ root_proto.T

        provo_rows = []

        for i, item in enumerate(external_items):
            pred_leaf_i = int(np.argmax(leaf_sim[i]))
            pred_leaf = external_items[pred_leaf_i]

            direct_root = root_names[int(np.argmax(root_sim[i]))]
            graph_root = pred_leaf["root"]

            provo_rows.append({
                "label": item["label"],
                "gold_root": item["root"],
                "comment": item["comment"],
                "pred_leaf": pred_leaf["label"],
                "leaf_grounding_correct": int(
                    pred_leaf["label"] == item["label"]
                ),
                "direct_text_root": direct_root,
                "direct_text_root_correct": int(
                    direct_root == item["root"]
                ),
                "graph_canonical_root": graph_root,
                "graph_canonical_root_correct": int(
                    graph_root == item["root"]
                ),
            })

        provo_rows_df = pd.DataFrame(provo_rows)

        provo_summary = pd.DataFrame([{
            "n": len(provo_rows_df),
            "leaf_grounding_accuracy": float(
                provo_rows_df["leaf_grounding_correct"].mean()
            ),
            "direct_text_root_accuracy": float(
                provo_rows_df["direct_text_root_correct"].mean()
            ),
            "ground_then_graph_root_accuracy": float(
                provo_rows_df[
                    "graph_canonical_root_correct"
                ].mean()
            ),
        }])

        print("\n[44] W3C PROV-O HUMAN-AUTHORED ONTOLOGY DIAGNOSTIC")
        display(provo_summary.round(4))

        provo_rows_df.to_csv(
            FINAL_TEST_DIR / "provo_external_ontology_results.csv",
            index=False
        )
        provo_summary.to_csv(
            FINAL_TEST_DIR / "provo_external_ontology_summary.csv",
            index=False
        )

    except Exception as exc:
        print(
            "PROV-O diagnostic skipped:",
            type(exc).__name__,
            str(exc)[:300]
        )



# =============================================================================
# 45/46 — G0 reader failure taxonomy + headline uncertainty audit
# =============================================================================

# -------------------------------------------------------------------------
# A. G0 arbitrary-reader failure taxonomy
# -------------------------------------------------------------------------

reader_failure_taxonomy = pd.DataFrame()
reader_failure_examples = pd.DataFrame()

if "reader_raw" in globals() and len(reader_raw):
    rr = reader_raw.copy()

    # Defensive columns for interrupted/legacy result files.
    defaults = {
        "parse_success": 0,
        "semantic_label_validity": 0,
        "task_success": 0,
        "overflow": 0,
        "error": "",
        "method": "UNKNOWN",
        "reader": "UNKNOWN",
        "split": "UNKNOWN",
    }
    for col, default in defaults.items():
        if col not in rr.columns:
            rr[col] = default

    rr["error_flag"] = rr["error"].fillna("").astype(str).str.len().gt(0).astype(int)
    rr["overflow_flag"] = rr["overflow"].fillna(0).astype(int)
    rr["parse_failure"] = (rr["parse_success"].fillna(0).astype(float) < 1).astype(int)
    rr["label_failure"] = (
        (rr["parse_success"].fillna(0).astype(float) >= 1)
        & (rr["semantic_label_validity"].fillna(0).astype(float) < 1)
    ).astype(int)
    rr["semantic_or_task_failure"] = (
        (rr["parse_success"].fillna(0).astype(float) >= 1)
        & (rr["semantic_label_validity"].fillna(0).astype(float) >= 1)
        & (rr["task_success"].fillna(0).astype(float) < 1)
    ).astype(int)

    reader_failure_taxonomy = (
        rr.groupby(["reader", "method", "split"], as_index=False)
        .agg(
            n=("task_success", "size"),
            error_rate=("error_flag", "mean"),
            overflow_rate=("overflow_flag", "mean"),
            parse_failure_rate=("parse_failure", "mean"),
            label_failure_rate=("label_failure", "mean"),
            semantic_or_task_failure_rate=("semantic_or_task_failure", "mean"),
            task_success=("task_success", "mean"),
        )
    )

    failmask = (
        (rr["error_flag"] == 1)
        | (rr["overflow_flag"] == 1)
        | (rr["parse_failure"] == 1)
        | (rr["label_failure"] == 1)
        | (rr["semantic_or_task_failure"] == 1)
    )

    keep_cols = [
        c for c in [
            "reader", "method", "split", "base_id",
            "composition_id", "parse_success",
            "semantic_label_validity", "task_success",
            "overflow", "error", "raw_output"
        ]
        if c in rr.columns
    ]

    reader_failure_examples = rr.loc[
        failmask, keep_cols
    ].head(100)

    print("\n[45A] G0 / ARBITRARY-READER FAILURE TAXONOMY")
    display(
        reader_failure_taxonomy.sort_values(
            ["task_success", "parse_failure_rate"]
        ).head(40).round(4)
    )

    reader_failure_taxonomy.to_csv(
        FINAL_TEST_DIR / "reader_failure_taxonomy.csv",
        index=False
    )
    reader_failure_examples.to_csv(
        FINAL_TEST_DIR / "reader_failure_examples.csv",
        index=False
    )
else:
    print("reader_raw unavailable; G0 taxonomy skipped.")

# -------------------------------------------------------------------------
# B. Grouped bootstrap confidence intervals
# -------------------------------------------------------------------------

def grouped_bootstrap_metric(
    df,
    group_col,
    metric_fn,
    n_boot=1000,
    seed=2026
):
    rng = np.random.default_rng(seed)
    groups = np.asarray(sorted(df[group_col].astype(str).unique()))
    if len(groups) < 2:
        return np.nan, np.nan, np.nan

    point = float(metric_fn(df))
    vals = []

    by_group = {
        g: df[df[group_col].astype(str) == g]
        for g in groups
    }

    for _ in range(n_boot):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        parts = []
        for j, g in enumerate(sampled):
            x = by_group[g].copy()
            # Make duplicated bootstrap groups unique for downstream GroupKFold.
            x["_boot_group"] = f"{j}::{g}"
            parts.append(x)

        b = pd.concat(parts, ignore_index=True)
        try:
            v = float(metric_fn(b))
            if np.isfinite(v):
                vals.append(v)
        except Exception:
            pass

    if not vals:
        return point, np.nan, np.nan

    return (
        point,
        float(np.quantile(vals, 0.025)),
        float(np.quantile(vals, 0.975)),
    )

def wilson_interval(successes, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = successes / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (
        z * math.sqrt(
            (p*(1-p)/n) + (z*z/(4*n*n))
        ) / denom
    )
    return p, center-half, center+half

uncertainty_rows = []

# Active-SCD controlled failure AUROC.
if "drift_df" in globals() and len(drift_df):
    def _drift_auc_metric(d):
        return _safe_auc_local(
            d["failure"],
            d["prepolicy_active_scd"]
        )

    p, lo, hi = grouped_bootstrap_metric(
        drift_df,
        "base_id",
        _drift_auc_metric,
        n_boot=1000,
        seed=SEED
    )
    uncertainty_rows.append({
        "metric": "Controlled Active-SCD failure AUROC",
        "point": p, "ci_low": lo, "ci_high": hi,
        "grouping": "base_id",
    })

# Strict router OOF AUROC.
if "strict_router_df" in globals() and len(strict_router_df):
    rtmp = strict_router_df.dropna(subset=["p_unseen_oof"]).copy()

    def _router_auc_metric(d):
        return _safe_auc_local(
            d["is_unseen"],
            d["p_unseen_oof"]
        )

    p, lo, hi = grouped_bootstrap_metric(
        rtmp,
        "group",
        _router_auc_metric,
        n_boot=1000,
        seed=SEED+1
    )
    uncertainty_rows.append({
        "metric": "Strict router OOF AUROC",
        "point": p, "ci_low": lo, "ci_high": hi,
        "grouping": "router group",
    })

# Naturalistic drift failure AUROC.
if "naturalistic_df" in globals() and len(naturalistic_df):
    def _nat_auc_metric(d):
        return _safe_auc_local(
            d["scale_failure"],
            d["prepolicy_active_scd"]
        )

    p, lo, hi = grouped_bootstrap_metric(
        naturalistic_df,
        "base_id",
        _nat_auc_metric,
        n_boot=1000,
        seed=SEED+2
    )
    uncertainty_rows.append({
        "metric": "Naturalistic version-drift failure AUROC",
        "point": p, "ci_low": lo, "ci_high": hi,
        "grouping": "base_id",
    })

# Independent learned executor, from previous reviewer-hardening cell.
if "independent_executor_df" in globals() and len(independent_executor_df):
    for executor, g in independent_executor_df.groupby("executor"):
        def _ind_auc_metric(d):
            return _safe_auc_local(
                d["failure"],
                d["prepolicy_active_scd"]
            )

        p, lo, hi = grouped_bootstrap_metric(
            g,
            "base_id",
            _ind_auc_metric,
            n_boot=1000,
            seed=SEED+3
        )
        uncertainty_rows.append({
            "metric": f"{executor} Active-SCD failure AUROC",
            "point": p, "ci_low": lo, "ci_high": hi,
            "grouping": "base_id",
        })

# LLM executor baseline-correct conditioned AUROC.
if "llm_executor_df" in globals() and len(llm_executor_df):
    for reader, g in llm_executor_df.groupby("reader"):
        clean = g[g["severity"] == 0][
            ["base_id", "task_success"]
        ].rename(columns={"task_success": "clean_success"})
        gg = g.merge(clean, on="base_id", how="left")
        gg = gg[gg["clean_success"] == 1].copy()

        if len(gg) and gg["failure"].nunique() >= 2:
            def _llm_auc_metric(d):
                return _safe_auc_local(
                    d["failure"],
                    d["prepolicy_active_scd"]
                )

            p, lo, hi = grouped_bootstrap_metric(
                gg,
                "base_id",
                _llm_auc_metric,
                n_boot=600,
                seed=SEED+4
            )
            uncertainty_rows.append({
                "metric": f"{reader} LLM executor Active-SCD AUROC",
                "point": p, "ci_low": lo, "ci_high": hi,
                "grouping": "baseline-correct base_id",
            })

# Semantic intervention rescue — exact small-sample interval.
if "intervention_summary" in globals() and len(intervention_summary):
    # Use the original detailed rows to avoid relying on summary schema.
    if "intervention_df" in globals():
        candidates = intervention_df[
            (intervention_df["intervention_type"] == "gold_active_correction")
            & (intervention_df["n_active_intervened"] == 3)
            & (intervention_df["baseline_success"] == 0)
        ]
        if len(candidates):
            successes = int(candidates["post_success"].sum())
            n = len(candidates)
            p, lo, hi = wilson_interval(successes, n)
            uncertainty_rows.append({
                "metric": f"Full active intervention rescue ({successes}/{n})",
                "point": p, "ci_low": lo, "ci_high": hi,
                "grouping": "failure case",
            })

uncertainty_df = pd.DataFrame(uncertainty_rows)

print("\n[45B] HEADLINE UNCERTAINTY AUDIT")
display(uncertainty_df.round(4))

uncertainty_df.to_csv(
    FINAL_TEST_DIR / "headline_uncertainty_intervals.csv",
    index=False
)



# =============================================================================
# 46/46 — Final paper-facing evidence report
# =============================================================================

def _fmt(x):
    try:
        if x is None or not np.isfinite(float(x)):
            return "NA"
        return f"{float(x):.4f}"
    except Exception:
        return "NA"

# Collect key results defensively.
strict_router_auc_r = (
    strict_router_metrics.iloc[0]["router_auc"]
    if "strict_router_metrics" in globals() and len(strict_router_metrics)
    else np.nan
)
strict_router_bal_r = (
    strict_router_metrics.iloc[0]["balanced_accuracy"]
    if "strict_router_metrics" in globals() and len(strict_router_metrics)
    else np.nan
)
strict_router_rec_r = (
    strict_router_metrics.iloc[0]["unseen_recall"]
    if "strict_router_metrics" in globals() and len(strict_router_metrics)
    else np.nan
)
strict_router_fp_r = (
    strict_router_metrics.iloc[0]["known_false_open_rate"]
    if "strict_router_metrics" in globals() and len(strict_router_metrics)
    else np.nan
)

nat_auc_r = (
    nat_failure_auc
    if "nat_failure_auc" in globals()
    else np.nan
)

llm_lines = []
if "llm_executor_summary" in globals() and len(llm_executor_summary):
    for _, r in llm_executor_summary.iterrows():
        llm_lines.append(
            f"- {r['reader']}: clean accuracy={_fmt(r['clean_accuracy'])}, "
            f"Active-SCD AUROC on baseline-correct bases="
            f"{_fmt(r['failure_auc_active_scd_baseline_correct'])}, "
            f"full-SCD={_fmt(r['failure_auc_full_scd_baseline_correct'])}, "
            f"confidence={_fmt(r['failure_auc_confidence_baseline_correct'])}."
        )
else:
    llm_lines.append("- LLM executor test unavailable or skipped.")

provo_line = "PROV-O external diagnostic unavailable or skipped."
if "provo_summary" in globals() and len(provo_summary):
    r = provo_summary.iloc[0]
    provo_line = (
        f"W3C PROV-O exploratory diagnostic: n={int(r['n'])}, "
        f"leaf grounding={_fmt(r['leaf_grounding_accuracy'])}, "
        f"direct text-to-root={_fmt(r['direct_text_root_accuracy'])}, "
        f"ground-then-graph root={_fmt(r['ground_then_graph_root_accuracy'])}."
    )

report = f"""
# SCALE — Final Additional Test Report

## 1. Corrected deployable hybrid routing

The strict identifier matcher requires an exact normalized match to one lexical
identifier span. It no longer concatenates the entire sentence before matching.

- strict router OOF AUROC: **{_fmt(strict_router_auc_r)}**
- balanced accuracy: **{_fmt(strict_router_bal_r)}**
- unseen recall: **{_fmt(strict_router_rec_r)}**
- known false-open rate: **{_fmt(strict_router_fp_r)}**

The semantic-alias false-ID assertion is enforced in code. If this cell completes,
the previously observed `semantic_alias -> symbolic_id` anomaly has been removed.

**Paper use:** Method 4.2 may now describe a deployable three-way route:
explicit ID -> deterministic normalizer; otherwise learned known/open-world router;
known -> CB+DualView; open-world -> SCALE ontology branch.

## 2. Naturalistic version evolution

Active-SCD failure AUROC on version-style prompt/schema/semantic evolution:
**{_fmt(nat_auc_r)}**

This experiment complements the original corruption families with prompt rewrites,
schema releases, authority-boundary changes, evidence reinterpretation, object
taxonomy changes, and compound releases.

**Paper use:** report as a robustness test, not as longitudinal production evidence.

## 3. Independent LLM downstream executors

{chr(10).join(llm_lines)}

These executors never receive SCALE contracts or SCD values.

**Paper use:** if Active-SCD remains predictive after conditioning on clean-correct
bases, this is the strongest response to the concern that the original 0.988/0.996
AUROC is mechanically induced by the deterministic semantic policy.

## 4. Human-authored ontology sanity check

{provo_line}

**Paper use:** exploratory external ontology evidence only. Do not call this a
full external SCALE benchmark because SCALE is not retrained on PROV-O.

## 5. G0 failure taxonomy

See:
`reader_failure_taxonomy.csv`

Use the taxonomy to distinguish:
- generation/runtime error,
- overflow,
- parse failure,
- vocabulary/semantic-label validity failure,
- valid parse + valid vocabulary but wrong downstream semantics/task.

**Paper use:** describe G0 as an arbitrary-reader robustness boundary, not as a
native SCALE interface failure.

## 6. Statistical uncertainty

See:
`headline_uncertainty_intervals.csv`

Grouped bootstrap uses task/base groups rather than treating correlated drift rows
as IID. Small intervention rescue counts use Wilson intervals.

## Final recommended claim discipline

### Strongly supported
- role-active semantic compatibility under controlled C1/C2 shifts;
- structural canonicalization of opaque unseen concepts with ontology propagation;
- deterministic runtime repair/blocking;
- Active-SCD as a failure-risk signal;
- mechanistic active-concept intervention;
- external CRM semantic-contract transfer;
- communication-cost advantage over Onto-RAG.

### Supported with qualification
- deployable non-oracle hybrid routing;
- Active-SCD transfer to independent learned/LLM executors;
- naturalistic version-drift robustness;
- real-producer recovery conditional on producer semantic fidelity.

### Not supported
- universal arbitrary-reader robustness (G0);
- general label-efficient directed invariance (G2);
- differentiable-logic benefit;
- universal ontology superiority;
- universal semantic-alias superiority over CB+DualView;
- consistent external end-task superiority;
- perfect unseen/open-set detection.

## Stop condition for experiments

If:
1. strict routing removes alias false-ID matches,
2. independent LLM-executor Active-SCD remains materially above chance, and
3. naturalistic version drift remains predictive,

then the paper has enough mechanism evidence. Additional synthetic baselines or
larger procedural ontologies are unlikely to add comparable value.

A truly independent, domain-specific longitudinal ontology/deployment benchmark
would still strengthen external validity, but it should be future work unless it can
be added cleanly without changing the paper's scope.
"""

(FINAL_TEST_DIR / "FINAL_ADDITIONAL_TEST_REPORT.md").write_text(
    report.strip() + "\n",
    encoding="utf-8"
)

from IPython.display import display, Markdown
display(Markdown(report))

print("\nSaved final test artifacts:")
for p in sorted(FINAL_TEST_DIR.iterdir()):
    print(" -", p.name)




In [ ]:

# =============================================================================
# FINAL PAPER-CLOSURE TESTS — SINGLE CONTINUATION SECTION
# Run only after every prior notebook cell has completed.
# =============================================================================

from pathlib import Path
import copy, gc, json, math, os, re, time, warnings
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)

from IPython.display import display, Markdown

PAPER_CLOSURE_DIR = ROOT / "paper_closure_tests"
PAPER_CLOSURE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SCALE — FINAL PAPER-CLOSURE TESTS")
print("Output:", PAPER_CLOSURE_DIR)
print("=" * 100)

# -----------------------------------------------------------------------------
# 0. PRE-FLIGHT
# -----------------------------------------------------------------------------

REQUIRED_PAPER_CLOSURE_OBJECTS = [
    "MODELS",
    "BASE_GRAPH",
    "LARGE_OPAQUE_GRAPH",
    "LARGE_OPAQUE_NODES",
    "large_ontology_message",
    "learned_large_prediction",
    "ensemble_predict",
    "realize_message",
    "shift_test_base",
    "dev_base",
    "train_base",
    "drift_base",
    "drift_df",
    "drift_messages",
    "semantic_encode",
    "dual_view_encode",
    "_router_features",
    "strict_explicit_id_lookup",
    "strict_router_threshold",
    "STRICT_FINAL_ROUTER",
    "canonicalize_lookup",
    "grouped_auc",
]

_missing = [
    x for x in REQUIRED_PAPER_CLOSURE_OBJECTS
    if x not in globals()
]
assert not _missing, (
    "Run the previous notebook cells first. Missing objects: "
    + ", ".join(_missing)
)

def pc_safe_auc(y, score):
    y = np.asarray(y, int)
    score = np.asarray(score, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, score))

def pc_wilson(successes, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = successes / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * math.sqrt(
        p*(1-p)/n + z*z/(4*n*n)
    ) / denom
    return float(p), float(center-half), float(center+half)

def pc_bootstrap_auc(
    df,
    y_col,
    score_col,
    group_col,
    n_boot=1000,
    seed=2026
):
    point = pc_safe_auc(df[y_col], df[score_col])
    groups = np.asarray(
        sorted(df[group_col].astype(str).unique())
    )

    if len(groups) < 2:
        return point, np.nan, np.nan

    rng = np.random.default_rng(seed)
    by_group = {
        g: df[df[group_col].astype(str) == g]
        for g in groups
    }
    vals = []

    for _ in range(n_boot):
        sample = rng.choice(
            groups,
            size=len(groups),
            replace=True
        )
        b = pd.concat(
            [by_group[g] for g in sample],
            ignore_index=True
        )
        v = pc_safe_auc(b[y_col], b[score_col])
        if np.isfinite(v):
            vals.append(v)

    if not vals:
        return point, np.nan, np.nan

    return (
        point,
        float(np.quantile(vals, 0.025)),
        float(np.quantile(vals, 0.975)),
    )

# =============================================================================
# 1. ARCHITECTURE-FAITHFUL DEPLOYABLE ROUTING
# =============================================================================
#
# Previous reviewer-hardening code intentionally used CB+DualView as a strong
# discriminative control for the known branch. That is useful as a baseline, but
# the paper states that the deployed SCALE architecture uses its OWN discriminative
# known-class head. Therefore the definitive deployable route must use Full SCALE
# on known inputs.
#
# Route:
#   explicit recoverable ID -> deterministic canonicalizer
#   otherwise:
#       p(open-world) >= tau -> SCALE ontology branch
#       p(open-world) <  tau -> Full SCALE known-class branch
# =============================================================================

def scale_hybrid_route_architecture_faithful(
    text,
    graph_ctx=LARGE_OPAQUE_GRAPH,
    threshold=None,
):
    if threshold is None:
        threshold = float(strict_router_threshold)

    # A. Explicit identifier path.
    id_hit = strict_explicit_id_lookup(text, graph_ctx)
    if id_hit is not None:
        return {
            "route": "symbolic_id",
            "predicted_node": id_hit,
            "canonical_root": canonicalize_lookup(
                id_hit, graph_ctx
            ),
            "p_unseen": 1.0,
        }

    # B. Learned open-set routing.
    p_unseen = float(
        STRICT_FINAL_ROUTER.predict_proba(
            _router_features(text).reshape(1, -1)
        )[0, 1]
    )

    if p_unseen >= threshold:
        node, root = learned_large_prediction(
            "SCALE-Full",
            text,
            graph_ctx,
        )
        return {
            "route": "open_world_ontology",
            "predicted_node": node,
            "canonical_root": root,
            "p_unseen": p_unseen,
        }

    # C. IMPORTANT: actual Full SCALE known decoder.
    pred = ensemble_predict(
        "SCALE",
        [text],
        BASE_GRAPH,
    )[0]
    root = pred["contract"].get("object")

    return {
        "route": "known_scale_decoder",
        "predicted_node": root,
        "canonical_root": root,
        "p_unseen": p_unseen,
    }

route_v2_rows = []

# Known C1/C2.
for base in shift_test_base:
    for impl in ["C1", "C2"]:
        txt = realize_message(
            base, "PLANNER", impl, 0
        )
        out_r = scale_hybrid_route_architecture_faithful(txt)

        route_v2_rows.append({
            "kind": "known",
            "base_or_node": base["base_id"],
            "surface": impl,
            "gold_root": base["object"],
            **out_r,
            "correct": int(
                out_r["canonical_root"] == base["object"]
            ),
        })

# Open-world four-surface stress.
PC_SURFACES = (
    list(LARGE_SURFACES)
    if "LARGE_SURFACES" in globals()
    else [
        "exact_id",
        "punctuated_id",
        "semantic_alias",
        "wrapped_id",
    ]
)

for node in LARGE_OPAQUE_NODES:
    for surface in PC_SURFACES:
        txt = large_ontology_message(
            node, surface
        )
        out_r = scale_hybrid_route_architecture_faithful(txt)

        route_v2_rows.append({
            "kind": "unseen",
            "base_or_node": node["label"],
            "surface": surface,
            "gold_root": node["canonical_root"],
            **out_r,
            "correct": int(
                out_r["canonical_root"]
                == node["canonical_root"]
            ),
        })

# Existing harder alias set, if available.
if "HARD_ALIAS_BY_ROOT" in globals():
    for root, aliases in HARD_ALIAS_BY_ROOT.items():
        for j, alias in enumerate(aliases):
            txt = (
                "Planner handoff: the target is "
                + alias
                + "."
            )
            out_r = scale_hybrid_route_architecture_faithful(txt)

            route_v2_rows.append({
                "kind": "unseen",
                "base_or_node": f"{root}::hard::{j}",
                "surface": "hard_semantic_alias",
                "gold_root": root,
                **out_r,
                "correct": int(
                    out_r["canonical_root"] == root
                ),
            })

route_v2_df = pd.DataFrame(route_v2_rows)

route_v2_summary = (
    route_v2_df
    .groupby(
        ["kind", "surface", "route"],
        as_index=False
    )
    .agg(
        canonical_root_accuracy=("correct", "mean"),
        n=("correct", "size"),
        mean_p_unseen=("p_unseen", "mean"),
    )
)

print("\n[1A] ARCHITECTURE-FAITHFUL DEPLOYABLE ROUTING")
display(route_v2_summary.round(4))

# Explicit regression checks.
_alias_symbolic = route_v2_df[
    route_v2_df["surface"].isin(
        ["semantic_alias", "hard_semantic_alias"]
    )
    & (route_v2_df["route"] == "symbolic_id")
]
assert len(_alias_symbolic) == 0, (
    "Regression: semantic alias entered symbolic ID route."
)

route_v2_df.to_csv(
    PAPER_CLOSURE_DIR /
    "architecture_faithful_routing.csv",
    index=False,
)
route_v2_summary.to_csv(
    PAPER_CLOSURE_DIR /
    "architecture_faithful_routing_summary.csv",
    index=False,
)

# Main paper checks.
known_c1_v2 = float(
    route_v2_df[
        (route_v2_df["kind"] == "known")
        & (route_v2_df["surface"] == "C1")
    ]["correct"].mean()
)
known_c2_v2 = float(
    route_v2_df[
        (route_v2_df["kind"] == "known")
        & (route_v2_df["surface"] == "C2")
    ]["correct"].mean()
)

# =============================================================================
# 2. LEAVE-ONE-ONTOLOGY-ROOT-OUT ROUTER GENERALIZATION
# =============================================================================
#
# Broad bootstrap intervals on the earlier router are partly caused by only a few
# semantically distinct ontology roots. Instead of pretending that more bootstrap
# resamples create more semantic diversity, this test asks the stronger question:
#
#   Can the router recognize an unseen semantic extension whose CANONICAL ROOT
#   was never represented among the open-world examples used to fit the router?
#
# Known-side training examples come from dev bases; known-side evaluation examples
# come from shift-test bases. Open-world training uses four roots and evaluation
# uses the fifth held-out root.
# =============================================================================

PC_ROOTS = sorted(
    {n["canonical_root"] for n in LARGE_OPAQUE_NODES}
)

def _pc_known_rows(bases, tag):
    rows = []
    impls = ["A", "B", "C1", "C2"]
    for i, base in enumerate(bases):
        impl = impls[i % len(impls)]
        txt = realize_message(
            base, "PLANNER", impl, 0
        )
        rows.append({
            "label": 0,
            "root": "KNOWN",
            "group": f"{tag}::{base['base_id']}",
            "text": txt,
        })
    return rows

known_train_rows = _pc_known_rows(
    dev_base,
    "known_train",
)
known_test_rows = _pc_known_rows(
    shift_test_base,
    "known_test",
)

open_rows = []
for node in LARGE_OPAQUE_NODES:
    open_rows.append({
        "label": 1,
        "root": node["canonical_root"],
        "group": f"node::{node['label']}",
        "text": large_ontology_message(
            node, "semantic_alias"
        ),
    })

# Add hard aliases, but keep them root-aware.
if "HARD_ALIAS_BY_ROOT" in globals():
    for root, aliases in HARD_ALIAS_BY_ROOT.items():
        for j, alias in enumerate(aliases):
            open_rows.append({
                "label": 1,
                "root": root,
                "group": f"hard::{root}::{j}",
                "text": (
                    "Planner handoff: the target is "
                    + alias
                    + "."
                ),
            })

router_root_generalization_rows = []

for held_root in PC_ROOTS:
    train_rows = list(known_train_rows) + [
        r for r in open_rows
        if r["root"] != held_root
    ]
    test_rows = list(known_test_rows) + [
        r for r in open_rows
        if r["root"] == held_root
    ]

    trdf = pd.DataFrame(train_rows)
    tedf = pd.DataFrame(test_rows)

    Xtr = np.stack([
        _router_features(t)
        for t in trdf["text"]
    ])
    ytr = trdf["label"].to_numpy(int)

    Xte = np.stack([
        _router_features(t)
        for t in tedf["text"]
    ])
    yte = tedf["label"].to_numpy(int)

    root_router = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])
    root_router.fit(Xtr, ytr)

    tr_score = root_router.predict_proba(
        Xtr
    )[:, 1]
    te_score = root_router.predict_proba(
        Xte
    )[:, 1]

    # Threshold chosen only on training rows.
    thresholds = np.unique(tr_score)
    best_t, best_b = 0.5, -1.0
    for t in thresholds:
        pred = (
            tr_score >= t
        ).astype(int)
        b = balanced_accuracy_score(
            ytr, pred
        )
        if b > best_b:
            best_t, best_b = float(t), float(b)

    te_pred = (
        te_score >= best_t
    ).astype(int)

    root_router_auc = pc_safe_auc(
        yte, te_score
    )
    root_balacc = float(
        balanced_accuracy_score(
            yte, te_pred
        )
    )

    unseen_mask = yte == 1
    known_mask = yte == 0

    router_root_generalization_rows.append({
        "held_out_root": held_root,
        "n_train": len(trdf),
        "n_test": len(tedf),
        "threshold_from_train": best_t,
        "test_auc": root_router_auc,
        "test_balanced_accuracy": root_balacc,
        "held_root_unseen_recall": float(
            te_pred[unseen_mask].mean()
        ),
        "known_false_open_rate": float(
            te_pred[known_mask].mean()
        ),
    })

router_root_generalization_df = pd.DataFrame(
    router_root_generalization_rows
)

router_root_aggregate = pd.DataFrame([{
    "mean_auc": float(
        router_root_generalization_df[
            "test_auc"
        ].mean()
    ),
    "min_auc": float(
        router_root_generalization_df[
            "test_auc"
        ].min()
    ),
    "mean_balanced_accuracy": float(
        router_root_generalization_df[
            "test_balanced_accuracy"
        ].mean()
    ),
    "min_unseen_recall": float(
        router_root_generalization_df[
            "held_root_unseen_recall"
        ].min()
    ),
    "max_known_false_open_rate": float(
        router_root_generalization_df[
            "known_false_open_rate"
        ].max()
    ),
}])

print("\n[2] LEAVE-ONE-ONTOLOGY-ROOT-OUT ROUTER")
display(router_root_generalization_df.round(4))
display(router_root_aggregate.round(4))

router_root_generalization_df.to_csv(
    PAPER_CLOSURE_DIR /
    "router_leave_one_root_out.csv",
    index=False,
)
router_root_aggregate.to_csv(
    PAPER_CLOSURE_DIR /
    "router_leave_one_root_out_summary.csv",
    index=False,
)

# =============================================================================
# 3. COMPETENCE-GATED INDEPENDENT LLM EXECUTOR
# =============================================================================
#
# The previous direct LLM executor test was not paper-ready because Granite and
# Qwen often failed even at severity 0. A drift robustness test is uninterpretable
# if the executor does not understand the clean task.
#
# This corrected protocol:
#
#   Stage A — TASK ADAPTATION
#       The LLM receives the downstream decision policy and a few clean demonstrations.
#       It still NEVER receives SCALE contracts, Active-SCD, ontology predictions,
#       or gold concepts at test time.
#
#   Stage B — CLEAN COMPETENCE GATE
#       Evaluate unseen clean dev tasks.
#       Required clean exact action+authority accuracy >= 0.70.
#
#   Stage C — DRIFT ROBUSTNESS
#       Only competent executors are evaluated under drift.
#       AUROC is computed after additionally conditioning on bases solved at severity 0.
#
# This separates "the executor cannot do the task" from "semantic drift broke the task".
# =============================================================================

RUN_COMPETENCE_GATED_LLM_EXECUTOR = True
LLM_CLEAN_GATE = 0.70
LLM_DEMO_COUNT = min(8, len(train_base))
LLM_DEV_COUNT = min(24, len(dev_base))
LLM_DRIFT_COUNT = min(24, len(drift_base))

PC_LLM_RESULT_FILE = (
    PAPER_CLOSURE_DIR /
    "competence_gated_llm_executor.jsonl"
)

def _pc_compact_handoffs(base, impl):
    msgs = {
        a: realize_message(
            base, a, impl, 0
        )
        for a in AGENTS
    }
    return msgs

def _pc_handoff_block(msgs, max_chars=420):
    return "\n".join(
        f"[{a}] {str(msgs[a])[:max_chars]}"
        for a in AGENTS
    )

# Explicit downstream policy is legitimate executor specification.
PC_EXECUTOR_POLICY = """
You are the final workflow executor.

Infer the operational object, evidence status, and authority from the three raw
agent handoffs and apply this workflow policy.

Evidence rule:
- blocking/failed evidence -> REJECT
- unresolved/uncertain evidence -> VERIFY
- passing evidence -> object-specific action

Object-specific passing action:
- purchase/procurement -> APPROVE
- privileged access -> APPROVE
- configuration/deployment change -> UPDATE
- service/security incident -> SEND
- compliance/control item -> APPROVE

Minimum authority:
- APPROVE or REJECT requires APPROVE authority
- UPDATE or SEND requires EXECUTE authority
- VERIFY requires RECOMMEND authority

If the inferred authority is lower than the action requires, output VERIFY instead.
Return the authority actually conveyed by the Policy handoff.

Return exactly:
{"action":"APPROVE|UPDATE|SEND|REJECT|VERIFY",
 "authority":"RECOMMEND|APPROVE|EXECUTE"}

No explanation and no markdown.
""".strip()

def _pc_build_demos():
    impl_cycle = [
        "A", "B", "C1", "C2"
    ]
    demos = []

    for i, base in enumerate(
        train_base[:LLM_DEMO_COUNT]
    ):
        impl = impl_cycle[
            i % len(impl_cycle)
        ]
        msgs = _pc_compact_handoffs(
            base, impl
        )

        demos.append(
            "EXAMPLE INPUT\n"
            + _pc_handoff_block(msgs, 260)
            + "\nEXAMPLE OUTPUT\n"
            + json.dumps({
                "action": base["final_action"],
                "authority": base["final_authority"],
            })
        )

    return "\n\n".join(demos)

PC_EXECUTOR_DEMOS = _pc_build_demos()

def pc_llm_executor_messages(raw_msgs):
    return [
        {
            "role": "system",
            "content": (
                PC_EXECUTOR_POLICY
                + "\n\n"
                + "Training examples:\n"
                + PC_EXECUTOR_DEMOS
            ),
        },
        {
            "role": "user",
            "content": (
                "TEST INPUT\n"
                + _pc_handoff_block(
                    raw_msgs, 420
                )
            ),
        },
    ]

def pc_parse_executor(text):
    obj = extract_json(text)

    if not isinstance(obj, dict):
        return None, None, False

    action = str(
        obj.get("action", "")
    ).strip().upper()
    authority = str(
        obj.get("authority", "")
    ).strip().upper()

    valid_actions = {
        "APPROVE",
        "UPDATE",
        "SEND",
        "REJECT",
        "VERIFY",
    }
    valid_auth = {
        "RECOMMEND",
        "APPROVE",
        "EXECUTE",
    }

    ok = (
        action in valid_actions
        and authority in valid_auth
    )
    return action, authority, bool(ok)

def pc_append_jsonl(path, row):
    with open(
        path, "a", encoding="utf-8"
    ) as f:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + "\n"
        )

def pc_read_jsonl(path):
    if not path.exists():
        return pd.DataFrame()
    rows = []
    with open(
        path, "r", encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(
                    json.loads(line)
                )
            except Exception:
                pass
    return pd.DataFrame(rows)

def _pc_generate_executor(
    tok, mdl, spec, raw_msgs
):
    return generate_text(
        tok,
        mdl,
        pc_llm_executor_messages(
            raw_msgs
        ),
        spec,
        max_input=1800,
        max_new=48,
    )

llm_competence_rows = []

if RUN_COMPETENCE_GATED_LLM_EXECUTOR:
    # Qwen and Granite are enough for a cross-family check.
    PC_EXECUTOR_MODELS = READER_MODELS[:2]

    existing_pc = pc_read_jsonl(
        PC_LLM_RESULT_FILE
    )
    already = set()

    if len(existing_pc):
        already = {
            (
                str(r["reader"]),
                str(r["stage"]),
                str(r["base_id"]),
                int(r.get("severity", -1)),
            )
            for _, r in existing_pc.iterrows()
        }

    for spec in PC_EXECUTOR_MODELS:
        mdl = None
        tok = None

        try:
            (
                model_id,
                primary_loaded,
                tok,
                mdl,
            ) = load_model_safe(spec)

            # --------------------------
            # A. Clean competence stage
            # --------------------------
            clean_impl_cycle = [
                "A", "B", "C1", "C2"
            ]

            for i, base in enumerate(
                dev_base[:LLM_DEV_COUNT]
            ):
                impl = clean_impl_cycle[
                    i % len(clean_impl_cycle)
                ]
                key = (
                    spec["label"],
                    "clean_gate",
                    base["base_id"],
                    -1,
                )
                if key in already:
                    continue

                msgs = _pc_compact_handoffs(
                    base, impl
                )
                gen = _pc_generate_executor(
                    tok, mdl, spec, msgs
                )

                action, authority, parse_ok = (
                    pc_parse_executor(
                        gen["text"]
                    )
                )

                success = int(
                    parse_ok
                    and action
                    == base["final_action"]
                    and authority
                    == base["final_authority"]
                )

                pc_append_jsonl(
                    PC_LLM_RESULT_FILE,
                    {
                        "reader": spec["label"],
                        "loaded_model": model_id,
                        "primary_model_loaded":
                            bool(primary_loaded),
                        "stage": "clean_gate",
                        "base_id": base["base_id"],
                        "implementation": impl,
                        "severity": -1,
                        "parse_ok": int(parse_ok),
                        "pred_action": action,
                        "pred_authority": authority,
                        "gold_action":
                            base["final_action"],
                        "gold_authority":
                            base["final_authority"],
                        "task_success": success,
                        "input_tokens":
                            int(gen["input_tokens"]),
                        "output_tokens":
                            int(gen["output_tokens"]),
                        "latency_s":
                            float(gen["latency_s"]),
                        "raw_output":
                            gen["text"][:500],
                    },
                )
                already.add(key)

            now = pc_read_jsonl(
                PC_LLM_RESULT_FILE
            )
            clean_reader = now[
                (now["reader"] == spec["label"])
                & (now["stage"] == "clean_gate")
            ]

            clean_accuracy = float(
                clean_reader[
                    "task_success"
                ].mean()
            )

            print(
                f"\nLLM competence gate — "
                f"{spec['label']}: "
                f"{clean_accuracy:.4f}"
            )

            # -----------------------------------------
            # B. Only competent executors see drift.
            # -----------------------------------------
            if clean_accuracy < LLM_CLEAN_GATE:
                print(
                    "  -> INCONCLUSIVE for drift: "
                    "clean competence gate not met."
                )
                continue

            for base in drift_base[:LLM_DRIFT_COUNT]:
                for severity in [
                    0, 1, 2, 3
                ]:
                    key = (
                        spec["label"],
                        "drift",
                        base["base_id"],
                        severity,
                    )
                    if key in already:
                        continue

                    raw_msgs, ops = drift_messages(
                        base, severity
                    )

                    gen = _pc_generate_executor(
                        tok,
                        mdl,
                        spec,
                        raw_msgs,
                    )

                    action, authority, parse_ok = (
                        pc_parse_executor(
                            gen["text"]
                        )
                    )

                    success = int(
                        parse_ok
                        and action
                        == base["final_action"]
                        and authority
                        == base["final_authority"]
                    )

                    drow = drift_df[
                        (
                            drift_df["base_id"]
                            == base["base_id"]
                        )
                        & (
                            drift_df["severity"]
                            == severity
                        )
                    ].iloc[0]

                    pc_append_jsonl(
                        PC_LLM_RESULT_FILE,
                        {
                            "reader":
                                spec["label"],
                            "loaded_model":
                                model_id,
                            "primary_model_loaded":
                                bool(primary_loaded),
                            "stage": "drift",
                            "base_id":
                                base["base_id"],
                            "implementation":
                                "drift",
                            "severity":
                                severity,
                            "operators":
                                "|".join(ops),
                            "parse_ok":
                                int(parse_ok),
                            "pred_action":
                                action,
                            "pred_authority":
                                authority,
                            "gold_action":
                                base["final_action"],
                            "gold_authority":
                                base["final_authority"],
                            "task_success":
                                success,
                            "failure":
                                1 - success,
                            "prepolicy_active_scd":
                                float(
                                    drow[
                                        "prepolicy_active_scd"
                                    ]
                                ),
                            "full_scd":
                                float(
                                    drow["full_scd"]
                                ),
                            "neg_confidence":
                                float(
                                    1.0
                                    - drow["confidence"]
                                ),
                            "input_tokens":
                                int(
                                    gen["input_tokens"]
                                ),
                            "output_tokens":
                                int(
                                    gen["output_tokens"]
                                ),
                            "latency_s":
                                float(
                                    gen["latency_s"]
                                ),
                            "raw_output":
                                gen["text"][:500],
                        },
                    )
                    already.add(key)

        finally:
            try:
                del mdl
            except Exception:
                pass
            try:
                del tok
            except Exception:
                pass
            cleanup_gpu()

pc_llm_df = pc_read_jsonl(
    PC_LLM_RESULT_FILE
)

pc_llm_summary_rows = []

if len(pc_llm_df):
    for reader in sorted(
        pc_llm_df["reader"].unique()
    ):
        clean = pc_llm_df[
            (pc_llm_df["reader"] == reader)
            & (
                pc_llm_df["stage"]
                == "clean_gate"
            )
        ].copy()

        clean_acc = (
            float(
                clean["task_success"].mean()
            )
            if len(clean)
            else np.nan
        )

        drift = pc_llm_df[
            (pc_llm_df["reader"] == reader)
            & (pc_llm_df["stage"] == "drift")
        ].copy()

        summary = {
            "reader": reader,
            "clean_n": len(clean),
            "clean_accuracy": clean_acc,
            "competence_gate": LLM_CLEAN_GATE,
            "gate_pass": int(
                np.isfinite(clean_acc)
                and clean_acc >= LLM_CLEAN_GATE
            ),
            "drift_n": len(drift),
            "baseline_correct_bases": 0,
            "active_scd_failure_auc": np.nan,
            "full_scd_failure_auc": np.nan,
            "confidence_failure_auc": np.nan,
        }

        if len(drift):
            baseline = drift[
                drift["severity"] == 0
            ][
                ["base_id", "task_success"]
            ].rename(
                columns={
                    "task_success":
                        "baseline_success"
                }
            )

            d2 = drift.merge(
                baseline,
                on="base_id",
                how="left",
            )
            d2 = d2[
                d2["baseline_success"] == 1
            ].copy()

            summary[
                "baseline_correct_bases"
            ] = int(
                d2["base_id"].nunique()
            )

            if (
                len(d2)
                and d2["failure"].nunique()
                >= 2
            ):
                summary[
                    "active_scd_failure_auc"
                ] = pc_safe_auc(
                    d2["failure"],
                    d2[
                        "prepolicy_active_scd"
                    ],
                )
                summary[
                    "full_scd_failure_auc"
                ] = pc_safe_auc(
                    d2["failure"],
                    d2["full_scd"],
                )
                summary[
                    "confidence_failure_auc"
                ] = pc_safe_auc(
                    d2["failure"],
                    d2["neg_confidence"],
                )

        pc_llm_summary_rows.append(
            summary
        )

pc_llm_summary = pd.DataFrame(
    pc_llm_summary_rows
)

print("\n[3] COMPETENCE-GATED LLM EXECUTORS")
if len(pc_llm_summary):
    display(pc_llm_summary.round(4))
else:
    print("No executor results.")

pc_llm_summary.to_csv(
    PAPER_CLOSURE_DIR /
    "competence_gated_llm_executor_summary.csv",
    index=False,
)

# =============================================================================
# 4. EXTERNAL HUMAN-AUTHORED ONTOLOGY — SCHEMA.ORG
# =============================================================================
#
# The earlier PROV-O diagnostic failed because the selected hierarchy contained
# too few unambiguous descendants. Schema.org provides a much broader, genuinely
# human-authored class hierarchy.
#
# This remains an EXTERNAL STRUCTURAL DIAGNOSTIC, not a claim that SCALE was
# trained or benchmarked end-to-end on Schema.org.
#
# Test:
#   human-written class description (label masked)
#     -> frozen BGE leaf grounding
#     -> Schema.org ancestry
#     -> canonical root
#
# Compare:
#   direct description -> root
#   description -> leaf -> graph root
#
# Internet failure is handled gracefully.
# =============================================================================

RUN_SCHEMAORG_EXTERNAL_ONTOLOGY = True
SCHEMAORG_URLS = [
    "https://schema.org/version/latest/schemaorg-current-https.ttl",
    "https://schema.org/version/latest/schemaorg-current-http.ttl",
]

schema_external_df = pd.DataFrame()
schema_external_summary = pd.DataFrame()

if RUN_SCHEMAORG_EXTERNAL_ONTOLOGY:
    try:
        from rdflib import (
            Graph,
            RDF,
            RDFS,
            URIRef,
            Namespace,
        )

        sg = Graph()
        loaded_schema_url = None
        last_schema_exc = None

        for url in SCHEMAORG_URLS:
            try:
                sg.parse(
                    url,
                    format="turtle",
                )
                loaded_schema_url = url
                break
            except Exception as exc:
                last_schema_exc = exc

        if loaded_schema_url is None:
            raise RuntimeError(
                "Could not load Schema.org: "
                + str(last_schema_exc)
            )

        SCHEMA = Namespace(
            "https://schema.org/"
        )

        ROOT_NAMES = [
            "CreativeWork",
            "Organization",
            "Place",
            "Product",
            "Event",
            "Action",
        ]
        ROOT_URIS = {
            name: SCHEMA[name]
            for name in ROOT_NAMES
        }

        def sc_label(uri):
            vals = list(
                sg.objects(uri, RDFS.label)
            )
            return (
                str(vals[0])
                if vals
                else str(uri).rstrip("/").split("/")[-1]
            )

        def sc_comment(uri):
            vals = list(
                sg.objects(uri, RDFS.comment)
            )
            return (
                str(vals[0]).strip()
                if vals else ""
            )

        # subclass graph
        parents = {}
        classes = (
            set(
                sg.subjects(
                    RDF.type,
                    RDFS.Class,
                )
            )
            | set(
                sg.subjects(
                    RDFS.subClassOf,
                    None,
                )
            )
        )

        for c in classes:
            if not isinstance(c, URIRef):
                continue

            parents[c] = [
                p
                for p in sg.objects(
                    c,
                    RDFS.subClassOf,
                )
                if isinstance(p, URIRef)
            ]

        def sc_ancestors(
            uri,
            max_hops=30,
        ):
            seen = set()
            frontier = [uri]

            for _ in range(max_hops):
                nxt = []
                for x in frontier:
                    for p in parents.get(
                        x, []
                    ):
                        if p not in seen:
                            seen.add(p)
                            nxt.append(p)
                if not nxt:
                    break
                frontier = nxt

            return seen

        external_items = []

        for c in sorted(
            classes,
            key=str,
        ):
            if (
                not isinstance(c, URIRef)
                or c in ROOT_URIS.values()
            ):
                continue

            anc = sc_ancestors(c)
            root_hits = [
                name
                for name, root_uri
                in ROOT_URIS.items()
                if root_uri in anc
            ]

            label = sc_label(c)
            comment = sc_comment(c)

            if (
                len(root_hits) == 1
                and len(comment) >= 45
                and len(label) >= 3
            ):
                # Mask exact class label and the canonical root name.
                query = re.sub(
                    re.escape(label),
                    "[MASKED CLASS]",
                    comment,
                    flags=re.I,
                )
                query = re.sub(
                    re.escape(root_hits[0]),
                    "[MASKED ROOT]",
                    query,
                    flags=re.I,
                )

                external_items.append({
                    "uri": c,
                    "label": label,
                    "comment": comment,
                    "query": query,
                    "root": root_hits[0],
                })

        # Balanced deterministic sample, max 20/root.
        balanced_items = []

        for root in ROOT_NAMES:
            xs = [
                x for x in external_items
                if x["root"] == root
            ]
            xs = sorted(
                xs,
                key=lambda z: z["label"],
            )[:20]
            balanced_items.extend(xs)

        external_items = balanced_items

        root_counts = pd.Series(
            [x["root"] for x in external_items]
        ).value_counts()

        if (
            len(external_items) < 30
            or root_counts.size < 4
        ):
            raise RuntimeError(
                "Insufficient balanced Schema.org hierarchy: "
                f"n={len(external_items)}, "
                f"roots={root_counts.to_dict()}"
            )

        # Leaf prototype uses only the human-authored class label.
        leaf_proto = semantic_encode([
            "Schema.org class: "
            + x["label"]
            for x in external_items
        ])
        leaf_proto = (
            leaf_proto
            / np.maximum(
                np.linalg.norm(
                    leaf_proto,
                    axis=1,
                    keepdims=True,
                ),
                1e-12,
            )
        )

        # Root prototype uses root name + human-authored root comment when available.
        root_texts = []

        for root in ROOT_NAMES:
            root_uri = ROOT_URIS[root]
            root_comment = sc_comment(
                root_uri
            )
            root_texts.append(
                f"Schema.org root {root}. "
                + root_comment
            )

        root_proto = semantic_encode(
            root_texts
        )
        root_proto = (
            root_proto
            / np.maximum(
                np.linalg.norm(
                    root_proto,
                    axis=1,
                    keepdims=True,
                ),
                1e-12,
            )
        )

        query_emb = semantic_encode([
            x["query"]
            for x in external_items
        ])
        query_emb = (
            query_emb
            / np.maximum(
                np.linalg.norm(
                    query_emb,
                    axis=1,
                    keepdims=True,
                ),
                1e-12,
            )
        )

        leaf_sim = (
            query_emb @ leaf_proto.T
        )
        root_sim = (
            query_emb @ root_proto.T
        )

        schema_rows = []

        for i, item in enumerate(
            external_items
        ):
            pred_leaf_i = int(
                np.argmax(
                    leaf_sim[i]
                )
            )
            pred_leaf = external_items[
                pred_leaf_i
            ]

            direct_root = ROOT_NAMES[
                int(
                    np.argmax(
                        root_sim[i]
                    )
                )
            ]

            graph_root = pred_leaf["root"]

            schema_rows.append({
                "class_label":
                    item["label"],
                "gold_root":
                    item["root"],
                "masked_description":
                    item["query"],
                "pred_leaf":
                    pred_leaf["label"],
                "leaf_grounding_correct":
                    int(
                        pred_leaf["label"]
                        == item["label"]
                    ),
                "direct_root":
                    direct_root,
                "direct_root_correct":
                    int(
                        direct_root
                        == item["root"]
                    ),
                "ground_then_graph_root":
                    graph_root,
                "ground_then_graph_correct":
                    int(
                        graph_root
                        == item["root"]
                    ),
            })

        schema_external_df = pd.DataFrame(
            schema_rows
        )

        n_schema = len(
            schema_external_df
        )
        direct_successes = int(
            schema_external_df[
                "direct_root_correct"
            ].sum()
        )
        graph_successes = int(
            schema_external_df[
                "ground_then_graph_correct"
            ].sum()
        )
        leaf_successes = int(
            schema_external_df[
                "leaf_grounding_correct"
            ].sum()
        )

        leaf_p, leaf_lo, leaf_hi = (
            pc_wilson(
                leaf_successes,
                n_schema,
            )
        )
        direct_p, direct_lo, direct_hi = (
            pc_wilson(
                direct_successes,
                n_schema,
            )
        )
        graph_p, graph_lo, graph_hi = (
            pc_wilson(
                graph_successes,
                n_schema,
            )
        )

        schema_external_summary = pd.DataFrame([{
            "source_url":
                loaded_schema_url,
            "n":
                n_schema,
            "n_roots":
                int(
                    schema_external_df[
                        "gold_root"
                    ].nunique()
                ),
            "leaf_grounding_accuracy":
                leaf_p,
            "leaf_ci_low":
                leaf_lo,
            "leaf_ci_high":
                leaf_hi,
            "direct_root_accuracy":
                direct_p,
            "direct_root_ci_low":
                direct_lo,
            "direct_root_ci_high":
                direct_hi,
            "ground_then_graph_accuracy":
                graph_p,
            "graph_ci_low":
                graph_lo,
            "graph_ci_high":
                graph_hi,
        }])

        schema_by_root = (
            schema_external_df
            .groupby(
                "gold_root",
                as_index=False,
            )
            .agg(
                n=(
                    "class_label",
                    "size",
                ),
                leaf_grounding_accuracy=(
                    "leaf_grounding_correct",
                    "mean",
                ),
                direct_root_accuracy=(
                    "direct_root_correct",
                    "mean",
                ),
                ground_then_graph_accuracy=(
                    "ground_then_graph_correct",
                    "mean",
                ),
            )
        )

        print(
            "\n[4] SCHEMA.ORG EXTERNAL "
            "HUMAN-AUTHORED ONTOLOGY DIAGNOSTIC"
        )
        display(
            schema_external_summary.round(4)
        )
        display(
            schema_by_root.round(4)
        )

        schema_external_df.to_csv(
            PAPER_CLOSURE_DIR /
            "schemaorg_external_ontology_results.csv",
            index=False,
        )
        schema_external_summary.to_csv(
            PAPER_CLOSURE_DIR /
            "schemaorg_external_ontology_summary.csv",
            index=False,
        )
        schema_by_root.to_csv(
            PAPER_CLOSURE_DIR /
            "schemaorg_external_ontology_by_root.csv",
            index=False,
        )

    except Exception as exc:
        print(
            "\nSchema.org external ontology "
            "diagnostic SKIPPED:",
            type(exc).__name__,
            str(exc)[:500],
        )

# =============================================================================
# 5. G0 READER NAME RESOLUTION + FAILURE TAXONOMY
# =============================================================================
#
# Previous taxonomy output could show UNKNOWN because result streams used
# heterogeneous field names. Resolve the reader name from the first available
# non-empty model identity column instead of assuming `reader`.
# =============================================================================

resolved_reader_taxonomy = pd.DataFrame()
reader_identity_audit = pd.DataFrame()

if "reader_raw" in globals() and len(reader_raw):
    rr = reader_raw.copy()

    candidate_identity_cols = [
        c for c in [
            "reader",
            "reader_label",
            "model_label",
            "label",
            "loaded_model",
            "model_id",
            "model",
            "reader_model",
        ]
        if c in rr.columns
    ]

    def pc_resolve_reader(row):
        for col in candidate_identity_cols:
            v = row.get(col, None)

            if (
                v is not None
                and str(v).strip()
                and str(v).strip().upper()
                not in {
                    "UNKNOWN",
                    "NONE",
                    "NAN",
                }
            ):
                return str(v).strip()

        return "UNRESOLVED"

    rr["reader_resolved"] = rr.apply(
        pc_resolve_reader,
        axis=1,
    )

    reader_identity_audit = pd.DataFrame({
        "candidate_column":
            candidate_identity_cols,
        "non_null_count": [
            int(
                rr[c]
                .notna()
                .sum()
            )
            for c in candidate_identity_cols
        ],
        "unique_non_null": [
            int(
                rr[c]
                .dropna()
                .astype(str)
                .nunique()
            )
            for c in candidate_identity_cols
        ],
    })

    # Defensive metric fields.
    for col, default in {
        "parse_success": 0,
        "semantic_label_validity": 0,
        "task_success": 0,
        "overflow": 0,
        "error": "",
        "method": "UNKNOWN",
        "split": "UNKNOWN",
    }.items():
        if col not in rr.columns:
            rr[col] = default

    rr["runtime_error"] = (
        rr["error"]
        .fillna("")
        .astype(str)
        .str.len()
        .gt(0)
        .astype(int)
    )
    rr["overflow_failure"] = (
        rr["overflow"]
        .fillna(0)
        .astype(int)
    )
    rr["parse_failure"] = (
        rr["parse_success"]
        .fillna(0)
        .astype(float)
        .lt(1)
        .astype(int)
    )
    rr["label_validity_failure"] = (
        (
            rr["parse_success"]
            .fillna(0)
            .astype(float)
            .ge(1)
        )
        & (
            rr[
                "semantic_label_validity"
            ]
            .fillna(0)
            .astype(float)
            .lt(1)
        )
    ).astype(int)
    rr["semantic_task_failure"] = (
        (
            rr["parse_success"]
            .fillna(0)
            .astype(float)
            .ge(1)
        )
        & (
            rr[
                "semantic_label_validity"
            ]
            .fillna(0)
            .astype(float)
            .ge(1)
        )
        & (
            rr["task_success"]
            .fillna(0)
            .astype(float)
            .lt(1)
        )
    ).astype(int)

    resolved_reader_taxonomy = (
        rr.groupby(
            [
                "reader_resolved",
                "method",
                "split",
            ],
            as_index=False,
        )
        .agg(
            n=("task_success", "size"),
            runtime_error_rate=(
                "runtime_error",
                "mean",
            ),
            overflow_rate=(
                "overflow_failure",
                "mean",
            ),
            parse_failure_rate=(
                "parse_failure",
                "mean",
            ),
            label_validity_failure_rate=(
                "label_validity_failure",
                "mean",
            ),
            semantic_task_failure_rate=(
                "semantic_task_failure",
                "mean",
            ),
            task_success=(
                "task_success",
                "mean",
            ),
        )
    )

    unresolved_rate = float(
        (
            rr["reader_resolved"]
            == "UNRESOLVED"
        ).mean()
    )

    print(
        "\n[5] G0 READER IDENTITY + FAILURE TAXONOMY"
    )
    print(
        "Unresolved reader identity rate:",
        round(unresolved_rate, 4),
    )
    display(
        reader_identity_audit
    )
    display(
        resolved_reader_taxonomy
        .sort_values(
            [
                "task_success",
                "parse_failure_rate",
            ]
        )
        .head(60)
        .round(4)
    )

    reader_identity_audit.to_csv(
        PAPER_CLOSURE_DIR /
        "reader_identity_audit.csv",
        index=False,
    )
    resolved_reader_taxonomy.to_csv(
        PAPER_CLOSURE_DIR /
        "resolved_reader_failure_taxonomy.csv",
        index=False,
    )

else:
    print(
        "\n[5] reader_raw unavailable; "
        "G0 taxonomy skipped."
    )

# =============================================================================
# 6. PAPER-CLOSURE UNCERTAINTY / SAFETY AUDIT
# =============================================================================

closure_uncertainty_rows = []

# A. Architecture-faithful routed known C1/C2 binomial intervals.
for surface in ["C1", "C2"]:
    g = route_v2_df[
        (route_v2_df["kind"] == "known")
        & (route_v2_df["surface"] == surface)
    ]
    s = int(g["correct"].sum())
    n = len(g)
    p, lo, hi = pc_wilson(
        s, n
    )

    closure_uncertainty_rows.append({
        "metric":
            f"Architecture-faithful routed {surface} accuracy",
        "point": p,
        "ci_low": lo,
        "ci_high": hi,
        "n": n,
        "interval":
            "Wilson 95%",
    })

# B. Small opaque ontology parent mechanism.
if "small_ontology_df" in globals():
    sdf = small_ontology_df.copy()

    possible_method_cols = [
        c for c in [
            "method",
            "Method",
        ]
        if c in sdf.columns
    ]

    if possible_method_cols:
        mcol = possible_method_cols[0]

        for method in [
            "SCALE-Full",
            "SCALE",
            "SCALE-NoOnt",
        ]:
            sub = sdf[
                sdf[mcol]
                .astype(str)
                .eq(method)
            ]

            if (
                "tier" in sub.columns
                and len(sub)
            ):
                sub = sub[
                    sub["tier"]
                    .astype(str)
                    .str.lower()
                    .eq("opaque")
                ]

            metric_col = None
            for c in [
                "canonical_parent_correct",
                "parent_correct",
                "canonical_parent_accuracy",
            ]:
                if c in sub.columns:
                    metric_col = c
                    break

            if (
                metric_col is not None
                and len(sub)
            ):
                vals = (
                    sub[metric_col]
                    .astype(float)
                )
                successes = int(
                    np.round(
                        vals.sum()
                    )
                )
                n = len(vals)

                p, lo, hi = pc_wilson(
                    successes, n
                )

                closure_uncertainty_rows.append({
                    "metric":
                        f"{method} opaque parent accuracy",
                    "point": p,
                    "ci_low": lo,
                    "ci_high": hi,
                    "n": n,
                    "interval":
                        "Wilson 95%",
                })

# C. Runtime enforcement exact-rate intervals if detailed table exists.
for candidate_name in [
    "constraint_stress_df",
    "constraint_df",
    "runtime_constraint_df",
]:
    if candidate_name in globals():
        cdf = globals()[
            candidate_name
        ]
        if isinstance(
            cdf, pd.DataFrame
        ) and len(cdf):
            for col in [
                "detected",
                "repaired",
                "blocked",
            ]:
                if col in cdf.columns:
                    vals = (
                        cdf[col]
                        .dropna()
                        .astype(int)
                    )
                    if len(vals):
                        p, lo, hi = pc_wilson(
                            int(vals.sum()),
                            len(vals),
                        )
                        closure_uncertainty_rows.append({
                            "metric":
                                "Runtime "
                                + col,
                            "point": p,
                            "ci_low": lo,
                            "ci_high": hi,
                            "n": len(vals),
                            "interval":
                                "Wilson 95%",
                        })
            break

closure_uncertainty_df = pd.DataFrame(
    closure_uncertainty_rows
)

print(
    "\n[6] PAPER-CLOSURE UNCERTAINTY AUDIT"
)
display(
    closure_uncertainty_df.round(4)
)

closure_uncertainty_df.to_csv(
    PAPER_CLOSURE_DIR /
    "paper_closure_uncertainty.csv",
    index=False,
)

# =============================================================================
# 7. FINAL CLAIM / EVIDENCE MATRIX
# =============================================================================

# Pull already-computed strong robustness evidence rather than rerunning it.
naturalistic_auc = (
    float(nat_failure_auc)
    if "nat_failure_auc" in globals()
    else np.nan
)

learned_executor_min_auc = np.nan
if (
    "independent_executor_summary"
    in globals()
    and len(
        independent_executor_summary
    )
):
    learned_executor_min_auc = float(
        independent_executor_summary[
            "failure_auc_active_scd"
        ].min()
    )

controlled_active_auc = (
    pc_safe_auc(
        drift_df["failure"],
        drift_df[
            "prepolicy_active_scd"
        ],
    )
    if len(drift_df)
    else np.nan
)

root_router_mean_auc = float(
    router_root_aggregate[
        "mean_auc"
    ].iloc[0]
)
root_router_min_auc = float(
    router_root_aggregate[
        "min_auc"
    ].iloc[0]
)

schema_graph_acc = (
    float(
        schema_external_summary[
            "ground_then_graph_accuracy"
        ].iloc[0]
    )
    if len(schema_external_summary)
    else np.nan
)
schema_direct_acc = (
    float(
        schema_external_summary[
            "direct_root_accuracy"
        ].iloc[0]
    )
    if len(schema_external_summary)
    else np.nan
)

claim_rows = [
    {
        "claim":
            "Controlled role-active semantic compatibility",
        "status":
            "SUPPORTED",
        "evidence":
            "Existing C1/C2 controlled representation results.",
        "main_paper":
            "YES",
    },
    {
        "claim":
            "Ontology enables opaque unseen structural canonicalization",
        "status":
            "SUPPORTED",
        "evidence":
            "Existing opaque-leaf Full vs NoOnt mechanism ablation.",
        "main_paper":
            "YES",
    },
    {
        "claim":
            "Active-SCD predicts controlled failure",
        "status":
            (
                "SUPPORTED"
                if controlled_active_auc >= 0.90
                else "QUALIFIED"
            ),
        "evidence":
            f"Controlled AUROC={controlled_active_auc:.4f}",
        "main_paper":
            "YES",
    },
    {
        "claim":
            "Active-SCD transfers to naturalistic version evolution",
        "status":
            (
                "SUPPORTED"
                if np.isfinite(
                    naturalistic_auc
                )
                and naturalistic_auc >= 0.85
                else "QUALIFIED"
            ),
        "evidence":
            (
                f"Naturalistic failure AUROC="
                f"{naturalistic_auc:.4f}"
                if np.isfinite(
                    naturalistic_auc
                )
                else "Unavailable"
            ),
        "main_paper":
            "YES",
    },
    {
        "claim":
            "Active-SCD transfers to independent learned executors",
        "status":
            (
                "SUPPORTED"
                if np.isfinite(
                    learned_executor_min_auc
                )
                and learned_executor_min_auc >= 0.75
                else "QUALIFIED"
            ),
        "evidence":
            (
                "Minimum learned-executor AUROC="
                f"{learned_executor_min_auc:.4f}"
                if np.isfinite(
                    learned_executor_min_auc
                )
                else "Unavailable"
            ),
        "main_paper":
            "YES",
    },
    {
        "claim":
            "Non-oracle open-world routing generalizes across ontology roots",
        "status":
            (
                "SUPPORTED"
                if root_router_mean_auc >= 0.75
                and root_router_min_auc >= 0.60
                else "QUALIFIED"
            ),
        "evidence":
            (
                f"LO-root mean AUROC="
                f"{root_router_mean_auc:.4f}; "
                f"minimum={root_router_min_auc:.4f}"
            ),
        "main_paper":
            "METHOD + APPENDIX",
    },
    {
        "claim":
            "Human-authored ontology structural transfer",
        "status":
            (
                "SUPPORTED"
                if np.isfinite(
                    schema_graph_acc
                )
                and schema_graph_acc
                    >= schema_direct_acc
                else "EXPLORATORY"
            ),
        "evidence":
            (
                f"Schema.org graph={schema_graph_acc:.4f}, "
                f"direct={schema_direct_acc:.4f}"
                if np.isfinite(
                    schema_graph_acc
                )
                else "External ontology test unavailable"
            ),
        "main_paper":
            "APPENDIX / LIMITATION",
    },
    {
        "claim":
            "Universal arbitrary-reader robustness",
        "status":
            "NOT SUPPORTED",
        "evidence":
            "Prespecified G0 remains failed.",
        "main_paper":
            "NEGATIVE BOUNDARY",
    },
    {
        "claim":
            "General label-efficient directed invariance",
        "status":
            "NOT SUPPORTED",
        "evidence":
            "Prespecified G2 remains failed.",
        "main_paper":
            "NEGATIVE BOUNDARY",
    },
    {
        "claim":
            "Universal external end-task improvement",
        "status":
            "NOT SUPPORTED",
        "evidence":
            "CRM task success remains executor-dependent.",
        "main_paper":
            "NEGATIVE BOUNDARY",
    },
]

claim_matrix_df = pd.DataFrame(
    claim_rows
)

print(
    "\n[7] FINAL CLAIM / EVIDENCE MATRIX"
)
display(claim_matrix_df)

claim_matrix_df.to_csv(
    PAPER_CLOSURE_DIR /
    "final_claim_evidence_matrix.csv",
    index=False,
)

# =============================================================================
# 8. PAPER-READY REPORT
# =============================================================================

def pc_fmt(x):
    try:
        if not np.isfinite(float(x)):
            return "NA"
        return f"{float(x):.4f}"
    except Exception:
        return "NA"

llm_report_lines = []
if len(pc_llm_summary):
    for _, r in pc_llm_summary.iterrows():
        llm_report_lines.append(
            f"- {r['reader']}: clean={pc_fmt(r['clean_accuracy'])}, "
            f"gate={'PASS' if int(r['gate_pass']) else 'FAIL'}, "
            f"baseline-correct drift bases={int(r['baseline_correct_bases'])}, "
            f"Active-SCD AUROC={pc_fmt(r['active_scd_failure_auc'])}."
        )
else:
    llm_report_lines.append(
        "- Competence-gated LLM executor unavailable."
    )

schema_line = (
    "Schema.org external ontology diagnostic unavailable."
)
if len(schema_external_summary):
    sr = schema_external_summary.iloc[0]
    schema_line = (
        f"Schema.org n={int(sr['n'])}, "
        f"roots={int(sr['n_roots'])}, "
        f"leaf grounding={pc_fmt(sr['leaf_grounding_accuracy'])}, "
        f"direct root={pc_fmt(sr['direct_root_accuracy'])}, "
        f"ground→graph root={pc_fmt(sr['ground_then_graph_accuracy'])}."
    )

report = f"""
# SCALE — Paper Closure Report

## A. Architecture-faithful hybrid routing

The final deployable router now uses **Full SCALE itself** for known concepts,
rather than the CB+DualView baseline proxy.

- Routed C1 known accuracy: **{pc_fmt(known_c1_v2)}**
- Routed C2 known accuracy: **{pc_fmt(known_c2_v2)}**
- Semantic-alias symbolic-ID false routes: **0 by assertion**

This is the routing result that should be aligned with Method 4.2.

## B. Leave-one-ontology-root-out open-set routing

- Mean held-root AUROC: **{pc_fmt(root_router_mean_auc)}**
- Minimum held-root AUROC: **{pc_fmt(root_router_min_auc)}**
- Mean balanced accuracy: **{pc_fmt(router_root_aggregate['mean_balanced_accuracy'].iloc[0])}**
- Minimum unseen recall: **{pc_fmt(router_root_aggregate['min_unseen_recall'].iloc[0])}**
- Maximum known false-open rate: **{pc_fmt(router_root_aggregate['max_known_false_open_rate'].iloc[0])}**

This is stronger evidence than narrowing a bootstrap CI by repeated resampling because
the held-out open-world root is semantically absent from router fitting.

## C. Competence-gated independent LLM executor

{chr(10).join(llm_report_lines)}

Only executors passing clean accuracy >= **{LLM_CLEAN_GATE:.2f}** are eligible for
the drift robustness claim. A failed competence gate is reported as **inconclusive**,
not as evidence against Active-SCD.

## D. Human-authored external ontology

{schema_line}

This is an exploratory structural sanity test over a genuine human-authored ontology.
It must not be described as a full external SCALE benchmark unless a future experiment
trains/evaluates the full SCALE architecture on that domain.

## E. G0 arbitrary-reader boundary

Reader/model identities are reconstructed from the actual stored result columns before
failure taxonomy is reported. Use `resolved_reader_failure_taxonomy.csv`.

The prespecified G0 result itself is unchanged.

## F. Existing strongest evidence retained

- Controlled Active-SCD failure AUROC: **{pc_fmt(controlled_active_auc)}**
- Naturalistic version-evolution failure AUROC: **{pc_fmt(naturalistic_auc)}**
- Minimum independent learned-executor Active-SCD AUROC: **{pc_fmt(learned_executor_min_auc)}**

No successful previous result is replaced by a weaker exploratory test.

## G. Recommended stopping rule

If the architecture-faithful route preserves C1/C2 competence and leave-one-root-out
routing is materially above chance, the routing objection is closed.

If at least one generative executor passes the clean competence gate, its drift result can
be reported in the appendix or robustness section. If none pass, keep the already-strong
independent learned-executor evidence and state that small generative executors did not
meet the clean competence threshold.

If Schema.org succeeds, it can be cited as an external ontology sanity diagnostic. If it
does not, external ontology validation remains a limitation; do not add more procedural
ontology experiments merely to obtain a positive result.

At this point, further label-efficiency losses, differentiable logic variants, additional
small LLMs, or larger synthetic ontologies are not recommended.
"""

(
    PAPER_CLOSURE_DIR /
    "PAPER_CLOSURE_REPORT.md"
).write_text(
    report.strip() + "\n",
    encoding="utf-8",
)

display(Markdown(report))

print(
    "\nSaved paper-closure artifacts:"
)
for p in sorted(
    PAPER_CLOSURE_DIR.iterdir()
):
    print(" -", p.name)


In [ ]:

# =============================================================================
# FINAL FAILURE RESOLUTION & DECISION AUDIT
# Run after the entire notebook above.
# =============================================================================

from pathlib import Path
import copy, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
)

from IPython.display import display, Markdown

RESOLVE_DIR = ROOT / "final_failure_resolution"
RESOLVE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SCALE — FINAL FAILURE RESOLUTION & DECISION AUDIT")
print("Output:", RESOLVE_DIR)
print("=" * 100)

# -----------------------------------------------------------------------------
# PRE-FLIGHT
# -----------------------------------------------------------------------------

_REQUIRED_FINAL = [
    "reader_raw",
    "reader_df",
    "native_df",
    "curve_df",
    "curve_paired_deltas",
    "MODELS",
    "BASE_GRAPH",
    "ensemble_predict",
    "ontology_repair",
    "realize_message",
    "shift_test_base",
    "core_test_base",
    "AGENTS",
    "hop_contract",
    "active_correct",
    "router_root_generalization_df",
    "strict_router_df",
    "_router_features",
]

_missing = [x for x in _REQUIRED_FINAL if x not in globals()]
assert not _missing, (
    "Run all previous notebook cells before this final section. Missing: "
    + ", ".join(_missing)
)

def _safe_auc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    return (
        float(roc_auc_score(y, s))
        if len(np.unique(y)) >= 2
        else np.nan
    )

def _wilson(successes, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = successes / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * math.sqrt(
        p*(1-p)/n + z*z/(4*n*n)
    ) / den
    return float(p), float(center-half), float(center+half)

# =============================================================================
# 1. G0 ROOT-CAUSE AUDIT
# =============================================================================
#
# G0 was deliberately defined as a strict arbitrary-reader criterion.
# It remains FAILED.  This section decomposes the failure into layers:
#
#   L0 generation/runtime
#   L1 parse
#   L2 allowed-label / vocabulary
#   L3 semantic interpretation
#   L4 final task
#
# The point is NOT to relabel G0; it is to identify where the arbitrary-reader
# assumption breaks.
# =============================================================================

rr = reader_raw.copy()

# Resolve the reader identity from the canonical stored column first.
if "reader_model" in rr.columns:
    rr["reader_final"] = rr["reader_model"].astype(str)
elif "reader_resolved" in rr.columns:
    rr["reader_final"] = rr["reader_resolved"].astype(str)
else:
    identity_cols = [
        c for c in [
            "reader", "reader_label", "model_label",
            "active_model_id", "loaded_model", "model_id"
        ]
        if c in rr.columns
    ]

    def _resolve_identity(row):
        for col in identity_cols:
            v = row.get(col)
            if (
                v is not None
                and str(v).strip()
                and str(v).strip().upper()
                not in {"UNKNOWN", "NONE", "NAN"}
            ):
                return str(v).strip()
        return "UNRESOLVED"

    rr["reader_final"] = rr.apply(_resolve_identity, axis=1)

for col, default in {
    "primary_model_loaded": True,
    "parse_success": 0,
    "semantic_label_validity": 0,
    "semantic_success": 0,
    "task_success": 0,
    "overflow": 0,
    "error": None,
    "method": "UNKNOWN",
    "split": "UNKNOWN",
}.items():
    if col not in rr.columns:
        rr[col] = default

# Only genuine requested reader models: fallback-model rows are not evidence
# about the named reader.
g0_df = rr[
    rr["primary_model_loaded"].fillna(False).astype(bool)
].copy()

g0_df["runtime_error"] = (
    g0_df["error"].notna()
    & g0_df["error"].astype(str).str.len().gt(0)
).astype(int)

g0_df["overflow_failure"] = (
    g0_df["overflow"].fillna(0).astype(int) > 0
).astype(int)

g0_df["parse_failure"] = (
    g0_df["parse_success"].fillna(0).astype(float) < 1
).astype(int)

g0_df["label_failure_given_parse"] = (
    (g0_df["parse_success"].fillna(0).astype(float) >= 1)
    & (
        g0_df["semantic_label_validity"]
        .fillna(0).astype(float) < 1
    )
).astype(int)

g0_df["semantic_failure_given_valid"] = (
    (g0_df["parse_success"].fillna(0).astype(float) >= 1)
    & (
        g0_df["semantic_label_validity"]
        .fillna(0).astype(float) >= 1
    )
    & (g0_df["semantic_success"].fillna(0).astype(float) < 1)
).astype(int)

g0_df["task_failure_given_semantic"] = (
    (g0_df["semantic_success"].fillna(0).astype(float) >= 1)
    & (g0_df["task_success"].fillna(0).astype(float) < 1)
).astype(int)

g0_layer_summary = (
    g0_df.groupby(
        ["reader_final", "split", "method"],
        as_index=False
    )
    .agg(
        n=("task_success", "size"),
        runtime_error_rate=("runtime_error", "mean"),
        overflow_rate=("overflow_failure", "mean"),
        parse_failure_rate=("parse_failure", "mean"),
        label_failure_given_parse=(
            "label_failure_given_parse", "mean"
        ),
        semantic_failure_given_valid=(
            "semantic_failure_given_valid", "mean"
        ),
        task_failure_given_semantic=(
            "task_failure_given_semantic", "mean"
        ),
        parse_success=("parse_success", "mean"),
        semantic_label_validity=(
            "semantic_label_validity", "mean"
        ),
        semantic_success=("semantic_success", "mean"),
        task_success=("task_success", "mean"),
    )
)

g0_reader_summary = (
    g0_df.groupby("reader_final", as_index=False)
    .agg(
        n=("task_success", "size"),
        runtime_error_rate=("runtime_error", "mean"),
        overflow_rate=("overflow_failure", "mean"),
        parse_failure_rate=("parse_failure", "mean"),
        label_failure_given_parse=(
            "label_failure_given_parse", "mean"
        ),
        semantic_failure_given_valid=(
            "semantic_failure_given_valid", "mean"
        ),
        semantic_success=("semantic_success", "mean"),
        task_success=("task_success", "mean"),
    )
)

print("\n[1A] G0 — READER-LEVEL FAILURE DECOMPOSITION")
display(g0_reader_summary.round(4))

print("\n[1B] G0 — WORST READER / SPLIT / METHOD CELLS")
display(
    g0_layer_summary
    .sort_values(
        [
            "task_success",
            "semantic_success",
            "parse_success",
        ]
    )
    .head(40)
    .round(4)
)

g0_layer_summary.to_csv(
    RESOLVE_DIR / "g0_layer_failure_summary.csv",
    index=False,
)
g0_reader_summary.to_csv(
    RESOLVE_DIR / "g0_reader_summary.csv",
    index=False,
)

# =============================================================================
# 2. G0 REMEDIATION TEST — TYPED/NATIVE ADAPTER
# =============================================================================
#
# Engineering interpretation of G0:
#
# Arbitrary generative consumption is a convenience layer, not the semantic
# contract itself.  SCALE already exposes a typed active payload.
#
# We compare:
#   A) arbitrary LLM reader (G0 path)
#   B) deterministic typed/native receiver (G1 path)
#
# IMPORTANT:
# This remediation does NOT retroactively pass G0.  It demonstrates that the
# failed boundary can be removed by using the interface in the way it was
# designed: typed consumption rather than unconstrained reinterpretation.
# =============================================================================

# Compare only splits represented in native_df and only SCALE.
native_scale = native_df[
    native_df["method"].astype(str).eq("SCALE")
].copy()

reader_scale = reader_df[
    reader_df["method"].astype(str).eq("SCALE")
].copy()

# Reader composition results are per reader; aggregate first at the case level.
reader_case = (
    reader_scale.groupby(
        [
            "reader_model", "split",
            "base_id", "composition_id"
        ],
        as_index=False
    )
    .agg(
        semantic_success=("semantic_success", "mean"),
        task_success=("task_success", "mean"),
        parse_success=("parse_success", "mean"),
    )
)

native_case = native_scale[
    [
        "split", "base_id", "composition_id",
        "semantic_success", "task_success"
    ]
].rename(
    columns={
        "semantic_success": "typed_semantic_success",
        "task_success": "typed_task_success",
    }
)

g0_rescue = reader_case.merge(
    native_case,
    on=["split", "base_id", "composition_id"],
    how="inner",
)

g0_rescue["semantic_rescue"] = (
    (
        g0_rescue["semantic_success"] < 1
    )
    & (
        g0_rescue["typed_semantic_success"] >= 1
    )
).astype(int)

g0_rescue["task_rescue"] = (
    (g0_rescue["task_success"] < 1)
    & (g0_rescue["typed_task_success"] >= 1)
).astype(int)

g0_rescue_summary = (
    g0_rescue.groupby(
        ["reader_model", "split"],
        as_index=False
    )
    .agg(
        n=("base_id", "size"),
        arbitrary_parse_success=("parse_success", "mean"),
        arbitrary_semantic_success=("semantic_success", "mean"),
        typed_semantic_success=("typed_semantic_success", "mean"),
        arbitrary_task_success=("task_success", "mean"),
        typed_task_success=("typed_task_success", "mean"),
        semantic_rescue_rate=("semantic_rescue", "mean"),
        task_rescue_rate=("task_rescue", "mean"),
    )
)

print("\n[2] G0 — TYPED ADAPTER REMEDIATION")
display(g0_rescue_summary.round(4))

g0_rescue.to_csv(
    RESOLVE_DIR / "g0_typed_adapter_rescue_cases.csv",
    index=False,
)
g0_rescue_summary.to_csv(
    RESOLVE_DIR / "g0_typed_adapter_rescue_summary.csv",
    index=False,
)

# Practical resolution criterion.
typed_min_task = (
    float(g0_rescue_summary["typed_task_success"].min())
    if len(g0_rescue_summary) else np.nan
)
typed_resolution_supported = bool(
    np.isfinite(typed_min_task)
    and typed_min_task >= 0.95
)

# =============================================================================
# 3. G2 — DIRECTED INVARIANCE DIAGNOSTIC
# =============================================================================
#
# G2 remains FAILED by the prespecified gate.
#
# Rather than adding another alignment loss after observing the failure, we ask:
#
#   * how often does SCALE beat SCALE-NoInv?
#   * how often does it lose?
#   * are signs consistent across seeds?
#   * is there any visibility regime where the advantage is stable?
#
# A reliable regularizer should show sign-consistent improvement, not isolated
# favorable cells.
# =============================================================================

g2 = curve_df[
    curve_df["method"].isin(
        ["SCALE", "SCALE-NoInv", "SCALE-NoOnt", "CB+DualView"]
    )
].copy()

# Seed-level paired SCALE - NoInv deltas.
scale_seed = (
    g2[g2["method"] == "SCALE"]
    .groupby(
        ["visibility", "seed", "split"],
        as_index=False
    )
    .agg(scale_accuracy=("active_correct", "mean"))
)

noinv_seed = (
    g2[g2["method"] == "SCALE-NoInv"]
    .groupby(
        ["visibility", "seed", "split"],
        as_index=False
    )
    .agg(noinv_accuracy=("active_correct", "mean"))
)

g2_seed_pair = scale_seed.merge(
    noinv_seed,
    on=["visibility", "seed", "split"],
    how="inner",
)
g2_seed_pair["delta_scale_minus_noinv"] = (
    g2_seed_pair["scale_accuracy"]
    - g2_seed_pair["noinv_accuracy"]
)

g2_sign_summary = (
    g2_seed_pair.groupby(
        ["visibility", "split"],
        as_index=False
    )
    .agg(
        mean_delta=("delta_scale_minus_noinv", "mean"),
        seed_sd=("delta_scale_minus_noinv", "std"),
        n_seeds=("seed", "nunique"),
        wins=(
            "delta_scale_minus_noinv",
            lambda x: int((x > 1e-12).sum())
        ),
        ties=(
            "delta_scale_minus_noinv",
            lambda x: int((np.abs(x) <= 1e-12).sum())
        ),
        losses=(
            "delta_scale_minus_noinv",
            lambda x: int((x < -1e-12).sum())
        ),
    )
)

# Exact paired case-level diagnostics with bootstrap CI.
g2_case_rows = []

for visibility in sorted(g2["visibility"].unique()):
    for split_name in sorted(g2["split"].unique()):
        sub = g2[
            (g2["visibility"] == visibility)
            & (g2["split"] == split_name)
        ]

        a = sub[sub["method"] == "SCALE"][
            ["seed", "base_id", "agent", "active_correct"]
        ].rename(columns={"active_correct": "scale"})

        b = sub[sub["method"] == "SCALE-NoInv"][
            ["seed", "base_id", "agent", "active_correct"]
        ].rename(columns={"active_correct": "noinv"})

        pair = a.merge(
            b,
            on=["seed", "base_id", "agent"],
            how="inner",
        )

        if not len(pair):
            continue

        diff = (
            pair["scale"].to_numpy(float)
            - pair["noinv"].to_numpy(float)
        )

        rng = np.random.default_rng(
            int(SEED + 10000*float(visibility) + len(g2_case_rows))
        )
        boots = []

        # Cluster bootstrap by seed + base, preserving agent correlation.
        pair["_cluster"] = (
            pair["seed"].astype(str)
            + "::"
            + pair["base_id"].astype(str)
        )
        clusters = pair["_cluster"].unique()

        for _ in range(2000):
            sampled = rng.choice(
                clusters,
                size=len(clusters),
                replace=True,
            )
            parts = [
                pair[pair["_cluster"] == c]
                for c in sampled
            ]
            bb = pd.concat(parts, ignore_index=True)
            boots.append(
                float((bb["scale"] - bb["noinv"]).mean())
            )

        lo, hi = np.quantile(boots, [0.025, 0.975])

        g2_case_rows.append({
            "visibility": float(visibility),
            "split": split_name,
            "n_pairs": len(pair),
            "delta_scale_minus_noinv": float(diff.mean()),
            "ci_low": float(lo),
            "ci_high": float(hi),
            "scale_only_correct": int(
                ((pair["scale"] == 1) & (pair["noinv"] == 0)).sum()
            ),
            "noinv_only_correct": int(
                ((pair["scale"] == 0) & (pair["noinv"] == 1)).sum()
            ),
            "both_correct": int(
                ((pair["scale"] == 1) & (pair["noinv"] == 1)).sum()
            ),
            "both_wrong": int(
                ((pair["scale"] == 0) & (pair["noinv"] == 0)).sum()
            ),
        })

g2_case_diag = pd.DataFrame(g2_case_rows)

print("\n[3A] G2 — SEED SIGN CONSISTENCY")
display(g2_sign_summary.round(4))

print("\n[3B] G2 — CASE-LEVEL SCALE VS SCALE-NoInv")
display(g2_case_diag.round(4))

g2_seed_pair.to_csv(
    RESOLVE_DIR / "g2_seed_pair_deltas.csv",
    index=False,
)
g2_sign_summary.to_csv(
    RESOLVE_DIR / "g2_sign_consistency.csv",
    index=False,
)
g2_case_diag.to_csv(
    RESOLVE_DIR / "g2_case_diagnostics.csv",
    index=False,
)

# A data-driven *interpretation*, not a new gate:
# invariance is "reliably useful" only if at least one cell has CI > 0 AND
# no cell has CI < 0.  This is deliberately conservative.
g2_positive_cells = (
    (g2_case_diag["ci_low"] > 0).sum()
    if len(g2_case_diag) else 0
)
g2_negative_cells = (
    (g2_case_diag["ci_high"] < 0).sum()
    if len(g2_case_diag) else 0
)

g2_reliable_regularizer = bool(
    g2_positive_cells > 0
    and g2_negative_cells == 0
)

# =============================================================================
# 4. G2 SIMPLIFICATION SAFETY CHECK
# =============================================================================
#
# If G2 fails, a reviewer may ask why the optional regularizer remains in the
# deployed method at all.
#
# We therefore compare the already-trained Full SCALE and SCALE-NoInv ensembles
# on the core supported CLOSED-SET semantic interface.
#
# This is NOT a replacement primary model and does not alter reported SCALE
# numbers.  It tells us whether removing invariance is a plausible future
# simplification, or whether it materially damages supported behavior.
# =============================================================================

simplify_rows = []

eval_specs = [
    ("known_A", core_test_base, "A"),
    ("known_B", core_test_base, "B"),
    ("c1_natural", shift_test_base, "C1"),
    ("c2_schema", shift_test_base, "C2"),
]

for split_name, bases, impl in eval_specs:
    for base in bases:
        for agent in AGENTS:
            msg = realize_message(
                base, agent, impl, 0
            )
            gold = hop_contract(base, agent)

            for method in ["SCALE", "SCALE-NoInv"]:
                pred = ensemble_predict(
                    method,
                    [msg],
                    BASE_GRAPH,
                )[0]["contract"]
                pred = ontology_repair(pred)

                simplify_rows.append({
                    "split": split_name,
                    "base_id": base["base_id"],
                    "agent": agent,
                    "method": method,
                    "active_correct": active_correct(
                        gold, pred, agent
                    ),
                })

g2_simplify_df = pd.DataFrame(simplify_rows)

g2_simplify_summary = (
    g2_simplify_df.groupby(
        ["split", "method"],
        as_index=False
    )
    .agg(
        active_accuracy=("active_correct", "mean"),
        n=("active_correct", "size"),
    )
)

# Paired deltas.
g2_simplify_delta_rows = []

for split_name in g2_simplify_df["split"].unique():
    sub = g2_simplify_df[
        g2_simplify_df["split"] == split_name
    ]

    a = sub[sub["method"] == "SCALE"][
        ["base_id", "agent", "active_correct"]
    ].rename(columns={"active_correct": "scale"})

    b = sub[sub["method"] == "SCALE-NoInv"][
        ["base_id", "agent", "active_correct"]
    ].rename(columns={"active_correct": "noinv"})

    p = a.merge(
        b,
        on=["base_id", "agent"],
        how="inner",
    )

    g2_simplify_delta_rows.append({
        "split": split_name,
        "n": len(p),
        "scale_accuracy": float(p["scale"].mean()),
        "noinv_accuracy": float(p["noinv"].mean()),
        "scale_minus_noinv": float(
            (p["scale"] - p["noinv"]).mean()
        ),
        "different_predictions": int(
            (p["scale"] != p["noinv"]).sum()
        ),
    })

g2_simplify_delta = pd.DataFrame(
    g2_simplify_delta_rows
)

print("\n[4A] G2 — OPTIONAL REGULARIZER SIMPLIFICATION CHECK")
display(g2_simplify_summary.round(4))
display(g2_simplify_delta.round(4))

g2_simplify_df.to_csv(
    RESOLVE_DIR / "g2_simplification_cases.csv",
    index=False,
)
g2_simplify_summary.to_csv(
    RESOLVE_DIR / "g2_simplification_summary.csv",
    index=False,
)
g2_simplify_delta.to_csv(
    RESOLVE_DIR / "g2_simplification_deltas.csv",
    index=False,
)

max_abs_noinv_delta = (
    float(
        g2_simplify_delta["scale_minus_noinv"]
        .abs().max()
    )
    if len(g2_simplify_delta)
    else np.nan
)

# =============================================================================
# 5. ROUTER FAILURE DIAGNOSIS — COMPLIANCE_ITEM
# =============================================================================
#
# The previous leave-one-root-out experiment found strong routing for four roots
# but near-chance behavior for COMPLIANCE_ITEM.
#
# We diagnose *why* using the four features already used by the router:
#
#   cb_maxprob
#   cb_margin
#   onto_advantage
#   best_unseen_score
#
# This is diagnosis only.  We do NOT add a root-specific threshold after seeing
# the failure, because that would invalidate the leave-one-root-out claim.
# =============================================================================

ROUTER_FEATURE_NAMES = [
    "cb_maxprob",
    "cb_margin",
    "onto_advantage",
    "best_unseen_score",
]

router_diag_rows = []

# Known reference examples.
for base in shift_test_base:
    for impl in ["C1", "C2"]:
        txt = realize_message(
            base, "PLANNER", impl, 0
        )
        f = _router_features(txt)

        router_diag_rows.append({
            "class": "KNOWN",
            "root": base["object"],
            "surface": impl,
            **{
                name: float(value)
                for name, value
                in zip(ROUTER_FEATURE_NAMES, f)
            },
        })

# Unseen root-specific semantic aliases.
for node in LARGE_OPAQUE_NODES:
    txt = large_ontology_message(
        node, "semantic_alias"
    )
    f = _router_features(txt)

    router_diag_rows.append({
        "class": "UNSEEN",
        "root": node["canonical_root"],
        "surface": "semantic_alias",
        **{
            name: float(value)
            for name, value
            in zip(ROUTER_FEATURE_NAMES, f)
        },
    })

router_feature_df = pd.DataFrame(
    router_diag_rows
)

router_feature_summary = (
    router_feature_df.groupby(
        ["class", "root"],
        as_index=False
    )
    .agg(
        n=("root", "size"),
        **{
            f"{name}_mean": (name, "mean")
            for name in ROUTER_FEATURE_NAMES
        },
        **{
            f"{name}_sd": (name, "std")
            for name in ROUTER_FEATURE_NAMES
        },
    )
)

# Standardized distance between each unseen root and KNOWN reference centroid.
known_X = router_feature_df[
    router_feature_df["class"] == "KNOWN"
][ROUTER_FEATURE_NAMES].to_numpy(float)

mu = known_X.mean(axis=0)
sd = known_X.std(axis=0)
sd = np.where(sd < 1e-9, 1.0, sd)

router_root_distance_rows = []

for root in sorted(
    router_feature_df[
        router_feature_df["class"] == "UNSEEN"
    ]["root"].unique()
):
    X = router_feature_df[
        (router_feature_df["class"] == "UNSEEN")
        & (router_feature_df["root"] == root)
    ][ROUTER_FEATURE_NAMES].to_numpy(float)

    z = (X - mu) / sd
    dist = np.linalg.norm(z, axis=1)

    # Extract the already-computed held-root performance.
    perf = router_root_generalization_df[
        router_root_generalization_df[
            "held_out_root"
        ].astype(str).eq(str(root))
    ]

    router_root_distance_rows.append({
        "root": root,
        "mean_standardized_distance_from_known": float(dist.mean()),
        "min_standardized_distance_from_known": float(dist.min()),
        "heldout_auc": (
            float(perf["test_auc"].iloc[0])
            if len(perf) else np.nan
        ),
        "heldout_unseen_recall": (
            float(
                perf["held_root_unseen_recall"].iloc[0]
            )
            if len(perf) else np.nan
        ),
    })

router_root_distance_df = pd.DataFrame(
    router_root_distance_rows
)

print("\n[5A] ROUTER — FEATURE DISTRIBUTIONS")
display(router_feature_summary.round(4))

print("\n[5B] ROUTER — ROOT DISTANCE VS HELD-OUT PERFORMANCE")
display(router_root_distance_df.round(4))

router_feature_df.to_csv(
    RESOLVE_DIR / "router_feature_cases.csv",
    index=False,
)
router_feature_summary.to_csv(
    RESOLVE_DIR / "router_feature_summary.csv",
    index=False,
)
router_root_distance_df.to_csv(
    RESOLVE_DIR / "router_root_distance_diagnostic.csv",
    index=False,
)

# Plot each feature separately (no subplots).
for feat in ROUTER_FEATURE_NAMES:
    fig, ax = plt.subplots(figsize=(7.4, 4.6))

    roots = ["KNOWN"] + sorted(
        router_feature_df[
            router_feature_df["class"] == "UNSEEN"
        ]["root"].unique()
    )

    data = []
    labels = []

    data.append(
        router_feature_df[
            router_feature_df["class"] == "KNOWN"
        ][feat].to_numpy(float)
    )
    labels.append("KNOWN")

    for root in roots[1:]:
        data.append(
            router_feature_df[
                (router_feature_df["class"] == "UNSEEN")
                & (router_feature_df["root"] == root)
            ][feat].to_numpy(float)
        )
        labels.append(root)

    ax.boxplot(data, tick_labels=labels)
    ax.set_ylabel(feat)
    ax.set_title(
        f"Open-set router diagnostic: {feat}"
    )
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(
        RESOLVE_DIR / f"router_{feat}_by_root.png",
        dpi=220,
    )
    plt.show()

# =============================================================================
# 6. NEGATIVE-RESULT DECISION TABLE
# =============================================================================
#
# Explicitly separate:
#   - original gate outcome
#   - post-hoc diagnosis
#   - engineering remediation
#   - what may be claimed in the paper
# =============================================================================

decision_rows = [
    {
        "item": "G0 strict arbitrary-reader robustness",
        "prespecified_status": "FAIL",
        "diagnosis": (
            "Reader-specific parse / label / semantic failures; "
            "native SCALE interface is a separate layer."
        ),
        "remediation": (
            "Consume role-active semantic contracts through the typed/native "
            "adapter instead of unconstrained generative reinterpretation."
        ),
        "remediation_supported": (
            "YES" if typed_resolution_supported else "PARTIAL"
        ),
        "paper_claim": (
            "G0 remains failed. Report typed adapter as an engineering "
            "resolution of the arbitrary-reader boundary, not as a gate pass."
        ),
    },
    {
        "item": "G2 label-efficient directed invariance",
        "prespecified_status": "FAIL",
        "diagnosis": (
            f"Positive CI cells={g2_positive_cells}; "
            f"negative CI cells={g2_negative_cells}; "
            "benefit is not sign-consistent across shifts/visibility."
        ),
        "remediation": (
            "Treat directed invariance as optional; do not make label-efficiency "
            "a contribution. Consider lambda_inv=0 as the simpler deployment "
            "default if the simplification check is neutral."
        ),
        "remediation_supported": (
            "YES"
            if (
                np.isfinite(max_abs_noinv_delta)
                and max_abs_noinv_delta <= 0.03
            )
            else "QUALIFIED"
        ),
        "paper_claim": (
            "G2 remains failed. Directed alignment is a diagnostic/optional "
            "regularizer, not a supported general advantage."
        ),
    },
    {
        "item": "Open-set router root generalization",
        "prespecified_status": "NOT A PRIMARY GATE",
        "diagnosis": (
            "Strong mean performance but severe semantic-root heterogeneity; "
            "COMPLIANCE_ITEM is the key failure family."
        ),
        "remediation": (
            "Do not introduce a post-hoc root-specific threshold. Report the "
            "heterogeneity and keep routing as a supporting mechanism."
        ),
        "remediation_supported": "BOUNDARY",
        "paper_claim": (
            "Practical non-oracle routing is demonstrated, but universal unseen "
            "root generalization is not."
        ),
    },
    {
        "item": "Generative LLM executor drift robustness",
        "prespecified_status": "INCONCLUSIVE",
        "diagnosis": (
            "Small generative executors did not meet the clean competence gate."
        ),
        "remediation": (
            "Retain independent learned-executor evidence; do not lower the "
            "clean competence threshold post hoc."
        ),
        "remediation_supported": "YES",
        "paper_claim": (
            "Do not claim generative-executor drift robustness."
        ),
    },
    {
        "item": "External ontology mechanism",
        "prespecified_status": "EXPLORATORY",
        "diagnosis": (
            "Schema.org supports structural canonicalization but is not a "
            "full end-to-end SCALE benchmark."
        ),
        "remediation": (
            "Use it as external mechanism evidence and retain full external "
            "ontology deployment as future work."
        ),
        "remediation_supported": "YES",
        "paper_claim": (
            "Human-authored ontology structure improves coarse structural "
            "placement; do not call this end-to-end external SCALE superiority."
        ),
    },
]

decision_df = pd.DataFrame(decision_rows)

print("\n[6] FINAL NEGATIVE-RESULT / REMEDIATION TABLE")
display(decision_df)

decision_df.to_csv(
    RESOLVE_DIR / "final_negative_result_decisions.csv",
    index=False,
)

# =============================================================================
# 7. FINAL PAPER-READY EVIDENCE SNAPSHOT
# =============================================================================

def _fmt(x):
    try:
        x = float(x)
        return f"{x:.4f}" if np.isfinite(x) else "NA"
    except Exception:
        return "NA"

# Pull headline existing results defensively.
controlled_auc = (
    _safe_auc(
        drift_df["failure"],
        drift_df["prepolicy_active_scd"],
    )
    if "drift_df" in globals() and len(drift_df)
    else np.nan
)

naturalistic_auc = (
    float(nat_failure_auc)
    if "nat_failure_auc" in globals()
    else np.nan
)

learned_executor_auc = np.nan
if (
    "independent_executor_summary" in globals()
    and len(independent_executor_summary)
):
    learned_executor_auc = float(
        independent_executor_summary[
            "failure_auc_active_scd"
        ].min()
    )

schema_graph = np.nan
schema_direct = np.nan
if (
    "schema_external_summary" in globals()
    and len(schema_external_summary)
):
    schema_graph = float(
        schema_external_summary[
            "ground_then_graph_accuracy"
        ].iloc[0]
    )
    schema_direct = float(
        schema_external_summary[
            "direct_root_accuracy"
        ].iloc[0]
    )

g0_reader_min = (
    float(g0_reader_summary["task_success"].min())
    if len(g0_reader_summary)
    else np.nan
)

report = f"""
# SCALE — Final Failure Resolution Report

## G0 — strict arbitrary-reader robustness

**Original gate outcome: FAIL. It must remain FAIL.**

Minimum arbitrary-reader aggregate task success across reader families:
**{_fmt(g0_reader_min)}**

The decomposition in `g0_layer_failure_summary.csv` separates runtime/overflow,
parse, allowed-label, semantic, and final-task failures.

The final engineering remediation is a **typed/native semantic adapter**, not
another generative reader. Across the matched SCALE core/C1/C2 cases, the minimum
typed-adapter task success is **{_fmt(typed_min_task)}**.

Interpretation:
- if the typed path remains strong while arbitrary LLM consumption fails,
  G0 is correctly localized to the consumer layer;
- do **not** retroactively mark G0 as passed;
- paper wording should say that arbitrary generative readers are not guaranteed
  consumers of the semantic contract and typed adapters are the reliable interface.

## G2 — label-efficient directed invariance

**Original gate outcome: FAIL. It must remain FAIL.**

Across post-hoc SCALE-vs-NoInv paired cells:
- cells with CI strictly above zero: **{g2_positive_cells}**
- cells with CI strictly below zero: **{g2_negative_cells}**
- reliable regularizer under the deliberately conservative diagnostic:
  **{"YES" if g2_reliable_regularizer else "NO"}**

Maximum absolute Full-SCALE vs NoInv accuracy difference in the closed-set
simplification check:
**{_fmt(max_abs_noinv_delta)}**

Interpretation:
- if the main supported behavior is unchanged when invariance is removed, the
  cleanest paper position is that directed invariance is optional and unnecessary
  for the supported claims;
- do not invent a new adaptive invariance gate after observing G2;
- do not claim label efficiency.

## Router root heterogeneity

See `router_root_distance_diagnostic.csv`.

The purpose of the diagnostic is to explain the already-observed COMPLIANCE_ITEM
failure through the router feature geometry. No root-specific threshold is fitted.

Paper position:
**non-oracle routing is practical but not uniformly root-general.**

## Evidence that remains primary

- Controlled Active-SCD failure AUROC: **{_fmt(controlled_auc)}**
- Naturalistic version-evolution failure AUROC: **{_fmt(naturalistic_auc)}**
- Minimum independent learned-executor Active-SCD AUROC: **{_fmt(learned_executor_auc)}**
- Schema.org direct root accuracy: **{_fmt(schema_direct)}**
- Schema.org ground→graph root accuracy: **{_fmt(schema_graph)}**

## Recommended final architecture story

1. Known semantics → Full SCALE discriminative decoder.
2. Explicit ontology IDs → deterministic normalization/canonicalization.
3. Candidate open-world semantics → learned router → ontology branch.
4. Canonical contract → deterministic validation.
5. Role-active projection → leakage-free Active-SCD monitoring.
6. Downstream production consumers should use a typed adapter; arbitrary free-form
   LLM readers remain an unsupported convenience layer.
7. Directed invariance is optional because G2 does not establish a general gain.

## Stop recommendation

After this section, do not add additional synthetic losses, larger procedural
ontologies, more label-efficiency curves, or a post-hoc root-specific router.

The remaining scientifically honest limitations are:
- arbitrary free-form reader robustness;
- root-dependent open-set routing;
- no demonstrated label-efficiency advantage from directed invariance;
- no competence-qualified small generative executor drift result;
- Schema.org is mechanism-level external evidence rather than a full external SCALE run.

These are appropriate limitations, not missing experiments that must be forced into
positive results.
"""

(
    RESOLVE_DIR / "FINAL_FAILURE_RESOLUTION_REPORT.md"
).write_text(
    report.strip() + "\n",
    encoding="utf-8",
)

display(Markdown(report))

print("\nSaved final-resolution artifacts:")
for p in sorted(RESOLVE_DIR.iterdir()):
    print(" -", p.name)


In [ ]:

# =============================================================================
# FINAL SAFETY-CRITICAL CLOSURE & SUBMISSION REGISTRY
# Run only after every previous notebook cell has completed.
# =============================================================================

from pathlib import Path
import copy, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

SAFETY_DIR = ROOT / "final_safety_closure"
SAFETY_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SCALE — FINAL SAFETY-CRITICAL CLOSURE")
print("Output:", SAFETY_DIR)
print("=" * 100)

_REQUIRED = [
    "shift_test_base",
    "drift_base",
    "AGENTS",
    "BASE_GRAPH",
    "LARGE_OPAQUE_GRAPH",
    "LARGE_OPAQUE_NODES",
    "large_ontology_message",
    "strict_explicit_id_lookup",
    "STRICT_FINAL_ROUTER",
    "strict_router_threshold",
    "_router_features",
    "learned_large_prediction",
    "ensemble_predict",
    "canonicalize_lookup",
    "global_semantics",
    "deterministic_policy",
    "drift_df",
]
_missing = [x for x in _REQUIRED if x not in globals()]
assert not _missing, (
    "Run the previous notebook first. Missing: " + ", ".join(_missing)
)

def _safe_num(x):
    try:
        x = float(x)
        return x if np.isfinite(x) else np.nan
    except Exception:
        return np.nan

def _wilson(successes, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = successes / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return float(p), float(center-half), float(center+half)

# =============================================================================
# 1. SELECTIVE SAFETY ENVELOPE FOR DEPLOYABLE ROUTING
# =============================================================================
#
# Motivation:
# In safety-critical systems, a component need not always return a forced answer.
# When the learned known/open router and the ontology branch disagree, a typed
# interface can return ABSTAIN / REVIEW rather than silently committing to a
# semantic interpretation.
#
# IMPORTANT:
# This is a post-hoc safety diagnostic, NOT a redefinition of the router gate.
# We report coverage together with accepted-case accuracy and error-capture rate.
#
# We test four fixed policies:
#   P0: no abstention
#   P1: abstain whenever known and ontology decoders disagree
#   P2: P1 OR router score lies within +/-0.10 of tau
#   P3: P1 OR router score lies within +/-0.20 of tau
#
# No root-specific threshold is fitted.
# =============================================================================

def _known_root_prediction(text):
    pred = ensemble_predict(
        "SCALE",
        [text],
        BASE_GRAPH,
    )[0]
    return pred["contract"].get("object")

def _onto_root_prediction(text):
    _, root = learned_large_prediction(
        "SCALE-Full",
        text,
        LARGE_OPAQUE_GRAPH,
    )
    return root

def _router_score(text):
    return float(
        STRICT_FINAL_ROUTER.predict_proba(
            _router_features(text).reshape(1, -1)
        )[0, 1]
    )

safety_route_rows = []

# Known inputs: C1/C2.
for base in shift_test_base:
    for surface in ["C1", "C2"]:
        text = realize_message(base, "PLANNER", surface, 0)

        id_hit = strict_explicit_id_lookup(
            text, LARGE_OPAQUE_GRAPH
        )
        p_open = _router_score(text)
        known_root = _known_root_prediction(text)
        onto_root = _onto_root_prediction(text)

        if id_hit is not None:
            default_root = canonicalize_lookup(
                id_hit, LARGE_OPAQUE_GRAPH
            )
            default_route = "symbolic_id"
        elif p_open >= strict_router_threshold:
            default_root = onto_root
            default_route = "open_world_ontology"
        else:
            default_root = known_root
            default_route = "known_scale_decoder"

        safety_route_rows.append({
            "kind": "known",
            "root": base["object"],
            "item_id": base["base_id"],
            "surface": surface,
            "gold_root": base["object"],
            "p_open": p_open,
            "known_root": known_root,
            "onto_root": onto_root,
            "decoder_disagreement": int(
                str(known_root) != str(onto_root)
            ),
            "router_distance_to_tau": abs(
                p_open - float(strict_router_threshold)
            ),
            "default_route": default_route,
            "default_root": default_root,
            "default_correct": int(
                default_root == base["object"]
            ),
        })

# Unseen semantic aliases are the relevant hard open-world regime.
for node in LARGE_OPAQUE_NODES:
    text = large_ontology_message(
        node, "semantic_alias"
    )

    id_hit = strict_explicit_id_lookup(
        text, LARGE_OPAQUE_GRAPH
    )
    p_open = _router_score(text)
    known_root = _known_root_prediction(text)
    onto_root = _onto_root_prediction(text)

    if id_hit is not None:
        default_root = canonicalize_lookup(
            id_hit, LARGE_OPAQUE_GRAPH
        )
        default_route = "symbolic_id"
    elif p_open >= strict_router_threshold:
        default_root = onto_root
        default_route = "open_world_ontology"
    else:
        default_root = known_root
        default_route = "known_scale_decoder"

    safety_route_rows.append({
        "kind": "unseen",
        "root": node["canonical_root"],
        "item_id": node["label"],
        "surface": "semantic_alias",
        "gold_root": node["canonical_root"],
        "p_open": p_open,
        "known_root": known_root,
        "onto_root": onto_root,
        "decoder_disagreement": int(
            str(known_root) != str(onto_root)
        ),
        "router_distance_to_tau": abs(
            p_open - float(strict_router_threshold)
        ),
        "default_route": default_route,
        "default_root": default_root,
        "default_correct": int(
            default_root == node["canonical_root"]
        ),
    })

# Optional harder aliases.
if "HARD_ALIAS_BY_ROOT" in globals():
    for root, aliases in HARD_ALIAS_BY_ROOT.items():
        for j, alias in enumerate(aliases):
            text = f"Planner handoff: the target is {alias}."

            id_hit = strict_explicit_id_lookup(
                text, LARGE_OPAQUE_GRAPH
            )
            p_open = _router_score(text)
            known_root = _known_root_prediction(text)
            onto_root = _onto_root_prediction(text)

            if id_hit is not None:
                default_root = canonicalize_lookup(
                    id_hit, LARGE_OPAQUE_GRAPH
                )
                default_route = "symbolic_id"
            elif p_open >= strict_router_threshold:
                default_root = onto_root
                default_route = "open_world_ontology"
            else:
                default_root = known_root
                default_route = "known_scale_decoder"

            safety_route_rows.append({
                "kind": "unseen",
                "root": root,
                "item_id": f"{root}::hard::{j}",
                "surface": "hard_semantic_alias",
                "gold_root": root,
                "p_open": p_open,
                "known_root": known_root,
                "onto_root": onto_root,
                "decoder_disagreement": int(
                    str(known_root) != str(onto_root)
                ),
                "router_distance_to_tau": abs(
                    p_open - float(strict_router_threshold)
                ),
                "default_route": default_route,
                "default_root": default_root,
                "default_correct": int(
                    default_root == root
                ),
            })

safety_route_df = pd.DataFrame(safety_route_rows)

SAFETY_POLICIES = {
    "P0_force_route": lambda d: np.zeros(len(d), dtype=bool),
    "P1_abstain_on_decoder_disagreement": (
        lambda d: d["decoder_disagreement"].to_numpy(bool)
    ),
    "P2_disagreement_or_tau_band_0.10": (
        lambda d: (
            d["decoder_disagreement"].to_numpy(bool)
            | (
                d["router_distance_to_tau"].to_numpy(float)
                <= 0.10
            )
        )
    ),
    "P3_disagreement_or_tau_band_0.20": (
        lambda d: (
            d["decoder_disagreement"].to_numpy(bool)
            | (
                d["router_distance_to_tau"].to_numpy(float)
                <= 0.20
            )
        )
    ),
}

selective_rows = []

for policy_name, policy_fn in SAFETY_POLICIES.items():
    abstain = policy_fn(safety_route_df)
    accepted = ~abstain

    total_errors = int(
        (safety_route_df["default_correct"] == 0).sum()
    )
    caught_errors = int(
        (
            abstain
            & (
                safety_route_df["default_correct"].to_numpy(int)
                == 0
            )
        ).sum()
    )

    accepted_correct = safety_route_df.loc[
        accepted, "default_correct"
    ]

    selective_rows.append({
        "policy": policy_name,
        "coverage": float(accepted.mean()),
        "abstention_rate": float(abstain.mean()),
        "accepted_accuracy": (
            float(accepted_correct.mean())
            if len(accepted_correct) else np.nan
        ),
        "accepted_error_rate": (
            float(1.0 - accepted_correct.mean())
            if len(accepted_correct) else np.nan
        ),
        "error_capture_rate": (
            caught_errors / total_errors
            if total_errors else np.nan
        ),
        "caught_errors": caught_errors,
        "total_default_errors": total_errors,
        "n": len(safety_route_df),
    })

selective_summary = pd.DataFrame(selective_rows)

# Per-root behavior under the simplest safety policy P1.
p1_abstain = SAFETY_POLICIES[
    "P1_abstain_on_decoder_disagreement"
](safety_route_df)

tmp = safety_route_df.copy()
tmp["p1_abstain"] = p1_abstain.astype(int)
tmp["p1_accept"] = 1 - tmp["p1_abstain"]

root_safety_summary = (
    tmp[tmp["kind"] == "unseen"]
    .groupby("root", as_index=False)
    .agg(
        n=("item_id", "size"),
        default_accuracy=("default_correct", "mean"),
        abstention_rate=("p1_abstain", "mean"),
        accepted_n=("p1_accept", "sum"),
    )
)

print("\n[1A] SELECTIVE ROUTING SAFETY ENVELOPE")
display(selective_summary.round(4))

print("\n[1B] P1 ABSTENTION BY UNSEEN ROOT")
display(root_safety_summary.round(4))

safety_route_df.to_csv(
    SAFETY_DIR / "selective_routing_cases.csv",
    index=False,
)
selective_summary.to_csv(
    SAFETY_DIR / "selective_routing_summary.csv",
    index=False,
)
root_safety_summary.to_csv(
    SAFETY_DIR / "selective_routing_by_root.csv",
    index=False,
)

# Risk-coverage plot.
plot_df = selective_summary.sort_values("coverage")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(
    plot_df["coverage"],
    plot_df["accepted_error_rate"],
    marker="o",
)
ax.set_xlabel("Coverage")
ax.set_ylabel("Accepted-case error rate")
ax.set_title("Selective semantic routing: risk–coverage")
fig.tight_layout()
fig.savefig(
    SAFETY_DIR / "selective_routing_risk_coverage.png",
    dpi=220,
)
plt.show()

# =============================================================================
# 2. ROLE-ACTIVE MASK NECESSITY / DECISION-SENSITIVITY AUDIT
# =============================================================================
#
# Reviewer concern:
# Role-active masks are task-defined.  Are the selected concepts genuinely
# decision-operative, or merely convenient labels?
#
# We intervene on each active semantic dimension in the deterministic downstream
# policy and measure how often changing that dimension changes the final action
# or final authority.  This is paired with the already-existing inactive-field
# invariance result.
#
# This does NOT claim that the masks are universally correct for every future
# downstream consumer.  It tests necessity for the evaluated decision function.
# =============================================================================

base_semantics = [
    global_semantics(b)
    for b in drift_base
]

object_values = sorted({
    s["object"] for s in base_semantics
})
evidence_values = sorted({
    s["evidence_state"] for s in base_semantics
})
authority_values = sorted({
    s["authority"] for s in base_semantics
})

ACTIVE_VALUE_SETS = {
    "object": object_values,
    "evidence_state": evidence_values,
    "authority": authority_values,
}

mask_rows = []

for base, sem in zip(drift_base, base_semantics):
    baseline_policy = deterministic_policy(sem)

    for field, values in ACTIVE_VALUE_SETS.items():
        alternatives = [
            v for v in values
            if v != sem[field]
        ]

        for alt in alternatives:
            pert = copy.deepcopy(sem)
            pert[field] = alt
            pert_policy = deterministic_policy(pert)

            mask_rows.append({
                "base_id": base["base_id"],
                "field": field,
                "original_value": sem[field],
                "perturbed_value": alt,
                "baseline_action": (
                    None if baseline_policy is None
                    else baseline_policy.get("action")
                ),
                "baseline_authority": (
                    None if baseline_policy is None
                    else baseline_policy.get("authority")
                ),
                "perturbed_action": (
                    None if pert_policy is None
                    else pert_policy.get("action")
                ),
                "perturbed_authority": (
                    None if pert_policy is None
                    else pert_policy.get("authority")
                ),
                "decision_changed": int(
                    pert_policy != baseline_policy
                ),
                "action_changed": int(
                    (
                        None if pert_policy is None
                        else pert_policy.get("action")
                    )
                    != (
                        None if baseline_policy is None
                        else baseline_policy.get("action")
                    )
                ),
                "authority_changed": int(
                    (
                        None if pert_policy is None
                        else pert_policy.get("authority")
                    )
                    != (
                        None if baseline_policy is None
                        else baseline_policy.get("authority")
                    )
                ),
            })

mask_sensitivity_df = pd.DataFrame(mask_rows)

mask_sensitivity_summary = (
    mask_sensitivity_df.groupby(
        "field",
        as_index=False,
    )
    .agg(
        n_interventions=("decision_changed", "size"),
        decision_change_rate=("decision_changed", "mean"),
        action_change_rate=("action_changed", "mean"),
        authority_change_rate=("authority_changed", "mean"),
    )
)

# Per-base necessity: does at least one legal alternative for the field alter
# the decision for this task?
mask_base_necessity = (
    mask_sensitivity_df.groupby(
        ["base_id", "field"],
        as_index=False,
    )
    .agg(
        any_decision_change=("decision_changed", "max"),
        mean_decision_change=("decision_changed", "mean"),
    )
)

mask_base_summary = (
    mask_base_necessity.groupby(
        "field",
        as_index=False,
    )
    .agg(
        bases=("base_id", "nunique"),
        fraction_bases_field_is_decision_relevant=(
            "any_decision_change", "mean"
        ),
        average_alternative_effect=(
            "mean_decision_change", "mean"
        ),
    )
)

print("\n[2A] ROLE-ACTIVE FIELD INTERVENTION SENSITIVITY")
display(mask_sensitivity_summary.round(4))

print("\n[2B] FRACTION OF BASES WHERE EACH FIELD CAN ALTER THE DECISION")
display(mask_base_summary.round(4))

mask_sensitivity_df.to_csv(
    SAFETY_DIR / "role_active_mask_interventions.csv",
    index=False,
)
mask_sensitivity_summary.to_csv(
    SAFETY_DIR / "role_active_mask_sensitivity.csv",
    index=False,
)
mask_base_summary.to_csv(
    SAFETY_DIR / "role_active_mask_base_necessity.csv",
    index=False,
)

# =============================================================================
# 3. CANONICAL FINAL RESULTS REGISTRY
# =============================================================================
#
# Purpose:
# The project now contains several iterations and reviewer-hardening patches.
# This registry is the single paper-facing numerical source of truth.
#
# It deliberately records whether an item is:
#   PRIMARY / SUPPORTING / QUALIFIED / NEGATIVE / EXPLORATORY
# and whether it was PRESPECIFIED or POST-HOC.
# =============================================================================

registry = []

def reg(
    key,
    value,
    claim_class,
    analysis_status,
    paper_location,
    note,
):
    registry.append({
        "key": key,
        "value": _safe_num(value),
        "claim_class": claim_class,
        "analysis_status": analysis_status,
        "paper_location": paper_location,
        "note": note,
    })

# Controlled representation.
reg(
    "scale_c1_active_accuracy",
    1.0,
    "PRIMARY",
    "PRESPECIFIED",
    "Main Table 1",
    "Natural paraphrase role-active accuracy.",
)
reg(
    "scale_c2_active_accuracy",
    0.8889,
    "PRIMARY",
    "PRESPECIFIED",
    "Main Table 1",
    "Schema/tool role-active accuracy.",
)

# Drift.
if len(drift_df):
    from sklearn.metrics import roc_auc_score
    controlled_auc = roc_auc_score(
        drift_df["failure"],
        drift_df["prepolicy_active_scd"],
    )
else:
    controlled_auc = np.nan

reg(
    "active_scd_controlled_failure_auc",
    controlled_auc,
    "PRIMARY",
    "PRESPECIFIED",
    "Main Table 2",
    "Leakage-free controlled failure prediction.",
)

if "nat_failure_auc" in globals():
    reg(
        "active_scd_naturalistic_failure_auc",
        nat_failure_auc,
        "PRIMARY",
        "POST-HOC ROBUSTNESS",
        "Main Table 2",
        "Constructed version-evolution robustness.",
    )

if (
    "independent_executor_summary" in globals()
    and len(independent_executor_summary)
):
    for _, row in independent_executor_summary.iterrows():
        reg(
            "active_scd_independent_executor_"
            + str(row["executor"]),
            row["failure_auc_active_scd"],
            "PRIMARY",
            "POST-HOC ROBUSTNESS",
            "Main Table 2 / Appendix",
            "Executor never consumes SCALE contracts.",
        )

# Routing.
if "router_root_aggregate" in globals() and len(router_root_aggregate):
    reg(
        "router_leave_one_root_mean_auc",
        router_root_aggregate["mean_auc"].iloc[0],
        "QUALIFIED",
        "POST-HOC DIAGNOSTIC",
        "Routing Appendix",
        "Mean root-held-out open-set AUROC.",
    )
    reg(
        "router_leave_one_root_min_auc",
        router_root_aggregate["min_auc"].iloc[0],
        "QUALIFIED",
        "POST-HOC DIAGNOSTIC",
        "Limitations / Routing Appendix",
        "Worst semantic-root open-set AUROC.",
    )

# Schema.org.
if "schema_external_summary" in globals() and len(schema_external_summary):
    srow = schema_external_summary.iloc[0]
    reg(
        "schemaorg_direct_root_accuracy",
        srow["direct_root_accuracy"],
        "SUPPORTING",
        "EXPLORATORY",
        "Ontology Appendix",
        "Direct description-to-root prediction.",
    )
    reg(
        "schemaorg_graph_root_accuracy",
        srow["ground_then_graph_accuracy"],
        "SUPPORTING",
        "EXPLORATORY",
        "Main ontology result / Appendix",
        "Ground-to-leaf then graph ancestry root placement.",
    )

# G0/G2.
reg(
    "gate_G0",
    0,
    "NEGATIVE",
    "PRESPECIFIED",
    "Gate Table",
    "Strict arbitrary-reader robustness remains failed.",
)
reg(
    "gate_G2",
    0,
    "NEGATIVE",
    "PRESPECIFIED",
    "Gate Table",
    "General label-efficient directed invariance remains failed.",
)

# New safety closure.
for _, row in selective_summary.iterrows():
    reg(
        "selective_" + row["policy"] + "_coverage",
        row["coverage"],
        "SUPPORTING",
        "POST-HOC SAFETY DIAGNOSTIC",
        "Appendix / future safety deployment",
        "Coverage under fixed abstention policy.",
    )
    reg(
        "selective_" + row["policy"] + "_accepted_accuracy",
        row["accepted_accuracy"],
        "SUPPORTING",
        "POST-HOC SAFETY DIAGNOSTIC",
        "Appendix / future safety deployment",
        "Accuracy among non-abstained cases.",
    )

for _, row in mask_base_summary.iterrows():
    reg(
        "role_mask_" + row["field"] + "_base_relevance",
        row["fraction_bases_field_is_decision_relevant"],
        "SUPPORTING",
        "POST-HOC MECHANISM DIAGNOSTIC",
        "Appendix / Limitations",
        "Fraction of evaluated bases where changing this active field can alter the downstream decision.",
    )

registry_df = pd.DataFrame(registry)
registry_df.to_csv(
    SAFETY_DIR / "FINAL_RESULTS_REGISTRY.csv",
    index=False,
)
(
    SAFETY_DIR / "FINAL_RESULTS_REGISTRY.json"
).write_text(
    json.dumps(
        registry,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n[3] CANONICAL PAPER-FACING RESULTS REGISTRY")
display(registry_df)

# =============================================================================
# 4. FINAL STOP / PAPER GUIDANCE REPORT
# =============================================================================

p1 = selective_summary[
    selective_summary["policy"]
    == "P1_abstain_on_decoder_disagreement"
].iloc[0]

report = f"""
# SCALE — Final Safety-Critical Closure Report

## 1. Selective routing

The default forced router remains the paper's reported deployable result.  A
post-hoc reject-option diagnostic evaluates whether unsafe semantic commitment
can be reduced by abstaining when the known and ontology decoders disagree.

For the simplest fixed policy (decoder disagreement only):

- coverage: **{p1['coverage']:.4f}**
- accepted-case accuracy: **{p1['accepted_accuracy']:.4f}**
- error-capture rate: **{p1['error_capture_rate']:.4f}**

Interpret this only as a safety-engineering diagnostic.  It does **not** repair
the original router result or alter the qualified routing claim.

If accepted-case risk drops materially at reasonable coverage, the practical
recommendation is to expose an explicit REVIEW/ABSTAIN state for critical
deployments instead of forcing every semantic extension into known/open routing.

## 2. Role-active mask necessity

The file `role_active_mask_base_necessity.csv` reports the fraction of evaluated
tasks for which changing object, evidence state, or authority can alter the
downstream decision.

This answers a remaining reviewer concern: the active variables are not merely
chosen because they are easy to decode; their interventions are behaviorally
operative for the evaluated policy.

However, role masks remain **consumer-dependent**. If a future downstream
component begins consuming a previously inactive contract field, the mask must
be revised and revalidated. This belongs in Limitations.

## 3. Submission numerical source of truth

Use:

`final_safety_closure/FINAL_RESULTS_REGISTRY.csv`

as the final manuscript-facing numerical registry.

Do not manually copy older numbers from intermediate notebook cells when a
registry entry exists.  The registry explicitly labels prespecified, post-hoc,
qualified, negative, and exploratory evidence.

## 4. What is still genuinely unresolved

The following are legitimate limitations rather than reasons for another
synthetic experiment:

1. **G0 remains failed** for arbitrary free-form generative consumers. Typed
   adapters are the intended production interface.
2. **G2 remains failed** as a general label-efficiency claim. Directed alignment
   is regime-dependent.
3. **Open-set routing remains root-dependent**, especially for known-like
   semantic extensions.
4. **Schema.org is mechanism-level external evidence**, not a full external
   retraining of SCALE.
5. **Constructed version evolution is not a longitudinal production log.**
6. **Small generative executors remain competence-inconclusive.**
7. **Role-active masks are task/consumer dependent** and must evolve when
   downstream consumption changes.

## 5. Final experiment recommendation

**Stop adding experiments after this section.**

The only future experiments likely to add qualitatively new evidence are:
- a longitudinal production deployment with real version history;
- a full external ontology/domain retraining;
- a competent larger generative executor satisfying the clean-task gate.

Those are future-work scale additions, not missing synthetic controls for the
current ICLR submission.
"""

(
    SAFETY_DIR / "FINAL_SAFETY_CLOSURE_REPORT.md"
).write_text(
    report.strip() + "\n",
    encoding="utf-8",
)

display(Markdown(report))

print("\nSaved final safety-closure artifacts:")
for p in sorted(SAFETY_DIR.iterdir()):
    print(" -", p.name)


In [ ]:

# =============================================================================
# FINAL CORRECTNESS PATCH
# Behavioral mask audit + canonical direct Active-SCD AUROC registry
# =============================================================================

from pathlib import Path
import json, copy
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

FINAL_PATCH_DIR = ROOT / "final_correctness_patch"
FINAL_PATCH_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SCALE — FINAL CORRECTNESS PATCH")
print("Output:", FINAL_PATCH_DIR)
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. BEHAVIORAL ROLE-ACTIVE MASK AUDIT
# -----------------------------------------------------------------------------
# We define the downstream behavioral decision as:
#     (action, authority)
# rather than comparing the entire policy dictionary.
# This avoids counting object-value changes as "behavior changes" merely because
# object is echoed in the returned dictionary.
# -----------------------------------------------------------------------------

assert "drift_base" in globals(), "Run previous cells first: drift_base missing."
assert "global_semantics" in globals(), "Run previous cells first: global_semantics missing."
assert "deterministic_policy" in globals(), "Run previous cells first: deterministic_policy missing."

def behavioral_decision(policy):
    if policy is None:
        return None
    return (
        policy.get("action"),
        policy.get("authority"),
    )

base_semantics = [global_semantics(b) for b in drift_base]

object_values = sorted({s["object"] for s in base_semantics})
evidence_values = sorted({s["evidence_state"] for s in base_semantics})
authority_values = sorted({s["authority"] for s in base_semantics})

ACTIVE_VALUE_SETS = {
    "object": object_values,
    "evidence_state": evidence_values,
    "authority": authority_values,
}

corrected_mask_rows = []

for base, sem in zip(drift_base, base_semantics):
    baseline_policy = deterministic_policy(sem)
    baseline_behavior = behavioral_decision(baseline_policy)

    for field, values in ACTIVE_VALUE_SETS.items():
        alternatives = [v for v in values if v != sem[field]]

        for alt in alternatives:
            pert = copy.deepcopy(sem)
            pert[field] = alt
            pert_policy = deterministic_policy(pert)
            pert_behavior = behavioral_decision(pert_policy)

            corrected_mask_rows.append({
                "base_id": base["base_id"],
                "field": field,
                "original_value": sem[field],
                "perturbed_value": alt,

                "baseline_action": (
                    None if baseline_policy is None
                    else baseline_policy.get("action")
                ),
                "baseline_authority": (
                    None if baseline_policy is None
                    else baseline_policy.get("authority")
                ),
                "perturbed_action": (
                    None if pert_policy is None
                    else pert_policy.get("action")
                ),
                "perturbed_authority": (
                    None if pert_policy is None
                    else pert_policy.get("authority")
                ),

                # Correct behavioral comparison:
                "behavior_changed": int(
                    pert_behavior != baseline_behavior
                ),

                # Component-level effects:
                "action_changed": int(
                    (
                        None if pert_policy is None
                        else pert_policy.get("action")
                    )
                    != (
                        None if baseline_policy is None
                        else baseline_policy.get("action")
                    )
                ),
                "authority_changed": int(
                    (
                        None if pert_policy is None
                        else pert_policy.get("authority")
                    )
                    != (
                        None if baseline_policy is None
                        else baseline_policy.get("authority")
                    )
                ),
            })

corrected_mask_df = pd.DataFrame(corrected_mask_rows)

corrected_mask_summary = (
    corrected_mask_df.groupby("field", as_index=False)
    .agg(
        n_interventions=("behavior_changed", "size"),
        behavior_change_rate=("behavior_changed", "mean"),
        action_change_rate=("action_changed", "mean"),
        authority_change_rate=("authority_changed", "mean"),
    )
)

corrected_mask_base = (
    corrected_mask_df.groupby(["base_id", "field"], as_index=False)
    .agg(
        any_behavior_change=("behavior_changed", "max"),
        mean_behavior_change=("behavior_changed", "mean"),
    )
)

corrected_mask_base_summary = (
    corrected_mask_base.groupby("field", as_index=False)
    .agg(
        bases=("base_id", "nunique"),
        fraction_bases_field_can_change_behavior=(
            "any_behavior_change", "mean"
        ),
        average_alternative_behavior_effect=(
            "mean_behavior_change", "mean"
        ),
    )
)

print("\n[1A] CORRECTED ROLE-ACTIVE BEHAVIORAL SENSITIVITY")
display(corrected_mask_summary.round(4))

print("\n[1B] CORRECTED BASE-LEVEL BEHAVIORAL RELEVANCE")
display(corrected_mask_base_summary.round(4))

corrected_mask_df.to_csv(
    FINAL_PATCH_DIR / "corrected_role_active_mask_interventions.csv",
    index=False,
)
corrected_mask_summary.to_csv(
    FINAL_PATCH_DIR / "corrected_role_active_mask_sensitivity.csv",
    index=False,
)
corrected_mask_base_summary.to_csv(
    FINAL_PATCH_DIR / "corrected_role_active_mask_base_necessity.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# 2. CANONICAL DIRECT Active-SCD -> FAILURE AUROC FOR INDEPENDENT EXECUTORS
# -----------------------------------------------------------------------------
# The manuscript claim is that Active-SCD itself predicts downstream failure.
# Therefore the canonical point estimate should be:
#     roc_auc_score(failure, active_scd)
# without fitting an auxiliary logistic regression.
# -----------------------------------------------------------------------------

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

candidate_frames = []

# Common names from previous cells / patches.
for name in [
    "independent_executor_predictions",
    "independent_executor_df",
    "ind_executor_df",
    "executor_drift_df",
    "independent_predictions",
]:
    obj = globals().get(name, None)
    if isinstance(obj, pd.DataFrame) and len(obj):
        candidate_frames.append((name, obj.copy()))

# Also search all globals for likely independent-executor dataframes.
for name, obj in list(globals().items()):
    if (
        isinstance(obj, pd.DataFrame)
        and len(obj)
        and name not in {x[0] for x in candidate_frames}
    ):
        cols = {str(c).lower() for c in obj.columns}
        if (
            any("executor" in c for c in cols)
            and any("failure" in c for c in cols)
            and any("scd" in c for c in cols)
        ):
            candidate_frames.append((name, obj.copy()))

direct_auc_rows = []

for source_name, df in candidate_frames:
    failure_col = find_col(
        df,
        ["failure", "failed", "is_failure", "executor_failure"]
    )
    active_col = find_col(
        df,
        [
            "prepolicy_active_scd",
            "active_scd",
            "Active-SCD",
            "active_scd_score",
        ]
    )
    executor_col = find_col(
        df,
        ["executor", "executor_name", "model", "reader"]
    )

    if failure_col is None or active_col is None:
        continue

    if executor_col is None:
        groups = [("unknown", df)]
    else:
        groups = list(df.groupby(executor_col))

    for executor_name, g in groups:
        y = pd.to_numeric(g[failure_col], errors="coerce")
        s = pd.to_numeric(g[active_col], errors="coerce")
        keep = y.notna() & s.notna()
        y = y[keep].astype(int)
        s = s[keep].astype(float)

        if len(y) == 0 or y.nunique() < 2:
            continue

        auc = float(roc_auc_score(y, s))
        direct_auc_rows.append({
            "source_dataframe": source_name,
            "executor": str(executor_name),
            "n": int(len(y)),
            "failures": int(y.sum()),
            "direct_active_scd_failure_auc": auc,
        })

direct_auc_df = pd.DataFrame(direct_auc_rows)

# Deduplicate by executor, preferring the largest matching dataframe.
if len(direct_auc_df):
    direct_auc_canonical = (
        direct_auc_df.sort_values(
            ["executor", "n"],
            ascending=[True, False]
        )
        .drop_duplicates("executor", keep="first")
        .reset_index(drop=True)
    )
else:
    direct_auc_canonical = pd.DataFrame(
        columns=[
            "source_dataframe",
            "executor",
            "n",
            "failures",
            "direct_active_scd_failure_auc",
        ]
    )

print("\n[2] CANONICAL DIRECT Active-SCD -> FAILURE AUROC")
display(direct_auc_canonical.round(4))

direct_auc_df.to_csv(
    FINAL_PATCH_DIR / "all_detected_independent_executor_auc_candidates.csv",
    index=False,
)
direct_auc_canonical.to_csv(
    FINAL_PATCH_DIR / "CANONICAL_INDEPENDENT_EXECUTOR_AUROC.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# 3. PATCH / REBUILD FINAL RESULTS REGISTRY
# -----------------------------------------------------------------------------

registry_path_candidates = [
    ROOT / "final_safety_closure" / "FINAL_RESULTS_REGISTRY.csv",
    ROOT / "FINAL_RESULTS_REGISTRY.csv",
]

existing_registry_path = next(
    (p for p in registry_path_candidates if p.exists()),
    None,
)

if existing_registry_path is not None:
    registry_df = pd.read_csv(existing_registry_path)
else:
    registry_df = pd.DataFrame(
        columns=[
            "key",
            "value",
            "claim_class",
            "analysis_status",
            "paper_location",
            "note",
        ]
    )

def upsert_registry(
    df,
    key,
    value,
    claim_class,
    analysis_status,
    paper_location,
    note,
):
    row = {
        "key": key,
        "value": float(value),
        "claim_class": claim_class,
        "analysis_status": analysis_status,
        "paper_location": paper_location,
        "note": note,
    }
    if "key" in df.columns and (df["key"] == key).any():
        for col, val in row.items():
            df.loc[df["key"] == key, col] = val
    else:
        df = pd.concat(
            [df, pd.DataFrame([row])],
            ignore_index=True,
        )
    return df

# Remove/replace earlier role-mask registry entries because they used full dict
# inequality rather than behavioral output.
if "key" in registry_df.columns:
    registry_df = registry_df[
        ~registry_df["key"].astype(str).str.startswith("role_mask_")
    ].copy()

for _, row in corrected_mask_base_summary.iterrows():
    registry_df = upsert_registry(
        registry_df,
        key=(
            "role_mask_"
            + str(row["field"])
            + "_behavioral_base_relevance"
        ),
        value=row[
            "fraction_bases_field_can_change_behavior"
        ],
        claim_class="SUPPORTING",
        analysis_status="POST-HOC MECHANISM DIAGNOSTIC",
        paper_location="Appendix / Limitations",
        note=(
            "Fraction of evaluated bases where changing this field "
            "can alter downstream (action, authority)."
        ),
    )

# Standardize independent-executor canonical AUROC values.
for _, row in direct_auc_canonical.iterrows():
    normalized_name = (
        str(row["executor"])
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    registry_df = upsert_registry(
        registry_df,
        key="active_scd_independent_executor_" + normalized_name,
        value=row["direct_active_scd_failure_auc"],
        claim_class="PRIMARY",
        analysis_status="POST-HOC ROBUSTNESS",
        paper_location="Main Table 2 / Appendix",
        note=(
            "Canonical point estimate = direct AUROC of Active-SCD "
            "score against executor failure; no auxiliary logistic "
            "regression fitted."
        ),
    )

patched_registry_csv = (
    FINAL_PATCH_DIR / "FINAL_RESULTS_REGISTRY_CORRECTED.csv"
)
patched_registry_json = (
    FINAL_PATCH_DIR / "FINAL_RESULTS_REGISTRY_CORRECTED.json"
)

registry_df.to_csv(
    patched_registry_csv,
    index=False,
)
patched_registry_json.write_text(
    registry_df.to_json(
        orient="records",
        indent=2,
    ),
    encoding="utf-8",
)

print("\n[3] CORRECTED PAPER-FACING REGISTRY")
display(registry_df)

# -----------------------------------------------------------------------------
# 4. FINAL PAPER GUIDANCE
# -----------------------------------------------------------------------------

paper_guidance = """
# Final correction guidance

## Role-active mask claim

Use the corrected behavioral comparison `(action, authority)`, not full policy
dictionary inequality.

Recommended interpretation:
- active semantic fields are **behaviorally heterogeneous**;
- authority and evidence can alter downstream behavior strongly;
- object changes should be described using the corrected action/authority effect,
  not a tautological 1.0 caused by the object being echoed in the policy output;
- role masks are consumer-dependent and must be revalidated when downstream
  consumption changes.

Do not claim that every active-field perturbation changes the final action.

## Independent executor AUROC

Use the canonical direct score:
`AUROC(failure, Active-SCD)`.

Do not use the auxiliary grouped/logistic-regression OOF point estimate as the
headline number unless the paper explicitly changes the metric definition.

The canonical CSV for the final manuscript is now:

`final_correctness_patch/FINAL_RESULTS_REGISTRY_CORRECTED.csv`

## Gate status

G0 remains FAIL.
G2 remains FAIL.
No prespecified gate is redefined by this patch.

## Experiment stop rule

After this correctness patch, stop adding synthetic experiments. Remaining
uncertainties are deployment/external-validity questions rather than missing
controls.
"""

(FINAL_PATCH_DIR / "FINAL_CORRECTION_GUIDANCE.md").write_text(
    paper_guidance.strip() + "\n",
    encoding="utf-8",
)

display(Markdown(paper_guidance))

print("\nSaved final correction artifacts:")
for p in sorted(FINAL_PATCH_DIR.iterdir()):
    print(" -", p.name)


# Final Reviewer-Stress Tests

These analyses are **post-hoc reviewer diagnostics** and do not redefine G0--G7.

They address five specific concerns without creating artificial new benchmark examples:

1. small-$n$ uncertainty and paired significance for C1/C2;
2. uncertainty of the opaque-ontology mechanism result;
3. architecture-faithful explicit-ID routing consistency (Table-12 ambiguity);
4. whether Active-SCD predicts failure beyond synthetic severity/version-family identity;
5. inference sensitivity to the dual-view raw/normalized mixture and component-complexity audit.

No competence threshold is relaxed and no synthetic 10× data expansion is performed.


In [ ]:

# =============================================================================
# FINAL REVIEWER-STRESS TESTS
# =============================================================================

from pathlib import Path
import copy, math, json
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from IPython.display import display, Markdown

REVIEWER_DIR = ROOT / "reviewer_stress_tests"
REVIEWER_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 104)
print("SCALE — FINAL REVIEWER-STRESS TESTS")
print("Output:", REVIEWER_DIR)
print("=" * 104)

_REQUIRED = [
    "rep_df", "ontology_df", "shift_test_base", "AGENTS",
    "realize_message", "prediction_for",
    "strict_hybrid_df", "drift_df", "naturalistic_df",
    "DUAL_VIEW_RAW_WEIGHT", "DUAL_VIEW_NORMALIZED_WEIGHT",
]
_missing = [x for x in _REQUIRED if x not in globals()]
assert not _missing, (
    "Run the full notebook first. Missing: " + ", ".join(_missing)
)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def wilson(successes, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = successes / n
    den = 1.0 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * math.sqrt(
        p*(1-p)/n + z*z/(4*n*n)
    ) / den
    return float(p), float(center-half), float(center+half)

def cluster_boot_diff(
    wide,
    a_col,
    b_col,
    group_col="base_id",
    n_boot=5000,
    seed=2026,
):
    d = wide.dropna(subset=[a_col, b_col, group_col]).copy()
    point = float((d[a_col] - d[b_col]).mean())
    groups = np.asarray(d[group_col].astype(str).unique())
    rng = np.random.default_rng(seed)
    by_group = {
        g: d[d[group_col].astype(str) == g]
        for g in groups
    }
    vals = []
    for _ in range(n_boot):
        sample = rng.choice(groups, size=len(groups), replace=True)
        b = pd.concat(
            [by_group[g] for g in sample],
            ignore_index=True,
        )
        vals.append(float((b[a_col] - b[b_col]).mean()))
    return (
        point,
        float(np.quantile(vals, 0.025)),
        float(np.quantile(vals, 0.975)),
    )

def exact_mcnemar(a, b):
    a = np.asarray(a, int)
    b = np.asarray(b, int)
    n10 = int(((a == 1) & (b == 0)).sum())
    n01 = int(((a == 0) & (b == 1)).sum())
    discordant = n10 + n01
    if discordant == 0:
        return n10, n01, 1.0
    p = float(
        binomtest(
            min(n10, n01),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )
    return n10, n01, p

def safe_auc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

# =============================================================================
# 1. SMALL-n PAIRED INFERENCE AUDIT
# =============================================================================
#
# This test deliberately does NOT invent more C1/C2 examples.
# It asks how much evidence the existing paired cases actually provide.
# Results should be used to calibrate the manuscript language:
# competitive / matches / modest numerical gain, not universal superiority.
# =============================================================================

scale_name = (
    "SCALE-Full"
    if "SCALE-Full" in set(rep_df["method"])
    else "SCALE"
)

smalln_rows = []

for split_name in ["c1_natural", "c2_schema"]:
    g = rep_df[
        rep_df["split"].eq(split_name)
        & rep_df["method"].isin(
            [scale_name, "CB+DualView", "CB-BGE"]
        )
    ].copy()

    if g.empty:
        continue

    key_cols = [
        c for c in [
            "base_id", "agent", "implementation", "variant"
        ]
        if c in g.columns
    ]

    wide = (
        g.pivot_table(
            index=key_cols,
            columns="method",
            values="active_correct",
            aggfunc="first",
        )
        .reset_index()
    )

    for baseline in ["CB+DualView", "CB-BGE"]:
        if scale_name not in wide.columns or baseline not in wide.columns:
            continue

        point, lo, hi = cluster_boot_diff(
            wide,
            scale_name,
            baseline,
            group_col="base_id",
            n_boot=5000,
            seed=SEED + len(smalln_rows),
        )

        n10, n01, p = exact_mcnemar(
            wide[scale_name],
            wide[baseline],
        )

        s_succ = int(wide[scale_name].sum())
        b_succ = int(wide[baseline].sum())
        n = len(wide)
        _, slo, shi = wilson(s_succ, n)
        _, blo, bhi = wilson(b_succ, n)

        smalln_rows.append({
            "split": split_name,
            "comparison": f"{scale_name} - {baseline}",
            "n_paired_records": n,
            "n_unique_bases": int(wide["base_id"].nunique()),
            "scale_accuracy": s_succ / n,
            "scale_wilson_low": slo,
            "scale_wilson_high": shi,
            "baseline_accuracy": b_succ / n,
            "baseline_wilson_low": blo,
            "baseline_wilson_high": bhi,
            "risk_difference": point,
            "cluster_boot_low": lo,
            "cluster_boot_high": hi,
            "scale_only_correct": n10,
            "baseline_only_correct": n01,
            "exact_mcnemar_p": p,
        })

smalln_df = pd.DataFrame(smalln_rows)

print("\n[1] SMALL-n PAIRED INFERENCE AUDIT")
display(smalln_df.round(4))

smalln_df.to_csv(
    REVIEWER_DIR / "small_n_paired_inference.csv",
    index=False,
)

# =============================================================================
# 2. OPAQUE ONTOLOGY UNCERTAINTY AUDIT
# =============================================================================
#
# The opaque-leaf result is a mechanism isolation, not a large-n benchmark.
# Quantify its uncertainty explicitly and compare SCALE-Full vs SCALE-NoOnt
# on the same bases.
# =============================================================================

opaque = ontology_df[
    ontology_df["tier"].astype(str).eq("opaque")
    & ontology_df["method"].isin(
        ["SCALE-Full", "SCALE-NoOnt", "CB+DualView", "Proto+Ancestor"]
    )
].copy()

onto_unc_rows = []
if len(opaque):
    for method, g in opaque.groupby("method"):
        k = int(g["canonical_parent_accuracy"].sum())
        n = len(g)
        p, lo, hi = wilson(k, n)
        onto_unc_rows.append({
            "method": method,
            "n": n,
            "correct": k,
            "accuracy": p,
            "wilson_low": lo,
            "wilson_high": hi,
        })

onto_unc_df = pd.DataFrame(onto_unc_rows)

onto_pair_df = pd.DataFrame()
if len(opaque):
    wide_o = (
        opaque.pivot_table(
            index="base_id",
            columns="method",
            values="canonical_parent_accuracy",
            aggfunc="first",
        )
        .reset_index()
    )
    if {"SCALE-Full", "SCALE-NoOnt"}.issubset(wide_o.columns):
        d, lo, hi = cluster_boot_diff(
            wide_o,
            "SCALE-Full",
            "SCALE-NoOnt",
            group_col="base_id",
            n_boot=5000,
            seed=SEED+17,
        )
        n10, n01, p = exact_mcnemar(
            wide_o["SCALE-Full"],
            wide_o["SCALE-NoOnt"],
        )
        onto_pair_df = pd.DataFrame([{
            "comparison": "SCALE-Full - SCALE-NoOnt",
            "n_bases": len(wide_o),
            "risk_difference": d,
            "cluster_boot_low": lo,
            "cluster_boot_high": hi,
            "scale_only_correct": n10,
            "noont_only_correct": n01,
            "exact_mcnemar_p": p,
        }])

print("\n[2A] OPAQUE ONTOLOGY UNCERTAINTY")
display(onto_unc_df.round(4))
print("\n[2B] OPAQUE ONTOLOGY PAIRED DIFFERENCE")
display(onto_pair_df.round(4))

onto_unc_df.to_csv(
    REVIEWER_DIR / "opaque_ontology_uncertainty.csv",
    index=False,
)
onto_pair_df.to_csv(
    REVIEWER_DIR / "opaque_ontology_paired_difference.csv",
    index=False,
)

# =============================================================================
# 3. ARCHITECTURE-CONSISTENCY / TABLE-12 RESOLUTION
# =============================================================================
#
# Reviewer concern:
# Method says recoverable explicit IDs go through deterministic f_id, whereas
# the old branch stress table reports low SCALE scores on punctuated/wrapped IDs.
#
# The definitive end-to-end architecture-faithful result must therefore be
# computed from strict_hybrid_df, i.e. AFTER the deterministic ID gate.
# =============================================================================

id_surfaces = ["exact_id", "punctuated_id", "wrapped_id"]
arch_id = strict_hybrid_df[
    strict_hybrid_df["kind"].eq("unseen")
    & strict_hybrid_df["surface"].isin(id_surfaces)
].copy()

arch_id_summary = (
    arch_id.groupby("surface", as_index=False)
    .agg(
        n=("correct", "size"),
        end_to_end_root_accuracy=("correct", "mean"),
        symbolic_id_route_rate=(
            "route",
            lambda s: float((s == "symbolic_id").mean())
        ),
    )
)

alias_arch = strict_hybrid_df[
    strict_hybrid_df["kind"].eq("unseen")
    & strict_hybrid_df["surface"].eq("semantic_alias")
].copy()

alias_arch_summary = pd.DataFrame([{
    "surface": "semantic_alias",
    "n": len(alias_arch),
    "end_to_end_root_accuracy": (
        float(alias_arch["correct"].mean())
        if len(alias_arch) else np.nan
    ),
    "symbolic_id_route_rate": (
        float((alias_arch["route"] == "symbolic_id").mean())
        if len(alias_arch) else np.nan
    ),
}])

print("\n[3A] ARCHITECTURE-FAITHFUL EXPLICIT-ID ROUTING")
display(arch_id_summary.round(4))
print("\n[3B] ARCHITECTURE-FAITHFUL SEMANTIC-ALIAS ROUTING")
display(alias_arch_summary.round(4))

# Integrity assertions: explicit recoverable IDs should never be evaluated as
# if they had bypassed the deterministic path.
for _, r in arch_id_summary.iterrows():
    assert abs(r["symbolic_id_route_rate"] - 1.0) < 1e-12, (
        f"{r['surface']} does not always use symbolic_id."
    )
    assert abs(r["end_to_end_root_accuracy"] - 1.0) < 1e-12, (
        f"{r['surface']} end-to-end root accuracy is not 1.0."
    )

arch_replacement = pd.concat(
    [arch_id_summary, alias_arch_summary],
    ignore_index=True,
)
arch_replacement["interpretation"] = np.where(
    arch_replacement["surface"].isin(id_surfaces),
    "final routed SCALE: deterministic ID path",
    "final routed SCALE: router + semantic decoder",
)

arch_replacement.to_csv(
    REVIEWER_DIR / "TABLE12_ARCHITECTURE_FAITHFUL_REPLACEMENT.csv",
    index=False,
)

# =============================================================================
# 4. DOES Active-SCD ADD INFORMATION BEYOND SEVERITY / VERSION FAMILY?
# =============================================================================
#
# This directly addresses the "constructed by design" concern.
#
# Controlled:
#   severity-only vs Active-SCD-only vs severity + Active-SCD.
#
# Naturalistic:
#   version-family-only vs Active-SCD-only vs family + Active-SCD.
#
# All estimates are grouped OOF by base_id.
# We are NOT using final policy outcomes as an input feature.
# =============================================================================

def grouped_oof_logistic_numeric(
    df,
    y_col,
    group_col,
    feature_cols,
    categorical_cols=None,
    n_splits=5,
):
    d = df.dropna(
        subset=[y_col, group_col] + list(feature_cols)
    ).copy()

    categorical_cols = categorical_cols or []
    groups = d[group_col].astype(str).to_numpy()
    y = d[y_col].astype(int).to_numpy()

    uniq_groups = np.unique(groups)
    splits = min(n_splits, len(uniq_groups))
    if splits < 2 or len(np.unique(y)) < 2:
        return None

    X = d[list(feature_cols)].copy()

    transformers = []
    num_cols = [
        c for c in feature_cols
        if c not in categorical_cols
    ]
    if num_cols:
        transformers.append(
            ("num", StandardScaler(), num_cols)
        )
    if categorical_cols:
        transformers.append(
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_cols,
            )
        )

    prep = ColumnTransformer(transformers)
    pipe = Pipeline([
        ("prep", prep),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])

    oof = np.full(len(d), np.nan)
    gkf = GroupKFold(n_splits=splits)

    for tr, te in gkf.split(X, y, groups):
        if len(np.unique(y[tr])) < 2:
            continue
        pipe.fit(X.iloc[tr], y[tr])
        oof[te] = pipe.predict_proba(X.iloc[te])[:, 1]

    keep = np.isfinite(oof)
    if keep.sum() == 0 or len(np.unique(y[keep])) < 2:
        return None

    yy = y[keep]
    pp = np.clip(oof[keep], 1e-6, 1-1e-6)

    return {
        "n": int(keep.sum()),
        "groups": int(len(np.unique(groups[keep]))),
        "auc": float(roc_auc_score(yy, pp)),
        "log_loss": float(log_loss(yy, pp)),
        "brier": float(brier_score_loss(yy, pp)),
    }

incremental_rows = []

# Controlled drift columns.
severity_col = next(
    (c for c in ["severity", "drift_severity"] if c in drift_df.columns),
    None
)
failure_col = next(
    (c for c in ["failure", "is_failure"] if c in drift_df.columns),
    None
)
active_col = next(
    (
        c for c in [
            "prepolicy_active_scd",
            "active_scd",
            "Active-SCD",
        ]
        if c in drift_df.columns
    ),
    None
)

if severity_col and failure_col and active_col and "base_id" in drift_df.columns:
    specs = [
        ("controlled_severity_only", [severity_col], []),
        ("controlled_active_scd_only", [active_col], []),
        (
            "controlled_severity_plus_active_scd",
            [severity_col, active_col],
            [],
        ),
    ]
    for label, feats, cats in specs:
        res = grouped_oof_logistic_numeric(
            drift_df,
            failure_col,
            "base_id",
            feats,
            cats,
        )
        if res:
            incremental_rows.append({
                "setting": label,
                **res,
            })

# Naturalistic version changes.
nat_failure_col = next(
    (
        c for c in ["scale_failure", "failure"]
        if c in naturalistic_df.columns
    ),
    None
)
nat_active_col = next(
    (
        c for c in [
            "prepolicy_active_scd",
            "active_scd",
        ]
        if c in naturalistic_df.columns
    ),
    None
)

if (
    nat_failure_col
    and nat_active_col
    and "family" in naturalistic_df.columns
    and "base_id" in naturalistic_df.columns
):
    specs = [
        (
            "naturalistic_family_only",
            ["family"],
            ["family"],
        ),
        (
            "naturalistic_active_scd_only",
            [nat_active_col],
            [],
        ),
        (
            "naturalistic_family_plus_active_scd",
            ["family", nat_active_col],
            ["family"],
        ),
    ]
    for label, feats, cats in specs:
        res = grouped_oof_logistic_numeric(
            naturalistic_df,
            nat_failure_col,
            "base_id",
            feats,
            cats,
        )
        if res:
            incremental_rows.append({
                "setting": label,
                **res,
            })

incremental_df = pd.DataFrame(incremental_rows)

print("\n[4] INCREMENTAL INFORMATION BEYOND SEVERITY / VERSION FAMILY")
display(incremental_df.round(4))

incremental_df.to_csv(
    REVIEWER_DIR / "active_scd_incremental_information.csv",
    index=False,
)

# =============================================================================
# 5. DUAL-VIEW INFERENCE SENSITIVITY + COMPONENT COMPLEXITY AUDIT
# =============================================================================
#
# Part A: vary raw/normalized interpolation ONLY AT INFERENCE.
# This is explicitly not a retraining hyperparameter sweep.
# It tests whether the final trained representation collapses if the 0.55/0.45
# mixture is perturbed.
#
# Part B: summarize existing trained ablations to identify which components
# have empirical support and which should remain secondary.
# =============================================================================

_original_raw = float(DUAL_VIEW_RAW_WEIGHT)
_original_norm = float(DUAL_VIEW_NORMALIZED_WEIGHT)

dual_sweep_rows = []

try:
    for raw_w in [0.0, 0.25, 0.45, 0.55, 0.75, 1.0]:
        globals()["DUAL_VIEW_RAW_WEIGHT"] = float(raw_w)
        globals()["DUAL_VIEW_NORMALIZED_WEIGHT"] = float(1.0 - raw_w)

        for split_name, impl in [
            ("c1_natural", "C1"),
            ("c2_schema", "C2"),
        ]:
            for method in ["CB+DualView", scale_name]:
                vals = []
                for base in shift_test_base:
                    for agent in AGENTS:
                        msg = realize_message(
                            base, agent, impl, 0
                        )
                        gold = hop_contract(base, agent)
                        pred, _, _ = prediction_for(
                            method,
                            msg,
                            agent,
                            BASE_GRAPH,
                            repair=True,
                        )
                        vals.append(
                            active_correct(gold, pred, agent)
                        )

                dual_sweep_rows.append({
                    "raw_weight": raw_w,
                    "normalized_weight": 1.0 - raw_w,
                    "split": split_name,
                    "method": method,
                    "active_accuracy": float(np.mean(vals)),
                    "n": len(vals),
                    "note": "inference-only sensitivity; model not retrained",
                })
finally:
    globals()["DUAL_VIEW_RAW_WEIGHT"] = _original_raw
    globals()["DUAL_VIEW_NORMALIZED_WEIGHT"] = _original_norm

dual_sweep_df = pd.DataFrame(dual_sweep_rows)

print("\n[5A] DUAL-VIEW MIXTURE INFERENCE SENSITIVITY")
display(dual_sweep_df.round(4))

dual_sweep_df.to_csv(
    REVIEWER_DIR / "dual_view_inference_sensitivity.csv",
    index=False,
)

# Existing trained component ablations.
component_methods = [
    m for m in [
        scale_name,
        "SCALE-NoInv",
        "SCALE-NoOnt",
        "SCALE-NoCal",
        "SCALE+Logic",
        "SCALE-Raw",
    ]
    if m in set(rep_df["method"])
]

component_df = (
    rep_df[
        rep_df["method"].isin(component_methods)
        & rep_df["split"].isin(
            ["known_ab", "c1_natural", "c2_schema"]
        )
    ]
    .groupby(["method", "split"], as_index=False)
    .agg(
        active_accuracy=("active_correct", "mean"),
        exact_contract=("exact_contract", "mean"),
        n=("active_correct", "size"),
    )
)

# Add calibration if available.
if "calibration_df" in globals() and len(calibration_df):
    cal = calibration_df[
        calibration_df["method"].isin(component_methods)
    ][["method", "split", "active_ece"]].copy()
    component_df = component_df.merge(
        cal,
        on=["method", "split"],
        how="left",
    )

print("\n[5B] TRAINED COMPONENT / COMPLEXITY AUDIT")
display(component_df.round(4))

component_df.to_csv(
    REVIEWER_DIR / "trained_component_complexity_audit.csv",
    index=False,
)

# =============================================================================
# 6. AUTOMATIC INTERPRETATION / STOP RULE
# =============================================================================

notes = []

if len(smalln_df):
    dual_c1 = smalln_df[
        smalln_df["split"].eq("c1_natural")
        & smalln_df["comparison"].str.contains("CB\\+DualView", regex=True)
    ]
    if len(dual_c1):
        r = dual_c1.iloc[0]
        if r["cluster_boot_low"] <= 0 <= r["cluster_boot_high"]:
            notes.append(
                "C1 SCALE-vs-CB+DualView clustered CI crosses zero: "
                "do not claim statistically established superiority."
            )
        else:
            notes.append(
                "C1 SCALE-vs-CB+DualView clustered CI excludes zero; "
                "still report the small number of independent bases."
            )

if len(onto_pair_df):
    r = onto_pair_df.iloc[0]
    notes.append(
        "Opaque ontology difference is a paired mechanism result; report its "
        f"small-n uncertainty explicitly (n={int(r['n_bases'])})."
    )

if len(arch_id_summary):
    ok = (
        np.allclose(
            arch_id_summary["symbolic_id_route_rate"], 1.0
        )
        and np.allclose(
            arch_id_summary["end_to_end_root_accuracy"], 1.0
        )
    )
    if ok:
        notes.append(
            "Final routed architecture resolves explicit exact/punctuated/wrapped "
            "IDs deterministically at 1.0. The old low-ID SCALE stress-table "
            "values must be labeled branch-conditioned/pre-routing or replaced."
        )

if len(incremental_df):
    notes.append(
        "Use the severity/family incremental OOF table to judge whether "
        "Active-SCD adds predictive information beyond construction labels; "
        "do not hide a negative result."
    )

notes.append(
    "The dual-view sweep is inference-only. It can support robustness of the "
    "representation mixture but cannot substitute for a retraining sweep of "
    "training-loss coefficients such as 0.70 concept weight."
)

report = "# Reviewer-stress interpretation\n\n" + "\n".join(
    f"- {x}" for x in notes
)

(REVIEWER_DIR / "REVIEWER_STRESS_INTERPRETATION.md").write_text(
    report + "\n",
    encoding="utf-8",
)

display(Markdown(report))

print("\nSaved reviewer-stress artifacts:")
for p in sorted(REVIEWER_DIR.iterdir()):
    print(" -", p.name)

print("\nSTOP RULE:")
print(
    "Do not add further synthetic benchmark families after these diagnostics. "
    "The only qualitatively new evidence would be a genuinely larger external "
    "dataset, real longitudinal version history, or a clean-competent generative "
    "executor. Those are future-work-scale additions."
)


# SCALE — Comprehensive Measurement & Reviewer-Closure Suite

This section is a **broad measurement program**, not a new redefinition of G0--G7.

It measures the final paper along six axes:

1. **Closed-set semantic compatibility at a larger held-out base count**
   using the already-held-out drift bases rather than inventing pseudo-independent role records.
2. **Baseline fairness** through development-only raw/normalized mixture selection.
3. **Active-SCD specificity** through within-stratum tests, inactive/permuted semantic controls,
   AUROC/AUPRC, calibration, and conditional-information analysis.
4. **Ontology and routing validity** through paired hierarchical tests, architecture-faithful
   routing, structural permutation controls, and selective-risk curves.
5. **Reader/runtime/external/complexity audits**.
6. A final **claim scorecard** that separates strong, supporting, qualified, negative,
   and inconclusive evidence.

All newly added analyses remain post-hoc reviewer-hardening unless explicitly based on a frozen
external source. No competence gate is relaxed and no gate outcome is relabeled.


In [ ]:
# =============================================================================
# 0. PRE-FLIGHT, HELPERS, AND DATA-PROVENANCE AUDIT
# =============================================================================

from pathlib import Path
import copy, json, math, hashlib
import numpy as np
import pandas as pd

from scipy.stats import binomtest
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from IPython.display import display, Markdown

MEASURE_DIR = ROOT / "comprehensive_measurement"
MEASURE_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_SLOTS = [
    "intent", "object", "action", "role",
    "authority", "provenance", "state",
]
ORIGINAL_ACTIVE_SLOT = {
    "PLANNER": "object",
    "RETRIEVER": "state",
    "POLICY": "authority",
}

REQ = [
    "train_base", "dev_base", "shift_test_base", "drift_base",
    "AGENTS", "BASE_GRAPH", "rep_df", "ontology_df",
    "large_ontology_df", "realize_message", "hop_contract",
    "active_correct", "exact_contract", "ensemble_predict",
    "ontology_repair", "drift_df", "drift_messages",
    "prepolicy_active_scd", "full_scd",
    "DUAL_VIEW_RAW_WEIGHT", "DUAL_VIEW_NORMALIZED_WEIGHT",
]
missing = [x for x in REQ if x not in globals()]
assert not missing, "Run all previous cells first. Missing: " + ", ".join(missing)

print("=" * 108)
print("SCALE — COMPREHENSIVE MEASUREMENT & REVIEWER CLOSURE")
print("Output:", MEASURE_DIR)
print("=" * 108)

def _wilson(k, n, z=1.959963984540054):
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = k / n
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return float(p), float(center-half), float(center+half)

def _ece(conf, corr, bins=10):
    conf = np.asarray(conf, float)
    corr = np.asarray(corr, float)
    ok = np.isfinite(conf) & np.isfinite(corr)
    conf, corr = conf[ok], corr[ok]
    if len(conf) == 0:
        return np.nan
    edges = np.linspace(0, 1, bins+1)
    value = 0.0
    for i in range(bins):
        if i == bins - 1:
            m = (conf >= edges[i]) & (conf <= edges[i+1])
        else:
            m = (conf >= edges[i]) & (conf < edges[i+1])
        if m.any():
            value += (m.mean()) * abs(conf[m].mean() - corr[m].mean())
    return float(value)

def _exact_mcnemar(a, b):
    a = np.asarray(a, int)
    b = np.asarray(b, int)
    n10 = int(((a == 1) & (b == 0)).sum())
    n01 = int(((a == 0) & (b == 1)).sum())
    d = n10 + n01
    p = 1.0 if d == 0 else float(
        binomtest(min(n10, n01), n=d, p=0.5, alternative="two-sided").pvalue
    )
    return n10, n01, p

def _group_boot_diff(
    df, value_a, value_b, group_col="base_id",
    n_boot=4000, seed=2027
):
    d = df.dropna(subset=[group_col, value_a, value_b]).copy()
    # Cluster means ensure role/variant records are not treated as independent.
    g = (
        d.groupby(group_col, as_index=False)[[value_a, value_b]]
        .mean()
    )
    point = float((g[value_a] - g[value_b]).mean())
    ids = np.asarray(g[group_col].astype(str))
    vals = []
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        idx = rng.integers(0, len(g), len(g))
        x = g.iloc[idx]
        vals.append(float((x[value_a] - x[value_b]).mean()))
    return (
        point,
        float(np.quantile(vals, .025)),
        float(np.quantile(vals, .975)),
        int(len(g)),
    )

def _safe_auc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    ok = np.isfinite(s)
    y, s = y[ok], s[ok]
    if len(y) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

def _safe_auprc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    ok = np.isfinite(s)
    y, s = y[ok], s[ok]
    if len(y) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, s))

def _group_boot_metric(
    df, y_col, score_col, group_col="base_id",
    metric="auc", n_boot=2500, seed=2028
):
    d = df.dropna(subset=[y_col, score_col, group_col]).copy()
    fn = _safe_auc if metric == "auc" else _safe_auprc
    point = fn(d[y_col], d[score_col])
    groups = np.asarray(sorted(d[group_col].astype(str).unique()))
    if len(groups) < 2:
        return point, np.nan, np.nan
    by = {g: d[d[group_col].astype(str) == g] for g in groups}
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        sample = rng.choice(groups, size=len(groups), replace=True)
        b = pd.concat([by[g] for g in sample], ignore_index=True)
        v = fn(b[y_col], b[score_col])
        if np.isfinite(v):
            vals.append(v)
    if not vals:
        return point, np.nan, np.nan
    return point, float(np.quantile(vals, .025)), float(np.quantile(vals, .975))

def _ids(xs):
    return {str(x["base_id"]) for x in xs}

sets = {
    "train": _ids(train_base),
    "dev": _ids(dev_base),
    "shift_test": _ids(shift_test_base),
    "drift": _ids(drift_base),
}
if "core_test_base" in globals():
    sets["core_test"] = _ids(core_test_base)
if "ontology_test_base" in globals():
    sets["ontology_test"] = _ids(ontology_test_base)

prov_rows = []
names = list(sets)
for i, a in enumerate(names):
    for b in names[i+1:]:
        prov_rows.append({
            "set_a": a,
            "set_b": b,
            "n_a": len(sets[a]),
            "n_b": len(sets[b]),
            "overlap": len(sets[a] & sets[b]),
        })
provenance_audit = pd.DataFrame(prov_rows)
display(provenance_audit)
provenance_audit.to_csv(
    MEASURE_DIR / "data_split_provenance_audit.csv", index=False
)

assert len(sets["train"] & sets["drift"]) == 0, (
    "Expanded held-out measurement invalid: drift_base overlaps train_base."
)
assert len(sets["dev"] & sets["drift"]) == 0, (
    "Expanded held-out measurement invalid: drift_base overlaps dev_base."
)

print("Preflight PASS — drift_base is independent of train/dev by base_id.")


In [ ]:
# =============================================================================
# 1. EXPANDED HELD-OUT C1/C2 COMPATIBILITY MEASUREMENT
# =============================================================================
#
# Reviewer concern: the original C1/C2 study contains only 12 independent bases.
#
# This analysis uses the already-held-out 96 drift bases as a NEW evaluation
# pool for representation compatibility. It does NOT count roles or variants as
# independent samples. Three textual variants are evaluated, but all uncertainty
# is clustered by base_id.
# =============================================================================

EXPANDED_METHODS = [
    "CB-BGE",
    "CB+DualView",
    "Proto-CB",
    "SCALE-NoInv",
    "SCALE-NoCal",
    "SCALE-Full",
]
EXPANDED_VARIANTS = [0, 1, 2]

eval_items = []
for base in drift_base:
    for impl, split in [("C1", "c1_natural"), ("C2", "c2_schema")]:
        for variant in EXPANDED_VARIANTS:
            for agent in AGENTS:
                eval_items.append({
                    "base_id": base["base_id"],
                    "workflow": base["object"],
                    "split": split,
                    "impl": impl,
                    "variant": variant,
                    "agent": agent,
                    "message": realize_message(base, agent, impl, variant),
                    "gold": hop_contract(base, agent),
                })

expanded_rows = []

for method in EXPANDED_METHODS:
    messages = [x["message"] for x in eval_items]
    model_name = {
        "SCALE-Full": "SCALE",
    }.get(method, method)

    preds = ensemble_predict(model_name, messages, BASE_GRAPH)

    repair = method in {
        "SCALE-Full", "SCALE-NoInv", "SCALE-NoCal"
    }

    for item, pobj in zip(eval_items, preds):
        pred = pobj["contract"]
        if repair:
            pred = ontology_repair(pred)

        expanded_rows.append({
            "base_id": item["base_id"],
            "workflow": item["workflow"],
            "split": item["split"],
            "variant": item["variant"],
            "agent": item["agent"],
            "method": method,
            "active_correct": active_correct(
                item["gold"], pred, item["agent"]
            ),
            "exact_contract": exact_contract(item["gold"], pred),
            "confidence": float(pobj["confidence"]),
        })

expanded_df = pd.DataFrame(expanded_rows)

expanded_summary = (
    expanded_df.groupby(["method", "split"], as_index=False)
    .agg(
        n_records=("active_correct", "size"),
        n_bases=("base_id", "nunique"),
        active_accuracy=("active_correct", "mean"),
        exact_contract_accuracy=("exact_contract", "mean"),
        mean_confidence=("confidence", "mean"),
    )
)
expanded_summary["active_ece"] = np.nan
expanded_summary["active_brier"] = np.nan

for i, row in expanded_summary.iterrows():
    g = expanded_df[
        (expanded_df["method"] == row["method"])
        & (expanded_df["split"] == row["split"])
    ]
    expanded_summary.loc[i, "active_ece"] = _ece(
        g["confidence"], g["active_correct"]
    )
    expanded_summary.loc[i, "active_brier"] = float(np.mean(
        (g["confidence"].to_numpy(float)
         - g["active_correct"].to_numpy(float))**2
    ))

print("\n[1A] EXPANDED HELD-OUT COMPATIBILITY")
display(expanded_summary.round(4))

expanded_by_role = (
    expanded_df.groupby(["method", "split", "agent"], as_index=False)
    .agg(
        n_bases=("base_id", "nunique"),
        active_accuracy=("active_correct", "mean"),
        exact_contract_accuracy=("exact_contract", "mean"),
    )
)
expanded_by_workflow = (
    expanded_df.groupby(["method", "split", "workflow"], as_index=False)
    .agg(
        n_bases=("base_id", "nunique"),
        active_accuracy=("active_correct", "mean"),
        exact_contract_accuracy=("exact_contract", "mean"),
    )
)

print("\n[1B] PER-ROLE")
display(expanded_by_role.round(4))
print("\n[1C] PER-WORKFLOW")
display(expanded_by_workflow.round(4))

# Paired base-clustered differences.
pair_rows = []
for split in ["c1_natural", "c2_schema"]:
    sub = expanded_df[expanded_df["split"] == split]
    base_method = (
        sub.groupby(["base_id", "method"], as_index=False)
        ["active_correct"].mean()
        .pivot(index="base_id", columns="method", values="active_correct")
        .reset_index()
    )

    for rival in ["CB+DualView", "CB-BGE", "Proto-CB"]:
        if {"SCALE-Full", rival}.issubset(base_method.columns):
            point, lo, hi, nbase = _group_boot_diff(
                base_method,
                "SCALE-Full",
                rival,
                group_col="base_id",
                n_boot=5000,
                seed=2029,
            )

            # Base-level strict success: all roles and all variants correct.
            strict = (
                sub[sub["method"].isin(["SCALE-Full", rival])]
                .groupby(["base_id", "method"], as_index=False)
                ["active_correct"].min()
                .pivot(index="base_id", columns="method", values="active_correct")
                .dropna()
            )
            n10, n01, p = _exact_mcnemar(
                strict["SCALE-Full"], strict[rival]
            )

            pair_rows.append({
                "split": split,
                "comparison": f"SCALE-Full - {rival}",
                "n_independent_bases": nbase,
                "mean_record_accuracy_difference": point,
                "cluster_boot_low": lo,
                "cluster_boot_high": hi,
                "scale_only_strict_base_success": n10,
                "rival_only_strict_base_success": n01,
                "exact_mcnemar_p_on_strict_base_success": p,
            })

expanded_pairwise = pd.DataFrame(pair_rows)
print("\n[1D] BASE-CLUSTERED PAIRED INFERENCE")
display(expanded_pairwise.round(4))

expanded_df.to_csv(MEASURE_DIR / "expanded_heldout_c1c2_records.csv", index=False)
expanded_summary.to_csv(MEASURE_DIR / "expanded_heldout_c1c2_summary.csv", index=False)
expanded_by_role.to_csv(MEASURE_DIR / "expanded_heldout_by_role.csv", index=False)
expanded_by_workflow.to_csv(MEASURE_DIR / "expanded_heldout_by_workflow.csv", index=False)
expanded_pairwise.to_csv(MEASURE_DIR / "expanded_heldout_pairwise.csv", index=False)


In [ ]:
# =============================================================================
# 2. DEVELOPMENT-ONLY DUAL-VIEW FAIRNESS AUDIT
# =============================================================================
#
# The earlier test-set sensitivity showed that CB+DualView can change materially
# with the raw/normalized interpolation. We therefore choose the inference
# interpolation ONLY on dev_base, freeze it, and then evaluate shift_test_base
# and the expanded drift-base set.
#
# IMPORTANT: these heads were trained under the original representation. This
# is an inference-selection fairness diagnostic, not a retraining sweep.
# =============================================================================

ALPHA_GRID = [0.00, 0.25, 0.45, 0.55, 0.75, 1.00]
FAIR_METHODS = ["CB+DualView", "SCALE-Full"]

orig_raw = float(DUAL_VIEW_RAW_WEIGHT)
orig_norm = float(DUAL_VIEW_NORMALIZED_WEIGHT)

def eval_weight(method, bases, alpha, variants=(0,), label="dev"):
    globals()["DUAL_VIEW_RAW_WEIGHT"] = float(alpha)
    globals()["DUAL_VIEW_NORMALIZED_WEIGHT"] = float(1.0-alpha)

    rows = []
    for base in bases:
        for impl, split in [("C1", "c1_natural"), ("C2", "c2_schema")]:
            for variant in variants:
                for agent in AGENTS:
                    msg = realize_message(base, agent, impl, variant)
                    gold = hop_contract(base, agent)
                    pred, conf, _ = prediction_for(
                        method, msg, agent, BASE_GRAPH, repair=True
                    )
                    rows.append({
                        "dataset": label,
                        "base_id": base["base_id"],
                        "split": split,
                        "agent": agent,
                        "variant": variant,
                        "method": method,
                        "alpha_raw": alpha,
                        "active_correct": active_correct(gold, pred, agent),
                        "exact_contract": exact_contract(gold, pred),
                        "confidence": conf,
                    })
    return pd.DataFrame(rows)

dev_alpha_frames = []
try:
    for method in FAIR_METHODS:
        for alpha in ALPHA_GRID:
            dev_alpha_frames.append(
                eval_weight(
                    method, dev_base, alpha,
                    variants=(0,), label="dev"
                )
            )
finally:
    globals()["DUAL_VIEW_RAW_WEIGHT"] = orig_raw
    globals()["DUAL_VIEW_NORMALIZED_WEIGHT"] = orig_norm

dev_alpha_df = pd.concat(dev_alpha_frames, ignore_index=True)

selection_rows = []
for method in FAIR_METHODS:
    for alpha in ALPHA_GRID:
        g = dev_alpha_df[
            (dev_alpha_df["method"] == method)
            & (dev_alpha_df["alpha_raw"] == alpha)
        ]
        selection_rows.append({
            "method": method,
            "alpha_raw": alpha,
            "alpha_norm": 1-alpha,
            "dev_active_accuracy": float(g["active_correct"].mean()),
            "dev_exact_accuracy": float(g["exact_contract"].mean()),
            "dev_ece": _ece(g["confidence"], g["active_correct"]),
            "distance_from_balanced": abs(alpha - 0.5),
        })

selection_df = pd.DataFrame(selection_rows)

selected = {}
for method in FAIR_METHODS:
    g = selection_df[selection_df["method"] == method].copy()
    g = g.sort_values(
        ["dev_active_accuracy", "dev_ece", "distance_from_balanced"],
        ascending=[False, True, True]
    )
    selected[method] = float(g.iloc[0]["alpha_raw"])

print("\n[2A] DEVELOPMENT-ONLY ALPHA SELECTION")
display(selection_df.round(4))
print("Selected alpha_raw:", selected)

frozen_eval_frames = []
try:
    for method in FAIR_METHODS:
        a = selected[method]
        frozen_eval_frames.append(
            eval_weight(
                method, shift_test_base, a,
                variants=(0,), label="original_shift_test"
            )
        )
        frozen_eval_frames.append(
            eval_weight(
                method, drift_base, a,
                variants=(0,1,2), label="expanded_heldout"
            )
        )
finally:
    globals()["DUAL_VIEW_RAW_WEIGHT"] = orig_raw
    globals()["DUAL_VIEW_NORMALIZED_WEIGHT"] = orig_norm

frozen_alpha_df = pd.concat(frozen_eval_frames, ignore_index=True)
frozen_alpha_summary = (
    frozen_alpha_df.groupby(
        ["dataset", "method", "split", "alpha_raw"], as_index=False
    )
    .agg(
        n_bases=("base_id", "nunique"),
        active_accuracy=("active_correct", "mean"),
        exact_contract_accuracy=("exact_contract", "mean"),
    )
)

print("\n[2B] FROZEN DEV-SELECTED ALPHA TEST RESULTS")
display(frozen_alpha_summary.round(4))

selection_df.to_csv(MEASURE_DIR / "dev_only_alpha_selection.csv", index=False)
frozen_alpha_df.to_csv(MEASURE_DIR / "dev_selected_alpha_records.csv", index=False)
frozen_alpha_summary.to_csv(MEASURE_DIR / "dev_selected_alpha_summary.csv", index=False)


In [ ]:
# =============================================================================
# 3. ACTIVE-SCD COMPREHENSIVE SPECIFICITY / PREDICTIVE-VALUE BATTERY
# =============================================================================
#
# New controls:
#   A. true role-active SCD
#   B. cyclically permuted active-role mapping
#   C. inactive-only semantic drift
#   D. full-contract SCD
#   E. negative confidence
#
# This asks whether the role-active semantic selection itself matters.
# =============================================================================

def _slot_semantic_distance(slot, a, b):
    if slot == "authority":
        return authority_distance(a, b)
    if slot == "state":
        return state_distance(a, b)
    return 0.0 if a == b else 1.0

def _mapped_scd(reference, current, mapping):
    posterior = []
    semantic = []
    for agent, slot in mapping.items():
        posterior.append(jsd(
            reference[agent]["probs"][slot],
            current[agent]["probs"][slot],
        ))
        semantic.append(_slot_semantic_distance(
            slot,
            reference[agent]["contract"].get(slot),
            current[agent]["contract"].get(slot),
        ))
    return .65*float(np.mean(posterior)) + .35*float(np.mean(semantic))

def _inactive_scd(reference, current):
    vals_js = []
    vals_sem = []
    for agent in AGENTS:
        active = ORIGINAL_ACTIVE_SLOT[agent]
        for slot in ORIGINAL_SLOTS:
            if slot == active:
                continue
            vals_js.append(jsd(
                reference[agent]["probs"][slot],
                current[agent]["probs"][slot],
            ))
            vals_sem.append(_slot_semantic_distance(
                slot,
                reference[agent]["contract"].get(slot),
                current[agent]["contract"].get(slot),
            ))
    return .65*float(np.mean(vals_js)) + .35*float(np.mean(vals_sem))

TRUE_MAP = {
    "PLANNER": "object",
    "RETRIEVER": "state",
    "POLICY": "authority",
}
PERMUTED_MAP = {
    "PLANNER": "state",
    "RETRIEVER": "authority",
    "POLICY": "object",
}

existing_outcomes = drift_df[
    ["base_id", "severity", "operators", "failure", "harmful_drift", "confidence"]
].copy()

score_rows = []
base_lookup = {b["base_id"]: b for b in drift_base}

for base in drift_base:
    reference = {
        agent: ensemble_predict(
            "SCALE",
            [realize_message(base, agent, "A", 0)],
            BASE_GRAPH,
        )[0]
        for agent in AGENTS
    }

    for severity in [0,1,2,3]:
        msgs, _ops = drift_messages(base, severity)
        current = {
            agent: ensemble_predict(
                "SCALE", [msgs[agent]], BASE_GRAPH
            )[0]
            for agent in AGENTS
        }
        repaired = {
            agent: {
                **current[agent],
                "contract": ontology_repair(current[agent]["contract"]),
            }
            for agent in AGENTS
        }

        score_rows.append({
            "base_id": base["base_id"],
            "severity": severity,
            "active_scd_recomputed": _mapped_scd(
                reference, repaired, TRUE_MAP
            ),
            "permuted_active_scd": _mapped_scd(
                reference, repaired, PERMUTED_MAP
            ),
            "inactive_only_scd": _inactive_scd(
                reference, repaired
            ),
            "full_scd_recomputed": full_scd(reference, repaired),
        })

scd_controls = pd.DataFrame(score_rows).merge(
    existing_outcomes,
    on=["base_id", "severity"],
    how="left",
)

scd_controls["neg_confidence"] = 1.0 - scd_controls["confidence"].astype(float)

metric_cols = [
    "active_scd_recomputed",
    "permuted_active_scd",
    "inactive_only_scd",
    "full_scd_recomputed",
    "neg_confidence",
]

pred_rows = []
for target in ["failure", "harmful_drift"]:
    for metric in metric_cols:
        auc, alo, ahi = _group_boot_metric(
            scd_controls, target, metric, metric="auc", n_boot=2500
        )
        ap, plo, phi = _group_boot_metric(
            scd_controls, target, metric, metric="ap", n_boot=2500
        )
        pred_rows.append({
            "target": target,
            "metric": metric,
            "auroc": auc,
            "auroc_ci_low": alo,
            "auroc_ci_high": ahi,
            "auprc": ap,
            "auprc_ci_low": plo,
            "auprc_ci_high": phi,
        })

scd_prediction_table = pd.DataFrame(pred_rows)

print("\n[3A] ACTIVE-SCD VS NEGATIVE CONTROLS")
display(scd_prediction_table.round(4))

# Within-severity and within-family discrimination.
within_rows = []
for sev, g in scd_controls.groupby("severity"):
    if g["failure"].nunique() >= 2:
        within_rows.append({
            "setting": "controlled",
            "stratum": f"severity_{sev}",
            "n": len(g),
            "failures": int(g["failure"].sum()),
            "active_scd_failure_auc": _safe_auc(
                g["failure"], g["active_scd_recomputed"]
            ),
            "permuted_failure_auc": _safe_auc(
                g["failure"], g["permuted_active_scd"]
            ),
            "inactive_failure_auc": _safe_auc(
                g["failure"], g["inactive_only_scd"]
            ),
        })

if "naturalistic_df" in globals() and len(naturalistic_df):
    nat_fail_col = (
        "scale_failure" if "scale_failure" in naturalistic_df.columns
        else "failure"
    )
    nat_scd_col = (
        "prepolicy_active_scd"
        if "prepolicy_active_scd" in naturalistic_df.columns
        else "active_scd"
    )
    if {"family", nat_fail_col, nat_scd_col}.issubset(naturalistic_df.columns):
        for fam, g in naturalistic_df.groupby("family"):
            if g[nat_fail_col].nunique() >= 2:
                within_rows.append({
                    "setting": "naturalistic",
                    "stratum": str(fam),
                    "n": len(g),
                    "failures": int(g[nat_fail_col].sum()),
                    "active_scd_failure_auc": _safe_auc(
                        g[nat_fail_col], g[nat_scd_col]
                    ),
                    "permuted_failure_auc": np.nan,
                    "inactive_failure_auc": np.nan,
                })

within_strata_df = pd.DataFrame(within_rows)
print("\n[3B] WITHIN-SEVERITY / WITHIN-FAMILY FAILURE DISCRIMINATION")
display(within_strata_df.round(4))

# Failure risk by Active-SCD quintile.
scd_controls["active_scd_quintile"] = pd.qcut(
    scd_controls["active_scd_recomputed"],
    5,
    labels=False,
    duplicates="drop",
)
scd_risk_gradient = (
    scd_controls.groupby("active_scd_quintile", as_index=False)
    .agg(
        n=("failure", "size"),
        failure_rate=("failure", "mean"),
        harmful_rate=("harmful_drift", "mean"),
        mean_active_scd=("active_scd_recomputed", "mean"),
    )
)

print("\n[3C] FAILURE-RISK GRADIENT")
display(scd_risk_gradient.round(4))

# Conditional OOF comparison: severity only vs SCD only vs severity + SCD.
def _grouped_oof(df, y_col, num_cols, cat_cols=None, group_col="base_id"):
    cat_cols = cat_cols or []
    d = df.dropna(subset=[y_col, group_col] + num_cols + cat_cols).copy()
    y = d[y_col].astype(int).to_numpy()
    groups = d[group_col].astype(str).to_numpy()
    if len(np.unique(y)) < 2 or len(np.unique(groups)) < 3:
        return None

    features = num_cols + cat_cols
    X = d[features].copy()
    trans = []
    if num_cols:
        trans.append(("num", StandardScaler(), num_cols))
    if cat_cols:
        trans.append((
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_cols,
        ))
    pipe = Pipeline([
        ("prep", ColumnTransformer(trans)),
        ("clf", LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=SEED
        )),
    ])
    k = min(5, len(np.unique(groups)))
    pred = np.full(len(d), np.nan)
    for tr, te in GroupKFold(k).split(X, y, groups):
        if len(np.unique(y[tr])) < 2:
            continue
        pipe.fit(X.iloc[tr], y[tr])
        pred[te] = pipe.predict_proba(X.iloc[te])[:,1]
    ok = np.isfinite(pred)
    if ok.sum() < 5 or len(np.unique(y[ok])) < 2:
        return None
    yy = y[ok]
    pp = np.clip(pred[ok], 1e-7, 1-1e-7)
    return {
        "n": int(ok.sum()),
        "groups": int(len(np.unique(groups[ok]))),
        "auroc": float(roc_auc_score(yy, pp)),
        "auprc": float(average_precision_score(yy, pp)),
        "log_loss": float(log_loss(yy, pp)),
        "brier": float(brier_score_loss(yy, pp)),
    }

conditional_rows = []
for label, nums in [
    ("severity_only", ["severity"]),
    ("active_scd_only", ["active_scd_recomputed"]),
    ("severity_plus_active_scd", ["severity", "active_scd_recomputed"]),
    ("severity_plus_permuted_scd", ["severity", "permuted_active_scd"]),
    ("severity_plus_inactive_scd", ["severity", "inactive_only_scd"]),
]:
    r = _grouped_oof(scd_controls, "failure", nums)
    if r:
        conditional_rows.append({"setting": label, **r})

conditional_df = pd.DataFrame(conditional_rows)
print("\n[3D] CONDITIONAL INFORMATION TEST")
display(conditional_df.round(4))

scd_controls.to_csv(MEASURE_DIR / "active_scd_control_records.csv", index=False)
scd_prediction_table.to_csv(MEASURE_DIR / "active_scd_predictive_battery.csv", index=False)
within_strata_df.to_csv(MEASURE_DIR / "active_scd_within_strata.csv", index=False)
scd_risk_gradient.to_csv(MEASURE_DIR / "active_scd_risk_gradient.csv", index=False)
conditional_df.to_csv(MEASURE_DIR / "active_scd_conditional_information.csv", index=False)


In [ ]:
# =============================================================================
# 4. ONTOLOGY, ROUTING, STRUCTURAL NEGATIVE CONTROLS, AND SELECTIVE RISK
# =============================================================================

# ---- 4A. Small opaque ontology paired result.
opaque = ontology_df[ontology_df["tier"].astype(str) == "opaque"].copy()

opaque_summary_rows = []
for method in ["CB+DualView", "Proto+Ancestor", "SCALE-NoOnt", "SCALE-Full"]:
    g = opaque[opaque["method"] == method]
    if len(g):
        k = int(g["canonical_parent_accuracy"].sum())
        p, lo, hi = _wilson(k, len(g))
        opaque_summary_rows.append({
            "method": method,
            "n": len(g),
            "accuracy": p,
            "wilson_low": lo,
            "wilson_high": hi,
        })
opaque_summary = pd.DataFrame(opaque_summary_rows)

opaque_pair = pd.DataFrame()
w = (
    opaque[opaque["method"].isin(["SCALE-Full", "SCALE-NoOnt"])]
    .pivot_table(
        index="base_id",
        columns="method",
        values="canonical_parent_accuracy",
        aggfunc="first",
    )
    .dropna()
)
if len(w):
    n10, n01, p = _exact_mcnemar(w["SCALE-Full"], w["SCALE-NoOnt"])
    opaque_pair = pd.DataFrame([{
        "n_bases": len(w),
        "scale_accuracy": float(w["SCALE-Full"].mean()),
        "noont_accuracy": float(w["SCALE-NoOnt"].mean()),
        "scale_only_correct": n10,
        "noont_only_correct": n01,
        "exact_mcnemar_p": p,
    }])

print("\n[4A] OPAQUE ONTOLOGY")
display(opaque_summary.round(4))
display(opaque_pair.round(4))

# ---- 4B. Large semantic-alias paired mechanism.
alias = large_ontology_df[
    (large_ontology_df["surface"] == "semantic_alias")
    & large_ontology_df["method"].isin(["SCALE-Full", "SCALE-NoOnt"])
].copy()

large_alias_pair = pd.DataFrame()
if len(alias):
    wa = (
        alias.pivot_table(
            index="gold_leaf",
            columns="method",
            values="canonical_root_accuracy",
            aggfunc="first",
        )
        .dropna()
    )
    if {"SCALE-Full", "SCALE-NoOnt"}.issubset(wa.columns):
        n10, n01, p = _exact_mcnemar(
            wa["SCALE-Full"], wa["SCALE-NoOnt"]
        )
        large_alias_pair = pd.DataFrame([{
            "n_unseen_nodes": len(wa),
            "scale_root_accuracy": float(wa["SCALE-Full"].mean()),
            "noont_root_accuracy": float(wa["SCALE-NoOnt"].mean()),
            "scale_only_correct": n10,
            "noont_only_correct": n01,
            "exact_mcnemar_p": p,
        }])

print("\n[4B] LARGE SEMANTIC-ALIAS PAIRED MECHANISM")
display(large_alias_pair.round(4))

# ---- 4C. Architecture-faithful routing provenance.
routing_tables = []

if "route_v2_df" in globals() and isinstance(route_v2_df, pd.DataFrame) and len(route_v2_df):
    rv = (
        route_v2_df.groupby(["kind", "surface", "route"], as_index=False)
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            mean_p_unseen=("p_unseen", "mean"),
        )
    )
    rv["implementation"] = "architecture_faithful_scale_known_branch"
    routing_tables.append(rv)

if "strict_hybrid_df" in globals() and isinstance(strict_hybrid_df, pd.DataFrame) and len(strict_hybrid_df):
    sh = (
        strict_hybrid_df.groupby(["kind", "surface", "route"], as_index=False)
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            mean_p_unseen=("p_unseen", "mean"),
        )
    )
    sh["implementation"] = "legacy_strict_cb_known_branch"
    routing_tables.append(sh)

routing_provenance_summary = (
    pd.concat(routing_tables, ignore_index=True)
    if routing_tables else pd.DataFrame()
)

print("\n[4C] ROUTING PROVENANCE — DO NOT MERGE DIFFERENT KNOWN BRANCHES")
display(routing_provenance_summary.round(4))

# Canonical explicit-ID assertions from architecture-faithful route.
if "route_v2_df" in globals() and len(route_v2_df):
    ids = route_v2_df[
        (route_v2_df["kind"] == "unseen")
        & route_v2_df["surface"].isin(
            ["exact_id", "punctuated_id", "wrapped_id"]
        )
    ]
    routed_id_summary = (
        ids.groupby("surface", as_index=False)
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            symbolic_id_rate=("route", lambda s: float((s=="symbolic_id").mean())),
        )
    )
else:
    routed_id_summary = pd.DataFrame()

print("\n[4D] FINAL EXPLICIT-ID PATH")
display(routed_id_summary.round(4))

# ---- 4E. Schema.org structural permutation negative control.
schema_perm_summary = pd.DataFrame()
if (
    "schema_external_df" in globals()
    and isinstance(schema_external_df, pd.DataFrame)
    and len(schema_external_df)
):
    sdf = schema_external_df.copy()
    observed_graph = float(sdf["ground_then_graph_correct"].mean())
    observed_direct = float(sdf["direct_root_correct"].mean())
    n10, n01, paired_p = _exact_mcnemar(
        sdf["ground_then_graph_correct"],
        sdf["direct_root_correct"],
    )

    leaf_to_root = (
        sdf[["class_label", "gold_root"]]
        .drop_duplicates()
        .set_index("class_label")["gold_root"]
        .to_dict()
    )
    labels = list(leaf_to_root.keys())
    roots = np.asarray([leaf_to_root[x] for x in labels], dtype=object)
    rng = np.random.default_rng(2030)
    null = []

    # Root-label permutation preserves the empirical root frequency but destroys
    # the semantic class-to-root structure.
    for _ in range(3000):
        perm = rng.permutation(roots)
        mp = dict(zip(labels, perm))
        pred = sdf["pred_leaf"].map(mp)
        null.append(float((pred == sdf["gold_root"]).mean()))

    null = np.asarray(null)
    perm_p = float((1 + np.sum(null >= observed_graph)) / (1 + len(null)))

    schema_perm_summary = pd.DataFrame([{
        "n": len(sdf),
        "observed_graph_root_accuracy": observed_graph,
        "direct_root_accuracy": observed_direct,
        "paired_graph_vs_direct_mcnemar_p": paired_p,
        "permuted_structure_mean_accuracy": float(null.mean()),
        "permuted_structure_95_low": float(np.quantile(null, .025)),
        "permuted_structure_95_high": float(np.quantile(null, .975)),
        "permutation_p_graph_ge_observed": perm_p,
    }])

print("\n[4E] SCHEMA.ORG STRUCTURAL NEGATIVE CONTROL")
display(schema_perm_summary.round(4))

# ---- 4F. Selective prediction / risk-coverage.
risk_coverage_rows = []
root_coverage = pd.DataFrame()

if (
    "safety_route_df" in globals()
    and isinstance(safety_route_df, pd.DataFrame)
    and len(safety_route_df)
):
    sr = safety_route_df.copy()
    sr["error"] = 1 - sr["default_correct"].astype(int)
    tau = float(strict_router_threshold)
    sr["risk_tau_closeness"] = -np.abs(sr["p_open"].astype(float) - tau)
    sr["risk_disagreement"] = sr["decoder_disagreement"].astype(float)

    # Normalize closeness for a combined diagnostic.
    x = sr["risk_tau_closeness"].to_numpy(float)
    xmin, xmax = np.min(x), np.max(x)
    closeness_norm = (
        (x - xmin)/(xmax-xmin)
        if xmax > xmin else np.zeros_like(x)
    )
    sr["risk_combined"] = (
        2.0*sr["risk_disagreement"].to_numpy(float) + closeness_norm
    )

    for signal in ["risk_tau_closeness", "risk_disagreement", "risk_combined"]:
        vals = []
        for coverage in np.linspace(.10, 1.00, 19):
            n_accept = max(1, int(round(coverage * len(sr))))
            # Lower risk is accepted first.
            idx = np.argsort(sr[signal].to_numpy(float))[:n_accept]
            err = float(sr.iloc[idx]["error"].mean())
            vals.append((n_accept/len(sr), err))
            risk_coverage_rows.append({
                "signal": signal,
                "coverage": n_accept/len(sr),
                "accepted_error_rate": err,
                "accepted_accuracy": 1-err,
                "n_accepted": n_accept,
            })

    if "root" in sr.columns:
        tmp = sr.copy()
        tmp["accept_disagreement_policy"] = 1 - tmp["decoder_disagreement"].astype(int)
        root_coverage = (
            tmp[tmp["kind"] == "unseen"]
            .groupby("root", as_index=False)
            .agg(
                n=("item_id", "size"),
                default_accuracy=("default_correct", "mean"),
                coverage=("accept_disagreement_policy", "mean"),
            )
        )

risk_coverage_df = pd.DataFrame(risk_coverage_rows)
print("\n[4F] SELECTIVE RISK-COVERAGE")
display(risk_coverage_df.round(4))
if len(root_coverage):
    print("\nPer-root disagreement coverage")
    display(root_coverage.round(4))

opaque_summary.to_csv(MEASURE_DIR / "opaque_ontology_summary.csv", index=False)
opaque_pair.to_csv(MEASURE_DIR / "opaque_ontology_paired.csv", index=False)
large_alias_pair.to_csv(MEASURE_DIR / "large_alias_paired.csv", index=False)
routing_provenance_summary.to_csv(MEASURE_DIR / "routing_provenance_summary.csv", index=False)
routed_id_summary.to_csv(MEASURE_DIR / "routed_explicit_id_summary.csv", index=False)
schema_perm_summary.to_csv(MEASURE_DIR / "schemaorg_structural_permutation.csv", index=False)
risk_coverage_df.to_csv(MEASURE_DIR / "routing_risk_coverage.csv", index=False)
root_coverage.to_csv(MEASURE_DIR / "routing_disagreement_per_root.csv", index=False)


In [ ]:
# =============================================================================
# 5. READER, CALIBRATION, RUNTIME, EXTERNAL TRANSFER, AND COMPLEXITY SCORECARD
# =============================================================================

# ---- 5A Reader robustness.
reader_measure = pd.DataFrame()
if "reader_df" in globals() and isinstance(reader_df, pd.DataFrame) and len(reader_df):
    cols = [
        c for c in [
            "reader_model", "split", "method",
            "parse_success", "semantic_label_validity",
            "semantic_success", "task_success",
            "input_tokens", "latency_s",
        ]
        if c in reader_df.columns
    ]
    if {"reader_model", "split", "method", "task_success"}.issubset(reader_df.columns):
        agg = {
            "task_success": "mean",
        }
        for c in [
            "parse_success", "semantic_label_validity",
            "semantic_success", "input_tokens", "latency_s"
        ]:
            if c in reader_df.columns:
                agg[c] = "mean"
        reader_measure = (
            reader_df.groupby(["reader_model", "split", "method"], as_index=False)
            .agg(agg)
        )

print("\n[5A] READER ROBUSTNESS")
display(reader_measure.round(4))

# ---- 5B Runtime enforcement.
runtime_measure = pd.DataFrame()
for name in [
    "constraint_stress_df",
    "constraint_df",
    "runtime_constraint_df",
    "runtime_df",
]:
    obj = globals().get(name, None)
    if isinstance(obj, pd.DataFrame) and len(obj):
        rows = []
        family_col = next(
            (c for c in ["family", "corruption", "violation_family", "type"] if c in obj.columns),
            None
        )
        groups = obj.groupby(family_col) if family_col else [("ALL", obj)]
        for fam, g in groups:
            row = {"source": name, "family": fam, "n": len(g)}
            for c in ["detected", "repaired", "blocked", "residual_violation"]:
                if c in g.columns:
                    row[c + "_rate"] = float(pd.to_numeric(g[c], errors="coerce").mean())
            rows.append(row)
        runtime_measure = pd.DataFrame(rows)
        break

print("\n[5B] RUNTIME ENFORCEMENT")
display(runtime_measure.round(4))

# ---- 5C Component complexity / ablations.
component_methods = [
    x for x in [
        "SCALE-Full", "SCALE-NoInv", "SCALE-NoOnt",
        "SCALE-NoCal", "SCALE+Logic", "SCALE-Raw"
    ]
    if x in set(rep_df["method"])
]

complexity_measure = (
    rep_df[
        rep_df["method"].isin(component_methods)
        & rep_df["split"].isin(["known_ab", "c1_natural", "c2_schema"])
    ]
    .groupby(["method", "split"], as_index=False)
    .agg(
        active_accuracy=("active_correct", "mean"),
        exact_contract_accuracy=("exact_contract", "mean"),
        active_ece=("active_ece", "mean") if "active_ece" in rep_df.columns else ("active_correct", "size"),
        n=("active_correct", "size"),
    )
)

print("\n[5C] COMPLEXITY / ABLATION AUDIT")
display(complexity_measure.round(4))

# ---- 5D CRM external boundary.
crm_measure = pd.DataFrame()
if "crm_sem_summary" in globals() and isinstance(crm_sem_summary, pd.DataFrame):
    crm_measure = crm_sem_summary.copy()

crm_retention_measure = pd.DataFrame()
if "crm_retention" in globals() and isinstance(crm_retention, pd.DataFrame):
    crm_retention_measure = crm_retention.copy()

print("\n[5D] CRM SEMANTIC TRANSFER")
display(crm_measure.round(4))
print("\nCRM RETENTION")
display(crm_retention_measure.round(4))

reader_measure.to_csv(MEASURE_DIR / "reader_measurement.csv", index=False)
runtime_measure.to_csv(MEASURE_DIR / "runtime_measurement.csv", index=False)
complexity_measure.to_csv(MEASURE_DIR / "complexity_measurement.csv", index=False)
crm_measure.to_csv(MEASURE_DIR / "crm_semantic_measurement.csv", index=False)
crm_retention_measure.to_csv(MEASURE_DIR / "crm_retention_measurement.csv", index=False)


In [ ]:
# =============================================================================
# 6. FINAL CLAIM SCORECARD + MANUSCRIPT-FACING SUMMARY
# =============================================================================

claim_rows = []

# Closed-set compatibility.
c1_pair = expanded_pairwise[
    expanded_pairwise["comparison"].eq("SCALE-Full - CB+DualView")
    & expanded_pairwise["split"].eq("c1_natural")
]
c2_pair = expanded_pairwise[
    expanded_pairwise["comparison"].eq("SCALE-Full - CB+DualView")
    & expanded_pairwise["split"].eq("c2_schema")
]

if len(c1_pair):
    r = c1_pair.iloc[0]
    c1_status = (
        "STRONG POSITIVE"
        if r["cluster_boot_low"] > 0
        else "COMPETITIVE / NO ESTABLISHED SUPERIORITY"
    )
    claim_rows.append({
        "claim": "Closed-set C1 semantic compatibility",
        "status": c1_status,
        "primary_evidence": (
            f"Expanded held-out n={int(r['n_independent_bases'])} bases; "
            f"paired diff={r['mean_record_accuracy_difference']:.4f}, "
            f"95% CI [{r['cluster_boot_low']:.4f},{r['cluster_boot_high']:.4f}]"
        ),
        "paper_action": "Use superiority language only if CI is strictly positive.",
    })

if len(c2_pair):
    r = c2_pair.iloc[0]
    c2_status = (
        "STRONG POSITIVE"
        if r["cluster_boot_low"] > 0
        else "COMPETITIVE / NO ESTABLISHED SUPERIORITY"
    )
    claim_rows.append({
        "claim": "Closed-set C2 semantic compatibility",
        "status": c2_status,
        "primary_evidence": (
            f"Expanded held-out n={int(r['n_independent_bases'])} bases; "
            f"paired diff={r['mean_record_accuracy_difference']:.4f}, "
            f"95% CI [{r['cluster_boot_low']:.4f},{r['cluster_boot_high']:.4f}]"
        ),
        "paper_action": "Frame as robustness/competitiveness unless CI excludes zero.",
    })

# Ontology mechanism.
if len(opaque_pair):
    r = opaque_pair.iloc[0]
    strong = (r["exact_mcnemar_p"] < .01 and r["scale_accuracy"] > r["noont_accuracy"])
    claim_rows.append({
        "claim": "Ontology structural canonicalization",
        "status": "STRONG MECHANISM EVIDENCE" if strong else "QUALIFIED",
        "primary_evidence": (
            f"Opaque paired n={int(r['n_bases'])}; "
            f"SCALE={r['scale_accuracy']:.3f}, NoOnt={r['noont_accuracy']:.3f}, "
            f"exact p={r['exact_mcnemar_p']:.4g}"
        ),
        "paper_action": "Keep as core contribution; report n and paired uncertainty.",
    })

if len(large_alias_pair):
    r = large_alias_pair.iloc[0]
    claim_rows.append({
        "claim": "Large-ontology semantic-alias structural recovery",
        "status": "SUPPORTING MECHANISM EVIDENCE",
        "primary_evidence": (
            f"n={int(r['n_unseen_nodes'])}; SCALE={r['scale_root_accuracy']:.3f}, "
            f"NoOnt={r['noont_root_accuracy']:.3f}, exact p={r['exact_mcnemar_p']:.4g}"
        ),
        "paper_action": "Use to support scale of ontology mechanism, not universal ontology superiority.",
    })

# Active-SCD.
a = scd_prediction_table[
    (scd_prediction_table["target"] == "failure")
    & (scd_prediction_table["metric"] == "active_scd_recomputed")
]
p = scd_prediction_table[
    (scd_prediction_table["target"] == "failure")
    & (scd_prediction_table["metric"] == "permuted_active_scd")
]
ina = scd_prediction_table[
    (scd_prediction_table["target"] == "failure")
    & (scd_prediction_table["metric"] == "inactive_only_scd")
]
if len(a):
    ar = a.iloc[0]
    pa = float(p.iloc[0]["auroc"]) if len(p) else np.nan
    ia = float(ina.iloc[0]["auroc"]) if len(ina) else np.nan
    claim_rows.append({
        "claim": "Decision-relevant Active-SCD failure observability",
        "status": "STRONG EMPIRICAL EVIDENCE",
        "primary_evidence": (
            f"AUROC={ar['auroc']:.3f} "
            f"[{ar['auroc_ci_low']:.3f},{ar['auroc_ci_high']:.3f}]; "
            f"permuted={pa:.3f}; inactive={ia:.3f}"
        ),
        "paper_action": "Make this the central empirical contribution.",
    })

# Routing.
router_status = "QUALIFIED"
if "router_root_generalization_df" in globals() and len(router_root_generalization_df):
    rr = router_root_generalization_df
    auc_col = next((c for c in ["auc", "auroc"] if c in rr.columns), None)
    if auc_col:
        worst = float(rr[auc_col].min())
        mean = float(rr[auc_col].mean())
        evidence = f"leave-root mean={mean:.3f}, worst={worst:.3f}"
    else:
        evidence = "root-generalization table available"
else:
    evidence = "root-generalization remains heterogeneous"
claim_rows.append({
    "claim": "Deployable open-set routing",
    "status": router_status,
    "primary_evidence": evidence,
    "paper_action": "Supporting engineering only; keep root-dependence limitation.",
})

# Reader gate.
claim_rows.append({
    "claim": "Arbitrary cross-reader robustness",
    "status": "NEGATIVE / G0 FAIL",
    "primary_evidence": "Typed/native path mitigates consumer failure; arbitrary Ministral rereading remains unsupported.",
    "paper_action": "Do not relabel G0; typed adapter is the intended interface.",
})

# Alignment.
claim_rows.append({
    "claim": "General label-efficiency from directed alignment",
    "status": "NEGATIVE / G2 FAIL",
    "primary_evidence": "Effects remain regime-dependent and sign-changing.",
    "paper_action": "Keep alignment outside the contribution list.",
})

# External ontology.
if len(schema_perm_summary):
    r = schema_perm_summary.iloc[0]
    claim_rows.append({
        "claim": "Human-authored ontology structural transfer",
        "status": "SUPPORTING EXTERNAL MECHANISM",
        "primary_evidence": (
            f"graph={r['observed_graph_root_accuracy']:.3f}, "
            f"direct={r['direct_root_accuracy']:.3f}, "
            f"permuted mean={r['permuted_structure_mean_accuracy']:.3f}, "
            f"perm p={r['permutation_p_graph_ge_observed']:.4g}"
        ),
        "paper_action": "Call this structural external evidence, not full external SCALE superiority.",
    })

claim_scorecard = pd.DataFrame(claim_rows)
print("\n[6] FINAL CLAIM SCORECARD")
display(claim_scorecard)

claim_scorecard.to_csv(
    MEASURE_DIR / "FINAL_COMPREHENSIVE_CLAIM_SCORECARD.csv", index=False
)

# Compose a concise machine-generated report.
def _fmt_table(df, cols=None, n=20):
    if df is None or len(df) == 0:
        return "_No data._"
    x = df.copy()
    if cols:
        x = x[[c for c in cols if c in x.columns]]
    return x.head(n).to_markdown(index=False)

report = f"""
# SCALE — Comprehensive Measurement Report

## Scientific status

This measurement suite treats independent **base/task** units as the statistical
unit and explicitly separates post-hoc reviewer diagnostics from the original
G0--G7 gates.

## Expanded C1/C2 evidence

{_fmt_table(expanded_summary, [
    "method","split","n_bases","active_accuracy",
    "exact_contract_accuracy","active_ece","active_brier"
])}

### Paired base-clustered comparisons

{_fmt_table(expanded_pairwise)}

## Development-only baseline-fairness diagnostic

Selected raw-view weights:
`{json.dumps(selected, sort_keys=True)}`

{_fmt_table(frozen_alpha_summary)}

## Active-SCD specificity

{_fmt_table(scd_prediction_table)}

### Conditional-information battery

{_fmt_table(conditional_df)}

### Within-stratum discrimination

{_fmt_table(within_strata_df)}

## Ontology mechanism

{_fmt_table(opaque_pair)}

{_fmt_table(large_alias_pair)}

## Routing provenance

The manuscript must not merge results produced by the legacy strict router whose
known branch uses CB+DualView with the architecture-faithful router whose known
branch uses SCALE's own discriminative decoder.

{_fmt_table(routing_provenance_summary)}

## Schema.org structural negative control

{_fmt_table(schema_perm_summary)}

## Final claim ledger

{_fmt_table(claim_scorecard)}

## Stop rule

After this suite, further synthetic benchmark families should not be added merely
to improve scores. The only qualitatively new evidence would be:

1. a competent generative downstream executor that passes the frozen clean gate;
2. a genuinely longitudinal public version-history case;
3. a second human-authored ontology with a usable hierarchy;
4. a separately frozen larger external compatibility dataset.

Negative or null outcomes from this suite must be reported rather than tuned away.
"""

(MEASURE_DIR / "FINAL_COMPREHENSIVE_MEASUREMENT_REPORT.md").write_text(
    report.strip() + "\n", encoding="utf-8"
)
display(Markdown(report))

manifest = {
    "status": "completed_when_all_cells_run",
    "output_dir": str(MEASURE_DIR),
    "analyses": [
        "data_split_provenance",
        "expanded_heldout_c1c2",
        "dev_only_dual_view_fairness",
        "active_scd_specificity_controls",
        "within_severity_and_family",
        "ontology_paired_tests",
        "architecture_faithful_routing_provenance",
        "schemaorg_structural_permutation",
        "selective_risk_coverage",
        "reader_runtime_crm_complexity",
        "final_claim_scorecard",
    ],
}
(MEASURE_DIR / "measurement_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print("\nComprehensive measurement suite complete.")
print("Primary output:", MEASURE_DIR / "FINAL_COMPREHENSIVE_MEASUREMENT_REPORT.md")
print("Claim scorecard:", MEASURE_DIR / "FINAL_COMPREHENSIVE_CLAIM_SCORECARD.csv")


# SCALE — External Gate-by-Gate Validation Suite

This suite is the final high-value validation layer after the comprehensive
measurement suite. It does **not** rewrite or relabel the original G0--G7 gates.

It asks whether each thesis survives stronger conditions:

- **G0**: stronger 7--8B heterogeneous readers, first-pass and one-repair modes,
  with NL, JSON, schema-only, and typed SCALE interfaces.
- **G1**: native SCALE competence on the larger 96-base held-out pool and all
  replacement compositions.
- **G2**: a fresh 10-seed Full-vs-NoInv retraining audit on the larger held-out pool.
- **G3**: a second, human-curated external ontology (EDAM), including real
  1.24->1.25 ontology evolution and structural permutation controls.
- **G4**: reuses the development-selected baseline fairness and expanded held-out
  utility analysis from the previous suite.
- **G5**: combinatorial semantic-constraint stress plus a JSON-Schema-only
  syntactic control.
- **G6**: competent generative downstream executors that never consume SCALE
  contracts, plus the existing conditional Active-SCD controls.
- **G7**: stronger real LLM producers, independently screened by NLI.

Additional exploratory evidence mines real MCP/A2A release histories without
turning them into a new gate or using them to tune SCALE.

Scientific rules:
1. original failed gates remain failed unless the paper explicitly defines a
   new external diagnostic rather than rewriting the prespecified gate;
2. no competence threshold is relaxed after seeing outcomes;
3. no fallback model silently substitutes for a failed strong model;
4. model, dataset, release, and checkpoint IDs are written to disk;
5. negative and inconclusive results are retained.


In [ ]:
# =============================================================================
# 0. CONFIGURATION, PRE-FLIGHT, STRICT MODEL LOADING
# =============================================================================

import os, re, gc, json, math, time, hashlib, itertools, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from scipy.stats import binomtest, wilcoxon
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)
from IPython.display import display, Markdown

# requests/jsonschema are installed by the robust bootstrap.
from rdflib import Graph, RDF, RDFS, OWL, URIRef, Literal
from transformers import AutoTokenizer, AutoModelForCausalLM

EXT_DIR = ROOT / "external_gate_validation"
EXT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STRONG_LLM_SUITE = bool(FULL_STRONG_LLM_CAPABLE)
if not RUN_STRONG_LLM_SUITE:
    print(
        "Strong 7–8B LLM suite SKIPPED/INCONCLUSIVE: preferred GPU VRAM "
        "is unavailable. No competence threshold is changed."
    )
RUN_G2_10SEED = True
RUN_EDAM_EXTERNAL = True
RUN_PROTOCOL_HISTORY = True
RUN_G5_COMBINATORIAL = True

# Full paper-strength defaults. Reduce only for debugging, never for manuscript runs.
STRONG_READER_BASE_LIMIT = min(96, len(drift_base))
STRONG_PRODUCER_BASE_LIMIT = min(48, len(drift_base))
GENERATIVE_EXECUTOR_BASE_LIMIT = min(96, len(drift_base))

# Frozen competence gate. Do not change after seeing drift results.
GEN_EXEC_CLEAN_TASK_GATE = 0.70
GEN_EXEC_PARSE_GATE = 0.95

# Three distinct open model families; no fallback substitution is permitted.
STRONG_LLM_SPECS = [
    {
        "label": "Qwen3-8B",
        "model_id": "Qwen/Qwen3-8B",
        "disable_thinking": True,
    },
    {
        "label": "Granite3.3-8B",
        "model_id": "ibm-granite/granite-3.3-8b-instruct",
        "disable_thinking": True,
    },
    {
        "label": "Mistral-7B-v0.3",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "disable_thinking": False,
    },
]

G2_EXTENDED_SEEDS = list(range(3100, 3110))

required = [
    "drift_base", "dev_base", "shift_test_base",
    "CORE_COMPOSITIONS", "C1_COMPOSITIONS", "C2_COMPOSITIONS",
    "AGENTS", "BASE_GRAPH", "global_semantics",
    "deterministic_policy", "realize_message", "build_interface",
    "reader_messages", "parse_semantics", "semantics_parse_valid",
    "generate_text", "extract_json", "cleanup_gpu",
    "producer_prompt", "active_hypothesis", "entailment_prob",
    "prediction_for", "ensemble_predict",
    "train_model", "single_predict", "MAIN_LABEL_MASK",
    "HEAD_EPOCHS", "ontology_repair", "hop_contract",
    "active_correct", "exact_contract",
    "final_contract", "corrupt_contract",
    "contract_violations", "drift_messages",
]
_missing = [x for x in required if x not in globals()]
assert not _missing, (
    "Run every previous notebook cell first. Missing: " + ", ".join(_missing)
)

assert len({b["base_id"] for b in drift_base} & {b["base_id"] for b in train_base}) == 0
assert len({b["base_id"] for b in drift_base} & {b["base_id"] for b in dev_base}) == 0

def strong_model_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _minor = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16

def load_strong_model_strict(spec):
    """
    No fallback. If the declared model cannot load, the result is recorded as
    unavailable rather than silently substituting another model.
    """
    cleanup_gpu()
    model_id = spec["model_id"]
    tok = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        trust_remote_code=False,
    )
    kwargs = dict(
        torch_dtype=strong_model_dtype(),
        low_cpu_mem_usage=True,
        device_map={"": 0} if torch.cuda.is_available() else {"": "cpu"},
        trust_remote_code=False,
    )
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            attn_implementation="sdpa",
            **kwargs,
        )
    except Exception:
        mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)

    mdl.eval()
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    return tok, mdl

def generate_strong(tok, mdl, messages, spec, max_input=1800, max_new=80):
    # Reuse the notebook's deterministic generation utility and chat-template logic.
    return generate_text(
        tok, mdl, messages, spec,
        max_input=max_input,
        max_new=max_new,
    )

def max_replacement_comp(comps):
    return sorted(
        comps,
        key=lambda x: (x["replacement_level"], x["composition_id"])
    )[-1]

G0_COMPS = {
    "core_BBB": max_replacement_comp(CORE_COMPOSITIONS),
    "c1_all_shifted": max_replacement_comp(C1_COMPOSITIONS),
    "c2_all_shifted": max_replacement_comp(C2_COMPOSITIONS),
}

def split_impl_from_name(name):
    if name == "core_BBB":
        return "core"
    if name == "c1_all_shifted":
        return "c1_natural"
    return "c2_schema"

def schema_only_payload(base, comp):
    """
    Strong syntactic control:
    exposes raw handoffs plus an explicit JSON/enum schema, but performs no
    semantic canonicalization for the reader.
    """
    packets = []
    for agent in AGENTS:
        impl = comp["mapping"][agent]
        packets.append({
            "agent": agent,
            "implementation": impl,
            "raw_message": realize_message(base, agent, impl, 0),
        })

    obj = {
        "handoffs": packets,
        "output_schema": {
            "type": "object",
            "required": ["object", "evidence_state", "authority"],
            "properties": {
                "object": {"enum": list(PASS_ACTION_BY_OBJECT.keys())},
                "evidence_state": {
                    "enum": [
                        "EVIDENCE_PASS",
                        "EVIDENCE_FAIL",
                        "EVIDENCE_UNCERTAIN",
                    ]
                },
                "authority": {"enum": list(AUTHORITIES)},
            },
        },
    }
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

def g0_interface(base, comp, interface):
    if interface == "NL":
        return build_interface(base, comp, "NL", BASE_GRAPH)
    if interface == "JSON_RAW":
        return build_interface(base, comp, "JSON", BASE_GRAPH)
    if interface == "SCHEMA_ONLY":
        return schema_only_payload(base, comp)
    if interface == "SCALE_TYPED":
        return build_interface(base, comp, "SCALE", BASE_GRAPH)
    raise ValueError(interface)

def downstream_task_success(base, sem):
    final = deterministic_policy(sem)
    return int(
        final is not None
        and final.get("object") == base["object"]
        and final.get("action") == base["final_action"]
        and final.get("authority") == base["final_authority"]
    )

def exact_json_semantics_prompt(payload):
    return [
        {"role": "system", "content": READER_SYSTEM},
        {"role": "user", "content": payload},
    ]

def parse_executor_output(text):
    obj = extract_json(text)
    if not isinstance(obj, dict):
        return None, False
    out = {
        "object": str(obj.get("object", "")).upper().strip(),
        "action": str(obj.get("action", "")).upper().strip(),
        "authority": str(obj.get("authority", "")).upper().strip(),
    }
    valid = (
        out["object"] in PASS_ACTION_BY_OBJECT
        and out["action"] in ACTIONS
        and out["authority"] in AUTHORITIES
    )
    return out, bool(valid)

EXECUTOR_SYSTEM = """
You are an independent downstream enterprise executor.
You do NOT receive SCALE contracts. You receive natural-language handoffs from
Planner, Retriever, and Policy agents.

Infer:
- workflow object,
- evidence state,
- authority level,
then execute the operational policy.

Rules:
1. Evidence FAIL -> desired action REJECT.
2. Evidence UNCERTAIN -> desired action VERIFY.
3. Evidence PASS:
   PURCHASE -> APPROVE
   ACCESS -> APPROVE
   CHANGE -> UPDATE
   INCIDENT -> SEND
   COMPLIANCE_ITEM -> APPROVE
4. If authority is below the required authority for the desired action, use VERIFY.
5. Preserve the observed authority value in the output.

Required authority:
PLAN=PROPOSE
SEARCH=NONE
ASSESS=NONE
APPROVE=APPROVE
REJECT=APPROVE
VERIFY=RECOMMEND
UPDATE=EXECUTE
SEND=EXECUTE

Return exactly one JSON object:
{"object":"...","action":"...","authority":"..."}
No explanation.
""".strip()

def executor_messages_from_handoffs(base, handoffs):
    block = "\n\n".join(
        f"[{agent}]\n{handoffs[agent]}"
        for agent in AGENTS
    )
    return [
        {"role": "system", "content": EXECUTOR_SYSTEM},
        {"role": "user", "content":
            f"REQUEST:\n{base['request']}\n\nHANDOFFS:\n{block}"
        },
    ]

print("=" * 108)
print("EXTERNAL GATE VALIDATION CONFIG")
print("Strong models:", [x["model_id"] for x in STRONG_LLM_SPECS])
print("G0 bases:", STRONG_READER_BASE_LIMIT)
print("Producer bases:", STRONG_PRODUCER_BASE_LIMIT)
print("Generative executor bases:", GENERATIVE_EXECUTOR_BASE_LIMIT)
print("G2 seeds:", G2_EXTENDED_SEEDS)
print("=" * 108)


In [ ]:
# =============================================================================
# SCALE — DETAILED IN-NOTEBOOK REPORTING HELPERS
# =============================================================================
#
# Every major analysis below writes:
#   1) a detailed Markdown interpretation directly into the notebook;
#   2) the same section to external_gate_validation/detailed_reports/*.md;
#   3) the section into an accumulated final report shown at notebook end.
# =============================================================================

from pathlib import Path
from IPython.display import display, Markdown
import numpy as np
import pandas as pd
import json, math, os

DETAIL_REPORT_DIR = EXT_DIR / "detailed_reports"
DETAIL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

SCALE_DETAILED_REPORT_SECTIONS = []

def _safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def _fmt(x, digits=3):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return "n/a"
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)

def _df_md(df, cols=None, max_rows=50, digits=4):
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return "_No rows available._"
    x = df.copy()
    if cols:
        cols = [c for c in cols if c in x.columns]
        x = x[cols]
    for c in x.select_dtypes(include=[np.number]).columns:
        x[c] = x[c].round(digits)
    return x.head(max_rows).to_markdown(index=False)

def _save_and_show_report(title, body, filename):
    text = f"# {title}\n\n{body.strip()}\n"
    path = DETAIL_REPORT_DIR / filename
    path.write_text(text, encoding="utf-8")
    SCALE_DETAILED_REPORT_SECTIONS.append(text)
    display(Markdown(text))
    print("Detailed notebook report saved:", path)
    return path

def _claim_language(delta=None, lo=None, hi=None):
    if delta is None:
        return "Use descriptive/qualified language."
    if lo is not None and np.isfinite(lo) and lo > 0:
        return "Paired uncertainty supports a positive effect; superiority language can be used cautiously."
    if hi is not None and np.isfinite(hi) and hi < 0:
        return "The comparison favors the baseline/ablation; do not claim superiority."
    return "The interval includes zero; use 'competitive', 'matches', or 'numerically higher/lower', not established superiority."

print("Detailed in-notebook reporting initialized.")


In [53]:
# =============================================================================
# 1. G0 + G6 + G7 — STRONG HETEROGENEOUS LLM SUITE
# =============================================================================
#
# One sequential model load performs:
#   G0: arbitrary vs typed cross-reader consumption
#   G6: clean-gated generative downstream execution under drift
#   G7: real upstream message generation and SCALE recovery
# =============================================================================

strong_reader_rows = []
strong_executor_rows = []
strong_producer_rows = []
strong_model_status = []

reader_bases = drift_base[:STRONG_READER_BASE_LIMIT]
producer_bases = drift_base[:STRONG_PRODUCER_BASE_LIMIT]
executor_bases = drift_base[:GENERATIVE_EXECUTOR_BASE_LIMIT]

# Existing Active-SCD column resolution.
DRIFT_ACTIVE_COL = next(
    (
        c for c in [
            "prepolicy_active_scd",
            "active_scd",
            "Active-SCD",
        ]
        if c in drift_df.columns
    ),
    None,
)
assert DRIFT_ACTIVE_COL is not None, "No Active-SCD column found in drift_df."

for spec in STRONG_LLM_SPECS:
    if not RUN_STRONG_LLM_SUITE:
        break

    print("\n" + "=" * 88)
    print("STRONG MODEL:", spec["label"], spec["model_id"])
    print("=" * 88)

    try:
        tok, mdl = load_strong_model_strict(spec)
        strong_model_status.append({
            "model": spec["label"],
            "model_id": spec["model_id"],
            "loaded": 1,
            "error": None,
        })
    except Exception as exc:
        strong_model_status.append({
            "model": spec["label"],
            "model_id": spec["model_id"],
            "loaded": 0,
            "error": f"{type(exc).__name__}:{str(exc)[:300]}",
        })
        print("LOAD FAILED:", type(exc).__name__, str(exc)[:300])
        continue

    # -------------------------------------------------------------------------
    # G0 — strong reader study
    # -------------------------------------------------------------------------
    for base in reader_bases:
        gold_sem = global_semantics(base)

        for case_name, comp in G0_COMPS.items():
            split = split_impl_from_name(case_name)

            for interface in ["NL", "JSON_RAW", "SCHEMA_ONLY", "SCALE_TYPED"]:
                payload = g0_interface(base, comp, interface)
                messages = exact_json_semantics_prompt(
                    f"INTERFACE={interface}\n\n{payload}"
                )

                try:
                    gen = generate_strong(
                        tok, mdl, messages, spec,
                        max_input=2200,
                        max_new=72,
                    )
                    sem_first, parse_first = parse_semantics(gen["text"])
                    label_valid_first = int(
                        parse_first and semantics_parse_valid(sem_first)
                    )

                    sem_final = sem_first
                    parse_final = parse_first
                    repaired = 0

                    # Uniform one-shot FORMAT repair only for malformed output.
                    if not parse_first:
                        repair_messages = reader_repair_messages(gen["text"])
                        regen = generate_strong(
                            tok, mdl, repair_messages, spec,
                            max_input=1000,
                            max_new=72,
                        )
                        sem2, parse2 = parse_semantics(regen["text"])
                        if parse2:
                            sem_final = sem2
                            parse_final = True
                            repaired = 1

                    valid_final = int(
                        parse_final and semantics_parse_valid(sem_final)
                    )
                    semantic_success = int(
                        valid_final and sem_final == gold_sem
                    )
                    task_success = (
                        downstream_task_success(base, sem_final)
                        if valid_final else 0
                    )

                    strong_reader_rows.append({
                        "model": spec["label"],
                        "model_id": spec["model_id"],
                        "base_id": base["base_id"],
                        "split": split,
                        "interface": interface,
                        "first_parse_success": int(parse_first),
                        "first_label_valid": label_valid_first,
                        "repair_used": repaired,
                        "final_parse_success": int(parse_final),
                        "final_label_valid": valid_final,
                        "semantic_success": semantic_success,
                        "task_success": task_success,
                        "input_tokens": gen.get("input_tokens", np.nan),
                        "latency_s": gen.get("latency_s", np.nan),
                        "error": None,
                    })
                except Exception as exc:
                    strong_reader_rows.append({
                        "model": spec["label"],
                        "model_id": spec["model_id"],
                        "base_id": base["base_id"],
                        "split": split,
                        "interface": interface,
                        "first_parse_success": 0,
                        "first_label_valid": 0,
                        "repair_used": 0,
                        "final_parse_success": 0,
                        "final_label_valid": 0,
                        "semantic_success": 0,
                        "task_success": 0,
                        "input_tokens": np.nan,
                        "latency_s": np.nan,
                        "error": f"{type(exc).__name__}:{str(exc)[:200]}",
                    })

    # -------------------------------------------------------------------------
    # G7 — stronger real upstream producers
    # -------------------------------------------------------------------------
    # NLI stays on CPU and is independent of the producer.
    load_nli()

    for base in producer_bases:
        for agent in AGENTS:
            try:
                gen = generate_strong(
                    tok, mdl,
                    producer_prompt(base, agent),
                    spec,
                    max_input=800,
                    max_new=64,
                )
                msg = gen["text"]
                ent = (
                    entailment_prob(msg, active_hypothesis(base, agent))
                    if msg else 0.0
                )
                accepted = int(
                    bool(msg)
                    and not gen.get("overflow", False)
                    and ent >= NLI_ENTAIL_THRESHOLD
                )

                for decoder in ["CB+DualView", "SCALE-Full"]:
                    pred, conf, _ = prediction_for(
                        decoder,
                        msg,
                        agent,
                        BASE_GRAPH,
                        repair=True,
                    )
                    gold = hop_contract(base, agent)
                    strong_producer_rows.append({
                        "producer_model": spec["label"],
                        "producer_model_id": spec["model_id"],
                        "base_id": base["base_id"],
                        "agent": agent,
                        "decoder": decoder,
                        "message": msg,
                        "nli_entailment": ent,
                        "accepted": accepted,
                        "active_correct": (
                            active_correct(gold, pred, agent)
                            if accepted else np.nan
                        ),
                        "confidence": conf,
                        "error": None,
                    })
            except Exception as exc:
                strong_producer_rows.append({
                    "producer_model": spec["label"],
                    "producer_model_id": spec["model_id"],
                    "base_id": base["base_id"],
                    "agent": agent,
                    "decoder": "generation_failed",
                    "message": "",
                    "nli_entailment": 0.0,
                    "accepted": 0,
                    "active_correct": np.nan,
                    "confidence": np.nan,
                    "error": f"{type(exc).__name__}:{str(exc)[:200]}",
                })

    # -------------------------------------------------------------------------
    # G6 — competent generative executor
    # -------------------------------------------------------------------------
    clean_records = []

    for base in executor_bases:
        clean_handoffs = {
            agent: realize_message(base, agent, "A", 0)
            for agent in AGENTS
        }
        try:
            gen = generate_strong(
                tok, mdl,
                executor_messages_from_handoffs(base, clean_handoffs),
                spec,
                max_input=1800,
                max_new=72,
            )
            pred, valid = parse_executor_output(gen["text"])
            correct = int(
                valid
                and pred["object"] == base["object"]
                and pred["action"] == base["final_action"]
                and pred["authority"] == base["final_authority"]
            )
        except Exception as exc:
            pred, valid, correct = None, False, 0
            gen = {"text": "", "input_tokens": np.nan, "latency_s": np.nan}

        clean_records.append({
            "model": spec["label"],
            "model_id": spec["model_id"],
            "base_id": base["base_id"],
            "severity": 0,
            "clean_gate_record": 1,
            "parse_valid": int(valid),
            "task_success": correct,
            "raw_output": gen.get("text", ""),
        })

    clean_df_local = pd.DataFrame(clean_records)
    clean_acc = float(clean_df_local["task_success"].mean())
    clean_parse = float(clean_df_local["parse_valid"].mean())
    passed_gate = (
        clean_acc >= GEN_EXEC_CLEAN_TASK_GATE
        and clean_parse >= GEN_EXEC_PARSE_GATE
    )

    print(
        f"{spec['label']} clean executor:"
        f" task={clean_acc:.3f}, parse={clean_parse:.3f}, pass={passed_gate}"
    )

    strong_executor_rows.extend(clean_records)

    # Frozen competence gate: run drift only if the model passes.
    if passed_gate:
        for base in executor_bases:
            for severity in [1, 2, 3]:
                handoffs, _ops = drift_messages(base, severity)

                try:
                    gen = generate_strong(
                        tok, mdl,
                        executor_messages_from_handoffs(base, handoffs),
                        spec,
                        max_input=1800,
                        max_new=72,
                    )
                    pred, valid = parse_executor_output(gen["text"])
                    correct = int(
                        valid
                        and pred["object"] == base["object"]
                        and pred["action"] == base["final_action"]
                        and pred["authority"] == base["final_authority"]
                    )
                except Exception as exc:
                    pred, valid, correct = None, False, 0
                    gen = {"text": ""}

                dd = drift_df[
                    (drift_df["base_id"] == base["base_id"])
                    & (drift_df["severity"] == severity)
                ].iloc[0]

                strong_executor_rows.append({
                    "model": spec["label"],
                    "model_id": spec["model_id"],
                    "base_id": base["base_id"],
                    "severity": severity,
                    "clean_gate_record": 0,
                    "parse_valid": int(valid),
                    "task_success": correct,
                    "failure": 1 - correct,
                    "active_scd": float(dd[DRIFT_ACTIVE_COL]),
                    "full_scd": float(dd["full_scd"]) if "full_scd" in dd else np.nan,
                    "neg_confidence": (
                        1.0 - float(dd["confidence"])
                        if "confidence" in dd else np.nan
                    ),
                    "raw_output": gen.get("text", ""),
                })

    del mdl, tok
    cleanup_gpu()

strong_reader_df = pd.DataFrame(strong_reader_rows)
strong_executor_df = pd.DataFrame(strong_executor_rows)
strong_producer_df = pd.DataFrame(strong_producer_rows)
strong_model_status_df = pd.DataFrame(strong_model_status)

# G0 summaries.
strong_reader_summary = pd.DataFrame()
if len(strong_reader_df):
    strong_reader_summary = (
        strong_reader_df.groupby(
            ["model", "split", "interface"], as_index=False
        )
        .agg(
            n_bases=("base_id", "nunique"),
            first_parse_success=("first_parse_success", "mean"),
            final_parse_success=("final_parse_success", "mean"),
            final_label_valid=("final_label_valid", "mean"),
            semantic_success=("semantic_success", "mean"),
            task_success=("task_success", "mean"),
            repair_rate=("repair_used", "mean"),
        )
    )

print("\n[G0+] STRONG READER SUMMARY")
display(strong_reader_summary.round(4))

# G7 summaries.
strong_producer_summary = pd.DataFrame()
if len(strong_producer_df):
    valid_dec = strong_producer_df[
        strong_producer_df["decoder"].isin(["CB+DualView", "SCALE-Full"])
    ]
    strong_producer_summary = (
        valid_dec.groupby(
            ["producer_model", "agent", "decoder"], as_index=False
        )
        .agg(
            n=("base_id", "size"),
            acceptance_rate=("accepted", "mean"),
            mean_nli=("nli_entailment", "mean"),
            active_accuracy_accepted=(
                "active_correct",
                lambda x: float(pd.to_numeric(x, errors="coerce").dropna().mean())
                if pd.to_numeric(x, errors="coerce").notna().any()
                else np.nan
            ),
        )
    )

print("\n[G7+] STRONG PRODUCER SUMMARY")
display(strong_producer_summary.round(4))

# G6 generative-executor summary.
gen_exec_summary_rows = []
if len(strong_executor_df):
    for model, g in strong_executor_df.groupby("model"):
        clean = g[g["clean_gate_record"] == 1]
        clean_task = float(clean["task_success"].mean()) if len(clean) else np.nan
        clean_parse = float(clean["parse_valid"].mean()) if len(clean) else np.nan
        passed = bool(
            clean_task >= GEN_EXEC_CLEAN_TASK_GATE
            and clean_parse >= GEN_EXEC_PARSE_GATE
        )

        drift_g = g[g["clean_gate_record"] == 0].copy()
        row = {
            "model": model,
            "clean_n": len(clean),
            "clean_task_accuracy": clean_task,
            "clean_parse_validity": clean_parse,
            "competence_gate_pass": int(passed),
            "drift_n": len(drift_g),
            "active_scd_failure_auc": np.nan,
            "active_scd_failure_auprc": np.nan,
            "full_scd_failure_auc": np.nan,
            "neg_conf_failure_auc": np.nan,
        }

        if passed and len(drift_g) and drift_g["failure"].nunique() >= 2:
            row["active_scd_failure_auc"] = float(
                roc_auc_score(drift_g["failure"], drift_g["active_scd"])
            )
            row["active_scd_failure_auprc"] = float(
                average_precision_score(
                    drift_g["failure"], drift_g["active_scd"]
                )
            )
            if drift_g["full_scd"].notna().all():
                row["full_scd_failure_auc"] = float(
                    roc_auc_score(drift_g["failure"], drift_g["full_scd"])
                )
            if drift_g["neg_confidence"].notna().all():
                row["neg_conf_failure_auc"] = float(
                    roc_auc_score(
                        drift_g["failure"], drift_g["neg_confidence"]
                    )
                )

        gen_exec_summary_rows.append(row)

generative_executor_summary = pd.DataFrame(gen_exec_summary_rows)

print("\n[G6+] COMPETENCE-GATED GENERATIVE EXECUTORS")
display(generative_executor_summary.round(4))

strong_model_status_df.to_csv(EXT_DIR / "strong_model_status.csv", index=False)
strong_reader_df.to_csv(EXT_DIR / "g0_strong_reader_records.csv", index=False)
strong_reader_summary.to_csv(EXT_DIR / "g0_strong_reader_summary.csv", index=False)
strong_producer_df.to_csv(EXT_DIR / "g7_strong_producer_records.csv", index=False)
strong_producer_summary.to_csv(EXT_DIR / "g7_strong_producer_summary.csv", index=False)
strong_executor_df.to_csv(EXT_DIR / "g6_generative_executor_records.csv", index=False)
generative_executor_summary.to_csv(
    EXT_DIR / "g6_generative_executor_summary.csv", index=False
)



STRONG MODEL: Qwen3-8B Qwen/Qwen3-8B
Qwen3-8B clean executor: task=0.729, parse=1.000, pass=True

STRONG MODEL: Granite3.3-8B ibm-granite/granite-3.3-8b-instruct
Granite3.3-8B clean executor: task=0.385, parse=0.531, pass=False

STRONG MODEL: Mistral-7B-v0.3 mistralai/Mistral-7B-Instruct-v0.3
Mistral-7B-v0.3 clean executor: task=0.406, parse=0.844, pass=False

[G0+] STRONG READER SUMMARY


,model,split,interface,n_bases,first_parse_success,final_parse_success,final_label_valid,semantic_success,task_success,repair_rate
0,Granite3.3-8B,c1_natural,JSON_RAW,96,1.0,1.0,1.0000,0.8021,0.8021,0.0
1,Granite3.3-8B,c1_natural,NL,96,1.0,1.0,1.0000,0.8333,0.8333,0.0
2,Granite3.3-8B,c1_natural,SCALE_TYPED,96,1.0,1.0,0.8125,0.8125,0.8125,0.0
3,Granite3.3-8B,c1_natural,SCHEMA_ONLY,96,1.0,1.0,1.0000,0.6875,0.6875,0.0
4,Granite3.3-8B,c2_schema,JSON_RAW,96,1.0,1.0,1.0000,0.5625,0.5625,0.0
5,Granite3.3-8B,c2_schema,NL,96,1.0,1.0,1.0000,0.5208,0.5208,0.0
6,Granite3.3-8B,c2_schema,SCALE_TYPED,96,1.0,1.0,1.0000,0.6667,0.6667,0.0
7,Granite3.3-8B,c2_schema,SCHEMA_ONLY,96,1.0,1.0,1.0000,0.3542,0.3542,0.0
8,Granite3.3-8B,core,JSON_RAW,96,1.0,1.0,1.0000,0.2500,0.2500,0.0
9,Granite3.3-8B,core,NL,96,1.0,1.0,1.0000,0.1562,0.1875,0.0



[G7+] STRONG PRODUCER SUMMARY


,producer_model,agent,decoder,n,acceptance_rate,mean_nli,active_accuracy_accepted
0,Granite3.3-8B,PLANNER,CB+DualView,48,0.7708,0.8073,0.7297
1,Granite3.3-8B,PLANNER,SCALE-Full,48,0.7708,0.8073,0.7297
2,Granite3.3-8B,POLICY,CB+DualView,48,0.7083,0.7413,1.0000
3,Granite3.3-8B,POLICY,SCALE-Full,48,0.7083,0.7413,1.0000
4,Granite3.3-8B,RETRIEVER,CB+DualView,48,0.3750,0.4427,1.0000
5,Granite3.3-8B,RETRIEVER,SCALE-Full,48,0.3750,0.4427,1.0000
6,Mistral-7B-v0.3,PLANNER,CB+DualView,48,0.8333,0.8513,1.0000
7,Mistral-7B-v0.3,PLANNER,SCALE-Full,48,0.8333,0.8513,1.0000
8,Mistral-7B-v0.3,POLICY,CB+DualView,48,0.6458,0.6529,1.0000
9,Mistral-7B-v0.3,POLICY,SCALE-Full,48,0.6458,0.6529,1.0000



[G6+] COMPETENCE-GATED GENERATIVE EXECUTORS


,model,clean_n,clean_task_accuracy,clean_parse_validity,competence_gate_pass,drift_n,active_scd_failure_auc,active_scd_failure_auprc,full_scd_failure_auc,neg_conf_failure_auc
0,Granite3.3-8B,96,0.3854,0.5312,0,0,NaN,NaN,NaN,NaN
1,Mistral-7B-v0.3,96,0.4062,0.8438,0,0,NaN,NaN,NaN,NaN
2,Qwen3-8B,96,0.7292,1.0000,1,288,NaN,NaN,NaN,NaN


In [54]:
# =============================================================================
# DETAILED RESULT REPORT — G0 / G6 / G7 STRONG LLM SUITE
# =============================================================================

parts = []

# ---------------- G0 ----------------
parts.append("## G0 — Strong heterogeneous reader consumption\n")

if isinstance(strong_model_status_df, pd.DataFrame) and len(strong_model_status_df):
    parts.append("### Model availability")
    parts.append(_df_md(
        strong_model_status_df,
        ["model", "model_id", "loaded", "error"],
        max_rows=20,
    ))

if isinstance(strong_reader_summary, pd.DataFrame) and len(strong_reader_summary):
    parts.append("\n### Full reader summary")
    parts.append(_df_md(
        strong_reader_summary,
        [
            "model", "split", "interface", "n_bases",
            "first_parse_success", "final_parse_success",
            "final_label_valid", "semantic_success",
            "task_success", "repair_rate"
        ],
        max_rows=100,
    ))

    # Aggregate interface comparison.
    agg = (
        strong_reader_summary.groupby("interface", as_index=False)
        .agg(
            mean_first_parse=("first_parse_success", "mean"),
            mean_final_parse=("final_parse_success", "mean"),
            mean_semantic=("semantic_success", "mean"),
            mean_task=("task_success", "mean"),
            mean_repair=("repair_rate", "mean"),
        )
        .sort_values("mean_semantic", ascending=False)
    )
    parts.append("\n### Interface-level aggregate")
    parts.append(_df_md(agg))

    typed = agg[agg["interface"] == "SCALE_TYPED"]
    nl = agg[agg["interface"] == "NL"]
    js = agg[agg["interface"] == "JSON_RAW"]
    schema = agg[agg["interface"] == "SCHEMA_ONLY"]

    def val(frame, col):
        return float(frame.iloc[0][col]) if len(frame) else np.nan

    typed_sem = val(typed, "mean_semantic")
    nl_sem = val(nl, "mean_semantic")
    json_sem = val(js, "mean_semantic")
    schema_sem = val(schema, "mean_semantic")

    parts.append("\n### Interpretation")
    parts.append(
        f"- Mean semantic success: **SCALE_TYPED {_fmt(typed_sem)}**, "
        f"NL {_fmt(nl_sem)}, JSON_RAW {_fmt(json_sem)}, SCHEMA_ONLY {_fmt(schema_sem)}."
    )

    if np.isfinite(typed_sem) and np.isfinite(schema_sem):
        diff = typed_sem - schema_sem
        parts.append(
            f"- Typed SCALE minus schema-only semantic success = **{_fmt(diff)}**."
        )
        if diff > 0.05:
            parts.append(
                "- This materially supports the claim that the typed SCALE interface is doing more than syntactic schema validation."
            )
        elif diff >= -0.02:
            parts.append(
                "- Typed SCALE is not clearly better than schema-only consumption in this aggregate; do not use this experiment as proof that typed semantics universally dominates JSON/schema."
            )
        else:
            parts.append(
                "- Schema-only consumption is stronger in this aggregate. This is a negative result for a broad typed-interface superiority claim."
            )

    parts.append(
        "- **Original G0 status must remain FAIL.** These stronger-reader results are a post-hoc localization/robustness diagnostic, not a redefinition of the prespecified gate."
    )
    parts.append(
        "- Manuscript language should separate arbitrary generative rereading from intended typed/native consumption."
    )
else:
    parts.append(
        "_Strong reader evaluation produced no usable rows. G0 external extension is inconclusive._"
    )

# ---------------- G6 ----------------
parts.append("\n## G6 — Competence-gated generative downstream executors\n")

if isinstance(generative_executor_summary, pd.DataFrame) and len(generative_executor_summary):
    parts.append(_df_md(
        generative_executor_summary,
        [
            "model", "clean_n", "clean_task_accuracy",
            "clean_parse_validity", "competence_gate_pass",
            "drift_n", "active_scd_failure_auc",
            "active_scd_failure_auprc",
            "full_scd_failure_auc", "neg_conf_failure_auc"
        ],
        max_rows=20,
    ))

    passed = generative_executor_summary[
        generative_executor_summary["competence_gate_pass"] == 1
    ]

    if len(passed):
        parts.append(
            f"- **{len(passed)} model(s)** passed the frozen clean competence gate "
            f"(task ≥ {GEN_EXEC_CLEAN_TASK_GATE:.2f}, parse ≥ {GEN_EXEC_PARSE_GATE:.2f})."
        )
        for r in passed.itertuples():
            parts.append(
                f"- {r.model}: clean task={_fmt(r.clean_task_accuracy)}, "
                f"parse={_fmt(r.clean_parse_validity)}, "
                f"Active-SCD failure AUROC={_fmt(r.active_scd_failure_auc)}, "
                f"AUPRC={_fmt(r.active_scd_failure_auprc)}, "
                f"Full-SCD AUROC={_fmt(r.full_scd_failure_auc)}, "
                f"negative-confidence AUROC={_fmt(r.neg_conf_failure_auc)}."
            )

        mean_auc = float(passed["active_scd_failure_auc"].dropna().mean()) if passed["active_scd_failure_auc"].notna().any() else np.nan
        if np.isfinite(mean_auc) and mean_auc >= 0.80:
            parts.append(
                "- This materially closes the strongest external-validity criticism: Active-SCD remains predictive for at least one clean-competent generative executor that never consumes SCALE contracts."
            )
        elif np.isfinite(mean_auc) and mean_auc >= 0.70:
            parts.append(
                "- The generative-executor evidence is positive but moderate; report it as supporting external validation rather than headline dominance."
            )
        else:
            parts.append(
                "- The competent generative-executor evidence is weak/null. Keep this as a limitation and do not average it with classifier-executor results."
            )
    else:
        parts.append(
            "- No declared strong model passed the **frozen** competence gate. The generative extension is therefore inconclusive; the gate must not be relaxed post hoc."
        )
else:
    parts.append("_No generative executor results available._")

# ---------------- G7 ----------------
parts.append("\n## G7 — Strong real LLM producers\n")

if isinstance(strong_producer_summary, pd.DataFrame) and len(strong_producer_summary):
    parts.append(_df_md(
        strong_producer_summary,
        [
            "producer_model", "agent", "decoder", "n",
            "acceptance_rate", "mean_nli",
            "active_accuracy_accepted"
        ],
        max_rows=100,
    ))

    scale_prod = strong_producer_summary[
        strong_producer_summary["decoder"] == "SCALE-Full"
    ]
    if len(scale_prod):
        accept = float(scale_prod["acceptance_rate"].mean())
        active = float(scale_prod["active_accuracy_accepted"].mean())
        parts.append(
            f"- Across strong producers, SCALE-conditioned mean producer acceptance = **{_fmt(accept)}** and active accuracy on accepted handoffs = **{_fmt(active)}**."
        )
        if accept >= 0.80 and active >= 0.95:
            parts.append(
                "- This is strong supporting evidence that semantic recovery survives more capable, naturally generated upstream handoffs."
            )
        else:
            parts.append(
                "- Producer fidelity or semantic recovery remains heterogeneous; preserve producer-side limitations and report per-role results."
            )
else:
    parts.append("_No strong-producer results available._")

_save_and_show_report(
    "G0 / G6 / G7 — Detailed Strong-LLM Validation",
    "\n".join(parts),
    "01_G0_G6_G7_strong_llm_detailed.md",
)


# G0 / G6 / G7 — Detailed Strong-LLM Validation

## G0 — Strong heterogeneous reader consumption

### Model availability
| model           | model_id                            |   loaded | error   |
|:----------------|:------------------------------------|---------:|:--------|
| Qwen3-8B        | Qwen/Qwen3-8B                       |        1 |         |
| Granite3.3-8B   | ibm-granite/granite-3.3-8b-instruct |        1 |         |
| Mistral-7B-v0.3 | mistralai/Mistral-7B-Instruct-v0.3  |        1 |         |

### Full reader summary
| model           | split      | interface   |   n_bases |   first_parse_success |   final_parse_success |   final_label_valid |   semantic_success |   task_success |   repair_rate |
|:----------------|:-----------|:------------|----------:|----------------------:|----------------------:|--------------------:|-------------------:|---------------:|--------------:|
| Granite3.3-8B   | c1_natural | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.8021 |         0.8021 |             0 |
| Granite3.3-8B   | c1_natural | NL          |        96 |                     1 |                     1 |              1      |             0.8333 |         0.8333 |             0 |
| Granite3.3-8B   | c1_natural | SCALE_TYPED |        96 |                     1 |                     1 |              0.8125 |             0.8125 |         0.8125 |             0 |
| Granite3.3-8B   | c1_natural | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.6875 |         0.6875 |             0 |
| Granite3.3-8B   | c2_schema  | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.5625 |         0.5625 |             0 |
| Granite3.3-8B   | c2_schema  | NL          |        96 |                     1 |                     1 |              1      |             0.5208 |         0.5208 |             0 |
| Granite3.3-8B   | c2_schema  | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             0.6667 |         0.6667 |             0 |
| Granite3.3-8B   | c2_schema  | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.3542 |         0.3542 |             0 |
| Granite3.3-8B   | core       | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.25   |         0.25   |             0 |
| Granite3.3-8B   | core       | NL          |        96 |                     1 |                     1 |              1      |             0.1562 |         0.1875 |             0 |
| Granite3.3-8B   | core       | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             1      |         1      |             0 |
| Granite3.3-8B   | core       | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.2083 |         0.2083 |             0 |
| Mistral-7B-v0.3 | c1_natural | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.7708 |         0.7708 |             0 |
| Mistral-7B-v0.3 | c1_natural | NL          |        96 |                     1 |                     1 |              1      |             0.9479 |         0.9479 |             0 |
| Mistral-7B-v0.3 | c1_natural | SCALE_TYPED |        96 |                     1 |                     1 |              0.8125 |             0.8125 |         0.8125 |             0 |
| Mistral-7B-v0.3 | c1_natural | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.8021 |         0.8333 |             0 |
| Mistral-7B-v0.3 | c2_schema  | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.2917 |         0.2917 |             0 |
| Mistral-7B-v0.3 | c2_schema  | NL          |        96 |                     1 |                     1 |              1      |             0.4167 |         0.4167 |             0 |
| Mistral-7B-v0.3 | c2_schema  | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             0.6667 |         0.6667 |             0 |
| Mistral-7B-v0.3 | c2_schema  | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.3958 |         0.3958 |             0 |
| Mistral-7B-v0.3 | core       | JSON_RAW    |        96 |                     1 |                     1 |              0.9792 |             0.1979 |         0.1979 |             0 |
| Mistral-7B-v0.3 | core       | NL          |        96 |                     1 |                     1 |              1      |             0.125  |         0.125  |             0 |
| Mistral-7B-v0.3 | core       | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             1      |         1      |             0 |
| Mistral-7B-v0.3 | core       | SCHEMA_ONLY |        96 |                     1 |                     1 |              0.8125 |             0.0938 |         0.0938 |             0 |
| Qwen3-8B        | c1_natural | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.9271 |         0.9479 |             0 |
| Qwen3-8B        | c1_natural | NL          |        96 |                     1 |                     1 |              1      |             0.9062 |         0.9271 |             0 |
| Qwen3-8B        | c1_natural | SCALE_TYPED |        96 |                     1 |                     1 |              0.8125 |             0.8125 |         0.8125 |             0 |
| Qwen3-8B        | c1_natural | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             1      |         1      |             0 |
| Qwen3-8B        | c2_schema  | JSON_RAW    |        96 |                     1 |                     1 |              1      |             0.375  |         0.375  |             0 |
| Qwen3-8B        | c2_schema  | NL          |        96 |                     1 |                     1 |              1      |             0.375  |         0.375  |             0 |
| Qwen3-8B        | c2_schema  | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             0.6667 |         0.6667 |             0 |
| Qwen3-8B        | c2_schema  | SCHEMA_ONLY |        96 |                     1 |                     1 |              1      |             0.3125 |         0.3125 |             0 |
| Qwen3-8B        | core       | JSON_RAW    |        96 |                     1 |                     1 |              0.8125 |             0.3958 |         0.4167 |             0 |
| Qwen3-8B        | core       | NL          |        96 |                     1 |                     1 |              0.8125 |             0.5521 |         0.5729 |             0 |
| Qwen3-8B        | core       | SCALE_TYPED |        96 |                     1 |                     1 |              1      |             1      |         1      |             0 |
| Qwen3-8B        | core       | SCHEMA_ONLY |        96 |                     1 |                     1 |              0.9792 |             0.4062 |         0.4062 |             0 |

### Interface-level aggregate
| interface   |   mean_first_parse |   mean_final_parse |   mean_semantic |   mean_task |   mean_repair |
|:------------|-------------------:|-------------------:|----------------:|------------:|--------------:|
| SCALE_TYPED |                  1 |                  1 |          0.8264 |      0.8264 |             0 |
| NL          |                  1 |                  1 |          0.537  |      0.5451 |             0 |
| JSON_RAW    |                  1 |                  1 |          0.5081 |      0.5127 |             0 |
| SCHEMA_ONLY |                  1 |                  1 |          0.4734 |      0.4769 |             0 |

### Interpretation
- Mean semantic success: **SCALE_TYPED 0.826**, NL 0.537, JSON_RAW 0.508, SCHEMA_ONLY 0.473.
- Typed SCALE minus schema-only semantic success = **0.353**.
- This materially supports the claim that the typed SCALE interface is doing more than syntactic schema validation.
- **Original G0 status must remain FAIL.** These stronger-reader results are a post-hoc localization/robustness diagnostic, not a redefinition of the prespecified gate.
- Manuscript language should separate arbitrary generative rereading from intended typed/native consumption.

## G6 — Competence-gated generative downstream executors

| model           |   clean_n |   clean_task_accuracy |   clean_parse_validity |   competence_gate_pass |   drift_n |   active_scd_failure_auc |   active_scd_failure_auprc |   full_scd_failure_auc |   neg_conf_failure_auc |
|:----------------|----------:|----------------------:|-----------------------:|-----------------------:|----------:|-------------------------:|---------------------------:|-----------------------:|-----------------------:|
| Granite3.3-8B   |        96 |                0.3854 |                 0.5312 |                      0 |         0 |                      nan |                        nan |                    nan |                    nan |
| Mistral-7B-v0.3 |        96 |                0.4062 |                 0.8438 |                      0 |         0 |                      nan |                        nan |                    nan |                    nan |
| Qwen3-8B        |        96 |                0.7292 |                 1      |                      1 |       288 |                      nan |                        nan |                    nan |                    nan |
- **1 model(s)** passed the frozen clean competence gate (task ≥ 0.70, parse ≥ 0.95).
- Qwen3-8B: clean task=0.729, parse=1.000, Active-SCD failure AUROC=n/a, AUPRC=n/a, Full-SCD AUROC=n/a, negative-confidence AUROC=n/a.
- The competent generative-executor evidence is weak/null. Keep this as a limitation and do not average it with classifier-executor results.

## G7 — Strong real LLM producers

| producer_model   | agent     | decoder     |   n |   acceptance_rate |   mean_nli |   active_accuracy_accepted |
|:-----------------|:----------|:------------|----:|------------------:|-----------:|---------------------------:|
| Granite3.3-8B    | PLANNER   | CB+DualView |  48 |            0.7708 |     0.8073 |                     0.7297 |
| Granite3.3-8B    | PLANNER   | SCALE-Full  |  48 |            0.7708 |     0.8073 |                     0.7297 |
| Granite3.3-8B    | POLICY    | CB+DualView |  48 |            0.7083 |     0.7413 |                     1      |
| Granite3.3-8B    | POLICY    | SCALE-Full  |  48 |            0.7083 |     0.7413 |                     1      |
| Granite3.3-8B    | RETRIEVER | CB+DualView |  48 |            0.375  |     0.4427 |                     1      |
| Granite3.3-8B    | RETRIEVER | SCALE-Full  |  48 |            0.375  |     0.4427 |                     1      |
| Mistral-7B-v0.3  | PLANNER   | CB+DualView |  48 |            0.8333 |     0.8513 |                     1      |
| Mistral-7B-v0.3  | PLANNER   | SCALE-Full  |  48 |            0.8333 |     0.8513 |                     1      |
| Mistral-7B-v0.3  | POLICY    | CB+DualView |  48 |            0.6458 |     0.6529 |                     1      |
| Mistral-7B-v0.3  | POLICY    | SCALE-Full  |  48 |            0.6458 |     0.6529 |                     1      |
| Mistral-7B-v0.3  | RETRIEVER | CB+DualView |  48 |            0      |     0.1911 |                   nan      |
| Mistral-7B-v0.3  | RETRIEVER | SCALE-Full  |  48 |            0      |     0.1911 |                   nan      |
| Qwen3-8B         | PLANNER   | CB+DualView |  48 |            1      |     0.9885 |                     1      |
| Qwen3-8B         | PLANNER   | SCALE-Full  |  48 |            1      |     0.9885 |                     1      |
| Qwen3-8B         | POLICY    | CB+DualView |  48 |            1      |     0.9935 |                     1      |
| Qwen3-8B         | POLICY    | SCALE-Full  |  48 |            1      |     0.9935 |                     1      |
| Qwen3-8B         | RETRIEVER | CB+DualView |  48 |            1      |     0.988  |                     1      |
| Qwen3-8B         | RETRIEVER | SCALE-Full  |  48 |            1      |     0.988  |                     1      |
- Across strong producers, SCALE-conditioned mean producer acceptance = **0.704** and active accuracy on accepted handoffs = **0.966**.
- Producer fidelity or semantic recovery remains heterogeneous; preserve producer-side limitations and report per-role results.


Detailed notebook report saved: /content/scale-iclr-full-master-from-scratch-v3.1.0-2171b05ab5d5/external_gate_validation/detailed_reports/01_G0_G6_G7_strong_llm_detailed.md


PosixPath('/content/scale-iclr-full-master-from-scratch-v3.1.0-2171b05ab5d5/external_gate_validation/detailed_reports/01_G0_G6_G7_strong_llm_detailed.md')

In [55]:
# =============================================================================
# 2. G1 — EXPANDED NATIVE SCALE COMPETENCE ACROSS ALL COMPOSITIONS
# =============================================================================

g1_rows = []

for base in drift_base:
    # All 8 known A/B combinations.
    for comp in CORE_COMPOSITIONS:
        sem = native_semantics(base, comp, "SCALE", BASE_GRAPH)
        g1_rows.append({
            "base_id": base["base_id"],
            "split": "known_ab",
            "composition_id": comp["composition_id"],
            "replacement_level": comp["replacement_level"],
            "semantic_success": int(sem == global_semantics(base)),
            "task_success": downstream_task_success(base, sem),
        })

    for split, comps in [
        ("c1_natural", C1_COMPOSITIONS),
        ("c2_schema", C2_COMPOSITIONS),
    ]:
        for comp in comps:
            sem = native_semantics(base, comp, "SCALE", BASE_GRAPH)
            g1_rows.append({
                "base_id": base["base_id"],
                "split": split,
                "composition_id": comp["composition_id"],
                "replacement_level": comp["replacement_level"],
                "semantic_success": int(sem == global_semantics(base)),
                "task_success": downstream_task_success(base, sem),
            })

g1_expanded_df = pd.DataFrame(g1_rows)
g1_expanded_summary = (
    g1_expanded_df.groupby(
        ["split", "replacement_level"], as_index=False
    )
    .agg(
        n_bases=("base_id", "nunique"),
        n_records=("task_success", "size"),
        semantic_success=("semantic_success", "mean"),
        task_success=("task_success", "mean"),
    )
)

print("\n[G1+] EXPANDED NATIVE SCALE COMPETENCE")
display(g1_expanded_summary.round(4))

g1_expanded_df.to_csv(EXT_DIR / "g1_expanded_native_records.csv", index=False)
g1_expanded_summary.to_csv(
    EXT_DIR / "g1_expanded_native_summary.csv", index=False
)



[G1+] EXPANDED NATIVE SCALE COMPETENCE


,split,replacement_level,n_bases,n_records,semantic_success,task_success
0,c1_natural,1,96,288,0.9375,0.9375
1,c1_natural,3,96,96,0.8125,0.8125
2,c2_schema,1,96,288,0.8889,0.8889
3,c2_schema,3,96,96,0.6667,0.6667
4,known_ab,0,96,96,1.0000,1.0000
5,known_ab,1,96,288,1.0000,1.0000
6,known_ab,2,96,288,1.0000,1.0000
7,known_ab,3,96,96,1.0000,1.0000


In [ ]:
# =============================================================================
# DETAILED RESULT REPORT — G1
# =============================================================================

parts = ["## G1 — Expanded native SCALE competence\n"]

if isinstance(g1_expanded_summary, pd.DataFrame) and len(g1_expanded_summary):
    parts.append(_df_md(g1_expanded_summary, max_rows=100))

    split_summary = (
        g1_expanded_df.groupby("split", as_index=False)
        .agg(
            n_bases=("base_id", "nunique"),
            semantic_success=("semantic_success", "mean"),
            task_success=("task_success", "mean"),
        )
    )
    parts.append("\n### Aggregate by split")
    parts.append(_df_md(split_summary))

    worst_sem = float(split_summary["semantic_success"].min())
    worst_task = float(split_summary["task_success"].min())
    parts.append(
        f"- Worst split semantic success = **{_fmt(worst_sem)}**; "
        f"worst split task success = **{_fmt(worst_task)}**."
    )

    if min(worst_sem, worst_task) >= 0.90:
        parts.append(
            "- The expanded held-out pool strongly supports native SCALE competence across independent replacement conditions."
        )
    elif min(worst_sem, worst_task) >= 0.80:
        parts.append(
            "- Native competence remains reasonably strong but is not uniformly near-ceiling; describe G1 as supported with boundary conditions."
        )
    else:
        parts.append(
            "- Expanded evaluation materially qualifies native competence. The main text should report the weak split explicitly."
        )

    parts.append(
        "- Replacement-level curves should be interpreted as compositional robustness, not proof of universal task superiority."
    )
else:
    parts.append("_No expanded G1 rows available._")

_save_and_show_report(
    "G1 — Detailed Expanded Native Competence",
    "\n".join(parts),
    "02_G1_expanded_native_detailed.md",
)


In [ ]:
# =============================================================================
# 3. G2 — FRESH 10-SEED FULL vs NOINV ROBUSTNESS AUDIT
# =============================================================================
#
# This does not change G2's original status. It asks whether the negative /
# regime-dependent conclusion survives a larger seed count and larger held-out set.
# =============================================================================

g2_ext_rows = []

if RUN_G2_10SEED:
    eval_items = []
    for base in drift_base:
        for agent in AGENTS:
            for impl, split in [
                ("B", "heldout_b"),
                ("C1", "c1_natural"),
                ("C2", "c2_schema"),
            ]:
                eval_items.append({
                    "base": base,
                    "agent": agent,
                    "impl": impl,
                    "split": split,
                    "message": realize_message(base, agent, impl, 0),
                    "gold": hop_contract(base, agent),
                })

    texts = [x["message"] for x in eval_items]

    for seed in G2_EXTENDED_SEEDS:
        print("G2 external seed", seed)

        full_model, _ = train_model(
            method="SCALE",
            seed=seed,
            label_masks=MAIN_LABEL_MASK,
            epochs=HEAD_EPOCHS,
            use_inv=True,
            use_logic=False,
            use_cal=True,
            graph_mode=True,
        )
        noinv_model, _ = train_model(
            method="SCALE-NoInv",
            seed=seed,
            label_masks=MAIN_LABEL_MASK,
            epochs=HEAD_EPOCHS,
            use_inv=False,
            use_logic=False,
            use_cal=True,
            graph_mode=True,
        )

        for label, model in [
            ("SCALE-Full", full_model),
            ("SCALE-NoInv", noinv_model),
        ]:
            preds = single_predict(model, texts, BASE_GRAPH)

            for item, pobj in zip(eval_items, preds):
                c = ontology_repair(pobj["contract"])
                g2_ext_rows.append({
                    "seed": seed,
                    "method": label,
                    "split": item["split"],
                    "base_id": item["base"]["base_id"],
                    "agent": item["agent"],
                    "active_correct": active_correct(
                        item["gold"], c, item["agent"]
                    ),
                    "exact_contract": exact_contract(item["gold"], c),
                    "confidence": pobj["confidence"],
                })

        del full_model, noinv_model
        cleanup_gpu()

g2_ext_df = pd.DataFrame(g2_ext_rows)

g2_seed_summary = pd.DataFrame()
g2_paired_summary = pd.DataFrame()

if len(g2_ext_df):
    g2_seed_summary = (
        g2_ext_df.groupby(
            ["seed", "method", "split"], as_index=False
        )
        .agg(
            active_accuracy=("active_correct", "mean"),
            exact_accuracy=("exact_contract", "mean"),
        )
    )

    paired_rows = []
    for split in ["heldout_b", "c1_natural", "c2_schema"]:
        w = (
            g2_seed_summary[g2_seed_summary["split"] == split]
            .pivot(index="seed", columns="method", values="active_accuracy")
            .dropna()
        )
        if {"SCALE-Full", "SCALE-NoInv"}.issubset(w.columns):
            delta = (
                w["SCALE-Full"] - w["SCALE-NoInv"]
            ).to_numpy(float)

            rng = np.random.default_rng(3111)
            boots = []
            for _ in range(10000):
                idx = rng.integers(0, len(delta), len(delta))
                boots.append(float(delta[idx].mean()))

            try:
                wp = float(
                    wilcoxon(
                        w["SCALE-Full"],
                        w["SCALE-NoInv"],
                        zero_method="wilcox",
                        alternative="two-sided",
                    ).pvalue
                )
            except Exception:
                wp = 1.0

            paired_rows.append({
                "split": split,
                "n_seeds": len(delta),
                "mean_delta_full_minus_noinv": float(delta.mean()),
                "bootstrap_low": float(np.quantile(boots, .025)),
                "bootstrap_high": float(np.quantile(boots, .975)),
                "positive_seeds": int((delta > 0).sum()),
                "negative_seeds": int((delta < 0).sum()),
                "ties": int((delta == 0).sum()),
                "wilcoxon_p": wp,
            })

    g2_paired_summary = pd.DataFrame(paired_rows)

print("\n[G2+] 10-SEED SUMMARY")
display(g2_seed_summary.round(4))
print("\n[G2+] PAIRED FULL-NOINV")
display(g2_paired_summary.round(4))

g2_ext_df.to_csv(EXT_DIR / "g2_10seed_records.csv", index=False)
g2_seed_summary.to_csv(EXT_DIR / "g2_10seed_summary.csv", index=False)
g2_paired_summary.to_csv(EXT_DIR / "g2_10seed_paired.csv", index=False)


In [58]:
# =============================================================================
# DETAILED RESULT REPORT — G2
# =============================================================================

parts = ["## G2 — Fresh 10-seed Full vs NoInv robustness audit\n"]

if isinstance(g2_paired_summary, pd.DataFrame) and len(g2_paired_summary):
    parts.append(_df_md(g2_paired_summary, max_rows=30))

    for r in g2_paired_summary.itertuples():
        parts.append(
            f"- **{r.split}**: mean Full−NoInv delta={_fmt(r.mean_delta_full_minus_noinv,4)}, "
            f"bootstrap CI=[{_fmt(r.bootstrap_low,4)}, {_fmt(r.bootstrap_high,4)}], "
            f"positive/tie/negative seeds={r.positive_seeds}/{r.ties}/{r.negative_seeds}, "
            f"Wilcoxon p={_fmt(r.wilcoxon_p,4)}."
        )
        parts.append(
            "  - " + _claim_language(
                r.mean_delta_full_minus_noinv,
                r.bootstrap_low,
                r.bootstrap_high,
            )
        )

    parts.append(
        "- **Original G2 remains FAIL regardless of this post-hoc audit.** The purpose is to test whether the earlier sign-changing conclusion persists with more seeds."
    )
    parts.append(
        "- If NoInv is noninferior or superior across the larger audit, simplify the final architecture narrative and demote alignment to a historical/checkpoint-specific regularizer."
    )
else:
    parts.append("_G2 10-seed audit was not run or produced no rows._")

_save_and_show_report(
    "G2 — Detailed 10-Seed Alignment Audit",
    "\n".join(parts),
    "03_G2_10seed_detailed.md",
)


# G2 — Detailed 10-Seed Alignment Audit

## G2 — Fresh 10-seed Full vs NoInv robustness audit

| split      |   n_seeds |   mean_delta_full_minus_noinv |   bootstrap_low |   bootstrap_high |   positive_seeds |   negative_seeds |   ties |   wilcoxon_p |
|:-----------|----------:|------------------------------:|----------------:|-----------------:|-----------------:|-----------------:|-------:|-------------:|
| heldout_b  |        10 |                        0.0656 |          0.0347 |           0.099  |               10 |                0 |      0 |       0.002  |
| c1_natural |        10 |                       -0.008  |         -0.0365 |           0.0139 |                1 |                2 |      7 |       0.75   |
| c2_schema  |        10 |                       -0.0156 |         -0.0823 |           0.0549 |                3 |                5 |      2 |       0.4375 |
- **heldout_b**: mean Full−NoInv delta=0.0656, bootstrap CI=[0.0347, 0.0990], positive/tie/negative seeds=10/0/0, Wilcoxon p=0.0020.
  - Paired uncertainty supports a positive effect; superiority language can be used cautiously.
- **c1_natural**: mean Full−NoInv delta=-0.0080, bootstrap CI=[-0.0365, 0.0139], positive/tie/negative seeds=1/7/2, Wilcoxon p=0.7500.
  - The interval includes zero; use 'competitive', 'matches', or 'numerically higher/lower', not established superiority.
- **c2_schema**: mean Full−NoInv delta=-0.0156, bootstrap CI=[-0.0823, 0.0549], positive/tie/negative seeds=3/2/5, Wilcoxon p=0.4375.
  - The interval includes zero; use 'competitive', 'matches', or 'numerically higher/lower', not established superiority.
- **Original G2 remains FAIL regardless of this post-hoc audit.** The purpose is to test whether the earlier sign-changing conclusion persists with more seeds.
- If NoInv is noninferior or superior across the larger audit, simplify the final architecture narrative and demote alignment to a historical/checkpoint-specific regularizer.


Detailed notebook report saved: /content/scale-iclr-full-master-from-scratch-v3.1.0-2171b05ab5d5/external_gate_validation/detailed_reports/03_G2_10seed_detailed.md


PosixPath('/content/scale-iclr-full-master-from-scratch-v3.1.0-2171b05ab5d5/external_gate_validation/detailed_reports/03_G2_10seed_detailed.md')

In [ ]:
# =============================================================================
# 4. G3 — SECOND EXTERNAL ONTOLOGY + REAL LONGITUDINAL ONTOLOGY EVOLUTION
# =============================================================================
#
# EDAM is evaluated in two ways:
#   A. static structural transfer on human-curated concepts;
#   B. real version evolution from EDAM 1.24 -> 1.25.
#
# Query text uses definitions/comments where available.
# Candidate grounding text uses labels/synonyms, reducing exact self-text leakage.
# =============================================================================

EDAM_URLS = {
    "1.24": "https://raw.githubusercontent.com/edamontology/edamontology/1.24/EDAM.owl",
    "1.25": "https://raw.githubusercontent.com/edamontology/edamontology/1.25/EDAM.owl",
}

def fetch_bytes(url, timeout=90):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.content

def load_owl_from_url(url):
    data = fetch_bytes(url)
    g = Graph()
    g.parse(data=data, format="xml")
    return g

def literal_values(g, uri, contains=None):
    vals = []
    for p, o in g.predicate_objects(uri):
        if not isinstance(o, Literal):
            continue
        ptxt = str(p).lower()
        if contains is None or any(x in ptxt for x in contains):
            txt = str(o).strip()
            if txt:
                vals.append(txt)
    return vals

def node_label(g, uri):
    labs = [str(x).strip() for x in g.objects(uri, RDFS.label)]
    return labs[0] if labs else str(uri).split("/")[-1]

def node_definition(g, uri):
    vals = literal_values(
        g, uri,
        contains=["definition", "comment", "description"]
    )
    # Prefer richer natural-language definitions.
    vals = sorted(set(vals), key=lambda x: (-len(x), x))
    return vals[0] if vals else ""

def node_synonyms(g, uri):
    vals = literal_values(
        g, uri,
        contains=["synonym", "alternative", "altlabel"]
    )
    return sorted(set(vals))[:8]

def is_deprecated(g, uri):
    for o in g.objects(uri, OWL.deprecated):
        if str(o).lower() in {"true", "1"}:
            return True
    return False

def class_nodes(g):
    out = set()
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):
            out.add(s)
    # EDAM may also expose subclass-only classes.
    for s, o in g.subject_objects(RDFS.subClassOf):
        if isinstance(s, URIRef):
            out.add(s)
        if isinstance(o, URIRef):
            out.add(o)
    return sorted(out, key=str)

def parent_map(g):
    pm = {}
    for s, o in g.subject_objects(RDFS.subClassOf):
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            pm.setdefault(s, set()).add(o)
    return pm

def find_named_roots(g):
    target = {"topic", "operation", "data", "format"}
    roots = {}
    for u in class_nodes(g):
        lab = node_label(g, u).strip().lower()
        if lab in target:
            roots[lab] = u
    return roots

def ancestor_closure(uri, pm, limit=100):
    seen = set()
    frontier = [uri]
    for _ in range(limit):
        nxt = []
        for x in frontier:
            for p in pm.get(x, set()):
                if p not in seen:
                    seen.add(p)
                    nxt.append(p)
        if not nxt:
            break
        frontier = nxt
    return seen

def root_of(uri, pm, roots):
    anc = ancestor_closure(uri, pm)
    for name, root in roots.items():
        if uri == root or root in anc:
            return name
    return None

def edam_records(g):
    pm = parent_map(g)
    roots = find_named_roots(g)

    rows = []
    for u in class_nodes(g):
        if is_deprecated(g, u):
            continue
        root = root_of(u, pm, roots)
        if root is None or u in roots.values():
            continue

        label = node_label(g, u)
        definition = node_definition(g, u)
        synonyms = node_synonyms(g, u)

        if len(definition.split()) < 5:
            continue

        candidate_text = " ; ".join(
            [label] + synonyms
        )[:1200]
        query_text = definition[:1800]

        rows.append({
            "uri": str(u),
            "label": label,
            "definition": definition,
            "candidate_text": candidate_text,
            "query_text": query_text,
            "root": root,
            "parents": "|".join(sorted(str(x) for x in pm.get(u, set()))),
        })
    return pd.DataFrame(rows), pm, roots

edam_static_df = pd.DataFrame()
edam_static_summary = pd.DataFrame()
edam_longitudinal_df = pd.DataFrame()
edam_longitudinal_summary = pd.DataFrame()

if RUN_EDAM_EXTERNAL:
    print("Loading EDAM 1.24 and 1.25...")
    edam24 = load_owl_from_url(EDAM_URLS["1.24"])
    edam25 = load_owl_from_url(EDAM_URLS["1.25"])

    rec24, pm24, roots24 = edam_records(edam24)
    rec25, pm25, roots25 = edam_records(edam25)

    # Balanced, deterministic static sample from EDAM 1.25.
    static_parts = []
    per_root = 100
    for root, g in rec25.groupby("root"):
        x = g.copy()
        x["hash"] = x["uri"].map(
            lambda s: hashlib.sha256(s.encode()).hexdigest()
        )
        static_parts.append(
            x.sort_values("hash").head(per_root).drop(columns="hash")
        )
    static = pd.concat(static_parts, ignore_index=True)

    candidate_emb = semantic_encode(
        static["candidate_text"].tolist(),
        batch_size=64,
    )
    query_emb = semantic_encode(
        static["query_text"].tolist(),
        batch_size=64,
    )

    # Root prototypes use root labels + available definitions.
    root_names = sorted(static["root"].unique())
    root_texts = []
    for name in root_names:
        ru = roots25[name]
        root_texts.append(
            node_label(edam25, ru) + ". " + node_definition(edam25, ru)
        )
    root_emb = semantic_encode(root_texts)

    # Direct text -> root.
    direct_idx = np.argmax(query_emb @ root_emb.T, axis=1)
    direct_root = [root_names[int(i)] for i in direct_idx]

    # Definition -> label/synonym leaf -> graph ancestry.
    leaf_idx = np.argmax(query_emb @ candidate_emb.T, axis=1)
    pred_leaf_root = static.iloc[leaf_idx]["root"].tolist()
    pred_leaf_uri = static.iloc[leaf_idx]["uri"].tolist()

    # Structural negative control: fixed root-label permutation.
    rng = np.random.default_rng(3250)
    root_arr = static["root"].to_numpy(object)
    permuted_roots = rng.permutation(root_arr)
    uri_to_perm_root = dict(zip(static["uri"], permuted_roots))
    shuffled_root = [
        uri_to_perm_root[u] for u in pred_leaf_uri
    ]

    edam_static_df = static.copy()
    edam_static_df["pred_leaf_uri"] = pred_leaf_uri
    edam_static_df["leaf_correct"] = (
        edam_static_df["pred_leaf_uri"] == edam_static_df["uri"]
    ).astype(int)
    edam_static_df["direct_root_pred"] = direct_root
    edam_static_df["direct_root_correct"] = (
        edam_static_df["direct_root_pred"] == edam_static_df["root"]
    ).astype(int)
    edam_static_df["graph_root_pred"] = pred_leaf_root
    edam_static_df["graph_root_correct"] = (
        edam_static_df["graph_root_pred"] == edam_static_df["root"]
    ).astype(int)
    edam_static_df["shuffled_root_pred"] = shuffled_root
    edam_static_df["shuffled_root_correct"] = (
        edam_static_df["shuffled_root_pred"] == edam_static_df["root"]
    ).astype(int)

    n10, n01, graph_vs_direct_p = _exact_mcnemar(
        edam_static_df["graph_root_correct"],
        edam_static_df["direct_root_correct"],
    )
    gn10, gn01, graph_vs_shuffle_p = _exact_mcnemar(
        edam_static_df["graph_root_correct"],
        edam_static_df["shuffled_root_correct"],
    )

    edam_static_summary = pd.DataFrame([{
        "n": len(edam_static_df),
        "roots": edam_static_df["root"].nunique(),
        "exact_leaf_accuracy": float(edam_static_df["leaf_correct"].mean()),
        "direct_root_accuracy": float(edam_static_df["direct_root_correct"].mean()),
        "graph_root_accuracy": float(edam_static_df["graph_root_correct"].mean()),
        "shuffled_root_accuracy": float(
            edam_static_df["shuffled_root_correct"].mean()
        ),
        "graph_vs_direct_mcnemar_p": graph_vs_direct_p,
        "graph_vs_shuffled_mcnemar_p": graph_vs_shuffle_p,
    }])

    # -------------------------------------------------------------------------
    # REAL longitudinal ontology evolution: EDAM 1.24 -> 1.25
    # -------------------------------------------------------------------------
    r24 = rec24.set_index("uri")
    r25 = rec25.set_index("uri")

    common = sorted(set(r24.index) & set(r25.index))
    added = sorted(set(r25.index) - set(r24.index))

    longitudinal_rows = []

    # Existing concepts whose label/definition/parents changed.
    for uri in common:
        a = r24.loc[uri]
        b = r25.loc[uri]
        changed = (
            a["label"] != b["label"]
            or a["definition"] != b["definition"]
            or a["parents"] != b["parents"]
        )
        if not changed:
            continue
        longitudinal_rows.append({
            "change_type": "existing_changed",
            "uri": uri,
            "old_root": a["root"],
            "new_root": b["root"],
            "old_candidate_text": a["candidate_text"],
            "new_query_text": b["query_text"],
        })

    # Truly new concepts in 1.25.
    for uri in added:
        b = r25.loc[uri]
        longitudinal_rows.append({
            "change_type": "new_concept",
            "uri": uri,
            "old_root": None,
            "new_root": b["root"],
            "old_candidate_text": None,
            "new_query_text": b["query_text"],
        })

    long_df = pd.DataFrame(longitudinal_rows)

    # Old ontology is the frozen consumer vocabulary.
    old_candidates = rec24.copy()
    old_candidate_emb = semantic_encode(
        old_candidates["candidate_text"].tolist(),
        batch_size=64,
    )
    old_root_names = sorted(old_candidates["root"].unique())
    old_root_texts = []
    for name in old_root_names:
        ru = roots24[name]
        old_root_texts.append(
            node_label(edam24, ru) + ". " + node_definition(edam24, ru)
        )
    old_root_emb = semantic_encode(old_root_texts)

    if len(long_df):
        q = semantic_encode(long_df["new_query_text"].tolist(), batch_size=64)

        li = np.argmax(q @ old_candidate_emb.T, axis=1)
        ground_old_root = old_candidates.iloc[li]["root"].tolist()
        ground_old_uri = old_candidates.iloc[li]["uri"].tolist()

        di = np.argmax(q @ old_root_emb.T, axis=1)
        direct_old_root = [old_root_names[int(i)] for i in di]

        long_df["grounded_old_uri"] = ground_old_uri
        long_df["ground_to_old_graph_root"] = ground_old_root
        long_df["direct_old_root"] = direct_old_root
        long_df["graph_root_retained"] = (
            long_df["ground_to_old_graph_root"] == long_df["new_root"]
        ).astype(int)
        long_df["direct_root_retained"] = (
            long_df["direct_old_root"] == long_df["new_root"]
        ).astype(int)

        edam_longitudinal_df = long_df
        edam_longitudinal_summary = (
            long_df.groupby("change_type", as_index=False)
            .agg(
                n=("uri", "size"),
                graph_root_retention=("graph_root_retained", "mean"),
                direct_root_retention=("direct_root_retained", "mean"),
            )
        )

print("\n[G3+] EDAM STATIC STRUCTURAL TRANSFER")
display(edam_static_summary.round(4))
print("\n[G3+] EDAM REAL VERSION EVOLUTION 1.24 -> 1.25")
display(edam_longitudinal_summary.round(4))

edam_static_df.to_csv(EXT_DIR / "g3_edam_static_records.csv", index=False)
edam_static_summary.to_csv(EXT_DIR / "g3_edam_static_summary.csv", index=False)
edam_longitudinal_df.to_csv(
    EXT_DIR / "g3_edam_longitudinal_records.csv", index=False
)
edam_longitudinal_summary.to_csv(
    EXT_DIR / "g3_edam_longitudinal_summary.csv", index=False
)


In [ ]:
# =============================================================================
# DETAILED RESULT REPORT — G3 / EDAM
# =============================================================================

parts = ["## G3 — Second external ontology and real ontology evolution\n"]

if isinstance(edam_static_summary, pd.DataFrame) and len(edam_static_summary):
    parts.append("### EDAM static structural transfer")
    parts.append(_df_md(edam_static_summary))

    r = edam_static_summary.iloc[0]
    graph = float(r["graph_root_accuracy"])
    direct = float(r["direct_root_accuracy"])
    shuffled = float(r["shuffled_root_accuracy"])
    leaf = float(r["exact_leaf_accuracy"])

    parts.append(
        f"- Exact leaf accuracy={_fmt(leaf)}; direct text→root={_fmt(direct)}; "
        f"ground→graph root={_fmt(graph)}; shuffled-structure root={_fmt(shuffled)}."
    )
    parts.append(
        f"- Structural gain over direct root prediction = **{_fmt(graph-direct)}**; "
        f"gain over shuffled structure = **{_fmt(graph-shuffled)}**."
    )

    if graph > direct and graph > shuffled:
        parts.append(
            "- This is strong external mechanism evidence because the correct human-curated hierarchy outperforms both direct root prediction and a structural negative control."
        )
    else:
        parts.append(
            "- EDAM does not provide a clean structural advantage under this protocol. Report it as qualified/null external evidence."
        )

if isinstance(edam_longitudinal_summary, pd.DataFrame) and len(edam_longitudinal_summary):
    parts.append("\n### EDAM 1.24 → 1.25 real version evolution")
    parts.append(_df_md(edam_longitudinal_summary))

    for r in edam_longitudinal_summary.itertuples():
        parts.append(
            f"- {r.change_type}: n={r.n}, graph-root retention={_fmt(r.graph_root_retention)}, "
            f"direct-root retention={_fmt(r.direct_root_retention)}."
        )

    weighted_graph = np.average(
        edam_longitudinal_summary["graph_root_retention"],
        weights=edam_longitudinal_summary["n"],
    )
    weighted_direct = np.average(
        edam_longitudinal_summary["direct_root_retention"],
        weights=edam_longitudinal_summary["n"],
    )
    parts.append(
        f"- Weighted real-version graph retention={_fmt(weighted_graph)} vs direct={_fmt(weighted_direct)}."
    )
    if weighted_graph > weighted_direct:
        parts.append(
            "- This materially strengthens the open-world/version-evolution thesis with a real ontology release history rather than a constructed synthetic shift."
        )
    else:
        parts.append(
            "- The real EDAM version-history result does not favor graph ancestry overall; preserve it as a boundary condition."
        )
else:
    parts.append(
        "\n_Real EDAM longitudinal evaluation produced no usable change rows or was not run._"
    )

_save_and_show_report(
    "G3 — Detailed EDAM Structural & Longitudinal Validation",
    "\n".join(parts),
    "04_G3_EDAM_detailed.md",
)


In [ ]:
# =============================================================================
# 5. G5 — COMBINATORIAL RUNTIME ENFORCEMENT + JSON-SCHEMA CONTROL
# =============================================================================

CONTRACT_JSON_SCHEMA = {
    "type": "object",
    "required": [
        "intent", "object", "action", "role",
        "authority", "provenance", "state",
    ],
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "object": {"type": "string", "enum": OBJECTS},
        "action": {"type": "string", "enum": ACTIONS},
        "role": {"type": "string", "enum": ROLES},
        "authority": {"type": "string", "enum": AUTHORITIES},
        "provenance": {"type": "string", "enum": PROVENANCE},
        "state": {"type": "string", "enum": STATES},
    },
    "additionalProperties": False,
}

def schema_accepts(c):
    try:
        jsonschema.validate(c, CONTRACT_JSON_SCHEMA)
        return 1
    except Exception:
        return 0

CORRUPTION_FAMILIES = [
    "authority_below_required",
    "role_mismatch",
    "state_mismatch",
    "provenance_corruption",
]

g5_rows = []

if RUN_G5_COMBINATORIAL:
    for base in drift_base:
        gold = final_contract(base)

        # Single, pair, and triple semantic corruptions.
        for k in [1, 2, 3]:
            for combo in itertools.combinations(CORRUPTION_FAMILIES, k):
                bad = dict(gold)
                for kind in combo:
                    bad = corrupt_contract(bad, kind)

                schema_ok = schema_accepts(bad)
                before = contract_violations(bad)
                repaired = ontology_repair(bad)
                after = contract_violations(repaired)

                # Provenance must not be fabricated; remaining violation => block.
                block = int(bool(after))

                g5_rows.append({
                    "base_id": base["base_id"],
                    "n_corruptions": k,
                    "corruptions": "|".join(combo),
                    "json_schema_accepts": schema_ok,
                    "semantic_violation_detected": int(bool(before)),
                    "n_before_violations": len(before),
                    "post_repair_semantic_valid": int(not bool(after)),
                    "block_required": block,
                    "n_after_violations": len(after),
                    "repaired_matches_original": int(repaired == gold),
                })

g5_combo_df = pd.DataFrame(g5_rows)
g5_combo_summary = pd.DataFrame()

if len(g5_combo_df):
    g5_combo_summary = (
        g5_combo_df.groupby("n_corruptions", as_index=False)
        .agg(
            n=("base_id", "size"),
            json_schema_acceptance=("json_schema_accepts", "mean"),
            semantic_detection=("semantic_violation_detected", "mean"),
            post_repair_valid=("post_repair_semantic_valid", "mean"),
            block_rate=("block_required", "mean"),
            exact_original_recovery=("repaired_matches_original", "mean"),
        )
    )

print("\n[G5+] JSON SCHEMA VS SEMANTIC ENFORCEMENT")
display(g5_combo_summary.round(4))

g5_combo_df.to_csv(EXT_DIR / "g5_combinatorial_records.csv", index=False)
g5_combo_summary.to_csv(EXT_DIR / "g5_combinatorial_summary.csv", index=False)


In [ ]:
# =============================================================================
# DETAILED RESULT REPORT — G5
# =============================================================================

parts = ["## G5 — Semantic enforcement versus JSON syntax\n"]

if isinstance(g5_combo_summary, pd.DataFrame) and len(g5_combo_summary):
    parts.append(_df_md(g5_combo_summary, max_rows=30))

    schema_accept = float(g5_combo_df["json_schema_accepts"].mean())
    semantic_detect = float(g5_combo_df["semantic_violation_detected"].mean())
    repair_valid = float(g5_combo_df["post_repair_semantic_valid"].mean())
    block = float(g5_combo_df["block_required"].mean())

    parts.append(
        f"- Overall JSON-schema acceptance of semantically corrupted contracts = **{_fmt(schema_accept)}**."
    )
    parts.append(
        f"- SCALE semantic-violation detection = **{_fmt(semantic_detect)}**; "
        f"post-repair semantic validity = **{_fmt(repair_valid)}**; "
        f"block-required rate = **{_fmt(block)}**."
    )

    if schema_accept >= 0.80 and semantic_detect >= 0.95:
        parts.append(
            "- This directly answers the 'typed adapter is just JSON schema' criticism: syntactically valid contracts can remain semantically invalid, while the semantic validator detects them."
        )
    else:
        parts.append(
            "- The JSON-vs-semantic separation is weaker than expected under this suite; avoid overstating semantic-enforcement superiority."
        )

    parts.append(
        "- Exact recovery should not be required for provenance cases that cannot be safely reconstructed; blocking is the correct safety outcome."
    )
else:
    parts.append("_No combinatorial G5 results available._")

_save_and_show_report(
    "G5 — Detailed Runtime Semantic-Enforcement Validation",
    "\n".join(parts),
    "05_G5_semantic_enforcement_detailed.md",
)


In [ ]:
# =============================================================================
# 6. EXPLORATORY REAL AGENT-PROTOCOL VERSION HISTORY: MCP + A2A
# =============================================================================
#
# This is deliberately NOT a new G6 gate. It creates a reproducible real-world
# version-history corpus from official changelogs and measures semantic magnitude
# of documented changes. It is external context for the paper's motivating problem.
# =============================================================================

PROTOCOL_SOURCES = {
    "MCP_2025_11_25": (
        "https://raw.githubusercontent.com/modelcontextprotocol/"
        "modelcontextprotocol/main/docs/specification/2025-11-25/changelog.mdx"
    ),
    "MCP_2026_07_28": (
        "https://raw.githubusercontent.com/modelcontextprotocol/"
        "modelcontextprotocol/main/docs/specification/2026-07-28/changelog.mdx"
    ),
    "A2A_CHANGELOG": (
        "https://raw.githubusercontent.com/a2aproject/A2A/main/CHANGELOG.md"
    ),
}

SEMANTIC_CATEGORIES = {
    "authority_security": [
        "authoriz", "permission", "scope", "credential", "security",
        "authentication", "consent",
    ],
    "state_session": [
        "session", "state", "taskstatus", "status", "initialize",
        "notification", "handle",
    ],
    "tool_schema": [
        "tool", "schema", "parameter", "argument", "json", "enum",
        "input", "output",
    ],
    "transport_protocol": [
        "transport", "http", "header", "json-rpc", "grpc", "protocol version",
    ],
    "capability_discovery": [
        "capability", "discover", "list", "resource", "prompt",
    ],
}

def release_bullets(text):
    rows = []
    section = ""
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("#"):
            section = re.sub(r"^#+\s*", "", s)
            continue
        if re.match(r"^[-*]\s+", s) or re.match(r"^\d+\.\s+", s):
            body = re.sub(r"^(?:[-*]|\d+\.)\s+", "", s).strip()
            if len(body) >= 15:
                rows.append((section, body))
    return rows

protocol_rows = []

if RUN_PROTOCOL_HISTORY:
    for source, url in PROTOCOL_SOURCES.items():
        try:
            text = requests.get(url, timeout=60).text
            if len(text) < 100:
                continue

            for i, (section, bullet) in enumerate(release_bullets(text)):
                low = (section + " " + bullet).lower()
                categories = [
                    cat for cat, kws in SEMANTIC_CATEGORIES.items()
                    if any(k in low for k in kws)
                ]
                official_breaking = int(
                    "breaking" in section.lower()
                    or "major change" in section.lower()
                    or "major changes" in section.lower()
                )

                protocol_rows.append({
                    "source": source,
                    "url": url,
                    "event_id": f"{source}:{i:04d}",
                    "section": section,
                    "change_text": bullet,
                    "official_major_or_breaking": official_breaking,
                    "semantic_categories": "|".join(categories),
                    "touches_semantic_interface": int(bool(categories)),
                })
        except Exception as exc:
            print("Protocol history fetch failed:", source, type(exc).__name__)

protocol_history_df = pd.DataFrame(protocol_rows)
protocol_history_summary = pd.DataFrame()

if len(protocol_history_df):
    # Embedding magnitude relative to the source's centroid provides a descriptive
    # semantic-change magnitude only. It is not called Active-SCD.
    emb = semantic_encode(protocol_history_df["change_text"].tolist())
    mags = np.zeros(len(protocol_history_df), dtype=float)

    for source, idx in protocol_history_df.groupby("source").groups.items():
        ii = np.asarray(list(idx), int)
        centroid = emb[ii].mean(axis=0)
        centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
        mags[ii] = 1.0 - (emb[ii] @ centroid)

    protocol_history_df["semantic_change_magnitude"] = mags

    protocol_history_summary = (
        protocol_history_df.groupby("source", as_index=False)
        .agg(
            n_events=("event_id", "size"),
            major_or_breaking_rate=("official_major_or_breaking", "mean"),
            semantic_interface_event_rate=("touches_semantic_interface", "mean"),
            mean_semantic_change=("semantic_change_magnitude", "mean"),
        )
    )

print("\n[EXTERNAL] REAL MCP/A2A VERSION-HISTORY CORPUS")
display(protocol_history_summary.round(4))

protocol_history_df.to_csv(
    EXT_DIR / "real_protocol_version_history_events.csv", index=False
)
protocol_history_summary.to_csv(
    EXT_DIR / "real_protocol_version_history_summary.csv", index=False
)


In [ ]:
# =============================================================================
# DETAILED RESULT REPORT — REAL PROTOCOL VERSION HISTORY
# =============================================================================

parts = ["## External context — Real MCP/A2A version-history corpus\n"]

if isinstance(protocol_history_summary, pd.DataFrame) and len(protocol_history_summary):
    parts.append(_df_md(protocol_history_summary, max_rows=30))

    if isinstance(protocol_history_df, pd.DataFrame) and len(protocol_history_df):
        category_counts = (
            protocol_history_df.assign(
                category=protocol_history_df["semantic_categories"]
                .fillna("")
                .str.split("|")
            )
            .explode("category")
        )
        category_counts = category_counts[
            category_counts["category"].astype(str).str.len() > 0
        ]
        if len(category_counts):
            cc = (
                category_counts.groupby("category", as_index=False)
                .agg(n_events=("event_id", "nunique"))
                .sort_values("n_events", ascending=False)
            )
            parts.append("\n### Semantic change categories")
            parts.append(_df_md(cc, max_rows=30))

        major = protocol_history_df[
            protocol_history_df["official_major_or_breaking"] == 1
        ]
        sem = protocol_history_df[
            protocol_history_df["touches_semantic_interface"] == 1
        ]

        parts.append(
            f"- Extracted {len(protocol_history_df)} official changelog events; "
            f"{len(major)} are marked major/breaking by the source context and "
            f"{len(sem)} touch one of the predefined semantic-interface categories."
        )

        parts.append(
            "- This corpus supports the *motivation* that real agent protocols evolve in state, authorization, tool/schema, transport, and capability semantics."
        )
        parts.append(
            "- It is **not** Active-SCD validation because downstream task failures are not available for these changelog events."
        )
else:
    parts.append("_No protocol-history rows available._")

_save_and_show_report(
    "External Protocol Evolution — Detailed MCP/A2A Report",
    "\n".join(parts),
    "06_protocol_history_detailed.md",
)


In [ ]:
# =============================================================================
# 7. FINAL G0--G7 EXTERNAL VALIDATION LEDGER
# =============================================================================

ledger = []

# G0
if len(strong_reader_summary):
    typed = strong_reader_summary[
        strong_reader_summary["interface"] == "SCALE_TYPED"
    ]
    nl = strong_reader_summary[
        strong_reader_summary["interface"] == "NL"
    ]
    ledger.append({
        "gate": "G0",
        "original_status": "FAIL",
        "external_question": "Do stronger heterogeneous readers consume the interface robustly?",
        "external_evidence": (
            f"typed mean semantic={typed['semantic_success'].mean():.3f}; "
            f"NL mean semantic={nl['semantic_success'].mean():.3f}"
        ),
        "external_interpretation": (
            "DIAGNOSTIC ONLY — original G0 is not relabeled. "
            "Use to localize reader-vs-interface failure."
        ),
    })
else:
    ledger.append({
        "gate": "G0",
        "original_status": "FAIL",
        "external_question": "Strong heterogeneous readers",
        "external_evidence": "not run / unavailable",
        "external_interpretation": "INCONCLUSIVE",
    })

# G1
g1_sem = float(g1_expanded_df["semantic_success"].mean())
g1_task = float(g1_expanded_df["task_success"].mean())
ledger.append({
    "gate": "G1",
    "original_status": "PASS",
    "external_question": "Does native SCALE remain competent on the 96-base held-out pool?",
    "external_evidence": f"semantic={g1_sem:.3f}; task={g1_task:.3f}",
    "external_interpretation": (
        "SUPPORTS G1" if min(g1_sem, g1_task) >= .90
        else "QUALIFIES G1"
    ),
})

# G2
if len(g2_paired_summary):
    c1row = g2_paired_summary[g2_paired_summary["split"] == "c1_natural"]
    ev = c1row.iloc[0].to_dict() if len(c1row) else {}
    ledger.append({
        "gate": "G2",
        "original_status": "FAIL",
        "external_question": "Does alignment show stable benefit across 10 fresh seeds?",
        "external_evidence": (
            f"C1 mean delta={ev.get('mean_delta_full_minus_noinv', np.nan):.4f}, "
            f"CI=[{ev.get('bootstrap_low', np.nan):.4f},"
            f"{ev.get('bootstrap_high', np.nan):.4f}]"
        ),
        "external_interpretation": (
            "DO NOT RELABEL ORIGINAL G2; use as robustness diagnosis."
        ),
    })

# G3
if len(edam_static_summary):
    r = edam_static_summary.iloc[0]
    long_txt = "no longitudinal rows"
    if len(edam_longitudinal_summary):
        long_txt = "; ".join(
            f"{x.change_type}:graph={x.graph_root_retention:.3f},direct={x.direct_root_retention:.3f}"
            for x in edam_longitudinal_summary.itertuples()
        )
    ledger.append({
        "gate": "G3",
        "original_status": "PASS",
        "external_question": "Does ontology structure transfer to a second human ontology and real releases?",
        "external_evidence": (
            f"EDAM graph root={r['graph_root_accuracy']:.3f}, "
            f"direct={r['direct_root_accuracy']:.3f}, "
            f"shuffled={r['shuffled_root_accuracy']:.3f}; {long_txt}"
        ),
        "external_interpretation": "STRONG EXTERNAL SUPPORT" if (
            r["graph_root_accuracy"] > r["direct_root_accuracy"]
            and r["graph_root_accuracy"] > r["shuffled_root_accuracy"]
        ) else "QUALIFIED EXTERNAL SUPPORT",
    })

# G4
if "expanded_pairwise" in globals() and len(expanded_pairwise):
    ledger.append({
        "gate": "G4",
        "original_status": "PASS",
        "external_question": "Competitive utility under expanded held-out and dev-selected fairness",
        "external_evidence": "See expanded_heldout_pairwise + dev_selected_alpha_summary.",
        "external_interpretation": (
            "Use 'superior' only where clustered paired CI excludes zero; otherwise competitive."
        ),
    })

# G5
if len(g5_combo_summary):
    r = g5_combo_summary
    ledger.append({
        "gate": "G5",
        "original_status": "PASS",
        "external_question": "Does semantic enforcement add value beyond syntactic JSON validity?",
        "external_evidence": (
            f"JSON schema acceptance={r['json_schema_acceptance'].mean():.3f}; "
            f"semantic detection={r['semantic_detection'].mean():.3f}; "
            f"post-repair valid={r['post_repair_valid'].mean():.3f}"
        ),
        "external_interpretation": "SUPPORTS SEMANTIC-OVER-SYNTACTIC ENFORCEMENT",
    })

# G6
if len(generative_executor_summary):
    passed = generative_executor_summary[
        generative_executor_summary["competence_gate_pass"] == 1
    ]
    if len(passed):
        best = passed.sort_values(
            "active_scd_failure_auc", ascending=False
        ).iloc[0]
        evidence = (
            f"{len(passed)} competent generative executor(s); "
            f"best Active-SCD failure AUROC={best['active_scd_failure_auc']:.3f}"
        )
        interp = "MATERIALLY STRENGTHENS G6 EXTERNAL VALIDITY"
    else:
        evidence = "No strong model passed the frozen clean competence gate."
        interp = "INCONCLUSIVE GENERATIVE EXTENSION; ORIGINAL G6 EVIDENCE REMAINS"
else:
    evidence = "Strong generative executor suite not run."
    interp = "INCONCLUSIVE"

ledger.append({
    "gate": "G6",
    "original_status": "PASS",
    "external_question": "Does Active-SCD predict failure of competent generative consumers?",
    "external_evidence": evidence,
    "external_interpretation": interp,
})

# G7
if len(strong_producer_summary):
    accepted = strong_producer_summary[
        strong_producer_summary["decoder"] == "SCALE-Full"
    ]
    ledger.append({
        "gate": "G7",
        "original_status": "PASS",
        "external_question": "Can SCALE recover semantics from stronger real LLM producers?",
        "external_evidence": (
            f"mean producer acceptance={accepted['acceptance_rate'].mean():.3f}; "
            f"accepted active accuracy={accepted['active_accuracy_accepted'].mean():.3f}"
        ),
        "external_interpretation": "STRONGER PRODUCER VALIDATION",
    })

external_gate_ledger = pd.DataFrame(ledger)
print("\nFINAL EXTERNAL GATE LEDGER")
display(external_gate_ledger)

external_gate_ledger.to_csv(
    EXT_DIR / "FINAL_EXTERNAL_GATE_LEDGER.csv", index=False
)

# ---------------------------------------------------------------------------
# Manuscript-facing evidence hierarchy
# ---------------------------------------------------------------------------

evidence_hierarchy = pd.DataFrame([
    {
        "claim": "Semantic interface formulation",
        "evidence_class": "CORE CONCEPTUAL",
        "preferred_evidence": "Formal role-active contract formulation + expanded native competence",
    },
    {
        "claim": "Closed-set compatibility",
        "evidence_class": "CORE BUT PERFORMANCE-QUALIFIED",
        "preferred_evidence": "Expanded held-out + dev-selected fair baseline + clustered paired CI",
    },
    {
        "claim": "Open-world structural canonicalization",
        "evidence_class": "CORE EMPIRICAL",
        "preferred_evidence": "Opaque paired test + Schema.org + EDAM + EDAM real 1.24->1.25 evolution",
    },
    {
        "claim": "Decision-relevant observability",
        "evidence_class": "PRIMARY EMPIRICAL",
        "preferred_evidence": "Conditional Active-SCD controls + independent classifiers + competent generative executor if gate passes",
    },
    {
        "claim": "Runtime semantic enforcement",
        "evidence_class": "SUPPORTING SYSTEMS EVIDENCE",
        "preferred_evidence": "JSON-Schema control + combinatorial semantic corruption",
    },
    {
        "claim": "Arbitrary reader robustness",
        "evidence_class": "NEGATIVE / QUALIFIED",
        "preferred_evidence": "Original G0 fail + strong-reader interface decomposition",
    },
    {
        "claim": "Directed alignment",
        "evidence_class": "NEGATIVE / NON-CORE",
        "preferred_evidence": "Original G2 fail + fresh 10-seed robustness audit",
    },
    {
        "claim": "External task superiority",
        "evidence_class": "NOT CLAIMED",
        "preferred_evidence": "CRMArena-Pro boundary result",
    },
])

evidence_hierarchy.to_csv(
    EXT_DIR / "FINAL_EVIDENCE_HIERARCHY.csv", index=False
)

# ---------------------------------------------------------------------------
# Final report
# ---------------------------------------------------------------------------

def _md(df, n=40):
    if df is None or len(df) == 0:
        return "_No result._"
    return df.head(n).to_markdown(index=False)

report = f"""
# SCALE — Final External Gate Validation Report

## Gate ledger

{_md(external_gate_ledger)}

## G0 stronger heterogeneous readers

{_md(strong_reader_summary)}

## G1 expanded native competence

{_md(g1_expanded_summary)}

## G2 fresh 10-seed alignment audit

{_md(g2_paired_summary)}

## G3 EDAM static structural transfer

{_md(edam_static_summary)}

## G3 real EDAM 1.24 -> 1.25 evolution

{_md(edam_longitudinal_summary)}

## G5 semantic enforcement vs JSON syntax

{_md(g5_combo_summary)}

## G6 competence-gated generative executors

{_md(generative_executor_summary)}

## G7 stronger producers

{_md(strong_producer_summary)}

## Real protocol version-history context

{_md(protocol_history_summary)}

## Scientific stop rule

This suite is the end of the synthetic/reviewer-hardening expansion.

After it is executed, additional tests should be added only if they introduce a
new independent source of evidence, not because an existing score is undesirable.

The paper must retain:
- original G0 FAIL if that was the prespecified result;
- original G2 FAIL if that was the prespecified result;
- any competent-generative-executor failure;
- any EDAM or protocol-history null result;
- any baseline-tuning result that eliminates a previous numerical advantage.

The strongest final paper should be organized around:
1. semantic interface drift as a systems-learning problem;
2. structural open-world canonicalization;
3. decision-relevant observability;
with routing, typed adapters, validation, and abstention as supporting engineering.
"""

(EXT_DIR / "FINAL_EXTERNAL_GATE_VALIDATION_REPORT.md").write_text(
    report.strip() + "\n",
    encoding="utf-8",
)
display(Markdown(report))

manifest = {
    "suite": "SCALE External Gate-by-Gate Validation",
    "original_gate_relabeling_allowed": False,
    "strong_models": [x["model_id"] for x in STRONG_LLM_SPECS],
    "generative_executor_gate": {
        "clean_task_accuracy": GEN_EXEC_CLEAN_TASK_GATE,
        "parse_validity": GEN_EXEC_PARSE_GATE,
    },
    "external_ontology": {
        "name": "EDAM",
        "versions": ["1.24", "1.25"],
        "urls": EDAM_URLS,
    },
    "real_protocol_history": list(PROTOCOL_SOURCES.keys()),
    "g2_extended_seeds": G2_EXTENDED_SEEDS,
}
(EXT_DIR / "external_validation_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\nFINAL EXTERNAL VALIDATION OUTPUTS")
for p in sorted(EXT_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
# =============================================================================
# FINAL DETAILED EXPERIMENTAL REPORT — NOTEBOOK EMBEDDED
# =============================================================================

parts = ["## Final gate-by-gate evidence ledger\n"]

if isinstance(external_gate_ledger, pd.DataFrame) and len(external_gate_ledger):
    parts.append(_df_md(external_gate_ledger, max_rows=30))

parts.append("\n## Final evidence hierarchy\n")
if isinstance(evidence_hierarchy, pd.DataFrame) and len(evidence_hierarchy):
    parts.append(_df_md(evidence_hierarchy, max_rows=30))

parts.append("\n## Submission-facing interpretation\n")
parts.append(
    "The original prespecified gates remain the authoritative gate outcomes. "
    "The external suite adds robustness, mechanism, and external-validity evidence without post-hoc relabeling."
)

# Dynamic, result-aware summary.
if isinstance(strong_reader_summary, pd.DataFrame) and len(strong_reader_summary):
    typed = strong_reader_summary[
        strong_reader_summary["interface"] == "SCALE_TYPED"
    ]
    if len(typed):
        parts.append(
            f"- Strong-reader typed SCALE mean semantic success: **{_fmt(typed['semantic_success'].mean())}**."
        )

if isinstance(g1_expanded_df, pd.DataFrame) and len(g1_expanded_df):
    parts.append(
        f"- Expanded G1 native semantic/task success: **{_fmt(g1_expanded_df['semantic_success'].mean())} / {_fmt(g1_expanded_df['task_success'].mean())}**."
    )

if isinstance(g2_paired_summary, pd.DataFrame) and len(g2_paired_summary):
    parts.append(
        "- G2 remains a negative/qualified contribution; the 10-seed audit should be cited to show whether the null/sign-changing result is robust."
    )

if isinstance(edam_static_summary, pd.DataFrame) and len(edam_static_summary):
    r = edam_static_summary.iloc[0]
    parts.append(
        f"- EDAM external graph/direct/shuffled root accuracy: **{_fmt(r['graph_root_accuracy'])} / {_fmt(r['direct_root_accuracy'])} / {_fmt(r['shuffled_root_accuracy'])}**."
    )

if isinstance(generative_executor_summary, pd.DataFrame) and len(generative_executor_summary):
    passed = generative_executor_summary[
        generative_executor_summary["competence_gate_pass"] == 1
    ]
    if len(passed):
        parts.append(
            f"- Competent generative executor count: **{len(passed)}**; mean Active-SCD failure AUROC among passing models: **{_fmt(passed['active_scd_failure_auc'].mean())}**."
        )
    else:
        parts.append(
            "- No strong generative executor passed the frozen gate; retain this as an explicit inconclusive limitation."
        )

if isinstance(g5_combo_df, pd.DataFrame) and len(g5_combo_df):
    parts.append(
        f"- JSON-schema acceptance of semantic corruptions: **{_fmt(g5_combo_df['json_schema_accepts'].mean())}** vs semantic detection **{_fmt(g5_combo_df['semantic_violation_detected'].mean())}**."
    )

parts.append("\n## Claim-writing rules for the manuscript\n")
parts.append(
    "1. **Closed-set compatibility:** say 'superior' only where a base-clustered paired 95% CI excludes zero; otherwise use 'competitive' or 'matches'.\n"
    "2. **Ontology:** core claim should rest on paired opaque-leaf evidence plus external human-curated ontology structure and real EDAM version evolution if positive.\n"
    "3. **Active-SCD:** strongest claim should combine conditional controls, independent learned executors, and competent generative executors only when the frozen competence gate is passed.\n"
    "4. **G0:** keep the original FAIL. Strong-reader results localize the failure but do not rewrite the gate.\n"
    "5. **G2:** keep the original FAIL and alignment outside the contribution list.\n"
    "6. **Runtime enforcement:** distinguish syntactic validation from semantic validation; blocking unverifiable provenance is a success mode, not failed repair.\n"
    "7. **External task gains:** do not claim universal task superiority; CRM remains a compatibility-versus-competence boundary test."
)

body = "\n".join(parts)

_save_and_show_report(
    "FINAL — Detailed Experimental & Claim Report",
    body,
    "99_FINAL_DETAILED_EXPERIMENTAL_REPORT.md",
)

# Also write a single merged report containing all earlier detailed sections.
merged = "\n\n---\n\n".join(SCALE_DETAILED_REPORT_SECTIONS)
merged_path = EXT_DIR / "FINAL_ALL_INNOTEBOOK_DETAILED_REPORTS.md"
merged_path.write_text(merged + "\n", encoding="utf-8")

print("\nMerged detailed report:", merged_path)
print("All major result interpretations are now embedded directly in the notebook.")


# SCALE — MASTER REVIEWER CLOSURE

This is the final integrated reviewer-closure layer.

It is intentionally designed around the major skeptical-reviewer objections rather
than around adding more favorable-looking scores.

## Reviewer issue → closure test

| Reviewer concern | Final closure analysis |
|---|---|
| C1/C2 has only 12 independent bases | 120-base × 12-surface-family frozen synthetic stress suite; inference clustered separately by base and surface family |
| Strong baseline may be equally good | development-selected fairness + paired difference + predeclared 3-point non-inferiority test |
| Typed adapter may be only JSON schema | strong-reader interface decomposition + combinatorial JSON-valid semantic corruptions |
| Active-SCD may merely encode authored corruption severity | within-severity/family controls + role-mask permutation + inactive-only control + held-out threshold/trajectory analysis + generative executor transfer |
| 11/11 correction may be tautological | message-level minimal semantic counterfactuals before decoding; target-slot selectivity and behavior sensitivity measured separately |
| Architecture is over-complex | 10-seed Full-vs-NoInv robustness + existing NoOnt/NoCal/Raw/Logic component audit |
| Table-12 explicit-ID numbers conflict with method | source-separated routing provenance + architecture-faithful deterministic-ID assertions + canonical result registry |
| External ontology evidence is too narrow | Schema.org + EDAM + real EDAM 1.24→1.25 evolution + W3C PROV-O/ODRL structural controls |
| No competent LLM downstream evidence | frozen-gate 7–8B generative executor suite from the previous external validation layer |
| Real evolution evidence is weak | real EDAM ontology release evolution + official MCP/A2A changelog corpus, clearly separated into validation vs motivation |
| Too many numbers / patch history | one canonical master registry and one manuscript-facing claim ledger |
| Statistical claims are too strong | all performance language is governed by paired uncertainty and predeclared non-inferiority rules |

## Scientific rules

- Original G0/G2 remain their original prespecified outcomes.
- Post-hoc analyses strengthen or qualify evidence; they never rewrite history.
- The statistical unit is the independent task/base and, where repeated surface
  families are used, a second surface-family uncertainty analysis is reported.
- Test-set hyperparameters are never selected from test performance.
- A positive claim must survive the relevant uncertainty analysis; otherwise the
  manuscript uses *competitive*, *matches*, *qualified*, or *inconclusive*.
- The final notebook produces its own detailed Markdown interpretation after every
  major block and a single final claim ledger at the end.


In [ ]:
# =============================================================================
# MASTER CLOSURE — CONFIGURATION / HELPERS
# =============================================================================

from pathlib import Path
import copy, json, math, hashlib, itertools, re
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, precision_score,
    recall_score, confusion_matrix
)
from IPython.display import display, Markdown

MASTER_DIR = ROOT / "master_reviewer_closure"
MASTER_DIR.mkdir(parents=True, exist_ok=True)
MASTER_REPORT_DIR = MASTER_DIR / "detailed_reports"
MASTER_REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_REPORT_SECTIONS = []

MASTER_EXPANDED_BASES = 120
MASTER_STYLE_FAMILIES = 12
NONINFERIORITY_MARGIN = 0.03
MASTER_SEED = 41717

def mr_fmt(x, d=4):
    try:
        x = float(x)
        if not np.isfinite(x):
            return "n/a"
        return f"{x:.{d}f}"
    except Exception:
        return str(x)

def mr_df(df, cols=None, n=80, digits=4):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return "_No rows available._"
    x = df.copy()
    if cols:
        cols = [c for c in cols if c in x.columns]
        x = x[cols]
    for c in x.select_dtypes(include=[np.number]).columns:
        x[c] = x[c].round(digits)
    return x.head(n).to_markdown(index=False)

def mr_report(title, body, filename):
    txt = f"# {title}\n\n{body.strip()}\n"
    (MASTER_REPORT_DIR / filename).write_text(txt, encoding="utf-8")
    MASTER_REPORT_SECTIONS.append(txt)
    display(Markdown(txt))

def mr_bootstrap_mean(values, n_boot=8000, seed=MASTER_SEED):
    x = np.asarray(values, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        vals.append(float(x[idx].mean()))
    return (
        float(x.mean()),
        float(np.quantile(vals, .025)),
        float(np.quantile(vals, .975)),
    )

def mr_paired_diff(
    df, method_a, method_b, metric="active_correct",
    cluster="base_id", n_boot=8000, seed=MASTER_SEED
):
    d = df[df["method"].isin([method_a, method_b])].copy()
    g = (
        d.groupby([cluster, "method"], as_index=False)[metric]
        .mean()
        .pivot(index=cluster, columns="method", values=metric)
        .dropna()
    )
    if not {method_a, method_b}.issubset(g.columns):
        return None
    delta = (g[method_a] - g[method_b]).to_numpy(float)
    point, lo, hi = mr_bootstrap_mean(delta, n_boot=n_boot, seed=seed)
    return {
        "cluster": cluster,
        "n_clusters": len(delta),
        "method_a": method_a,
        "method_b": method_b,
        "mean_difference": point,
        "ci_low": lo,
        "ci_high": hi,
        "noninferior_margin": NONINFERIORITY_MARGIN,
        "noninferior": int(np.isfinite(lo) and lo > -NONINFERIORITY_MARGIN),
        "superior": int(np.isfinite(lo) and lo > 0),
    }

def mr_exact_mcnemar(a, b):
    a = np.asarray(a, int)
    b = np.asarray(b, int)
    n10 = int(((a == 1) & (b == 0)).sum())
    n01 = int(((a == 0) & (b == 1)).sum())
    disc = n10 + n01
    p = 1.0 if disc == 0 else float(
        binomtest(min(n10, n01), n=disc, p=.5, alternative="two-sided").pvalue
    )
    return n10, n01, p

def mr_hash_style(base_id, split, n=MASTER_STYLE_FAMILIES):
    h = hashlib.sha256(f"{base_id}|{split}".encode()).hexdigest()
    return int(h[:8], 16) % n

def mr_safe_auc(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    ok = np.isfinite(s)
    y, s = y[ok], s[ok]
    if len(y) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))

print("MASTER REVIEWER CLOSURE initialized.")
print("Expanded bases:", MASTER_EXPANDED_BASES)
print("Surface families:", MASTER_STYLE_FAMILIES)
print("Non-inferiority margin:", NONINFERIORITY_MARGIN)


In [ ]:
# =============================================================================
# P0 — CANONICAL NUMERIC / ROUTING / REGISTRY AUDIT
# =============================================================================

audit_rows = []

# 1. Registry duplicate/conflict audit.
if "registry_df" in globals() and isinstance(registry_df, pd.DataFrame) and len(registry_df):
    key_col = "key" if "key" in registry_df.columns else None
    if key_col:
        for key, g in registry_df.groupby(key_col):
            vals = pd.to_numeric(g["value"], errors="coerce").dropna().unique()
            audit_rows.append({
                "audit_family": "registry",
                "item": key,
                "n_rows": len(g),
                "n_unique_values": len(vals),
                "conflict": int(len(vals) > 1),
                "values": "|".join(f"{x:.8f}" for x in vals),
            })

# 2. Routing-source separation.
if "routing_provenance_summary" in globals() and isinstance(routing_provenance_summary, pd.DataFrame):
    for r in routing_provenance_summary.itertuples():
        audit_rows.append({
            "audit_family": "routing_source",
            "item": f"{getattr(r,'implementation','unknown')}::{getattr(r,'kind','')}::{getattr(r,'surface','')}::{getattr(r,'route','')}",
            "n_rows": getattr(r, "n", np.nan),
            "n_unique_values": 1,
            "conflict": 0,
            "values": mr_fmt(getattr(r, "accuracy", np.nan), 6),
        })

# 3. Architecture-faithful explicit ID integrity.
id_integrity = pd.DataFrame()
if "route_v2_df" in globals() and isinstance(route_v2_df, pd.DataFrame) and len(route_v2_df):
    x = route_v2_df[
        (route_v2_df["kind"] == "unseen")
        & route_v2_df["surface"].isin(["exact_id","punctuated_id","wrapped_id"])
    ].copy()
    if len(x):
        id_integrity = (
            x.groupby("surface", as_index=False)
            .agg(
                n=("correct","size"),
                accuracy=("correct","mean"),
                symbolic_route_rate=("route", lambda s: float((s=="symbolic_id").mean()))
            )
        )

canonical_audit_df = pd.DataFrame(audit_rows)

print("\n[P0] Canonical registry/routing audit")
display(canonical_audit_df.head(200))
print("\nArchitecture-faithful explicit-ID integrity")
display(id_integrity)

canonical_audit_df.to_csv(MASTER_DIR / "P0_canonical_audit.csv", index=False)
id_integrity.to_csv(MASTER_DIR / "P0_explicit_id_integrity.csv", index=False)

conflicts = (
    canonical_audit_df[
        (canonical_audit_df["audit_family"] == "registry")
        & (canonical_audit_df["conflict"] == 1)
    ]
    if len(canonical_audit_df) else pd.DataFrame()
)

body = []
body.append("## What this audit checks")
body.append(
    "The final paper must have one numerical source of truth. This block flags duplicated registry keys with conflicting values and keeps legacy-vs-architecture-faithful routing implementations explicitly separated."
)
if len(conflicts):
    body.append(f"\n- **{len(conflicts)} registry key(s) still contain conflicting values.** These must not be copied into the manuscript until resolved.")
    body.append(mr_df(conflicts, n=100))
else:
    body.append("\n- No conflicting duplicate registry values were detected in the currently loaded canonical registry.")

if len(id_integrity):
    good = bool(
        np.allclose(id_integrity["accuracy"], 1.0)
        and np.allclose(id_integrity["symbolic_route_rate"], 1.0)
    )
    body.append(
        f"\n- Final explicit-ID architecture integrity: **{'PASS' if good else 'FAIL'}**."
    )
    body.append(mr_df(id_integrity))
    if good:
        body.append(
            "- Exact, punctuated, and wrapped explicit IDs are routed deterministically and recover their canonical root perfectly in the architecture-faithful implementation. Any older low SCALE ID scores must be labeled branch-conditioned/pre-routing or removed from the final end-to-end table."
        )

mr_report(
    "P0 — Canonical Numerical & Routing Consistency",
    "\n".join(body),
    "00_P0_canonical_consistency.md",
)


In [ ]:
# =============================================================================
# P1 — 120-BASE × 12-SURFACE-FAMILY FROZEN C1/C2 STRESS SUITE
# =============================================================================
#
# This expands statistical power and surface diversity.
# It remains synthetic and is never presented as external real-world evidence.
#
# Every base is evaluated under all 12 frozen surface families.
# We report uncertainty clustered separately by:
#   (a) independent base
#   (b) surface family
# A superiority claim requires BOTH clustered CIs to be strictly positive.
# Noninferiority uses the frozen 0.03 margin.
# =============================================================================

MASTER_OBJECT_PHRASES = {
    "PURCHASE": [
        "purchase request", "procurement case", "buying request",
        "acquisition workflow", "purchase authorization item",
        "procurement transaction"
    ],
    "ACCESS": [
        "privileged access request", "entitlement request", "access-control case",
        "privilege request", "identity-access workflow", "authorization request"
    ],
    "CHANGE": [
        "deployment modification", "system change request", "release modification",
        "change-control case", "deployment change", "configuration change"
    ],
    "INCIDENT": [
        "operational incident", "service incident", "incident-response case",
        "operational event", "response incident", "service disruption case"
    ],
    "COMPLIANCE_ITEM": [
        "assurance control item", "compliance-control case", "governance control item",
        "compliance review item", "assurance requirement", "control-assessment item"
    ],
}

MASTER_EVID_PHRASES = {
    "PASS": [
        "all checks are satisfied", "the evidence clears every required check",
        "review found no blocking issue", "the record satisfies the evidence criteria",
        "validation completed successfully", "evidence is sufficient"
    ],
    "FAIL": [
        "a blocking issue remains", "the evidence fails a mandatory check",
        "review found a disqualifying condition", "the record does not satisfy requirements",
        "validation failed", "evidence is insufficient"
    ],
    "UNCERTAIN": [
        "the evidence is unresolved", "review cannot establish a conclusive result",
        "the record remains ambiguous", "validation is inconclusive",
        "the evidence needs further verification", "the assessment is uncertain"
    ],
}

MASTER_AUTH_PHRASES = {
    "RECOMMEND": [
        "may recommend but cannot authorize",
        "holds recommendation authority only",
        "can propose a recommendation without approval power",
        "is permitted to recommend",
        "has advisory authority",
        "may issue a recommendation"
    ],
    "APPROVE": [
        "has approval authority",
        "is permitted to authorize approval",
        "holds approval permission",
        "may approve the operation",
        "has an approval mandate",
        "can authorize the decision"
    ],
    "EXECUTE": [
        "has execution authority",
        "is permitted to carry out the operation",
        "holds execution permission",
        "may execute the approved action",
        "has operational execution rights",
        "can perform the operation"
    ],
}

def master_phrase(mapping, key, style):
    vals = mapping[key]
    return vals[style % len(vals)]

def master_c1_message(base, agent, style):
    if agent == "PLANNER":
        obj = master_phrase(MASTER_OBJECT_PHRASES, base["object"], style)
        templates = [
            f"The planner identifies the workflow as a {obj}.",
            f"For downstream handling, interpret the target as a {obj}.",
            f"Planning assessment: the case concerns a {obj}.",
            f"The object that should govern the next stage is the {obj}.",
            f"Route the request according to the semantics of a {obj}.",
            f"From the planning perspective, this is a {obj}.",
            f"Operationally, the subject belongs to the {obj} category.",
            f"The planner's handoff classifies this item as a {obj}.",
            f"Treat the requested workflow object as {obj}.",
            f"Planning semantics indicate {obj} as the relevant entity.",
            f"The case being planned is best understood as a {obj}.",
            f"Semantic handoff from planning: target object = {obj}."
        ]
        return templates[style % len(templates)]

    if agent == "RETRIEVER":
        ev = master_phrase(MASTER_EVID_PHRASES, base["evidence_status"], style)
        templates = [
            f"Evidence review concludes that {ev}.",
            f"Retrieval assessment: {ev}.",
            f"The retrieved record indicates that {ev}.",
            f"Based on the available evidence, {ev}.",
            f"Evidence-state handoff: {ev}.",
            f"The retriever reports that {ev}.",
            f"Record verification finds that {ev}.",
            f"Evidence semantics should be read as follows: {ev}.",
            f"The supporting material shows that {ev}.",
            f"Review status from retrieval: {ev}.",
            f"The evidence package indicates that {ev}.",
            f"Downstream evidence interpretation: {ev}."
        ]
        return templates[style % len(templates)]

    auth = master_phrase(MASTER_AUTH_PHRASES, base["authority"], style)
    templates = [
        f"The policy role {auth}.",
        f"Governance assessment: the actor {auth}.",
        f"Authority semantics indicate that the actor {auth}.",
        f"For the downstream decision, the policy holder {auth}.",
        f"The delegated policy scope means the actor {auth}.",
        f"Policy handoff: the responsible role {auth}.",
        f"The current permission boundary says the actor {auth}.",
        f"Governance semantics: the policy actor {auth}.",
        f"The applicable authority level means the actor {auth}.",
        f"Decision-rights assessment: the actor {auth}.",
        f"The policy context establishes that the actor {auth}.",
        f"Downstream authority interpretation: the actor {auth}."
    ]
    return templates[style % len(templates)]

def master_c2_message(base, agent, style):
    if agent == "PLANNER":
        value = OBJ_C2[base["object"]]
        templates = [
            f'planner.v5={{"workflowClass":"{value}","next":"retrieval"}}',
            f'route::planner(workflow_category="{value}",next="lookup")',
            f'<planner workflow-category="{value}" next="retrieval"/>',
            f'planner_v5 | class={value} | handoff=retriever',
            f'{{"service":"planner","payload":{{"workflow":"{value}"}}}}',
            f'planner.schema.6/workflowType/{value}/next/retrieval',
            f'PLANNER[workflow={value};transition=retrieve]',
            f'plannerMessage(workflow="{value}", phase="handoff")',
            f'ns:planner:workflow-category={value};target=retriever',
            f'workflow_router_v6 -> category:{value} -> retrieval',
            f'planner_payload={{workflow_category:{value},version:6}}',
            f'event.planner.workflow_class="{value}"'
        ]
    elif agent == "RETRIEVER":
        value = EVID_C2[base["evidence_status"]]
        templates = [
            f'retriever.v5={{"evidenceAssessment":"{value}","next":"policy"}}',
            f'lookup::evidence(verdict="{value}",next="governance")',
            f'<retriever evidence-assessment="{value}" next="policy"/>',
            f'retriever_v5 | assessment={value} | handoff=policy',
            f'{{"service":"retriever","payload":{{"evidence":"{value}"}}}}',
            f'evidence.schema.6/assessment/{value}/next/governance',
            f'RETRIEVER[state={value};transition=policy]',
            f'evidenceMessage(assessment="{value}", phase="handoff")',
            f'ns:retriever:evidence-state={value};target=policy',
            f'evidence_engine_v6 -> assessment:{value} -> governance',
            f'retrieval_payload={{evidence_assessment:{value},version:6}}',
            f'event.retriever.evidence_assessment="{value}"'
        ]
    else:
        value = AUTH_C2[base["authority"]]
        templates = [
            f'policy.v5={{"permissionScope":"{value}","next":"execution"}}',
            f'governance::policy(scope="{value}",next="execute")',
            f'<policy permission-scope="{value}" next="execution"/>',
            f'policy_v5 | scope={value} | handoff=execution',
            f'{{"service":"policy","payload":{{"authority":"{value}"}}}}',
            f'policy.schema.6/permissionScope/{value}/next/execution',
            f'POLICY[authority={value};transition=execute]',
            f'policyMessage(scope="{value}", phase="handoff")',
            f'ns:policy:authority-scope={value};target=executor',
            f'governance_v6 -> permission:{value} -> execution',
            f'policy_payload={{permission_scope:{value},version:6}}',
            f'event.policy.permission_scope="{value}"'
        ]
    return templates[style % len(templates)]

master_review_bases = generate_tasks(
    MASTER_EXPANDED_BASES,
    MASTER_SEED,
    "master-review",
)

# Freeze dataset before model evaluation.
frozen_manifest_rows = []
for base in master_review_bases:
    for split in ["c1_natural", "c2_schema"]:
        for style in range(MASTER_STYLE_FAMILIES):
            for agent in AGENTS:
                msg = (
                    master_c1_message(base, agent, style)
                    if split == "c1_natural"
                    else master_c2_message(base, agent, style)
                )
                frozen_manifest_rows.append({
                    "base_id": base["base_id"],
                    "split": split,
                    "surface_family": style,
                    "agent": agent,
                    "message_sha256": hashlib.sha256(msg.encode()).hexdigest(),
                    "message": msg,
                })

master_surface_manifest = pd.DataFrame(frozen_manifest_rows)
master_surface_manifest.to_csv(
    MASTER_DIR / "P1_FROZEN_120x12_SURFACE_MANIFEST.csv",
    index=False,
)

assert not (
    set(master_surface_manifest["base_id"])
    & {str(x["base_id"]) for x in train_base + dev_base + shift_test_base + drift_base}
)

MASTER_METHODS = ["CB-BGE", "CB+DualView", "Proto-CB", "SCALE-Full"]
eval_rows = []

for split in ["c1_natural", "c2_schema"]:
    for style in range(MASTER_STYLE_FAMILIES):
        items = []
        for base in master_review_bases:
            for agent in AGENTS:
                msg = (
                    master_c1_message(base, agent, style)
                    if split == "c1_natural"
                    else master_c2_message(base, agent, style)
                )
                items.append((base, agent, msg))

        messages = [x[2] for x in items]

        for method in MASTER_METHODS:
            preds = ensemble_predict(
                "SCALE" if method == "SCALE-Full" else method,
                messages,
                BASE_GRAPH,
            )
            for (base, agent, msg), pobj in zip(items, preds):
                pred = pobj["contract"]
                if method == "SCALE-Full":
                    pred = ontology_repair(pred)
                gold = hop_contract(base, agent)
                eval_rows.append({
                    "base_id": base["base_id"],
                    "workflow": base["object"],
                    "split": split,
                    "surface_family": style,
                    "agent": agent,
                    "method": method,
                    "active_correct": active_correct(gold, pred, agent),
                    "exact_contract": exact_contract(gold, pred),
                    "confidence": pobj["confidence"],
                })

master_surface_df = pd.DataFrame(eval_rows)

master_surface_summary = (
    master_surface_df.groupby(["method","split"], as_index=False)
    .agg(
        n_bases=("base_id","nunique"),
        n_surface_families=("surface_family","nunique"),
        n_records=("active_correct","size"),
        active_accuracy=("active_correct","mean"),
        exact_contract_accuracy=("exact_contract","mean"),
    )
)

pairwise_rows = []
for split in ["c1_natural","c2_schema"]:
    d = master_surface_df[master_surface_df["split"] == split]

    for rival in ["CB+DualView","CB-BGE","Proto-CB"]:
        for cluster in ["base_id","surface_family"]:
            res = mr_paired_diff(
                d, "SCALE-Full", rival,
                metric="active_correct",
                cluster=cluster,
                n_boot=8000,
                seed=MASTER_SEED + len(pairwise_rows),
            )
            if res:
                res["split"] = split
                res["comparison"] = f"SCALE-Full - {rival}"
                pairwise_rows.append(res)

master_surface_pairwise = pd.DataFrame(pairwise_rows)

# Combined claim status requires both base and surface-family uncertainty.
claim_rows = []
for split in ["c1_natural","c2_schema"]:
    for rival in ["CB+DualView","CB-BGE","Proto-CB"]:
        q = master_surface_pairwise[
            (master_surface_pairwise["split"] == split)
            & (master_surface_pairwise["comparison"] == f"SCALE-Full - {rival}")
        ]
        if len(q) == 2:
            base_r = q[q["cluster"]=="base_id"].iloc[0]
            surf_r = q[q["cluster"]=="surface_family"].iloc[0]
            claim_rows.append({
                "split": split,
                "comparison": f"SCALE-Full - {rival}",
                "base_diff": base_r["mean_difference"],
                "base_ci_low": base_r["ci_low"],
                "base_ci_high": base_r["ci_high"],
                "surface_diff": surf_r["mean_difference"],
                "surface_ci_low": surf_r["ci_low"],
                "surface_ci_high": surf_r["ci_high"],
                "superiority_both": int(
                    base_r["ci_low"] > 0 and surf_r["ci_low"] > 0
                ),
                "noninferior_both_margin_003": int(
                    base_r["ci_low"] > -NONINFERIORITY_MARGIN
                    and surf_r["ci_low"] > -NONINFERIORITY_MARGIN
                ),
            })

master_surface_claims = pd.DataFrame(claim_rows)

# Per-style and per-workflow diagnostics.
master_surface_by_style = (
    master_surface_df.groupby(
        ["split","surface_family","method"], as_index=False
    )
    .agg(active_accuracy=("active_correct","mean"))
)
master_surface_by_workflow = (
    master_surface_df.groupby(
        ["split","workflow","method"], as_index=False
    )
    .agg(active_accuracy=("active_correct","mean"))
)

print("\n[P1] 120-base × 12-surface-family summary")
display(master_surface_summary.round(4))
print("\nPaired clustered inference")
display(master_surface_pairwise.round(4))
print("\nClaim-level combined base+surface inference")
display(master_surface_claims.round(4))

master_surface_df.to_csv(MASTER_DIR / "P1_surface_stress_records.csv", index=False)
master_surface_summary.to_csv(MASTER_DIR / "P1_surface_stress_summary.csv", index=False)
master_surface_pairwise.to_csv(MASTER_DIR / "P1_surface_stress_pairwise.csv", index=False)
master_surface_claims.to_csv(MASTER_DIR / "P1_surface_stress_claims.csv", index=False)
master_surface_by_style.to_csv(MASTER_DIR / "P1_surface_stress_by_style.csv", index=False)
master_surface_by_workflow.to_csv(MASTER_DIR / "P1_surface_stress_by_workflow.csv", index=False)

body = []
body.append(
    f"This frozen synthetic stress suite contains **{MASTER_EXPANDED_BASES} independent bases**, "
    f"**{MASTER_STYLE_FAMILIES} independently specified surface families**, two shift regimes, and three role handoffs."
)
body.append(
    "It addresses statistical power and surface diversity, but it remains synthetic and therefore does not replace the EDAM/Schema.org/CRM/generative-executor external-validity evidence."
)
body.append("\n### Aggregate results")
body.append(mr_df(master_surface_summary))
body.append("\n### Paired uncertainty")
body.append(mr_df(master_surface_claims))

for r in master_surface_claims.itertuples():
    if r.superiority_both:
        language = "Both base-clustered and surface-family-clustered intervals support a positive effect."
    elif r.noninferior_both_margin_003:
        language = "Superiority is not established, but the result satisfies the predeclared 3-point non-inferiority criterion under both clustering views."
    else:
        language = "Neither robust superiority nor the predeclared non-inferiority criterion is established under both clustering views."
    body.append(f"- **{r.split}, {r.comparison}:** {language}")

body.append(
    "\n**Manuscript rule:** closed-set performance is never the central novelty. "
    "Use *superior* only where both clustering analyses support it; otherwise use "
    "*noninferior/competitive/matches* as warranted."
)

mr_report(
    "P1 — Large Frozen C1/C2 Surface-Stress Measurement",
    "\n".join(body),
    "01_P1_large_surface_stress.md",
)


In [ ]:
# =============================================================================
# P2 — MESSAGE-LEVEL MINIMAL SEMANTIC COUNTERFACTUALS
# =============================================================================
#
# Stronger than the prior "correct the decoded active slot" intervention:
# intervention happens in the incoming message BEFORE SCALE decoding.
#
# We separately measure:
# - target semantic slot reaction,
# - non-target semantic stability,
# - whether the downstream policy changes when the gold semantic intervention
#   is expected to change behavior.
#
# This is still a controlled behavioral sensitivity test, not universal causality.
# =============================================================================

OBJ_ORDER = list(PASS_ACTION_BY_OBJECT.keys())
EVID_ORDER = ["PASS","UNCERTAIN","FAIL"]
AUTH_ORDER = ["RECOMMEND","APPROVE","EXECUTE"]

def mr_next(vals, current):
    i = vals.index(current)
    return vals[(i+1) % len(vals)]

def mr_alt_base(base, agent):
    b = copy.deepcopy(base)
    if agent == "PLANNER":
        b["object"] = mr_next(OBJ_ORDER, base["object"])
    elif agent == "RETRIEVER":
        b["evidence_status"] = mr_next(EVID_ORDER, base["evidence_status"])
        # keep a concise semantically aligned evidence note
        b["evidence"] = EVIDENCE_TEXT[b["evidence_status"]][0]
    elif agent == "POLICY":
        b["authority"] = mr_next(AUTH_ORDER, base["authority"])
    else:
        raise ValueError(agent)
    return b

intervention_rows = []

for split, msg_fn in [
    ("c1_natural", master_c1_message),
    ("c2_schema", master_c2_message),
]:
    for base in drift_base:
        style = mr_hash_style(base["base_id"], split)
        # Decode all original role messages.
        original_contracts = {}
        for agent in AGENTS:
            msg = msg_fn(base, agent, style)
            pred, conf, _ = prediction_for(
                "SCALE-Full", msg, agent, BASE_GRAPH, repair=True
            )
            original_contracts[agent] = pred

        orig_sem = active_semantics_from_contracts(original_contracts)
        orig_behavior = deterministic_policy(orig_sem)

        for agent in AGENTS:
            alt = mr_alt_base(base, agent)
            alt_msg = msg_fn(alt, agent, style)
            alt_pred, conf, _ = prediction_for(
                "SCALE-Full", alt_msg, agent, BASE_GRAPH, repair=True
            )

            cf_contracts = copy.deepcopy(original_contracts)
            cf_contracts[agent] = alt_pred
            cf_sem = active_semantics_from_contracts(cf_contracts)
            cf_behavior = deterministic_policy(cf_sem)

            gold_orig = global_semantics(base)
            gold_alt = copy.deepcopy(gold_orig)

            if agent == "PLANNER":
                target_key = "object"
                gold_alt["object"] = alt["object"]
            elif agent == "RETRIEVER":
                target_key = "evidence_state"
                gold_alt["evidence_state"] = EVIDENCE_TO_STATE[alt["evidence_status"]]
            else:
                target_key = "authority"
                gold_alt["authority"] = alt["authority"]

            gold_orig_behavior = deterministic_policy(gold_orig)
            gold_alt_behavior = deterministic_policy(gold_alt)

            other_keys = [
                k for k in ["object","evidence_state","authority"]
                if k != target_key
            ]

            intervention_rows.append({
                "split": split,
                "base_id": base["base_id"],
                "agent": agent,
                "target_key": target_key,
                "surface_family": style,
                "target_changed_in_prediction": int(
                    cf_sem[target_key] != orig_sem[target_key]
                ),
                "target_matches_counterfactual_gold": int(
                    cf_sem[target_key] == gold_alt[target_key]
                ),
                "non_target_active_stable": int(
                    all(cf_sem[k] == orig_sem[k] for k in other_keys)
                ),
                "predicted_behavior_changed": int(cf_behavior != orig_behavior),
                "gold_behavior_should_change": int(
                    gold_alt_behavior != gold_orig_behavior
                ),
                "behavior_change_correct_when_expected": int(
                    (cf_behavior != orig_behavior)
                    == (gold_alt_behavior != gold_orig_behavior)
                ),
            })

message_cf_df = pd.DataFrame(intervention_rows)
message_cf_summary = (
    message_cf_df.groupby(["split","agent","target_key"], as_index=False)
    .agg(
        n=("base_id","size"),
        target_reaction=("target_changed_in_prediction","mean"),
        target_counterfactual_accuracy=("target_matches_counterfactual_gold","mean"),
        non_target_stability=("non_target_active_stable","mean"),
        predicted_behavior_change=("predicted_behavior_changed","mean"),
        expected_behavior_change=("gold_behavior_should_change","mean"),
        behavior_sensitivity_agreement=("behavior_change_correct_when_expected","mean"),
    )
)

print("\n[P2] Message-level semantic counterfactuals")
display(message_cf_summary.round(4))

message_cf_df.to_csv(MASTER_DIR / "P2_message_counterfactual_records.csv", index=False)
message_cf_summary.to_csv(MASTER_DIR / "P2_message_counterfactual_summary.csv", index=False)

body = []
body.append(
    "The intervention now occurs at the **message boundary before decoding**, rather than by directly overwriting the downstream semantic state."
)
body.append(mr_df(message_cf_summary))
body.append(
    "\nInterpretation should separate three questions: (1) does SCALE decode the intended changed concept, "
    "(2) do unrelated active concepts remain stable, and (3) does downstream behavior change only where "
    "the gold semantic intervention should matter?"
)
body.append(
    "\nThis addresses the tautology criticism substantially better than 11/11 direct semantic correction, "
    "but it remains a controlled sensitivity experiment rather than proof of universal causal semantics."
)

mr_report(
    "P2 — Message-Level Counterfactual Semantic Sensitivity",
    "\n".join(body),
    "02_P2_message_counterfactuals.md",
)


In [ ]:
# =============================================================================
# P3 — ACTIVE-SCD HELD-OUT THRESHOLD TRANSFER + TRAJECTORY WARNING UTILITY
# =============================================================================
#
# AUROC alone does not show deployment utility. This block:
# 1. splits bases deterministically into calibration/test halves;
# 2. selects ONE threshold on calibration deterministic failures;
# 3. evaluates that frozen threshold on held-out controlled trajectories;
# 4. transfers the same threshold to competent generative executors, if any.
#
# "Lead steps" are severity-trajectory steps, not wall-clock time.
# =============================================================================

def mr_split_base(base_id):
    h = int(hashlib.sha256(str(base_id).encode()).hexdigest()[:8], 16)
    return "cal" if h % 2 == 0 else "test"

scd_col = next(
    (c for c in ["prepolicy_active_scd","active_scd","Active-SCD"] if c in drift_df.columns),
    None
)
assert scd_col is not None

thr_df = drift_df.copy()
thr_df["master_half"] = thr_df["base_id"].map(mr_split_base)

cal = thr_df[thr_df["master_half"]=="cal"].copy()
test = thr_df[thr_df["master_half"]=="test"].copy()

thresholds = np.unique(cal[scd_col].to_numpy(float))
best = None
for t in thresholds:
    pred = (cal[scd_col].to_numpy(float) >= t).astype(int)
    bal = balanced_accuracy_score(cal["failure"], pred)
    if best is None or bal > best[0] or (bal == best[0] and t > best[1]):
        best = (bal, float(t))

MASTER_SCD_THRESHOLD = best[1]

def mr_threshold_metrics(df, y_col, score_col, threshold):
    y = df[y_col].astype(int).to_numpy()
    pred = (df[score_col].astype(float).to_numpy() >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "n": len(df),
        "prevalence": float(y.mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y,pred)),
        "precision": float(precision_score(y,pred,zero_division=0)),
        "recall": float(recall_score(y,pred,zero_division=0)),
        "specificity": float(tn/max(tn+fp,1)),
        "false_positive_rate": float(fp/max(tn+fp,1)),
    }

threshold_rows = [
    {"setting":"controlled_cal", **mr_threshold_metrics(cal,"failure",scd_col,MASTER_SCD_THRESHOLD)},
    {"setting":"controlled_test", **mr_threshold_metrics(test,"failure",scd_col,MASTER_SCD_THRESHOLD)},
]

# Severity-trajectory alert timing on held-out bases.
trajectory_rows = []
for bid, g in test.groupby("base_id"):
    g = g.sort_values("severity")
    fail_rows = g[g["failure"]==1]
    alert_rows = g[g[scd_col] >= MASTER_SCD_THRESHOLD]

    first_failure = (
        int(fail_rows["severity"].iloc[0]) if len(fail_rows) else np.nan
    )
    first_alert = (
        int(alert_rows["severity"].iloc[0]) if len(alert_rows) else np.nan
    )

    if np.isfinite(first_failure):
        on_or_before = int(np.isfinite(first_alert) and first_alert <= first_failure)
        lead = (
            int(first_failure - first_alert)
            if np.isfinite(first_alert) else np.nan
        )
    else:
        on_or_before = np.nan
        lead = np.nan

    trajectory_rows.append({
        "base_id": bid,
        "first_failure_severity": first_failure,
        "first_alert_severity": first_alert,
        "alert_on_or_before_failure": on_or_before,
        "lead_severity_steps": lead,
    })

trajectory_df = pd.DataFrame(trajectory_rows)
failure_trajectory = trajectory_df[
    trajectory_df["first_failure_severity"].notna()
]

# Threshold transfer to competent generative executors.
gen_transfer_rows = []
if (
    "generative_executor_summary" in globals()
    and isinstance(generative_executor_summary, pd.DataFrame)
    and len(generative_executor_summary)
    and "strong_executor_df" in globals()
    and isinstance(strong_executor_df, pd.DataFrame)
):
    passed_models = set(
        generative_executor_summary[
            generative_executor_summary["competence_gate_pass"]==1
        ]["model"]
    )
    for model in sorted(passed_models):
        g = strong_executor_df[
            (strong_executor_df["model"]==model)
            & (strong_executor_df["clean_gate_record"]==0)
        ].copy()
        if len(g):
            g = g[g["base_id"].map(mr_split_base)=="test"]
            if len(g) and "failure" in g.columns:
                m = mr_threshold_metrics(
                    g, "failure", "active_scd", MASTER_SCD_THRESHOLD
                )
                gen_transfer_rows.append({
                    "setting": f"generative::{model}",
                    **m
                })

threshold_summary = pd.DataFrame(threshold_rows + gen_transfer_rows)

trajectory_summary = pd.DataFrame([{
    "n_test_bases": int(test["base_id"].nunique()),
    "n_bases_with_failure": len(failure_trajectory),
    "alert_on_or_before_failure_rate": (
        float(failure_trajectory["alert_on_or_before_failure"].mean())
        if len(failure_trajectory) else np.nan
    ),
    "mean_lead_severity_steps": (
        float(failure_trajectory["lead_severity_steps"].dropna().mean())
        if failure_trajectory["lead_severity_steps"].notna().any() else np.nan
    ),
    "median_lead_severity_steps": (
        float(failure_trajectory["lead_severity_steps"].dropna().median())
        if failure_trajectory["lead_severity_steps"].notna().any() else np.nan
    ),
    "frozen_threshold": MASTER_SCD_THRESHOLD,
}])

print("\n[P3] Frozen Active-SCD threshold utility")
display(threshold_summary.round(4))
print("\nTrajectory warning summary")
display(trajectory_summary.round(4))

threshold_summary.to_csv(MASTER_DIR / "P3_scd_threshold_transfer.csv", index=False)
trajectory_df.to_csv(MASTER_DIR / "P3_scd_trajectory_records.csv", index=False)
trajectory_summary.to_csv(MASTER_DIR / "P3_scd_trajectory_summary.csv", index=False)

body = []
body.append(
    f"A single Active-SCD threshold (**{MASTER_SCD_THRESHOLD:.4f}**) is selected on one deterministic calibration half and frozen before evaluation on the held-out half."
)
body.append("\n### Threshold performance")
body.append(mr_df(threshold_summary))
body.append("\n### Severity-trajectory monitoring utility")
body.append(mr_df(trajectory_summary))
body.append(
    "\nThe lead metric is measured in controlled severity steps, not real elapsed time. "
    "Its purpose is to show whether the pre-policy signal crosses an operational threshold "
    "before or at the first downstream failure along a held-out degradation trajectory."
)
if len(gen_transfer_rows):
    body.append(
        "\nThe identical deterministic-calibrated threshold is also applied to competent generative executors. "
        "This is stronger than fitting a new threshold to each executor."
    )
else:
    body.append(
        "\nNo competent generative executor was available for threshold transfer, so this part remains inconclusive without relaxing the frozen competence gate."
    )

mr_report(
    "P3 — Active-SCD Operational Threshold & Warning Utility",
    "\n".join(body),
    "03_P3_active_scd_operational_utility.md",
)


In [ ]:
# =============================================================================
# P4 — ADDITIONAL HUMAN-AUTHORED EXTERNAL ONTOLOGIES: PROV-O + ODRL
# =============================================================================
#
# This is a static structural mechanism test, not an end-to-end SCALE benchmark.
# It complements Schema.org and EDAM with two W3C ontologies tied directly to
# provenance and permissions/governance semantics.
# =============================================================================

EXTERNAL_W3C_ONTOLOGIES = {
    "PROV-O": "https://www.w3.org/ns/prov.ttl",
    "ODRL": "https://www.w3.org/ns/odrl/2/ODRL22.ttl",
}

def mr_parse_graph_url(url):
    rr = requests.get(
        url,
        timeout=60,
        headers={"Accept":"text/turtle, application/rdf+xml;q=0.8"}
    )
    rr.raise_for_status()
    g = Graph()
    ctype = rr.headers.get("content-type","").lower()
    fmt = "turtle" if ("turtle" in ctype or url.endswith(".ttl")) else "xml"
    g.parse(data=rr.content, format=fmt)
    return g

def mr_generic_class_table(g):
    classes = set()
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):
            classes.add(s)
    for s,o in g.subject_objects(RDFS.subClassOf):
        if isinstance(s, URIRef): classes.add(s)
        if isinstance(o, URIRef): classes.add(o)

    pm = {}
    for s,o in g.subject_objects(RDFS.subClassOf):
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            pm.setdefault(s,set()).add(o)

    # roots = classes without a parent inside this class vocabulary
    roots = [
        c for c in classes
        if not any(p in classes for p in pm.get(c,set()))
    ]

    def anc(uri):
        seen=set()
        q=[uri]
        while q:
            x=q.pop()
            for p in pm.get(x,set()):
                if p not in seen:
                    seen.add(p); q.append(p)
        return seen

    rows=[]
    for u in classes:
        if u in roots:
            continue
        labs=[str(x).strip() for x in g.objects(u,RDFS.label) if str(x).strip()]
        defs=[]
        for p,o in g.predicate_objects(u):
            if isinstance(o,Literal) and any(
                z in str(p).lower()
                for z in ["comment","definition","description"]
            ):
                t=str(o).strip()
                if len(t.split())>=4:
                    defs.append(t)
        if not labs or not defs:
            continue

        aa=anc(u)
        candidate_roots=[r for r in roots if r in aa]
        if not candidate_roots:
            continue

        # deterministic root selection when multiple roots exist:
        # use lexically first root URI; report this as coarse structural placement.
        root=sorted(candidate_roots,key=str)[0]
        root_lab=[str(x).strip() for x in g.objects(root,RDFS.label) if str(x).strip()]
        root_label=root_lab[0] if root_lab else str(root).split("#")[-1].split("/")[-1]

        rows.append({
            "uri":str(u),
            "label":labs[0],
            "definition":sorted(defs,key=lambda x:-len(x))[0],
            "root_uri":str(root),
            "root_label":root_label,
        })
    return pd.DataFrame(rows)

w3c_rows=[]
w3c_summary_rows=[]

for name,url in EXTERNAL_W3C_ONTOLOGIES.items():
    try:
        g=mr_parse_graph_url(url)
        t=mr_generic_class_table(g)

        if len(t)<10 or t["root_uri"].nunique()<2:
            w3c_summary_rows.append({
                "ontology":name,
                "status":"INSUFFICIENT_HIERARCHY",
                "n":len(t),
                "roots":t["root_uri"].nunique() if len(t) else 0,
            })
            continue

        q=semantic_encode(t["definition"].tolist(),batch_size=64)
        leaf=semantic_encode(t["label"].tolist(),batch_size=64)

        root_tbl=t[["root_uri","root_label"]].drop_duplicates().reset_index(drop=True)
        r_emb=semantic_encode(root_tbl["root_label"].tolist(),batch_size=64)

        li=np.argmax(q@leaf.T,axis=1)
        pred_leaf_root=t.iloc[li]["root_uri"].tolist()

        ri=np.argmax(q@r_emb.T,axis=1)
        direct_root=root_tbl.iloc[ri]["root_uri"].tolist()

        rng=np.random.default_rng(MASTER_SEED + len(w3c_summary_rows))
        perm_roots=rng.permutation(t["root_uri"].to_numpy(object))
        uri_to_perm=dict(zip(t["uri"],perm_roots))
        pred_leaf_uri=t.iloc[li]["uri"].tolist()
        shuffled=[uri_to_perm[u] for u in pred_leaf_uri]

        out_t=t.copy()
        out_t["pred_leaf_uri"]=pred_leaf_uri
        out_t["leaf_correct"]=(out_t["pred_leaf_uri"]==out_t["uri"]).astype(int)
        out_t["graph_root_pred"]=pred_leaf_root
        out_t["graph_root_correct"]=(out_t["graph_root_pred"]==out_t["root_uri"]).astype(int)
        out_t["direct_root_pred"]=direct_root
        out_t["direct_root_correct"]=(out_t["direct_root_pred"]==out_t["root_uri"]).astype(int)
        out_t["shuffled_root_pred"]=shuffled
        out_t["shuffled_root_correct"]=(out_t["shuffled_root_pred"]==out_t["root_uri"]).astype(int)
        out_t["ontology"]=name

        w3c_rows.append(out_t)

        g10,g01,gp=mr_exact_mcnemar(
            out_t["graph_root_correct"],out_t["direct_root_correct"]
        )
        s10,s01,sp=mr_exact_mcnemar(
            out_t["graph_root_correct"],out_t["shuffled_root_correct"]
        )

        w3c_summary_rows.append({
            "ontology":name,
            "status":"OK",
            "n":len(out_t),
            "roots":out_t["root_uri"].nunique(),
            "leaf_accuracy":out_t["leaf_correct"].mean(),
            "direct_root_accuracy":out_t["direct_root_correct"].mean(),
            "graph_root_accuracy":out_t["graph_root_correct"].mean(),
            "shuffled_root_accuracy":out_t["shuffled_root_correct"].mean(),
            "graph_vs_direct_mcnemar_p":gp,
            "graph_vs_shuffled_mcnemar_p":sp,
        })
    except Exception as exc:
        w3c_summary_rows.append({
            "ontology":name,
            "status":f"ERROR:{type(exc).__name__}",
            "n":0,
            "roots":0,
        })

w3c_external_df=(
    pd.concat(w3c_rows,ignore_index=True)
    if w3c_rows else pd.DataFrame()
)
w3c_external_summary=pd.DataFrame(w3c_summary_rows)

print("\n[P4] W3C external ontology controls")
display(w3c_external_summary.round(4))

w3c_external_df.to_csv(MASTER_DIR/"P4_w3c_external_records.csv",index=False)
w3c_external_summary.to_csv(MASTER_DIR/"P4_w3c_external_summary.csv",index=False)

body=[]
body.append(
    "PROV-O and ODRL are additional **human-authored external ontologies** selected because provenance and permission semantics are directly relevant to SCALE's contract vocabulary."
)
body.append(mr_df(w3c_external_summary))
body.append(
    "\nThese are structural mechanism controls rather than full end-to-end SCALE retraining. "
    "A positive result requires the correct ontology ancestry to beat both direct root prediction "
    "and a shuffled structural negative control. Null results remain external boundary evidence."
)

mr_report(
    "P4 — Additional External Ontology Structural Controls",
    "\n".join(body),
    "04_P4_W3C_external_ontologies.md",
)


In [ ]:
# =============================================================================
# P5 — MASTER STATISTICAL / CLAIM-STRENGTH AUDIT
# =============================================================================

claim_rows=[]

# Closed-set large stress.
if isinstance(master_surface_claims,pd.DataFrame) and len(master_surface_claims):
    for r in master_surface_claims.itertuples():
        if r.superiority_both:
            status="SUPPORTED SUPERIORITY"
            lang="may state a positive advantage, with synthetic-suite scope explicit"
        elif r.noninferior_both_margin_003:
            status="SUPPORTED NONINFERIORITY"
            lang="use noninferior/competitive; do not state established superiority"
        else:
            status="QUALIFIED"
            lang="use descriptive result only"
        claim_rows.append({
            "claim":"closed_set_"+r.split+"_"+r.comparison.replace(" ","_"),
            "status":status,
            "evidence":(
                f"base CI [{r.base_ci_low:.4f},{r.base_ci_high:.4f}], "
                f"surface CI [{r.surface_ci_low:.4f},{r.surface_ci_high:.4f}]"
            ),
            "manuscript_language":lang,
        })

# Message counterfactual.
if isinstance(message_cf_summary,pd.DataFrame) and len(message_cf_summary):
    target=float(message_cf_summary["target_counterfactual_accuracy"].mean())
    stable=float(message_cf_summary["non_target_stability"].mean())
    beh=float(message_cf_summary["behavior_sensitivity_agreement"].mean())
    claim_rows.append({
        "claim":"message_level_semantic_sensitivity",
        "status":"SUPPORTING" if min(target,stable,beh)>=.80 else "QUALIFIED",
        "evidence":f"target={target:.3f}; non-target stability={stable:.3f}; behavior agreement={beh:.3f}",
        "manuscript_language":"supporting behavioral-sensitivity evidence; not universal causal proof",
    })

# Active-SCD operational.
if isinstance(threshold_summary,pd.DataFrame) and len(threshold_summary):
    tst=threshold_summary[threshold_summary["setting"]=="controlled_test"]
    if len(tst):
        r=tst.iloc[0]
        claim_rows.append({
            "claim":"active_scd_operational_threshold",
            "status":"SUPPORTING" if r["balanced_accuracy"]>=.80 else "QUALIFIED",
            "evidence":(
                f"held-out balanced acc={r['balanced_accuracy']:.3f}, "
                f"recall={r['recall']:.3f}, specificity={r['specificity']:.3f}"
            ),
            "manuscript_language":"post-hoc operational diagnostic; keep primary claim AUROC-based",
        })

# EDAM.
if "edam_static_summary" in globals() and isinstance(edam_static_summary,pd.DataFrame) and len(edam_static_summary):
    r=edam_static_summary.iloc[0]
    claim_rows.append({
        "claim":"external_EDAM_structure",
        "status":"STRONG SUPPORT" if (
            r["graph_root_accuracy"]>r["direct_root_accuracy"]
            and r["graph_root_accuracy"]>r["shuffled_root_accuracy"]
        ) else "QUALIFIED",
        "evidence":(
            f"graph={r['graph_root_accuracy']:.3f}; direct={r['direct_root_accuracy']:.3f}; "
            f"shuffled={r['shuffled_root_accuracy']:.3f}"
        ),
        "manuscript_language":"external structural mechanism evidence",
    })

# W3C ontologies.
if isinstance(w3c_external_summary,pd.DataFrame) and len(w3c_external_summary):
    for r in w3c_external_summary.itertuples():
        if getattr(r,"status","")!="OK":
            st="INCONCLUSIVE"
            ev=f"status={r.status}"
        else:
            good=(
                r.graph_root_accuracy>r.direct_root_accuracy
                and r.graph_root_accuracy>r.shuffled_root_accuracy
            )
            st="SUPPORTING" if good else "QUALIFIED/NULL"
            ev=(
                f"graph={r.graph_root_accuracy:.3f}; direct={r.direct_root_accuracy:.3f}; "
                f"shuffled={r.shuffled_root_accuracy:.3f}"
            )
        claim_rows.append({
            "claim":f"external_{r.ontology}_structure",
            "status":st,
            "evidence":ev,
            "manuscript_language":"secondary external mechanism evidence",
        })

# G0/G2 preserve negative status.
claim_rows.extend([
    {
        "claim":"G0_arbitrary_reader_robustness",
        "status":"ORIGINAL FAIL — DO NOT RELABEL",
        "evidence":"use strong-reader interface decomposition only as post-hoc localization",
        "manuscript_language":"typed/native intended interface; arbitrary generative rereading unsupported",
    },
    {
        "claim":"G2_general_alignment_label_efficiency",
        "status":"ORIGINAL FAIL — DO NOT RELABEL",
        "evidence":"use 10-seed Full-vs-NoInv audit as robustness diagnosis",
        "manuscript_language":"alignment is regime-dependent/non-core",
    },
    {
        "claim":"universal_end_task_superiority",
        "status":"NOT SUPPORTED / NOT CLAIMED",
        "evidence":"CRMArena-Pro remains executor-dependent",
        "manuscript_language":"compatibility is distinct from competence",
    },
])

master_claim_audit=pd.DataFrame(claim_rows)
display(master_claim_audit)

master_claim_audit.to_csv(
    MASTER_DIR/"P5_MASTER_CLAIM_STRENGTH_AUDIT.csv",index=False
)

body=[]
body.append(
    "This table is the final rule-set for manuscript claim strength. "
    "It prevents a positive post-hoc diagnostic from silently becoming a new primary claim."
)
body.append(mr_df(master_claim_audit,n=200))
body.append(
    "\n**Key principle:** the final manuscript should become *narrower and stronger* as evidence accumulates, "
    "not broader. Negative G0/G2 results remain part of the scientific contribution through transparent boundary definition."
)

mr_report(
    "P5 — Master Statistical & Claim-Strength Audit",
    "\n".join(body),
    "05_P5_master_claim_audit.md",
)


In [ ]:
# =============================================================================
# P6 — SINGLE CANONICAL MASTER REGISTRY + FINAL MANUSCRIPT MAP
# =============================================================================

master_registry_rows=[]

def mr_reg(key,value,source,status,location,note):
    master_registry_rows.append({
        "key":key,
        "value":float(value) if value is not None and np.isfinite(float(value)) else np.nan,
        "source":source,
        "status":status,
        "paper_location":location,
        "note":note,
    })

# Retain current corrected registry as historical source, deduplicated by newest row.
if "registry_df" in globals() and isinstance(registry_df,pd.DataFrame) and len(registry_df):
    if "key" in registry_df.columns:
        hist=registry_df.drop_duplicates("key",keep="last")
        for r in hist.itertuples():
            try:
                val=float(r.value)
            except Exception:
                val=np.nan
            mr_reg(
                r.key,val,
                "corrected_previous_registry",
                getattr(r,"analysis_status",""),
                getattr(r,"paper_location",""),
                getattr(r,"note",""),
            )

# Append new master evidence under distinct keys so old prespecified numbers are
# not overwritten by post-hoc stress results.
for r in master_surface_summary.itertuples():
    mr_reg(
        f"master_surface_{r.method}_{r.split}_active_accuracy",
        r.active_accuracy,
        "master_120x12_surface_stress",
        "POST-HOC REVIEWER CLOSURE",
        "Appendix / robustness",
        f"{r.n_bases} bases, {r.n_surface_families} surface families."
    )

if isinstance(message_cf_summary,pd.DataFrame):
    mr_reg(
        "master_message_counterfactual_target_accuracy",
        message_cf_summary["target_counterfactual_accuracy"].mean(),
        "message_level_counterfactuals",
        "POST-HOC MECHANISM",
        "Appendix",
        "Message intervention occurs before semantic decoding."
    )
    mr_reg(
        "master_message_counterfactual_nontarget_stability",
        message_cf_summary["non_target_stability"].mean(),
        "message_level_counterfactuals",
        "POST-HOC MECHANISM",
        "Appendix",
        "Non-target active semantics remain stable."
    )

if isinstance(trajectory_summary,pd.DataFrame) and len(trajectory_summary):
    rr=trajectory_summary.iloc[0]
    mr_reg(
        "master_scd_alert_on_or_before_failure_rate",
        rr["alert_on_or_before_failure_rate"],
        "heldout_threshold_trajectory",
        "POST-HOC OPERATIONAL",
        "Appendix / Discussion",
        "Severity-step trajectory diagnostic, not wall-clock lead time."
    )

if "edam_static_summary" in globals() and isinstance(edam_static_summary,pd.DataFrame) and len(edam_static_summary):
    rr=edam_static_summary.iloc[0]
    mr_reg(
        "external_edam_graph_root_accuracy",
        rr["graph_root_accuracy"],
        "EDAM_static",
        "EXTERNAL MECHANISM",
        "Main/Appendix",
        "Human-curated external ontology structural transfer."
    )

for r in w3c_external_summary.itertuples():
    if getattr(r,"status","")=="OK":
        mr_reg(
            f"external_{r.ontology.lower().replace('-','_')}_graph_root_accuracy",
            r.graph_root_accuracy,
            f"W3C_{r.ontology}",
            "EXTERNAL MECHANISM",
            "Appendix",
            "Additional human-authored ontology structural control."
        )

master_registry=pd.DataFrame(master_registry_rows)

# Detect accidental duplicate keys in the master registry.
dup_master=(
    master_registry.groupby("key")
    .agg(n=("value","size"),n_values=("value","nunique"))
    .reset_index()
)
bad_master=dup_master[(dup_master["n"]>1)&(dup_master["n_values"]>1)]

assert len(bad_master)==0, (
    "Master registry contains conflicting duplicate keys:\n"
    + bad_master.to_string(index=False)
)

master_registry=master_registry.drop_duplicates("key",keep="last")
master_registry.to_csv(
    MASTER_DIR/"FINAL_CANONICAL_MASTER_RESULTS_REGISTRY.csv",index=False
)

# Manuscript section map.
manuscript_map=pd.DataFrame([
    {
        "section":"Abstract",
        "include":"problem formulation; competitive closed-set compatibility; opaque/external ontology mechanism; Active-SCD evidence ladder",
        "exclude":"patch history; G0/G2 details; routing engineering detail; every post-hoc score",
    },
    {
        "section":"Introduction",
        "include":"semantic interface drift; role-active consumer-dependent interface; three core contributions",
        "exclude":"universal reader/task superiority",
    },
    {
        "section":"Method",
        "include":"strict ID -> known discriminative -> open ontology; validator; Active-SCD; typed consumption",
        "exclude":"logic as contribution; alignment as essential mechanism",
    },
    {
        "section":"Results",
        "include":"fair closed-set comparison; paired ontology mechanism; Active-SCD evidence ladder; concise external evidence",
        "exclude":"revision-history language; stale branch-conditioned routing numbers",
    },
    {
        "section":"Discussion",
        "include":"compatibility vs competence; where ontology helps; observability; safety/boundaries",
        "exclude":"defensive enumeration of every diagnostic",
    },
    {
        "section":"Limitations",
        "include":"original G0/G2 failures; root-dependent routing; consumer-dependent masks; generative gate outcome; constructed-vs-real evidence distinction",
        "exclude":"attempts to explain away negative results",
    },
    {
        "section":"Appendix",
        "include":"large 120x12 suite; counterfactuals; threshold transfer; W3C ontologies; strong readers/producers; full uncertainty",
        "exclude":"conflicting duplicate canonical numbers",
    },
])

manuscript_map.to_csv(MASTER_DIR/"FINAL_MANUSCRIPT_EVIDENCE_MAP.csv",index=False)

body=[]
body.append(
    f"The canonical master registry contains **{len(master_registry)} unique keys** and no conflicting duplicate key values."
)
body.append("\n### Manuscript evidence map")
body.append(mr_df(manuscript_map,n=50))
body.append(
    "\nThe original prespecified numerical results are preserved under their original keys. "
    "New reviewer-closure results receive new keys rather than silently overwriting history."
)

mr_report(
    "P6 — Canonical Master Registry & Manuscript Evidence Map",
    "\n".join(body),
    "06_P6_master_registry_and_manuscript_map.md",
)


In [ ]:
# =============================================================================
# FINAL MASTER REVIEWER-CLOSURE REPORT
# =============================================================================

sections=[]

sections.append("## 1. Reviewer objections and current closure status\n")
reviewer_matrix=pd.DataFrame([
    ["Small C1/C2 n","P1 120×12 surface suite","Measured with base- and surface-family-clustered uncertainty"],
    ["Baseline not beaten","P1 + prior dev-only tuning","Superiority only if both CIs >0; otherwise noninferiority/competitive"],
    ["Typed adapter is JSON","Strong-reader decomposition + G5 combinatorial corruption","Syntactic and semantic validity measured separately"],
    ["Synthetic Active-SCD","Conditional controls + P3 threshold transfer + generative executor + EDAM real versions","Synthetic mechanism and external evidence explicitly separated"],
    ["Intervention tautology","P2 message-level counterfactuals","Intervention moved before decoder; selectivity/stability measured"],
    ["Too complex","10-seed G2 + component ablations","Alignment/logic remain non-core; calibration/ontology supported only where evidence exists"],
    ["Table-12 inconsistency","P0 provenance + deterministic-ID integrity","Legacy branch and final architecture separated"],
    ["External ontology weak","Schema.org + EDAM + PROV-O + ODRL","Correct structure compared with direct/shuffled controls"],
    ["No LLM executor evidence","Frozen-gate strong generative executors","Positive only if clean competence gate passes"],
    ["No real evolution","EDAM 1.24→1.25 + MCP/A2A release corpus","Ontology version evolution is validation; changelog corpus is motivation only"],
])
reviewer_matrix.columns=["reviewer_concern","closure_analysis","scientific_resolution_rule"]
sections.append(mr_df(reviewer_matrix,n=50))

sections.append("\n## 2. Final claim hierarchy\n")
sections.append(mr_df(master_claim_audit,n=100))

sections.append("\n## 3. Closed-set stress conclusions\n")
sections.append(mr_df(master_surface_claims,n=50))

sections.append("\n## 4. Message-level intervention conclusions\n")
sections.append(mr_df(message_cf_summary,n=50))

sections.append("\n## 5. Monitoring utility\n")
sections.append(mr_df(threshold_summary,n=50))
sections.append(mr_df(trajectory_summary,n=10))

sections.append("\n## 6. Additional external ontologies\n")
sections.append(mr_df(w3c_external_summary,n=20))

if "edam_static_summary" in globals():
    sections.append("\n## 7. EDAM external evidence\n")
    sections.append(mr_df(edam_static_summary,n=10))
    sections.append(mr_df(edam_longitudinal_summary,n=20))

if "generative_executor_summary" in globals():
    sections.append("\n## 8. Generative executor evidence\n")
    sections.append(mr_df(generative_executor_summary,n=20))

sections.append("\n## 9. What the final paper should claim\n")
sections.append(
    """
1. **Core conceptual claim:** semantic interface drift is distinct from syntactic/protocol compatibility.
2. **Core mechanism claim:** ontology structure is useful specifically for structural open-world canonicalization, not ordinary closed-set decoding.
3. **Primary empirical claim:** decision-active semantic drift provides a pre-policy failure signal that survives multiple negative controls and independent consumers.
4. **Supporting systems claim:** typed contracts and semantic validation separate syntactic validity from operational semantic validity.
5. **Qualified engineering claim:** routing and abstention are useful but root-dependent; they are not universal open-set detection.
6. **Negative result:** arbitrary generative reader robustness remains bounded by the original G0 result.
7. **Negative result:** directed invariance is not a general label-efficiency contribution (G2).
8. **Boundary claim:** semantic compatibility does not imply downstream task competence or universal task improvement.
"""
)

sections.append("\n## 10. Stop rule\n")
sections.append(
    """
After this master suite, **do not add another synthetic family merely to move a score**.

A genuinely new experiment is justified only if it provides a new independent source:
- production/longitudinal agent logs with downstream outcomes;
- another independent domain with task-level execution;
- a new competent generative executor family;
- independent human annotation of real semantic-interface changes.

Otherwise, the next work item is manuscript consolidation and statistical/claim consistency.
"""
)

final_body="\n".join(sections)

mr_report(
    "FINAL — SCALE Master Reviewer-Closure Report",
    final_body,
    "99_FINAL_MASTER_REVIEWER_CLOSURE_REPORT.md",
)

# Merge every master detailed report in execution order.
merged="\n\n---\n\n".join(MASTER_REPORT_SECTIONS)
(MASTER_DIR/"FINAL_ALL_MASTER_DETAILED_REPORTS.md").write_text(
    merged+"\n",encoding="utf-8"
)

reviewer_matrix.to_csv(
    MASTER_DIR/"FINAL_REVIEWER_CONCERN_CLOSURE_MATRIX.csv",index=False
)

print("\nMASTER REVIEWER CLOSURE COMPLETE")
print("Primary report:", MASTER_REPORT_DIR/"99_FINAL_MASTER_REVIEWER_CLOSURE_REPORT.md")
print("Canonical registry:", MASTER_DIR/"FINAL_CANONICAL_MASTER_RESULTS_REGISTRY.csv")
print("Reviewer matrix:", MASTER_DIR/"FINAL_REVIEWER_CONCERN_CLOSURE_MATRIX.csv")


# FINAL FULL RESULTS BOOK

The cell below executes only after every earlier test family has had a chance to
run. It does not hide missing experiments: incomplete or failed external tests
remain visible in the completion audit.


In [ ]:
# =============================================================================
# SCALE FULL MASTER — FINAL RESULTS BOOK, COMPLETION AUDIT, ARTIFACT PACK
# =============================================================================

from pathlib import Path
import json, os, shutil, platform
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

FULL_BOOK_DIR = ROOT / "FULL_MASTER_RESULTS_BOOK"
FULL_BOOK_DIR.mkdir(parents=True, exist_ok=True)

def fm_exists(name):
    return name in globals()

def fm_nonempty(name):
    if name not in globals():
        return False
    obj = globals()[name]
    if isinstance(obj, pd.DataFrame):
        return len(obj) > 0
    if isinstance(obj, (list, tuple, dict, set)):
        return len(obj) > 0
    return obj is not None

def fm_md_table(df, cols=None, max_rows=60, digits=4):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return "_No rows available._"
    x = df.copy()
    if cols:
        cols = [c for c in cols if c in x.columns]
        x = x[cols]
    for c in x.select_dtypes(include=[np.number]).columns:
        x[c] = x[c].round(digits)
    return x.head(max_rows).to_markdown(index=False)

def fm_section(title, df=None, cols=None, prose=None, max_rows=60):
    out = [f"## {title}"]
    if prose:
        out.append(prose)
    if isinstance(df, pd.DataFrame):
        out.append(fm_md_table(df, cols=cols, max_rows=max_rows))
    return "\n\n".join(out)

# -------------------------------------------------------------------------
# A. Completion audit
# -------------------------------------------------------------------------

completion_rows = []
for family, variable in FULL_MASTER_TEST_FAMILIES:
    completion_rows.append({
        "test_family": family,
        "expected_variable": variable,
        "object_exists": int(fm_exists(variable)),
        "nonempty": int(fm_nonempty(variable)),
        "status": (
            "COMPLETE"
            if fm_nonempty(variable)
            else ("EMPTY/INCONCLUSIVE" if fm_exists(variable) else "NOT_REACHED")
        ),
    })

full_completion_df = pd.DataFrame(completion_rows)
full_completion_df.to_csv(
    FULL_BOOK_DIR / "FULL_TEST_COMPLETION_AUDIT.csv",
    index=False,
)

display(Markdown("# SCALE — Full Master Completion Audit"))
display(full_completion_df)


# Runtime-dependent evaluation availability.
if "GEN_RUNTIME_STATUS" in globals():
    GEN_RUNTIME_STATUS.to_csv(
        FULL_BOOK_DIR / "GEN_RUNTIME_STATUS.csv",
        index=False,
    )
    display(Markdown("## Runtime-dependent test availability"))
    display(GEN_RUNTIME_STATUS)

# -------------------------------------------------------------------------
# B. Environment / reproducibility manifest
# -------------------------------------------------------------------------

environment_manifest = {
    "full_master_run_id": FULL_MASTER_RUN_ID,
    "experiment_version": EXPERIMENT_VERSION,
    "config_hash": CONFIG_HASH,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "head_seeds": list(HEAD_SEEDS),
    "curve_seeds": list(CURVE_SEEDS),
    "semantic_encoder": SEMANTIC_ENCODER_ID,
    "nli_validator": NLI_VALIDATOR_ID,
    "reader_models": READER_MODELS,
    "producer_models": PRODUCER_MODELS,
    "strong_llm_specs": (
        STRONG_LLM_SPECS if "STRONG_LLM_SPECS" in globals() else []
    ),
    "generative_executor_gate": {
        "clean_task": (
            GEN_EXEC_CLEAN_TASK_GATE
            if "GEN_EXEC_CLEAN_TASK_GATE" in globals() else None
        ),
        "parse": (
            GEN_EXEC_PARSE_GATE
            if "GEN_EXEC_PARSE_GATE" in globals() else None
        ),
    },
}

(FULL_BOOK_DIR / "FULL_ENVIRONMENT_MANIFEST.json").write_text(
    json.dumps(environment_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# C. Comprehensive notebook-visible results book
# -------------------------------------------------------------------------

sections = []

sections.append(
    "# SCALE — FULL MASTER RESULTS BOOK\n\n"
    "This report is generated automatically from the objects produced by the "
    "complete notebook. Missing/failed external tests remain visible rather "
    "than being silently omitted."
)

sections.append(
    fm_section(
        "0. Test completion",
        full_completion_df,
        max_rows=100,
        prose=(
            "A test family is marked COMPLETE only when its expected canonical "
            "object exists and is non-empty."
        ),
    )
)

# Core representation.
if fm_nonempty("rep_summary"):
    sections.append(fm_section(
        "1. Core representation benchmark",
        rep_summary,
        max_rows=80,
    ))
elif fm_nonempty("rep_df"):
    tmp = (
        rep_df.groupby(["method", "split"], as_index=False)
        .agg(
            n=("active_correct", "size"),
            active_accuracy=("active_correct", "mean"),
            exact_contract=("exact_contract", "mean"),
        )
    )
    sections.append(fm_section("1. Core representation benchmark", tmp))

# Gates.
if fm_nonempty("external_gate_ledger"):
    sections.append(fm_section(
        "2. G0–G7 external validation ledger",
        external_gate_ledger,
        max_rows=30,
    ))
elif fm_nonempty("gates"):
    sections.append(
        "## 2. Original gate ledger\n\n" +
        "\n".join(f"- {x[0]}: **{'PASS' if bool(x[1]) else 'FAIL'}**" for x in gates)
    )

# Large master C1/C2.
if fm_nonempty("master_surface_summary"):
    sections.append(fm_section(
        "3. Final large C1/C2 surface-stress benchmark",
        master_surface_summary,
        max_rows=50,
        prose=(
            "120 independent bases × 12 frozen surface families. "
            "Claim strength is governed by both base-clustered and "
            "surface-family-clustered uncertainty."
        ),
    ))
if fm_nonempty("master_surface_claims"):
    sections.append(fm_section(
        "3.1 Paired claim-level C1/C2 inference",
        master_surface_claims,
        max_rows=50,
    ))

# Ontology.
if fm_nonempty("opaque_pair"):
    sections.append(fm_section(
        "4. Opaque ontology paired mechanism",
        opaque_pair,
    ))
if fm_nonempty("schema_perm_summary"):
    sections.append(fm_section(
        "4.1 Schema.org structural negative control",
        schema_perm_summary,
    ))
if fm_nonempty("edam_static_summary"):
    sections.append(fm_section(
        "4.2 EDAM external structural transfer",
        edam_static_summary,
    ))
if fm_nonempty("edam_longitudinal_summary"):
    sections.append(fm_section(
        "4.3 Real EDAM ontology evolution 1.24→1.25",
        edam_longitudinal_summary,
        max_rows=30,
    ))
if fm_nonempty("w3c_external_summary"):
    sections.append(fm_section(
        "4.4 PROV-O / ODRL external structural controls",
        w3c_external_summary,
        max_rows=30,
    ))

# Drift / Active-SCD.
if fm_nonempty("scd_prediction_table"):
    sections.append(fm_section(
        "5. Active-SCD versus semantic negative controls",
        scd_prediction_table,
        max_rows=40,
    ))
if fm_nonempty("conditional_df"):
    sections.append(fm_section(
        "5.1 Conditional information beyond severity",
        conditional_df,
        max_rows=30,
    ))
if fm_nonempty("within_strata_df"):
    sections.append(fm_section(
        "5.2 Within-severity / within-family discrimination",
        within_strata_df,
        max_rows=60,
    ))
if fm_nonempty("threshold_summary"):
    sections.append(fm_section(
        "5.3 Frozen Active-SCD threshold transfer",
        threshold_summary,
        max_rows=30,
    ))
if fm_nonempty("trajectory_summary"):
    sections.append(fm_section(
        "5.4 Severity-trajectory warning utility",
        trajectory_summary,
    ))

# Generative downstream.
if fm_nonempty("generative_executor_summary"):
    sections.append(fm_section(
        "6. Competence-gated generative downstream executors",
        generative_executor_summary,
        max_rows=30,
    ))

# Readers / producers.
if fm_nonempty("strong_reader_summary"):
    sections.append(fm_section(
        "7. Strong heterogeneous readers",
        strong_reader_summary,
        max_rows=100,
    ))
if fm_nonempty("strong_producer_summary"):
    sections.append(fm_section(
        "8. Strong LLM producers",
        strong_producer_summary,
        max_rows=100,
    ))

# Runtime semantic control.
if fm_nonempty("g5_combo_summary"):
    sections.append(fm_section(
        "9. JSON-schema versus semantic enforcement",
        g5_combo_summary,
        max_rows=30,
    ))

# Counterfactuals.
if fm_nonempty("message_cf_summary"):
    sections.append(fm_section(
        "10. Message-level semantic counterfactuals",
        message_cf_summary,
        max_rows=50,
    ))

# Alignment.
if fm_nonempty("g2_paired_summary"):
    sections.append(fm_section(
        "11. Fresh 10-seed Full-vs-NoInv audit",
        g2_paired_summary,
        max_rows=30,
    ))

# CRM.
if fm_nonempty("crm_retention"):
    sections.append(fm_section(
        "12. CRMArena-Pro end-task boundary",
        crm_retention,
        max_rows=50,
    ))

# Protocol evolution.
if fm_nonempty("protocol_history_summary"):
    sections.append(fm_section(
        "13. Real MCP/A2A protocol evolution corpus",
        protocol_history_summary,
        max_rows=30,
        prose=(
            "This supports the real-world motivation for evolving semantic "
            "interfaces; without downstream outcomes it is not treated as "
            "Active-SCD validation."
        ),
    ))

# Final claim audits.
if fm_nonempty("master_claim_audit"):
    sections.append(fm_section(
        "14. Final scientific claim-strength audit",
        master_claim_audit,
        max_rows=100,
    ))
if fm_nonempty("reviewer_matrix"):
    sections.append(fm_section(
        "15. Reviewer concern → closure matrix",
        reviewer_matrix,
        max_rows=100,
    ))
if fm_nonempty("manuscript_map"):
    sections.append(fm_section(
        "16. Manuscript evidence map",
        manuscript_map,
        max_rows=50,
    ))

# -------------------------------------------------------------------------
# D. Final scientific interpretation
# -------------------------------------------------------------------------

interpretation = """
## 17. Final scientific interpretation rules

The notebook deliberately separates **what SCALE is supported to do** from
what the current evidence does not support.

### Core claims that may remain central if the corresponding tests complete

1. **Semantic interface drift** is a systems-learning problem distinct from
   mere transport/schema compatibility.
2. **Open-world structural canonicalization** is the regime where ontology
   structure is intended to add value; ontology is not claimed to improve
   ordinary closed-set classification universally.
3. **Decision-active semantic observability** is the primary empirical thesis:
   Active-SCD should remain predictive after controlling for severity/family
   labels, semantic-mask permutations, inactive semantics, and independent
   downstream consumers.

### Supporting engineering

- strict deterministic explicit-ID canonicalization;
- typed/native downstream interfaces;
- semantic runtime validation;
- selective abstention / review;
- deployable routing, with its root-dependent limitation preserved.

### Results that must remain negative or qualified where applicable

- original **G0** arbitrary-reader criterion;
- original **G2** general label-efficiency / invariance criterion;
- universal task-level superiority;
- universal open-set routing;
- competent-generative-executor evidence when no declared model passes the
  frozen clean competence gate.

### Statistical language

- use **superior** only when the relevant paired interval excludes zero;
- use **noninferior** only when the predeclared margin is satisfied;
- otherwise use **competitive**, **matches**, **numerically higher/lower**,
  or **inconclusive**.
"""

sections.append(interpretation)

full_results_book = "\n\n---\n\n".join(sections)

full_book_path = FULL_BOOK_DIR / "SCALE_FULL_MASTER_RESULTS_BOOK.md"
full_book_path.write_text(full_results_book + "\n", encoding="utf-8")

display(Markdown(full_results_book))

# -------------------------------------------------------------------------
# E. Canonical registry and artifact inventory
# -------------------------------------------------------------------------

if "master_registry" in globals() and isinstance(master_registry, pd.DataFrame):
    master_registry.to_csv(
        FULL_BOOK_DIR / "FINAL_CANONICAL_MASTER_RESULTS_REGISTRY.csv",
        index=False,
    )

artifact_rows = []
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        try:
            rel = str(p.relative_to(ROOT))
        except Exception:
            rel = str(p)
        artifact_rows.append({
            "path": rel,
            "bytes": p.stat().st_size,
            "suffix": p.suffix.lower(),
        })

artifact_inventory = pd.DataFrame(artifact_rows)
artifact_inventory.to_csv(
    FULL_BOOK_DIR / "FULL_ARTIFACT_INVENTORY.csv",
    index=False,
)

# Avoid zipping a previous zip into itself.
zip_base = Path("/content") / f"{EXPERIMENT_VERSION}-{CONFIG_HASH}-FULL_MASTER_RESULTS"
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(ROOT),
)

print("\n" + "=" * 100)
print("FULL MASTER NOTEBOOK COMPLETE")
print("Results book:", full_book_path)
print("Artifact inventory:", FULL_BOOK_DIR / "FULL_ARTIFACT_INVENTORY.csv")
print("Results archive:", zip_path)
print("Completed test families:",
      int((full_completion_df["status"] == "COMPLETE").sum()),
      "/",
      len(full_completion_df))
print("=" * 100)


In [ ]:
# =============================================================================
# FINAL FROM-SCRATCH INTEGRITY SENTINEL
# =============================================================================

from pathlib import Path
import json
import pandas as pd

_SENTINEL_DIR = ROOT / "FULL_MASTER_RESULTS_BOOK"
_SENTINEL_DIR.mkdir(parents=True, exist_ok=True)

_required_final_objects = [
    "rep_df",
    "ontology_df",
    "drift_df",
    "native_df",
    "gates",
    "master_claim_audit",
    "master_registry",
    "full_completion_df",
]

_integrity_rows = []
for name in _required_final_objects:
    exists = name in globals()
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        nonempty = len(obj) > 0
        size = len(obj)
    elif isinstance(obj, (list, tuple, dict, set)):
        nonempty = len(obj) > 0
        size = len(obj)
    else:
        nonempty = obj is not None
        size = None
    _integrity_rows.append({
        "object": name,
        "exists": int(exists),
        "nonempty": int(nonempty),
        "size": size,
    })

FROM_SCRATCH_INTEGRITY = pd.DataFrame(_integrity_rows)
display(FROM_SCRATCH_INTEGRITY)

core_ok = bool(
    FROM_SCRATCH_INTEGRITY[
        FROM_SCRATCH_INTEGRITY["object"].isin(
            ["rep_df", "ontology_df", "drift_df", "native_df", "gates"]
        )
    ]["nonempty"].all()
)

status = {
    "experiment_version": EXPERIMENT_VERSION,
    "config_hash": CONFIG_HASH,
    "core_pipeline_complete": core_ok,
    "integrity_rows": _integrity_rows,
}

(_SENTINEL_DIR / "FROM_SCRATCH_EXECUTION_SENTINEL.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8",
)

print("=" * 100)
print("FROM-SCRATCH MASTER EXECUTION STATUS")
print("Core pipeline complete:", core_ok)
print("Sentinel:", _SENTINEL_DIR / "FROM_SCRATCH_EXECUTION_SENTINEL.json")
print("=" * 100)
